In [1]:
# ==================================================================================================
# PROJECT 11 — CELL 1 / STEP 0
# FRESH-RUNTIME BOOTSTRAP, REGISTRY VALIDATION, DATASET RESTORATION,
# AND UNREGISTERED CANDIDATE DISCOVERY
#
# PURPOSE:
# - verify Google Drive and the thesis experiment root
# - verify Projects 1–10 are COMPLETE_AND_FROZEN
# - verify Project 11 is not already registered
# - restore the source archive into this fresh Colab runtime
# - inventory all source-project directories
# - exclude already completed Projects 1–10
# - produce the remaining Project 11 candidate list
#
# SAFETY:
# - does not modify the completion registry
# - does not access or modify frozen raw-result directories
# - does not rerun any completed project
# - does not select Project 11 prematurely
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import shutil
import sys
import tarfile
import time

import pandas as pd


print("=" * 132)
print("=== PROJECT 11 CELL 1 / STEP 0: FRESH-RUNTIME BOOTSTRAP AND CANDIDATE DISCOVERY ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 11

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

DATA_ROOT = (
    THESIS_ROOT
    / "Data"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SOURCE_ARCHIVE_PATH = (
    DATA_ROOT
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

LOCAL_DATASET_ROOT = Path(
    "/content/datasets"
)

PROJECT_11_BOOTSTRAP_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_11_bootstrap"
)

PROJECT_11_SOURCE_INVENTORY_PATH = (
    PROJECT_11_BOOTSTRAP_ROOT
    / "project_11_local_source_inventory.csv"
)

PROJECT_11_REMAINING_CANDIDATES_PATH = (
    PROJECT_11_BOOTSTRAP_ROOT
    / "project_11_remaining_candidate_inventory.csv"
)

PROJECT_11_BOOTSTRAP_REPORT_PATH = (
    PROJECT_11_BOOTSTRAP_ROOT
    / "project_11_bootstrap_report.json"
)

PROJECT_11_BOOTSTRAP_STATUS_PATH = (
    PROJECT_11_BOOTSTRAP_ROOT
    / "project_11_bootstrap_status.json"
)

BOOTSTRAP_STATUS = (
    "PASS_PROJECT_11_FRESH_RUNTIME_BOOTSTRAPPED_"
    "AND_UNREGISTERED_CANDIDATES_DISCOVERED"
)

EXPECTED_COMPLETE_PROJECTS = 10
EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def safe_extract_tar(
    archive_path,
    destination_root,
):
    archive_path = Path(
        archive_path
    )

    destination_root = Path(
        destination_root
    )

    destination_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    destination_resolved = destination_root.resolve()

    with tarfile.open(
        archive_path,
        mode="r:gz",
    ) as archive:
        members = archive.getmembers()

        for member in members:
            target_path = (
                destination_root
                / member.name
            ).resolve()

            try:
                target_path.relative_to(
                    destination_resolved
                )

            except ValueError:
                raise RuntimeError(
                    "Unsafe archive member blocked:\n"
                    f"{member.name}"
                )

        archive.extractall(
            destination_root
        )

    return len(
        members
    )


def count_files_by_suffix(
    root,
    suffixes,
):
    suffixes = {
        suffix.lower()
        for suffix in suffixes
    }

    return sum(
        1
        for path in Path(root).rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in suffixes
        )
    )


def directory_size_bytes(root):
    return int(
        sum(
            path.stat().st_size
            for path in Path(root).rglob("*")
            if path.is_file()
        )
    )


def resolve_exact_column(
    dataframe,
    expected_name,
):
    matches = [
        column
        for column in dataframe.columns
        if str(column).strip().lower()
        == expected_name.lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve registry column "
            f"{expected_name!r}.\n"
            f"Matches: {matches}\n"
            f"Columns: {dataframe.columns.tolist()}"
        )

    return matches[0]


# --------------------------------------------------------------------------------------------------
# 3. VERIFY DRIVE AND REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_directories = [
    THESIS_ROOT,
    DATA_ROOT,
    NOTES_ROOT,
    RESULTS_ROOT,
]

missing_directories = [
    str(path)
    for path in required_directories
    if not path.is_dir()
]

if missing_directories:
    raise FileNotFoundError(
        "Required Google Drive directories are missing:\n"
        + "\n".join(
            missing_directories
        )
    )


required_files = [
    REGISTRY_PATH,
    SOURCE_ARCHIVE_PATH,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Required files are missing:\n"
        + "\n".join(
            missing_files
        )
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

archive_sha256 = sha256_file(
    SOURCE_ARCHIVE_PATH
)

archive_size_bytes = int(
    SOURCE_ARCHIVE_PATH.stat().st_size
)


print("\nRequired inputs:")

print(
    "Thesis root:",
    THESIS_ROOT,
)

print(
    "Completion registry:",
    REGISTRY_PATH,
)

print(
    "Registry SHA-256:",
    registry_sha256_before,
)

print(
    "Source archive:",
    SOURCE_ARCHIVE_PATH,
)

print(
    "Archive bytes:",
    archive_size_bytes,
)

print(
    "Archive SHA-256:",
    archive_sha256,
)


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE COMPLETION REGISTRY
# --------------------------------------------------------------------------------------------------

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_exact_column(
    registry,
    "ProjectNumber",
)

project_column = resolve_exact_column(
    registry,
    "Project",
)

project_slug_column = resolve_exact_column(
    registry,
    "ProjectSlug",
)

status_column = resolve_exact_column(
    registry,
    "Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


registered_project_numbers = sorted(
    registry_project_numbers.tolist()
)

registered_project_names = set(
    registry[
        project_column
    ]
    .astype(str)
    .str.strip()
    .tolist()
)

registered_project_slugs = set(
    registry[
        project_slug_column
    ]
    .astype(str)
    .str.strip()
    .tolist()
)


complete_project_count = int(
    registry[
        status_column
    ]
    .astype(str)
    .str.strip()
    .eq(
        EXPECTED_COMPLETE_STATUS
    )
    .sum()
)


if len(registry) != EXPECTED_COMPLETE_PROJECTS:
    raise RuntimeError(
        "The completion registry does not contain exactly "
        "10 rows.\n"
        f"Actual rows: {len(registry)}"
    )


if registered_project_numbers != list(
    range(1, 11)
):
    raise RuntimeError(
        "The completion registry does not contain exactly "
        "Projects 1–10.\n"
        f"Actual: {registered_project_numbers}"
    )


if complete_project_count != EXPECTED_COMPLETE_PROJECTS:
    raise RuntimeError(
        "Not all Projects 1–10 are COMPLETE_AND_FROZEN.\n"
        f"Complete projects: {complete_project_count}"
    )


if PROJECT_NUMBER in set(
    registered_project_numbers
):
    raise RuntimeError(
        "Project 11 is already present in the registry."
    )


print("\nCompletion registry validation:")

display(
    registry[
        [
            project_number_column,
            project_column,
            project_slug_column,
            status_column,
        ]
    ]
)


print(
    "\nRegistry rows:",
    len(registry),
)

print(
    "Registered project numbers:",
    registered_project_numbers,
)

print(
    "COMPLETE_AND_FROZEN projects:",
    complete_project_count,
)

print(
    "Project 11 registry rows:",
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
)


# --------------------------------------------------------------------------------------------------
# 5. RESTORE THE DATASET IN THIS FRESH RUNTIME
# --------------------------------------------------------------------------------------------------

extraction_started = time.perf_counter()

existing_source_directories_before = sorted(
    [
        path
        for path in LOCAL_DATASET_ROOT.iterdir()
        if path.is_dir()
    ],
    key=lambda path:
        path.name.lower(),
) if LOCAL_DATASET_ROOT.is_dir() else []


archive_extraction_performed = False
archive_member_count = None


# A new Colab runtime normally has no /content/datasets directory.
# If it already contains source directories, preserve them and avoid unnecessary extraction.

if not existing_source_directories_before:
    if LOCAL_DATASET_ROOT.exists():
        if not LOCAL_DATASET_ROOT.is_dir():
            raise RuntimeError(
                "The local dataset path exists but is not a directory:\n"
                f"{LOCAL_DATASET_ROOT}"
            )

        shutil.rmtree(
            LOCAL_DATASET_ROOT
        )

    LOCAL_DATASET_ROOT.mkdir(
        parents=True,
        exist_ok=False,
    )

    archive_member_count = safe_extract_tar(
        SOURCE_ARCHIVE_PATH,
        LOCAL_DATASET_ROOT,
    )

    archive_extraction_performed = True


extraction_seconds = float(
    time.perf_counter()
    - extraction_started
)


# --------------------------------------------------------------------------------------------------
# 6. RESOLVE THE ACTUAL SOURCE-DIRECTORY LEVEL
# --------------------------------------------------------------------------------------------------

def immediate_directories(root):
    return sorted(
        [
            path
            for path in Path(root).iterdir()
            if path.is_dir()
        ],
        key=lambda path:
            path.name.lower(),
    )


top_level_directories = immediate_directories(
    LOCAL_DATASET_ROOT
)


if not top_level_directories:
    raise RuntimeError(
        "No source directories were found after dataset restoration."
    )


# Some archives wrap all projects in one container directory.
# Detect that safely.

directories_with_at_symbol = [
    path
    for path in top_level_directories
    if "@" in path.name
]


if directories_with_at_symbol:
    SOURCE_PROJECTS_ROOT = (
        LOCAL_DATASET_ROOT
    )

else:
    candidate_wrappers = []

    for directory in top_level_directories:
        children = immediate_directories(
            directory
        )

        at_children = [
            child
            for child in children
            if "@" in child.name
        ]

        if at_children:
            candidate_wrappers.append(
                (
                    directory,
                    len(at_children),
                )
            )


    if len(candidate_wrappers) == 1:
        SOURCE_PROJECTS_ROOT = (
            candidate_wrappers[0][0]
        )

    elif len(candidate_wrappers) > 1:
        SOURCE_PROJECTS_ROOT = max(
            candidate_wrappers,
            key=lambda item:
                item[1],
        )[0]

    else:
        # Fall back to the local dataset root.
        SOURCE_PROJECTS_ROOT = (
            LOCAL_DATASET_ROOT
        )


source_project_directories = immediate_directories(
    SOURCE_PROJECTS_ROOT
)


# Exclude non-project support directories.

source_project_directories = [
    path
    for path in source_project_directories
    if (
        not path.name.startswith(".")
        and path.name
        not in {
            "__MACOSX",
            "sample_data",
        }
    )
]


if not source_project_directories:
    raise RuntimeError(
        "No candidate source-project directories were found."
    )


# --------------------------------------------------------------------------------------------------
# 7. BUILD LOCAL SOURCE INVENTORY
# --------------------------------------------------------------------------------------------------

inventory_records = []


for source_directory in source_project_directories:
    project_name = source_directory.name

    all_files = [
        path
        for path in source_directory.rglob("*")
        if path.is_file()
    ]

    csv_files = [
        path
        for path in all_files
        if path.suffix.lower() == ".csv"
    ]

    parquet_files = [
        path
        for path in all_files
        if path.suffix.lower()
        in {
            ".parquet",
            ".pq",
        }
    ]

    json_files = [
        path
        for path in all_files
        if path.suffix.lower() == ".json"
    ]

    file_names_lower = [
        path.name.lower()
        for path in all_files
    ]

    likely_build_files = [
        path
        for path in all_files
        if "build" in path.name.lower()
    ]

    likely_test_files = [
        path
        for path in all_files
        if "test" in path.name.lower()
    ]

    source_bytes = int(
        sum(
            path.stat().st_size
            for path in all_files
        )
    )

    inventory_records.append({
        "SourceProject":
            project_name,

        "SourceDirectory":
            str(
                source_directory
            ),

        "RegisteredByProjectName":
            project_name
            in registered_project_names,

        "RegisteredByProjectSlug":
            project_name
            in registered_project_slugs,

        "AlreadyRegistered":
            (
                project_name
                in registered_project_names
                or project_name
                in registered_project_slugs
            ),

        "FileCount":
            len(
                all_files
            ),

        "SourceBytes":
            source_bytes,

        "CSVFiles":
            len(
                csv_files
            ),

        "ParquetFiles":
            len(
                parquet_files
            ),

        "JSONFiles":
            len(
                json_files
            ),

        "FilesContainingBuild":
            len(
                likely_build_files
            ),

        "FilesContainingTest":
            len(
                likely_test_files
            ),

        "HasTabularFiles":
            bool(
                csv_files
                or parquet_files
            ),

        "TopLevelFileSample":
            " | ".join(
                sorted(
                    [
                        path.relative_to(
                            source_directory
                        ).as_posix()
                        for path in all_files
                    ]
                )[:10]
            ),
    })


source_inventory = (
    pd.DataFrame(
        inventory_records
    )
    .sort_values(
        [
            "AlreadyRegistered",
            "SourceProject",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


remaining_candidates = (
    source_inventory[
        ~source_inventory[
            "AlreadyRegistered"
        ]
        & source_inventory[
            "HasTabularFiles"
        ]
    ]
    .copy()
    .sort_values(
        [
            "FileCount",
            "SourceBytes",
            "SourceProject",
        ],
        ascending=[
            False,
            False,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


remaining_candidates.insert(
    0,
    "CandidateOrder",
    range(
        1,
        len(remaining_candidates) + 1,
    ),
)


# --------------------------------------------------------------------------------------------------
# 8. VALIDATION
# --------------------------------------------------------------------------------------------------

registered_source_matches = int(
    source_inventory[
        "AlreadyRegistered"
    ].sum()
)

unregistered_source_count = int(
    (
        ~source_inventory[
            "AlreadyRegistered"
        ]
    ).sum()
)

remaining_tabular_candidate_count = len(
    remaining_candidates
)


validation_records = [
    {
        "Check":
            "Registry rows",

        "Expected":
            10,

        "Actual":
            len(registry),

        "Pass":
            len(registry) == 10,
    },

    {
        "Check":
            "Registered project numbers",

        "Expected":
            list(range(1, 11)),

        "Actual":
            registered_project_numbers,

        "Pass":
            registered_project_numbers
            == list(range(1, 11)),
    },

    {
        "Check":
            "COMPLETE_AND_FROZEN projects",

        "Expected":
            10,

        "Actual":
            complete_project_count,

        "Pass":
            complete_project_count == 10,
    },

    {
        "Check":
            "Project 11 registry rows",

        "Expected":
            0,

        "Actual":
            int(
                registry_project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ),

        "Pass":
            int(
                registry_project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ) == 0,
    },

    {
        "Check":
            "Source project directories",

        "Expected":
            "> 10",

        "Actual":
            len(source_inventory),

        "Pass":
            len(source_inventory) > 10,
    },

    {
        "Check":
            "Remaining tabular candidates",

        "Expected":
            "> 0",

        "Actual":
            remaining_tabular_candidate_count,

        "Pass":
            remaining_tabular_candidate_count > 0,
    },
]


validation = pd.DataFrame(
    validation_records
)

failed_validation = validation[
    ~validation[
        "Pass"
    ]
]


print("\nBootstrap validation:")

display(
    validation
)


if not failed_validation.empty:
    raise RuntimeError(
        "PROJECT 11 FRESH-RUNTIME BOOTSTRAP FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 9. WRITE PROJECT 11 BOOTSTRAP EVIDENCE
# --------------------------------------------------------------------------------------------------

PROJECT_11_BOOTSTRAP_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    PROJECT_11_SOURCE_INVENTORY_PATH,
    source_inventory,
)

atomic_write_csv(
    PROJECT_11_REMAINING_CANDIDATES_PATH,
    remaining_candidates,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        BOOTSTRAP_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "ThesisRoot":
        str(
            THESIS_ROOT
        ),

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "RegisteredProjectNumbers":
        registered_project_numbers,

    "CompleteAndFrozenProjects":
        complete_project_count,

    "Project11RegistryRows":
        int(
            registry_project_numbers.eq(
                PROJECT_NUMBER
            ).sum()
        ),

    "SourceArchive":
        str(
            SOURCE_ARCHIVE_PATH
        ),

    "SourceArchiveBytes":
        archive_size_bytes,

    "SourceArchiveSHA256":
        archive_sha256,

    "ArchiveExtractionPerformed":
        archive_extraction_performed,

    "ArchiveMemberCount":
        archive_member_count,

    "ExtractionSeconds":
        extraction_seconds,

    "LocalDatasetRoot":
        str(
            LOCAL_DATASET_ROOT
        ),

    "SourceProjectsRoot":
        str(
            SOURCE_PROJECTS_ROOT
        ),

    "SourceProjectDirectories":
        len(
            source_inventory
        ),

    "RegisteredSourceMatches":
        registered_source_matches,

    "UnregisteredSourceDirectories":
        unregistered_source_count,

    "RemainingTabularCandidates":
        remaining_tabular_candidate_count,

    "SourceInventory":
        str(
            PROJECT_11_SOURCE_INVENTORY_PATH
        ),

    "RemainingCandidateInventory":
        str(
            PROJECT_11_REMAINING_CANDIDATES_PATH
        ),

    "RegistryModified":
        False,

    "RawResultsModified":
        False,

    "CompletedProjectsRerun":
        False,
}


atomic_write_json(
    PROJECT_11_BOOTSTRAP_REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        BOOTSTRAP_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "CompleteAndFrozenProjects":
        complete_project_count,

    "SourceProjectDirectories":
        len(
            source_inventory
        ),

    "RemainingTabularCandidates":
        remaining_tabular_candidate_count,

    "RegistrySHA256":
        registry_sha256_before,
}


atomic_write_json(
    PROJECT_11_BOOTSTRAP_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 10. FINAL REGISTRY IMMUTABILITY CHECK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 11 bootstrap."
    )


# --------------------------------------------------------------------------------------------------
# 11. DISPLAY
# --------------------------------------------------------------------------------------------------

print("\nRegistered Projects 1–10:")

display(
    registry[
        [
            project_number_column,
            project_column,
            project_slug_column,
            status_column,
        ]
    ]
)


print("\nLocal source inventory:")

display(
    source_inventory[
        [
            "SourceProject",
            "AlreadyRegistered",
            "FileCount",
            "SourceBytes",
            "CSVFiles",
            "ParquetFiles",
            "FilesContainingBuild",
            "FilesContainingTest",
            "HasTabularFiles",
        ]
    ]
)


print("\nRemaining unregistered tabular candidates:")

display(
    remaining_candidates[
        [
            "CandidateOrder",
            "SourceProject",
            "SourceDirectory",
            "FileCount",
            "SourceBytes",
            "CSVFiles",
            "ParquetFiles",
            "FilesContainingBuild",
            "FilesContainingTest",
        ]
    ]
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 132)
print("=== PROJECT 11 CELL 1 / STEP 0 RESULT ===")
print("=" * 132)


print("\nRuntime bootstrap:")

print(
    "Archive extraction performed:",
    archive_extraction_performed,
)

print(
    "Archive extraction seconds:",
    round(
        extraction_seconds,
        2,
    ),
)

print(
    "Local dataset root:",
    LOCAL_DATASET_ROOT,
)

print(
    "Resolved source-project root:",
    SOURCE_PROJECTS_ROOT,
)


print("\nCompletion registry:")

print(
    "Registry rows:",
    len(registry),
)

print(
    "Registered project numbers:",
    registered_project_numbers,
)

print(
    "COMPLETE_AND_FROZEN projects:",
    complete_project_count,
)

print(
    "Project 11 registry rows:",
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
)

print(
    "Registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)


print("\nCandidate discovery:")

print(
    "Source project directories:",
    len(source_inventory),
)

print(
    "Already registered source matches:",
    registered_source_matches,
)

print(
    "Unregistered source directories:",
    unregistered_source_count,
)

print(
    "Remaining tabular candidates:",
    remaining_tabular_candidate_count,
)


print("\nOutputs:")

print(
    PROJECT_11_SOURCE_INVENTORY_PATH
)

print(
    PROJECT_11_REMAINING_CANDIDATES_PATH
)

print(
    PROJECT_11_BOOTSTRAP_REPORT_PATH
)


print(
    "\nSTATUS:",
    BOOTSTRAP_STATUS,
)

print("=" * 132)

=== PROJECT 11 CELL 1 / STEP 0: FRESH-RUNTIME BOOTSTRAP AND CANDIDATE DISCOVERY ===

Required inputs:
Thesis root: /content/drive/MyDrive/Thesis_Experiment
Completion registry: /content/drive/MyDrive/Thesis_Experiment/Notes/completed_project_registry.csv
Registry SHA-256: 847bdcbee16c8757fde34e9489cd3abbf0781a319644175472597265b1df2750
Source archive: /content/drive/MyDrive/Thesis_Experiment/Data/Raw/TCP-CI-main-dataset.tar.gz
Archive bytes: 237104379
Archive SHA-256: 92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e

Completion registry validation:


,ProjectNumber,Project,ProjectSlug,Status
0,1,Angel-ML@angel,Angel-ML__angel,COMPLETE_AND_FROZEN
1,2,apache@airavata,apache__airavata,COMPLETE_AND_FROZEN
2,3,b2ihealthcare@snow-owl,b2ihealthcare__snow-owl,COMPLETE_AND_FROZEN
3,4,eclipse@paho.mqtt.java,eclipse__paho.mqtt.java,COMPLETE_AND_FROZEN
4,5,thinkaurelius@titan,thinkaurelius__titan,COMPLETE_AND_FROZEN
5,6,eclipse@jetty.project,eclipse__jetty.project,COMPLETE_AND_FROZEN
6,7,CompEvol@beast2,CompEvol__beast2,COMPLETE_AND_FROZEN
7,8,optimatika@ojAlgo,optimatika__ojAlgo,COMPLETE_AND_FROZEN
8,9,camunda@camunda-bpm-platform,camunda__camunda-bpm-platform,COMPLETE_AND_FROZEN
9,10,spring-cloud@spring-cloud-dataflow,spring-cloud__spring-cloud-dataflow,COMPLETE_AND_FROZEN



Registry rows: 10
Registered project numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
COMPLETE_AND_FROZEN projects: 10
Project 11 registry rows: 0


/tmp/ipykernel_424/1758688739.py:250: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(



Bootstrap validation:


...


Local source inventory:


,SourceProject,AlreadyRegistered,FileCount,SourceBytes,CSVFiles,ParquetFiles,FilesContainingBuild,FilesContainingTest,HasTabularFiles
0,EMResearch@EvoMaster,False,6,16276377,6,0,1,0,True
1,Graylog2@graylog2-server,False,6,33623440,6,0,1,0,True
2,JMRI@JMRI,False,6,810273839,6,0,1,0,True
3,SonarSource@sonarqube,False,6,374503252,6,0,1,0,True
4,apache@curator,False,6,12666992,6,0,1,0,True
5,apache@logging-log4j2,False,6,101286750,6,0,1,0,True
6,apache@rocketmq,False,6,10163815,6,0,1,0,True
7,apache@shardingsphere,False,6,142849042,6,0,1,0,True
8,apache@sling,False,6,105842962,6,0,1,0,True
9,cantaloupe-project@cantaloupe,False,6,14535925,6,0,1,0,True



Remaining unregistered tabular candidates:


,CandidateOrder,SourceProject,SourceDirectory,FileCount,SourceBytes,CSVFiles,ParquetFiles,FilesContainingBuild,FilesContainingTest
0,1,JMRI@JMRI,/content/datasets/datasets/JMRI@JMRI,6,810273839,6,0,1,0
1,2,SonarSource@sonarqube,/content/datasets/datasets/SonarSource@sonarqube,6,374503252,6,0,1,0
2,3,apache@shardingsphere,/content/datasets/datasets/apache@shardingsphere,6,142849042,6,0,1,0
3,4,zolyfarkas@spf4j,/content/datasets/datasets/zolyfarkas@spf4j,6,124393976,6,0,1,0
4,5,facebook@buck,/content/datasets/datasets/facebook@buck,6,105909660,6,0,1,0
5,6,apache@sling,/content/datasets/datasets/apache@sling,6,105842962,6,0,1,0
6,7,apache@logging-log4j2,/content/datasets/datasets/apache@logging-log4j2,6,101286750,6,0,1,0
7,8,Graylog2@graylog2-server,/content/datasets/datasets/Graylog2@graylog2-s...,6,33623440,6,0,1,0
8,9,jcabi@jcabi-github,/content/datasets/datasets/jcabi@jcabi-github,6,17772649,6,0,1,0
9,10,EMResearch@EvoMaster,/content/datasets/datasets/EMResearch@EvoMaster,6,16276377,6,0,1,0




=== PROJECT 11 CELL 1 / STEP 0 RESULT ===

Runtime bootstrap:
Archive extraction performed: True
Archive extraction seconds: 28.19
Local dataset root: /content/datasets
Resolved source-project root: /content/datasets/datasets

Completion registry:
Registry rows: 10
Registered project numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
COMPLETE_AND_FROZEN projects: 10
Project 11 registry rows: 0
Registry unchanged: True

Candidate discovery:
Source project directories: 25
Already registered source matches: 10
Unregistered source directories: 15
Remaining tabular candidates: 15

Outputs:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/project_11_bootstrap/project_11_local_source_inventory.csv
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/project_11_bootstrap/project_11_remaining_candidate_inventory.csv
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/project_11_bootstrap/project_11_bootstrap_report.json

STATUS: PASS_PROJECT_11_FRESH_RUNTIME_BOOTSTRAPPED_AND_U

In [2]:
# ==================================================================================================
# PROJECT 11 — CELL 2 / STEP 1A
# CANDIDATE ELIGIBILITY SCAN, DETERMINISTIC RANKING,
# AND PROVISIONAL PROJECT 11 SELECTION
#
# PURPOSE:
# - scan the 15 remaining unregistered TCP-CI projects
# - validate required source files and schemas
# - impose the frozen chronological 75/25 split
# - count raw and model-ready training/evaluation failures
# - enforce the thesis eligibility rules
# - rank eligible candidates using the same deterministic policy
#   used for Projects 9 and 10
# - create a provisional Project 11 selection
#
# SAFETY:
# - does not modify the completion registry
# - does not modify Projects 1–10
# - does not create Project 11 raw experiment outputs
# - does not yet freeze the selected source
# - supports candidate-level resume using a Drive progress file
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import math
import os
import time
import warnings

import numpy as np
import pandas as pd


print("=" * 122)
print("=== PROJECT 11 CELL 2 / STEP 1A: CANDIDATE DISCOVERY, ELIGIBILITY AND RANKING ===")
print("=" * 122)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PROJECT 11 CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 11

EXPECTED_COMPLETED_PROJECTS = 10
EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_BOOTSTRAP_STATUS = (
    "PASS_PROJECT_11_FRESH_RUNTIME_BOOTSTRAPPED_"
    "AND_UNREGISTERED_CANDIDATES_DISCOVERED"
)

STEP1A_PASS_STATUS = (
    "PASS_PROJECT_11_CANDIDATE_DISCOVERY_COMPLETE"
)

EXPECTED_REGISTRY_SHA256 = (
    "847bdcbee16c8757fde34e9489cd3abb"
    "f0781a319644175472597265b1df2750"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9"
    "acb25aad982a1ff9fbef7103cb9e36e"
)

TRAIN_FRACTION = 0.75
EVALUATION_FRACTION = 0.25

# Thesis-plan operational eligibility rule:
# at least ten failing builds in the 75% training partition.
MIN_FAILING_TRAINING_BUILDS = 10

EXPECTED_REMAINING_CANDIDATES = 15

CHUNK_SIZE = 500_000

REQUIRED_SOURCE_FILES = [
    "builds.csv",
    "contributors.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "exe.csv",
    "id_map.csv",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

BOOTSTRAP_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_11_bootstrap"
)

BOOTSTRAP_STATUS_PATH = (
    BOOTSTRAP_ROOT
    / "project_11_bootstrap_status.json"
)

BOOTSTRAP_REPORT_PATH = (
    BOOTSTRAP_ROOT
    / "project_11_bootstrap_report.json"
)

REMAINING_CANDIDATES_PATH = (
    BOOTSTRAP_ROOT
    / "project_11_remaining_candidate_inventory.csv"
)

PROJECT_11_SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_11_selection"
)

SCAN_PROGRESS_PATH = (
    PROJECT_11_SELECTION_ROOT
    / "project_11_candidate_scan_progress.csv"
)

SOURCE_SCHEMA_AUDIT_PATH = (
    PROJECT_11_SELECTION_ROOT
    / "project_11_source_schema_audit.csv"
)

CANDIDATE_INVENTORY_PATH = (
    PROJECT_11_SELECTION_ROOT
    / "project_11_candidate_inventory.csv"
)

ELIGIBLE_CANDIDATES_PATH = (
    PROJECT_11_SELECTION_ROOT
    / "project_11_eligible_candidates_ranked.csv"
)

INELIGIBLE_CANDIDATES_PATH = (
    PROJECT_11_SELECTION_ROOT
    / "project_11_ineligible_candidates.csv"
)

PROVISIONAL_SELECTION_PATH = (
    PROJECT_11_SELECTION_ROOT
    / "project_11_provisional_selection.json"
)

STEP1A_REPORT_PATH = (
    PROJECT_11_SELECTION_ROOT
    / "project_11_step1a_report.json"
)

STEP1A_STATUS_PATH = (
    PROJECT_11_SELECTION_ROOT
    / "project_11_step1a_status.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def resolve_exact_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {label}.\n"
            f"Expected: {expected!r}\n"
            f"Matches: {matches}\n"
            f"Columns: {list(columns)}"
        )

    return matches[0]


def normalise_project_slug(project_name):
    return str(
        project_name
    ).replace(
        "@",
        "__",
        1,
    )


def parse_build_ids(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    missing_count = int(
        numeric.isna().sum()
    )

    if missing_count:
        raise RuntimeError(
            f"{label} contains {missing_count} "
            "missing or non-numeric build IDs."
        )

    fractional = (
        numeric
        - np.floor(
            numeric
        )
    )

    if not np.isclose(
        fractional.to_numpy(
            dtype=float
        ),
        0.0,
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral build IDs."
        )

    return numeric.astype(
        "int64"
    )


def verdict_failure_mask(
    values,
    label,
):
    """
    Convert TCP-CI verdicts into a binary failure mask.

    Numeric encoding:
      0 = pass
      any non-zero value = failure subtype

    String fallbacks are included only for defensive validation.
    """

    series = pd.Series(
        values
    )

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    non_missing_source = series.notna()

    numeric_coverage = int(
        numeric.notna().sum()
    )

    source_coverage = int(
        non_missing_source.sum()
    )

    if numeric_coverage == source_coverage:
        if numeric.isna().any():
            raise RuntimeError(
                f"{label} contains missing verdicts."
            )

        return numeric.ne(0)


    normalised = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    pass_values = {
        "0",
        "0.0",
        "pass",
        "passed",
        "success",
        "successful",
        "false",
    }

    failure_values = {
        "1",
        "1.0",
        "2",
        "2.0",
        "3",
        "3.0",
        "fail",
        "failed",
        "failure",
        "error",
        "errors",
        "true",
    }

    known_mask = normalised.isin(
        pass_values
        | failure_values
    )

    if not known_mask.all():
        unknown_values = sorted(
            normalised[
                ~known_mask
            ]
            .unique()
            .tolist()
        )

        raise RuntimeError(
            f"{label} contains unsupported verdict values:\n"
            + "\n".join(
                unknown_values[:20]
            )
        )

    return normalised.isin(
        failure_values
    )


def inspect_csv_header(path):
    return pd.read_csv(
        path,
        nrows=0,
    ).columns.tolist()


def scan_partitioned_table(
    path,
    build_column,
    verdict_column,
    all_build_ids,
    training_build_ids,
    evaluation_build_ids,
    label,
):
    total_rows = 0

    training_rows = 0
    evaluation_rows = 0
    unlinked_rows = 0

    training_failures = 0
    evaluation_failures = 0

    training_passes = 0
    evaluation_passes = 0

    failing_training_builds = set()
    failing_evaluation_builds = set()

    observed_verdict_values = set()


    reader = pd.read_csv(
        path,
        usecols=[
            build_column,
            verdict_column,
        ],
        chunksize=CHUNK_SIZE,
        low_memory=False,
    )


    for chunk in reader:
        build_ids = parse_build_ids(
            chunk[
                build_column
            ],
            f"{label}.{build_column}",
        )

        failure_mask = verdict_failure_mask(
            chunk[
                verdict_column
            ],
            f"{label}.{verdict_column}",
        )

        observed_verdict_values.update(
            chunk[
                verdict_column
            ]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )

        total_rows += len(
            chunk
        )

        linked_mask = build_ids.isin(
            all_build_ids
        )

        unlinked_rows += int(
            (
                ~linked_mask
            ).sum()
        )

        training_mask = build_ids.isin(
            training_build_ids
        )

        evaluation_mask = build_ids.isin(
            evaluation_build_ids
        )

        training_rows += int(
            training_mask.sum()
        )

        evaluation_rows += int(
            evaluation_mask.sum()
        )

        training_failure_mask = (
            training_mask
            & failure_mask
        )

        evaluation_failure_mask = (
            evaluation_mask
            & failure_mask
        )

        training_failures += int(
            training_failure_mask.sum()
        )

        evaluation_failures += int(
            evaluation_failure_mask.sum()
        )

        training_passes += int(
            (
                training_mask
                & ~failure_mask
            ).sum()
        )

        evaluation_passes += int(
            (
                evaluation_mask
                & ~failure_mask
            ).sum()
        )

        failing_training_builds.update(
            build_ids[
                training_failure_mask
            ].astype(int).tolist()
        )

        failing_evaluation_builds.update(
            build_ids[
                evaluation_failure_mask
            ].astype(int).tolist()
        )


    accounted_rows = (
        training_rows
        + evaluation_rows
        + unlinked_rows
    )

    if accounted_rows != total_rows:
        raise RuntimeError(
            f"{label}: row accounting differs.\n"
            f"Total: {total_rows}\n"
            f"Training: {training_rows}\n"
            f"Evaluation: {evaluation_rows}\n"
            f"Unlinked: {unlinked_rows}"
        )


    return {
        "Rows":
            int(
                total_rows
            ),

        "TrainingRows":
            int(
                training_rows
            ),

        "EvaluationRows":
            int(
                evaluation_rows
            ),

        "UnlinkedRows":
            int(
                unlinked_rows
            ),

        "TrainingFailures":
            int(
                training_failures
            ),

        "EvaluationFailures":
            int(
                evaluation_failures
            ),

        "TrainingPasses":
            int(
                training_passes
            ),

        "EvaluationPasses":
            int(
                evaluation_passes
            ),

        "FailingTrainingBuilds":
            int(
                len(
                    failing_training_builds
                )
            ),

        "FailingEvaluationBuilds":
            int(
                len(
                    failing_evaluation_builds
                )
            ),

        "ObservedVerdictValues":
            " | ".join(
                sorted(
                    observed_verdict_values
                )
            ),
    }


def inspect_candidate(
    candidate_order,
    project_name,
    source_directory,
):
    started = time.perf_counter()

    source_directory = Path(
        source_directory
    )

    project_slug = normalise_project_slug(
        project_name
    )

    result = {
        "OriginalCandidateOrder":
            int(
                candidate_order
            ),

        "Project":
            project_name,

        "ProjectSlug":
            project_slug,

        "SourceDirectory":
            str(
                source_directory
            ),

        "RequiredFilesPresent":
            False,

        "BuildsPath":
            str(
                source_directory
                / "builds.csv"
            ),

        "ExecutionsPath":
            str(
                source_directory
                / "exe.csv"
            ),

        "DatasetPath":
            str(
                source_directory
                / "dataset.csv"
            ),

        "IDMapPath":
            str(
                source_directory
                / "id_map.csv"
            ),

        "EntityHistoryPath":
            str(
                source_directory
                / "entity_change_history.csv"
            ),

        "ContributorsPath":
            str(
                source_directory
                / "contributors.csv"
            ),

        "InspectionStatus":
            "ERROR",

        "InspectionError":
            "",

        "ProtocolEligible":
            False,

        "EligibilityReason":
            "",
    }


    try:
        missing_files = [
            filename
            for filename in REQUIRED_SOURCE_FILES
            if not (
                source_directory
                / filename
            ).is_file()
        ]

        if missing_files:
            result[
                "EligibilityReason"
            ] = (
                "Missing required source files: "
                + ", ".join(
                    missing_files
                )
            )

            result[
                "InspectionStatus"
            ] = "INELIGIBLE"

            result[
                "ElapsedSeconds"
            ] = float(
                time.perf_counter()
                - started
            )

            return result


        result[
            "RequiredFilesPresent"
        ] = True


        builds_path = (
            source_directory
            / "builds.csv"
        )

        exe_path = (
            source_directory
            / "exe.csv"
        )

        dataset_path = (
            source_directory
            / "dataset.csv"
        )


        builds_columns = inspect_csv_header(
            builds_path
        )

        exe_columns = inspect_csv_header(
            exe_path
        )

        dataset_columns = inspect_csv_header(
            dataset_path
        )


        build_id_column = resolve_exact_column(
            builds_columns,
            "id",
            f"{project_name} builds.csv build ID",
        )

        started_at_column = resolve_exact_column(
            builds_columns,
            "started_at",
            f"{project_name} builds.csv timestamp",
        )

        execution_build_column = resolve_exact_column(
            exe_columns,
            "build",
            f"{project_name} exe.csv build",
        )

        execution_verdict_column = resolve_exact_column(
            exe_columns,
            "verdict",
            f"{project_name} exe.csv verdict",
        )

        dataset_build_column = resolve_exact_column(
            dataset_columns,
            "Build",
            f"{project_name} dataset.csv build",
        )

        dataset_verdict_column = resolve_exact_column(
            dataset_columns,
            "Verdict",
            f"{project_name} dataset.csv verdict",
        )


        result.update({
            "BuildIDColumn":
                build_id_column,

            "StartedAtColumn":
                started_at_column,

            "ExecutionBuildColumn":
                execution_build_column,

            "ExecutionVerdictColumn":
                execution_verdict_column,

            "DatasetBuildColumn":
                dataset_build_column,

            "DatasetVerdictColumn":
                dataset_verdict_column,

            "BuildColumnCount":
                len(
                    builds_columns
                ),

            "ExecutionColumnCount":
                len(
                    exe_columns
                ),

            "DatasetColumnCount":
                len(
                    dataset_columns
                ),
        })


        builds = pd.read_csv(
            builds_path,
            usecols=[
                build_id_column,
                started_at_column,
            ],
            low_memory=False,
        )


        builds[
            build_id_column
        ] = parse_build_ids(
            builds[
                build_id_column
            ],
            f"{project_name} builds.csv.{build_id_column}",
        )


        builds[
            started_at_column
        ] = pd.to_datetime(
            builds[
                started_at_column
            ],
            errors="coerce",
            utc=True,
        )


        missing_timestamps = int(
            builds[
                started_at_column
            ].isna().sum()
        )

        if missing_timestamps:
            raise RuntimeError(
                f"builds.csv contains {missing_timestamps} "
                "missing or invalid timestamps."
            )


        duplicate_build_rows = int(
            builds[
                build_id_column
            ].duplicated(
                keep=False
            ).sum()
        )

        if duplicate_build_rows:
            raise RuntimeError(
                f"builds.csv contains {duplicate_build_rows} "
                "rows participating in duplicate build IDs."
            )


        # Frozen chronology:
        # timestamp ascending, Build ID descending for timestamp ties.
        builds = (
            builds.sort_values(
                [
                    started_at_column,
                    build_id_column,
                ],
                ascending=[
                    True,
                    False,
                ],
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )


        build_count = len(
            builds
        )

        training_build_count = int(
            math.floor(
                TRAIN_FRACTION
                * build_count
            )
        )

        evaluation_build_count = int(
            build_count
            - training_build_count
        )


        if (
            training_build_count <= 0
            or evaluation_build_count <= 0
        ):
            raise RuntimeError(
                "The chronological 75/25 split produced "
                "an empty partition."
            )


        training_build_ids = set(
            builds.iloc[
                :training_build_count
            ][
                build_id_column
            ]
            .astype(int)
            .tolist()
        )

        evaluation_build_ids = set(
            builds.iloc[
                training_build_count:
            ][
                build_id_column
            ]
            .astype(int)
            .tolist()
        )

        all_build_ids = (
            training_build_ids
            | evaluation_build_ids
        )


        if (
            training_build_ids
            & evaluation_build_ids
        ):
            raise RuntimeError(
                "Training and evaluation build sets overlap."
            )


        raw_counts = scan_partitioned_table(
            path=exe_path,
            build_column=execution_build_column,
            verdict_column=execution_verdict_column,
            all_build_ids=all_build_ids,
            training_build_ids=training_build_ids,
            evaluation_build_ids=evaluation_build_ids,
            label=f"{project_name}.exe.csv",
        )


        model_counts = scan_partitioned_table(
            path=dataset_path,
            build_column=dataset_build_column,
            verdict_column=dataset_verdict_column,
            all_build_ids=all_build_ids,
            training_build_ids=training_build_ids,
            evaluation_build_ids=evaluation_build_ids,
            label=f"{project_name}.dataset.csv",
        )


        result.update({
            "Builds":
                int(
                    build_count
                ),

            "TrainingBuilds":
                int(
                    training_build_count
                ),

            "EvaluationBuilds":
                int(
                    evaluation_build_count
                ),

            "FirstBuildID":
                int(
                    builds.iloc[0][
                        build_id_column
                    ]
                ),

            "LastBuildID":
                int(
                    builds.iloc[-1][
                        build_id_column
                    ]
                ),

            "FirstBuildTimestampUTC":
                builds.iloc[0][
                    started_at_column
                ].isoformat(),

            "LastBuildTimestampUTC":
                builds.iloc[-1][
                    started_at_column
                ].isoformat(),

            "RawExecutionRows":
                raw_counts[
                    "Rows"
                ],

            "RawTrainingRows":
                raw_counts[
                    "TrainingRows"
                ],

            "RawEvaluationRows":
                raw_counts[
                    "EvaluationRows"
                ],

            "RawTrainFailures":
                raw_counts[
                    "TrainingFailures"
                ],

            "RawEvaluationFailures":
                raw_counts[
                    "EvaluationFailures"
                ],

            "RawTrainPasses":
                raw_counts[
                    "TrainingPasses"
                ],

            "RawEvaluationPasses":
                raw_counts[
                    "EvaluationPasses"
                ],

            "RawFailingTrainingBuilds":
                raw_counts[
                    "FailingTrainingBuilds"
                ],

            "RawFailingEvaluationBuilds":
                raw_counts[
                    "FailingEvaluationBuilds"
                ],

            "RawUnlinkedRows":
                raw_counts[
                    "UnlinkedRows"
                ],

            "RawVerdictValues":
                raw_counts[
                    "ObservedVerdictValues"
                ],

            "ModelReadyRows":
                model_counts[
                    "Rows"
                ],

            "ModelTrainingRows":
                model_counts[
                    "TrainingRows"
                ],

            "ModelEvaluationRows":
                model_counts[
                    "EvaluationRows"
                ],

            "ModelTrainFailures":
                model_counts[
                    "TrainingFailures"
                ],

            "ModelEvaluationFailures":
                model_counts[
                    "EvaluationFailures"
                ],

            "ModelTrainPasses":
                model_counts[
                    "TrainingPasses"
                ],

            "ModelEvaluationPasses":
                model_counts[
                    "EvaluationPasses"
                ],

            "ModelFailingTrainingBuilds":
                model_counts[
                    "FailingTrainingBuilds"
                ],

            "ModelFailingEvaluationBuilds":
                model_counts[
                    "FailingEvaluationBuilds"
                ],

            "ModelUnlinkedRows":
                model_counts[
                    "UnlinkedRows"
                ],

            "ModelVerdictValues":
                model_counts[
                    "ObservedVerdictValues"
                ],
        })


        eligibility_reasons = []


        if (
            raw_counts[
                "UnlinkedRows"
            ] != 0
        ):
            eligibility_reasons.append(
                "Raw execution rows reference builds "
                "missing from builds.csv"
            )


        if (
            model_counts[
                "UnlinkedRows"
            ] != 0
        ):
            eligibility_reasons.append(
                "Model-ready rows reference builds "
                "missing from builds.csv"
            )


        if (
            raw_counts[
                "FailingTrainingBuilds"
            ]
            < MIN_FAILING_TRAINING_BUILDS
        ):
            eligibility_reasons.append(
                "Fewer than 10 failing raw training builds"
            )


        if (
            model_counts[
                "FailingTrainingBuilds"
            ]
            < MIN_FAILING_TRAINING_BUILDS
        ):
            eligibility_reasons.append(
                "Fewer than 10 failing model-ready "
                "training builds"
            )


        if (
            raw_counts[
                "EvaluationFailures"
            ] <= 0
        ):
            eligibility_reasons.append(
                "No raw evaluation failures"
            )


        if (
            model_counts[
                "EvaluationFailures"
            ] <= 0
        ):
            eligibility_reasons.append(
                "No model-ready evaluation failures"
            )


        if (
            model_counts[
                "FailingEvaluationBuilds"
            ] <= 0
        ):
            eligibility_reasons.append(
                "No failing model-ready evaluation builds"
            )


        if (
            model_counts[
                "TrainingFailures"
            ] <= 0
            or model_counts[
                "TrainingPasses"
            ] <= 0
        ):
            eligibility_reasons.append(
                "Model training partition does not contain "
                "both classes"
            )


        if (
            model_counts[
                "EvaluationRows"
            ] <= 0
        ):
            eligibility_reasons.append(
                "No model-ready evaluation rows"
            )


        protocol_eligible = (
            len(
                eligibility_reasons
            )
            == 0
        )


        result[
            "ProtocolEligible"
        ] = bool(
            protocol_eligible
        )

        result[
            "EligibilityReason"
        ] = (
            "Eligible"
            if protocol_eligible
            else "; ".join(
                eligibility_reasons
            )
        )

        result[
            "InspectionStatus"
        ] = (
            "ELIGIBLE"
            if protocol_eligible
            else "INELIGIBLE"
        )


    except Exception as error:
        result[
            "InspectionStatus"
        ] = "ERROR"

        result[
            "ProtocolEligible"
        ] = False

        result[
            "EligibilityReason"
        ] = "Inspection error"

        result[
            "InspectionError"
        ] = (
            f"{type(error).__name__}: "
            f"{error}"
        )


    result[
        "ElapsedSeconds"
    ] = float(
        time.perf_counter()
        - started
    )

    return result


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUT VALIDATION
# --------------------------------------------------------------------------------------------------

required_files = [
    REGISTRY_PATH,
    BOOTSTRAP_STATUS_PATH,
    BOOTSTRAP_REPORT_PATH,
    REMAINING_CANDIDATES_PATH,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Required Project 11 bootstrap files are missing:\n"
        + "\n".join(
            missing_files
        )
    )


bootstrap_status = load_json(
    BOOTSTRAP_STATUS_PATH
)

bootstrap_report = load_json(
    BOOTSTRAP_REPORT_PATH
)


if bootstrap_status.get(
    "Status"
) != EXPECTED_BOOTSTRAP_STATUS:
    raise RuntimeError(
        "Project 11 bootstrap status differs.\n"
        f"Expected: {EXPECTED_BOOTSTRAP_STATUS}\n"
        f"Actual:   {bootstrap_status.get('Status')}"
    )


if bootstrap_report.get(
    "Status"
) != EXPECTED_BOOTSTRAP_STATUS:
    raise RuntimeError(
        "Project 11 bootstrap report status differs."
    )


if bootstrap_report.get(
    "SourceArchiveSHA256"
) != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "The Project 11 source-archive SHA-256 differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "The completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}\n\n"
        "Do not run candidate selection while another "
        "registry writer is active."
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_exact_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_exact_column(
    registry.columns,
    "Project",
    "registry Project",
)

project_slug_column = resolve_exact_column(
    registry.columns,
    "ProjectSlug",
    "registry ProjectSlug",
)

status_column = resolve_exact_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


if len(
    registry
) != EXPECTED_COMPLETED_PROJECTS:
    raise RuntimeError(
        "The registry does not contain exactly ten projects."
    )


if sorted(
    registry_project_numbers.tolist()
) != list(
    range(1, 11)
):
    raise RuntimeError(
        "The registry does not contain exactly Projects 1–10."
    )


if not registry[
    status_column
].astype(str).eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "One or more Projects 1–10 are not COMPLETE_AND_FROZEN."
    )


if int(
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).sum()
) != 0:
    raise RuntimeError(
        "Project 11 is already registered."
    )


remaining_candidates_source = pd.read_csv(
    REMAINING_CANDIDATES_PATH,
    low_memory=False,
)


required_candidate_columns = [
    "CandidateOrder",
    "SourceProject",
    "SourceDirectory",
]


missing_candidate_columns = [
    column
    for column in required_candidate_columns
    if column not in remaining_candidates_source.columns
]


if missing_candidate_columns:
    raise RuntimeError(
        "The remaining-candidate inventory is missing columns:\n"
        + "\n".join(
            missing_candidate_columns
        )
    )


remaining_candidates_source = (
    remaining_candidates_source.sort_values(
        "CandidateOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if len(
    remaining_candidates_source
) != EXPECTED_REMAINING_CANDIDATES:
    raise RuntimeError(
        "The remaining-candidate count differs.\n"
        f"Expected: {EXPECTED_REMAINING_CANDIDATES}\n"
        f"Actual:   {len(remaining_candidates_source)}"
    )


registered_names = set(
    registry[
        project_column
    ]
    .astype(str)
    .str.strip()
)

registered_slugs = set(
    registry[
        project_slug_column
    ]
    .astype(str)
    .str.strip()
)


candidate_registered_overlap = [
    project
    for project in remaining_candidates_source[
        "SourceProject"
    ].astype(str)
    if (
        project in registered_names
        or normalise_project_slug(
            project
        ) in registered_slugs
    )
]


if candidate_registered_overlap:
    raise RuntimeError(
        "The remaining-candidate inventory contains "
        "already registered projects:\n"
        + "\n".join(
            candidate_registered_overlap
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. RESUMABLE CANDIDATE SCAN
# --------------------------------------------------------------------------------------------------

PROJECT_11_SELECTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


candidate_names = (
    remaining_candidates_source[
        "SourceProject"
    ]
    .astype(str)
    .tolist()
)


completed_progress_records = {}


if SCAN_PROGRESS_PATH.is_file():
    existing_progress = pd.read_csv(
        SCAN_PROGRESS_PATH,
        low_memory=False,
    )

    if {
        "Project",
        "InspectionStatus",
    }.issubset(
        existing_progress.columns
    ):
        for row in existing_progress.to_dict(
            orient="records"
        ):
            project = str(
                row.get(
                    "Project",
                    "",
                )
            )

            inspection_status = str(
                row.get(
                    "InspectionStatus",
                    "",
                )
            )

            if (
                project in candidate_names
                and inspection_status
                in {
                    "ELIGIBLE",
                    "INELIGIBLE",
                }
            ):
                completed_progress_records[
                    project
                ] = row


if completed_progress_records:
    print(
        "\nReusable candidate checkpoints found:",
        len(
            completed_progress_records
        ),
    )

else:
    print(
        "\nNo reusable Project 11 candidate scan progress found."
    )


scan_records = []


for row in remaining_candidates_source.itertuples(
    index=False
):
    candidate_order = int(
        row.CandidateOrder
    )

    project_name = str(
        row.SourceProject
    )

    source_directory = str(
        row.SourceDirectory
    )


    print("-" * 122)

    print(
        f"[{candidate_order:02d}/"
        f"{EXPECTED_REMAINING_CANDIDATES:02d}] "
        f"Inspecting: {project_name}"
    )


    if project_name in completed_progress_records:
        result = completed_progress_records[
            project_name
        ]

        print(
            "    Reused checkpoint:",
            result.get(
                "InspectionStatus"
            ),
        )

    else:
        result = inspect_candidate(
            candidate_order=candidate_order,
            project_name=project_name,
            source_directory=source_directory,
        )


    scan_records.append(
        result
    )


    current_progress = (
        pd.DataFrame(
            scan_records
        )
        .sort_values(
            "OriginalCandidateOrder",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )


    atomic_write_csv(
        SCAN_PROGRESS_PATH,
        current_progress,
    )


    print(
        "    Status:",
        result.get(
            "InspectionStatus"
        ),
    )

    print(
        "    Resolved columns:",
        (
            result.get(
                "BuildIDColumn"
            ),
            result.get(
                "StartedAtColumn"
            ),
            result.get(
                "ExecutionBuildColumn"
            ),
            result.get(
                "ExecutionVerdictColumn"
            ),
            result.get(
                "DatasetBuildColumn"
            ),
            result.get(
                "DatasetVerdictColumn"
            ),
        ),
    )

    print(
        "    Builds:",
        result.get(
            "Builds"
        ),
        "| Model rows:",
        result.get(
            "ModelReadyRows"
        ),
        "| Model eval failures:",
        result.get(
            "ModelEvaluationFailures"
        ),
        "| Seconds:",
        round(
            float(
                result.get(
                    "ElapsedSeconds",
                    0.0,
                )
            ),
            2,
        ),
    )


candidate_inventory = (
    pd.DataFrame(
        scan_records
    )
    .sort_values(
        "OriginalCandidateOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 6. SCHEMA AUDIT
# --------------------------------------------------------------------------------------------------

schema_columns = [
    "Project",
    "ProjectSlug",
    "BuildIDColumn",
    "StartedAtColumn",
    "ExecutionBuildColumn",
    "ExecutionVerdictColumn",
    "DatasetBuildColumn",
    "DatasetVerdictColumn",
    "InspectionStatus",
    "InspectionError",
]


for column in schema_columns:
    if column not in candidate_inventory.columns:
        candidate_inventory[
            column
        ] = ""


source_schema_audit = candidate_inventory[
    schema_columns
].copy()


# --------------------------------------------------------------------------------------------------
# 7. ELIGIBILITY AND DETERMINISTIC RANKING
# --------------------------------------------------------------------------------------------------

inspection_errors = candidate_inventory[
    candidate_inventory[
        "InspectionStatus"
    ].astype(str).eq(
        "ERROR"
    )
].copy()


eligible_candidates = candidate_inventory[
    candidate_inventory[
        "ProtocolEligible"
    ].fillna(False).astype(bool)
].copy()


ineligible_candidates = candidate_inventory[
    ~candidate_inventory[
        "ProtocolEligible"
    ].fillna(False).astype(bool)
].copy()


if eligible_candidates.empty:
    raise RuntimeError(
        "No protocol-eligible Project 11 candidates were found."
    )


ranking_numeric_columns = [
    "ModelEvaluationFailures",
    "ModelFailingEvaluationBuilds",
    "ModelEvaluationRows",
    "RawEvaluationFailures",
    "Builds",
]


for column in ranking_numeric_columns:
    eligible_candidates[
        column
    ] = pd.to_numeric(
        eligible_candidates[
            column
        ],
        errors="raise",
    )


# Frozen candidate ranking policy:
# 1. More model-ready evaluation failure rows
# 2. More failing model-ready evaluation builds
# 3. More model-ready evaluation rows
# 4. More raw evaluation failure rows
# 5. More total builds
# 6. Project name ascending as deterministic final tie-break
eligible_candidates = (
    eligible_candidates.sort_values(
        [
            "ModelEvaluationFailures",
            "ModelFailingEvaluationBuilds",
            "ModelEvaluationRows",
            "RawEvaluationFailures",
            "Builds",
            "Project",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            False,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


eligible_candidates.insert(
    0,
    "CandidateRank",
    range(
        1,
        len(
            eligible_candidates
        )
        + 1,
    ),
)


provisional = eligible_candidates.iloc[
    0
].to_dict()


# --------------------------------------------------------------------------------------------------
# 8. VALIDATION
# --------------------------------------------------------------------------------------------------

eligible_count = len(
    eligible_candidates
)

ineligible_count = len(
    ineligible_candidates
)

inspection_error_count = len(
    inspection_errors
)


validation_records = [
    {
        "Check":
            "Completion registry rows",

        "Expected":
            10,

        "Actual":
            len(
                registry
            ),

        "Pass":
            len(
                registry
            )
            == 10,
    },

    {
        "Check":
            "COMPLETE_AND_FROZEN projects",

        "Expected":
            10,

        "Actual":
            int(
                registry[
                    status_column
                ].eq(
                    EXPECTED_COMPLETE_STATUS
                ).sum()
            ),

        "Pass":
            registry[
                status_column
            ].eq(
                EXPECTED_COMPLETE_STATUS
            ).all(),
    },

    {
        "Check":
            "Project 11 registry rows",

        "Expected":
            0,

        "Actual":
            int(
                registry_project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            ),

        "Pass":
            int(
                registry_project_numbers.eq(
                    PROJECT_NUMBER
                ).sum()
            )
            == 0,
    },

    {
        "Check":
            "Candidates inspected",

        "Expected":
            EXPECTED_REMAINING_CANDIDATES,

        "Actual":
            len(
                candidate_inventory
            ),

        "Pass":
            len(
                candidate_inventory
            )
            == EXPECTED_REMAINING_CANDIDATES,
    },

    {
        "Check":
            "Inspection errors",

        "Expected":
            0,

        "Actual":
            inspection_error_count,

        "Pass":
            inspection_error_count
            == 0,
    },

    {
        "Check":
            "Protocol-eligible candidates",

        "Expected":
            "> 0",

        "Actual":
            eligible_count,

        "Pass":
            eligible_count
            > 0,
    },

    {
        "Check":
            "Candidate ranks unique",

        "Expected":
            eligible_count,

        "Actual":
            int(
                eligible_candidates[
                    "CandidateRank"
                ].nunique()
            ),

        "Pass":
            int(
                eligible_candidates[
                    "CandidateRank"
                ].nunique()
            )
            == eligible_count,
    },

    {
        "Check":
            "Top candidate rank",

        "Expected":
            1,

        "Actual":
            int(
                provisional[
                    "CandidateRank"
                ]
            ),

        "Pass":
            int(
                provisional[
                    "CandidateRank"
                ]
            )
            == 1,
    },

    {
        "Check":
            "Top candidate protocol eligible",

        "Expected":
            True,

        "Actual":
            bool(
                provisional[
                    "ProtocolEligible"
                ]
            ),

        "Pass":
            bool(
                provisional[
                    "ProtocolEligible"
                ]
            ),
    },
]


validation = pd.DataFrame(
    validation_records
)

failed_validation = validation[
    ~validation[
        "Pass"
    ]
]


print("\nProject 11 Step 1A validation:")

display(
    validation
)


if not failed_validation.empty:
    raise RuntimeError(
        "PROJECT 11 STEP 1A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 9. WRITE SELECTION EVIDENCE
# --------------------------------------------------------------------------------------------------

atomic_write_csv(
    SOURCE_SCHEMA_AUDIT_PATH,
    source_schema_audit,
)

atomic_write_csv(
    CANDIDATE_INVENTORY_PATH,
    candidate_inventory,
)

atomic_write_csv(
    ELIGIBLE_CANDIDATES_PATH,
    eligible_candidates,
)

atomic_write_csv(
    INELIGIBLE_CANDIDATES_PATH,
    ineligible_candidates,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


provisional_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "SelectionState":
        "PROVISIONAL_NOT_YET_FROZEN",

    "CandidateRank":
        int(
            provisional[
                "CandidateRank"
            ]
        ),

    "Project":
        provisional[
            "Project"
        ],

    "ProjectSlug":
        provisional[
            "ProjectSlug"
        ],

    "SourceDirectory":
        provisional[
            "SourceDirectory"
        ],

    "BuildIDColumn":
        provisional[
            "BuildIDColumn"
        ],

    "StartedAtColumn":
        provisional[
            "StartedAtColumn"
        ],

    "ExecutionBuildColumn":
        provisional[
            "ExecutionBuildColumn"
        ],

    "ExecutionVerdictColumn":
        provisional[
            "ExecutionVerdictColumn"
        ],

    "DatasetBuildColumn":
        provisional[
            "DatasetBuildColumn"
        ],

    "DatasetVerdictColumn":
        provisional[
            "DatasetVerdictColumn"
        ],

    "Builds":
        int(
            provisional[
                "Builds"
            ]
        ),

    "TrainingBuilds":
        int(
            provisional[
                "TrainingBuilds"
            ]
        ),

    "EvaluationBuilds":
        int(
            provisional[
                "EvaluationBuilds"
            ]
        ),

    "RawExecutionRows":
        int(
            provisional[
                "RawExecutionRows"
            ]
        ),

    "RawTrainingRows":
        int(
            provisional[
                "RawTrainingRows"
            ]
        ),

    "RawEvaluationRows":
        int(
            provisional[
                "RawEvaluationRows"
            ]
        ),

    "RawTrainFailures":
        int(
            provisional[
                "RawTrainFailures"
            ]
        ),

    "RawEvaluationFailures":
        int(
            provisional[
                "RawEvaluationFailures"
            ]
        ),

    "RawFailingTrainingBuilds":
        int(
            provisional[
                "RawFailingTrainingBuilds"
            ]
        ),

    "RawFailingEvaluationBuilds":
        int(
            provisional[
                "RawFailingEvaluationBuilds"
            ]
        ),

    "ModelReadyRows":
        int(
            provisional[
                "ModelReadyRows"
            ]
        ),

    "ModelTrainingRows":
        int(
            provisional[
                "ModelTrainingRows"
            ]
        ),

    "ModelEvaluationRows":
        int(
            provisional[
                "ModelEvaluationRows"
            ]
        ),

    "ModelTrainFailures":
        int(
            provisional[
                "ModelTrainFailures"
            ]
        ),

    "ModelEvaluationFailures":
        int(
            provisional[
                "ModelEvaluationFailures"
            ]
        ),

    "ModelFailingTrainingBuilds":
        int(
            provisional[
                "ModelFailingTrainingBuilds"
            ]
        ),

    "ModelFailingEvaluationBuilds":
        int(
            provisional[
                "ModelFailingEvaluationBuilds"
            ]
        ),

    "RawUnlinkedRows":
        int(
            provisional[
                "RawUnlinkedRows"
            ]
        ),

    "ModelUnlinkedRows":
        int(
            provisional[
                "ModelUnlinkedRows"
            ]
        ),

    "EligibilityPolicy": {
        "Chronology":
            (
                "started_at ascending; build ID descending "
                "for timestamp ties"
            ),

        "Split":
            (
                "floor(0.75 × total builds) training; "
                "remaining builds clean evaluation"
            ),

        "MinimumFailingTrainingBuilds":
            MIN_FAILING_TRAINING_BUILDS,

        "RequiresRawEvaluationFailures":
            True,

        "RequiresModelEvaluationFailures":
            True,

        "RequiresBothModelTrainingClasses":
            True,

        "RequiresZeroUnlinkedRows":
            True,
    },

    "RankingPolicy": [
        "ModelEvaluationFailures descending",
        "ModelFailingEvaluationBuilds descending",
        "ModelEvaluationRows descending",
        "RawEvaluationFailures descending",
        "Builds descending",
        "Project ascending",
    ],

    "CreatedAtUTC":
        completed_at_utc,
}


atomic_write_json(
    PROVISIONAL_SELECTION_PATH,
    provisional_payload,
)


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        STEP1A_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "CompletedProjects":
        EXPECTED_COMPLETED_PROJECTS,

    "RemainingCandidatesInspected":
        len(
            candidate_inventory
        ),

    "ProtocolEligibleCandidates":
        eligible_count,

    "ProtocolIneligibleCandidates":
        ineligible_count,

    "InspectionErrors":
        inspection_error_count,

    "ProvisionalCandidateRank":
        int(
            provisional[
                "CandidateRank"
            ]
        ),

    "ProvisionalProject":
        provisional[
            "Project"
        ],

    "ProvisionalProjectSlug":
        provisional[
            "ProjectSlug"
        ],

    "ProvisionalSourceDirectory":
        provisional[
            "SourceDirectory"
        ],

    "CandidateInventory":
        str(
            CANDIDATE_INVENTORY_PATH
        ),

    "EligibleCandidatesRanked":
        str(
            ELIGIBLE_CANDIDATES_PATH
        ),

    "IneligibleCandidates":
        str(
            INELIGIBLE_CANDIDATES_PATH
        ),

    "SourceSchemaAudit":
        str(
            SOURCE_SCHEMA_AUDIT_PATH
        ),

    "ProvisionalSelection":
        str(
            PROVISIONAL_SELECTION_PATH
        ),

    "RegistryModified":
        False,

    "Projects1To10Modified":
        False,

    "Project11RawResultsCreated":
        False,

    "Project12Accessed":
        False,

    "Project12WriteAttempted":
        False,
}


atomic_write_json(
    STEP1A_REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        STEP1A_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "CandidatesInspected":
        len(
            candidate_inventory
        ),

    "EligibleCandidates":
        eligible_count,

    "IneligibleCandidates":
        ineligible_count,

    "InspectionErrors":
        inspection_error_count,

    "ProvisionalProject":
        provisional[
            "Project"
        ],

    "ProvisionalProjectSlug":
        provisional[
            "ProjectSlug"
        ],

    "RegistrySHA256":
        registry_sha256_before,
}


atomic_write_json(
    STEP1A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 10. FINAL IMMUTABILITY CHECK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during "
        "Project 11 Step 1A."
    )


# --------------------------------------------------------------------------------------------------
# 11. DISPLAY
# --------------------------------------------------------------------------------------------------

ranked_display_columns = [
    "CandidateRank",
    "Project",
    "ProjectSlug",
    "Builds",
    "TrainingBuilds",
    "EvaluationBuilds",
    "RawExecutionRows",
    "RawTrainFailures",
    "RawEvaluationFailures",
    "RawFailingEvaluationBuilds",
    "ModelReadyRows",
    "ModelTrainingRows",
    "ModelEvaluationRows",
    "ModelTrainFailures",
    "ModelEvaluationFailures",
    "ModelFailingEvaluationBuilds",
    "RawUnlinkedRows",
    "ModelUnlinkedRows",
]


print("\nRanked eligible Project 11 candidates:")

display(
    eligible_candidates[
        ranked_display_columns
    ]
)


print("\nProtocol-ineligible remaining projects:")

display_columns_ineligible = [
    "Project",
    "Builds",
    "RawFailingTrainingBuilds",
    "RawEvaluationFailures",
    "ModelFailingTrainingBuilds",
    "ModelEvaluationFailures",
    "EligibilityReason",
    "InspectionStatus",
]


for column in display_columns_ineligible:
    if column not in ineligible_candidates.columns:
        ineligible_candidates[
            column
        ] = ""


display(
    ineligible_candidates[
        display_columns_ineligible
    ]
)


print("\nResolved TCP-CI schemas:")

display(
    source_schema_audit
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 122)
print("=== PROJECT 11 CELL 2 / STEP 1A RESULT ===")
print("=" * 122)


print("\nThesis progress:")

print(
    "Completed projects:",
    EXPECTED_COMPLETED_PROJECTS,
)

print(
    "Remaining before Project 11:",
    len(
        candidate_inventory
    ),
)


print("\nCandidate discovery:")

print(
    "Candidates inspected:",
    len(
        candidate_inventory
    ),
)

print(
    "Protocol-eligible candidates:",
    eligible_count,
)

print(
    "Protocol-ineligible candidates:",
    ineligible_count,
)

print(
    "Inspection errors:",
    inspection_error_count,
)


print("\nProvisional Project 11 candidate:")

print(
    "Candidate rank:",
    int(
        provisional[
            "CandidateRank"
        ]
    ),
)

print(
    "Project:",
    provisional[
        "Project"
    ],
)

print(
    "Project slug:",
    provisional[
        "ProjectSlug"
    ],
)

print(
    "Source directory:",
    provisional[
        "SourceDirectory"
    ],
)


print("\nResolved schema:")

print(
    "builds.csv build ID:",
    provisional[
        "BuildIDColumn"
    ],
)

print(
    "builds.csv timestamp:",
    provisional[
        "StartedAtColumn"
    ],
)

print(
    "exe.csv build:",
    provisional[
        "ExecutionBuildColumn"
    ],
)

print(
    "exe.csv verdict:",
    provisional[
        "ExecutionVerdictColumn"
    ],
)

print(
    "dataset.csv build:",
    provisional[
        "DatasetBuildColumn"
    ],
)

print(
    "dataset.csv verdict:",
    provisional[
        "DatasetVerdictColumn"
    ],
)


print("\nCandidate dimensions:")

print(
    "Builds:",
    int(
        provisional[
            "Builds"
        ]
    ),
)

print(
    "Training / evaluation builds:",
    int(
        provisional[
            "TrainingBuilds"
        ]
    ),
    "/",
    int(
        provisional[
            "EvaluationBuilds"
        ]
    ),
)

print(
    "Raw execution rows:",
    int(
        provisional[
            "RawExecutionRows"
        ]
    ),
)

print(
    "Raw training / evaluation rows:",
    int(
        provisional[
            "RawTrainingRows"
        ]
    ),
    "/",
    int(
        provisional[
            "RawEvaluationRows"
        ]
    ),
)

print(
    "Raw train / evaluation failures:",
    int(
        provisional[
            "RawTrainFailures"
        ]
    ),
    "/",
    int(
        provisional[
            "RawEvaluationFailures"
        ]
    ),
)

print(
    "Model-ready rows:",
    int(
        provisional[
            "ModelReadyRows"
        ]
    ),
)

print(
    "Model training / evaluation rows:",
    int(
        provisional[
            "ModelTrainingRows"
        ]
    ),
    "/",
    int(
        provisional[
            "ModelEvaluationRows"
        ]
    ),
)

print(
    "Model train / evaluation failures:",
    int(
        provisional[
            "ModelTrainFailures"
        ]
    ),
    "/",
    int(
        provisional[
            "ModelEvaluationFailures"
        ]
    ),
)

print(
    "Model failing evaluation builds:",
    int(
        provisional[
            "ModelFailingEvaluationBuilds"
        ]
    ),
)


print("\nSaved outputs:")

for output_path in [
    SCAN_PROGRESS_PATH,
    SOURCE_SCHEMA_AUDIT_PATH,
    CANDIDATE_INVENTORY_PATH,
    ELIGIBLE_CANDIDATES_PATH,
    INELIGIBLE_CANDIDATES_PATH,
    PROVISIONAL_SELECTION_PATH,
    STEP1A_REPORT_PATH,
    STEP1A_STATUS_PATH,
]:
    print(
        output_path
    )


print("\nIsolation:")

print(
    "Completion registry modified:",
    0,
)

print(
    "Projects 1–10 modified:",
    0,
)

print(
    "Project 12 accessed:",
    False,
)


print(
    "\nSTATUS:",
    STEP1A_PASS_STATUS,
)

print("=" * 122)

=== PROJECT 11 CELL 2 / STEP 1A: CANDIDATE DISCOVERY, ELIGIBILITY AND RANKING ===

No reusable Project 11 candidate scan progress found.
--------------------------------------------------------------------------------------------------------------------------
[01/15] Inspecting: JMRI@JMRI
    Status: ELIGIBLE
    Resolved columns: ('id', 'started_at', 'build', 'verdict', 'Build', 'Verdict')
    Builds: 1481 | Model rows: 410395 | Model eval failures: 73 | Seconds: 16.45
--------------------------------------------------------------------------------------------------------------------------
[02/15] Inspecting: SonarSource@sonarqube
    Status: ELIGIBLE
    Resolved columns: ('id', 'started_at', 'build', 'verdict', 'Build', 'Verdict')
    Builds: 4286 | Model rows: 224550 | Model eval failures: 20 | Seconds: 6.91
--------------------------------------------------------------------------------------------------------------------------
[03/15] Inspecting: apache@shardingsphere
    Status:

,Check,Expected,Actual,Pass
0,Completion registry rows,10,10,True
1,COMPLETE_AND_FROZEN projects,10,10,True
2,Project 11 registry rows,0,0,True
3,Candidates inspected,15,15,True
4,Inspection errors,0,0,True
5,Protocol-eligible candidates,> 0,14,True
6,Candidate ranks unique,14,14,True
7,Top candidate rank,1,1,True
8,Top candidate protocol eligible,True,True,True



Ranked eligible Project 11 candidates:


,CandidateRank,Project,ProjectSlug,Builds,TrainingBuilds,EvaluationBuilds,RawExecutionRows,RawTrainFailures,RawEvaluationFailures,RawFailingEvaluationBuilds,ModelReadyRows,ModelTrainingRows,ModelEvaluationRows,ModelTrainFailures,ModelEvaluationFailures,ModelFailingEvaluationBuilds,RawUnlinkedRows,ModelUnlinkedRows
0,1,apache@shardingsphere,apache__shardingsphere,1049,786,263,833541,1193,171,25,91042,78035,13007,1188,171,25,0,0
1,2,zolyfarkas@spf4j,zolyfarkas__spf4j,587,440,147,68787,297,101,91,26762,18406,8356,296,101,91,0,0
2,3,jcabi@jcabi-github,jcabi__jcabi-github,809,606,203,140526,91,78,44,10437,2357,8080,90,78,44,0,0
3,4,JMRI@JMRI,JMRI__JMRI,1481,1110,371,6469640,240,73,24,410395,303251,107144,239,73,24,0,0
4,5,EMResearch@EvoMaster,EMResearch__EvoMaster,583,437,146,59155,286,68,41,14460,9907,4553,284,68,41,0,0
5,6,apache@sling,apache__sling,1403,1052,351,265459,767,49,48,113175,107157,6018,765,49,48,0,0
6,7,yamcs@Yamcs,yamcs__Yamcs,504,378,126,58101,147,45,10,7533,6452,1081,145,45,10,0,0
7,8,apache@logging-log4j2,apache__logging-log4j2,441,330,111,240253,208,40,39,117968,95812,22156,207,40,39,0,0
8,9,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,450,337,113,67108,144,31,12,10580,8232,2348,142,31,12,0,0
9,10,SonarSource@sonarqube,SonarSource__sonarqube,4286,3214,1072,5635027,1778,20,17,224550,205696,18854,1777,20,17,0,0



Protocol-ineligible remaining projects:


,Project,Builds,RawFailingTrainingBuilds,RawEvaluationFailures,ModelFailingTrainingBuilds,ModelEvaluationFailures,EligibilityReason,InspectionStatus
7,Graylog2@graylog2-server,3668,125,0,124,0,No raw evaluation failures; No model-ready eva...,INELIGIBLE



Resolved TCP-CI schemas:


,Project,ProjectSlug,BuildIDColumn,StartedAtColumn,ExecutionBuildColumn,ExecutionVerdictColumn,DatasetBuildColumn,DatasetVerdictColumn,InspectionStatus,InspectionError
0,JMRI@JMRI,JMRI__JMRI,id,started_at,build,verdict,Build,Verdict,ELIGIBLE,
1,SonarSource@sonarqube,SonarSource__sonarqube,id,started_at,build,verdict,Build,Verdict,ELIGIBLE,
2,apache@shardingsphere,apache__shardingsphere,id,started_at,build,verdict,Build,Verdict,ELIGIBLE,
3,zolyfarkas@spf4j,zolyfarkas__spf4j,id,started_at,build,verdict,Build,Verdict,ELIGIBLE,
4,facebook@buck,facebook__buck,id,started_at,build,verdict,Build,Verdict,ELIGIBLE,
5,apache@sling,apache__sling,id,started_at,build,verdict,Build,Verdict,ELIGIBLE,
6,apache@logging-log4j2,apache__logging-log4j2,id,started_at,build,verdict,Build,Verdict,ELIGIBLE,
7,Graylog2@graylog2-server,Graylog2__graylog2-server,id,started_at,build,verdict,Build,Verdict,INELIGIBLE,
8,jcabi@jcabi-github,jcabi__jcabi-github,id,started_at,build,verdict,Build,Verdict,ELIGIBLE,
9,EMResearch@EvoMaster,EMResearch__EvoMaster,id,started_at,build,verdict,Build,Verdict,ELIGIBLE,




=== PROJECT 11 CELL 2 / STEP 1A RESULT ===

Thesis progress:
Completed projects: 10
Remaining before Project 11: 15

Candidate discovery:
Candidates inspected: 15
Protocol-eligible candidates: 14
Protocol-ineligible candidates: 1
Inspection errors: 0

Provisional Project 11 candidate:
Candidate rank: 1
Project: apache@shardingsphere
Project slug: apache__shardingsphere
Source directory: /content/datasets/datasets/apache@shardingsphere

Resolved schema:
builds.csv build ID: id
builds.csv timestamp: started_at
exe.csv build: build
exe.csv verdict: verdict
dataset.csv build: Build
dataset.csv verdict: Verdict

Candidate dimensions:
Builds: 1049
Training / evaluation builds: 786 / 263
Raw execution rows: 833541
Raw training / evaluation rows: 609619 / 223922
Raw train / evaluation failures: 1193 / 171
Model-ready rows: 91042
Model training / evaluation rows: 78035 / 13007
Model train / evaluation failures: 1188 / 171
Model failing evaluation builds: 25

Saved outputs:
/content/drive/MyDr

In [3]:
# ==================================================================================================
# PROJECT 11 — CELL 3 / STEP 1B
# FINAL PROJECT SELECTION, SOURCE MANIFEST, CHRONOLOGY,
# PARTITION VALIDATION, AND SELECTION CHECKPOINT
#
# SELECTED PROJECT:
#   apache@shardingsphere
#
# PURPOSE:
# - validate the deterministic Step 1A selection
# - freeze all six selected source files by size and SHA-256
# - calculate the canonical selected-source root SHA-256
# - freeze the exact build chronology
# - freeze the chronological 75/25 training/evaluation partition
# - revalidate raw and model-ready dimensions
# - create the authoritative Project 11 selection checkpoint
#
# SAFETY:
# - does not modify the completion registry
# - does not modify Projects 1–10
# - does not access or write Project 12
# - does not create Project 11 experiment/raw-result conditions
# - does not train any model
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import math
import os
import time

import numpy as np
import pandas as pd


print("=" * 128)
print("=== PROJECT 11 CELL 3 / STEP 1B: FINAL SELECTION AND SOURCE FREEZE ===")
print("=" * 128)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN IDENTITY AND EXPECTED DIMENSIONS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 11

PROJECT_NAME = (
    "apache@shardingsphere"
)

PROJECT_SLUG = (
    "apache__shardingsphere"
)

SOURCE_DIRECTORY = Path(
    "/content/datasets/datasets/apache@shardingsphere"
)


EXPECTED_REGISTRY_SHA256 = (
    "847bdcbee16c8757fde34e9489cd3abb"
    "f0781a319644175472597265b1df2750"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9"
    "acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_STEP1A_STATUS = (
    "PASS_PROJECT_11_CANDIDATE_DISCOVERY_COMPLETE"
)

STEP1B_PASS_STATUS = (
    "PASS_PROJECT_11_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_COMPLETE_STATUS = (
    "COMPLETE_AND_FROZEN"
)


TRAIN_FRACTION = 0.75

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 142_849_042

EXPECTED_BUILDS = 1_049
EXPECTED_TRAINING_BUILDS = 786
EXPECTED_EVALUATION_BUILDS = 263

EXPECTED_RAW_ROWS = 833_541
EXPECTED_RAW_TRAINING_ROWS = 609_619
EXPECTED_RAW_EVALUATION_ROWS = 223_922
EXPECTED_RAW_TRAINING_FAILURES = 1_193
EXPECTED_RAW_EVALUATION_FAILURES = 171
EXPECTED_RAW_FAILING_EVALUATION_BUILDS = 25

EXPECTED_MODEL_ROWS = 91_042
EXPECTED_MODEL_TRAINING_ROWS = 78_035
EXPECTED_MODEL_EVALUATION_ROWS = 13_007
EXPECTED_MODEL_TRAINING_FAILURES = 1_188
EXPECTED_MODEL_EVALUATION_FAILURES = 171
EXPECTED_MODEL_FAILING_EVALUATION_BUILDS = 25

EXPECTED_BUILD_ID_COLUMN = "id"
EXPECTED_BUILD_TIMESTAMP_COLUMN = "started_at"
EXPECTED_EXE_BUILD_COLUMN = "build"
EXPECTED_EXE_VERDICT_COLUMN = "verdict"
EXPECTED_DATASET_BUILD_COLUMN = "Build"
EXPECTED_DATASET_VERDICT_COLUMN = "Verdict"

CHUNK_SIZE = 500_000


REQUIRED_SOURCE_FILES = [
    "builds.csv",
    "contributors.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "exe.csv",
    "id_map.csv",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

DATA_ROOT = (
    THESIS_ROOT
    / "Data"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SOURCE_ARCHIVE_PATH = (
    DATA_ROOT
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)


SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_11_selection"
)

STEP1A_STATUS_PATH = (
    SELECTION_ROOT
    / "project_11_step1a_status.json"
)

STEP1A_REPORT_PATH = (
    SELECTION_ROOT
    / "project_11_step1a_report.json"
)

PROVISIONAL_SELECTION_PATH = (
    SELECTION_ROOT
    / "project_11_provisional_selection.json"
)

ELIGIBLE_CANDIDATES_PATH = (
    SELECTION_ROOT
    / "project_11_eligible_candidates_ranked.csv"
)

CANDIDATE_INVENTORY_PATH = (
    SELECTION_ROOT
    / "project_11_candidate_inventory.csv"
)


FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_11_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_11_fixed_chronological_builds.csv"
)

PARTITION_AUDIT_PATH = (
    SELECTION_ROOT
    / "project_11_partition_audit.csv"
)

DIMENSION_AUDIT_PATH = (
    SELECTION_ROOT
    / "project_11_dimension_audit.csv"
)

FINAL_SCHEMA_AUDIT_PATH = (
    SELECTION_ROOT
    / "project_11_final_source_schema_audit.csv"
)

STEP1B_VALIDATION_PATH = (
    SELECTION_ROOT
    / "project_11_step1b_validation.csv"
)

STEP1B_REPORT_PATH = (
    SELECTION_ROOT
    / "project_11_step1b_report.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_11_step1b_status.json"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_11_selection_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def canonical_root_hash(
    manifest,
):
    required_columns = [
        "RelativePath",
        "SizeBytes",
        "SHA256",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in manifest.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Cannot calculate a root hash. Missing columns:\n"
            + "\n".join(
                missing_columns
            )
        )

    digest = hashlib.sha256()

    ordered = manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_exact_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {label}.\n"
            f"Expected: {expected!r}\n"
            f"Matches: {matches}\n"
            f"Columns: {list(columns)}"
        )

    return matches[0]


def parse_build_ids(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains missing or non-numeric build IDs."
        )

    values_array = numeric.to_numpy(
        dtype=float
    )

    if not np.equal(
        values_array,
        np.floor(
            values_array
        ),
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral build IDs."
        )

    return numeric.astype(
        "int64"
    )


def verdict_failure_mask(
    values,
    label,
):
    series = pd.Series(
        values
    )

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if (
        numeric.notna().sum()
        == series.notna().sum()
    ):
        if numeric.isna().any():
            raise RuntimeError(
                f"{label} contains missing verdicts."
            )

        return numeric.ne(0)


    normalised = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    pass_values = {
        "0",
        "0.0",
        "pass",
        "passed",
        "success",
        "successful",
        "false",
    }

    failure_values = {
        "1",
        "1.0",
        "2",
        "2.0",
        "3",
        "3.0",
        "fail",
        "failed",
        "failure",
        "error",
        "errors",
        "true",
    }

    known_values = (
        pass_values
        | failure_values
    )

    unknown_mask = (
        ~normalised.isin(
            known_values
        )
    )

    if unknown_mask.any():
        unknown_values = sorted(
            normalised[
                unknown_mask
            ]
            .unique()
            .tolist()
        )

        raise RuntimeError(
            f"{label} contains unsupported verdict values:\n"
            + "\n".join(
                unknown_values[:20]
            )
        )

    return normalised.isin(
        failure_values
    )


def count_csv_rows(
    path,
):
    path = Path(path)

    columns = pd.read_csv(
        path,
        nrows=0,
    ).columns.tolist()

    if not columns:
        return 0

    first_column = columns[0]

    rows = 0

    for chunk in pd.read_csv(
        path,
        usecols=[
            first_column
        ],
        chunksize=CHUNK_SIZE,
        low_memory=False,
    ):
        rows += len(
            chunk
        )

    return int(
        rows
    )


def scan_partitioned_verdict_table(
    path,
    build_column,
    verdict_column,
    all_build_ids,
    training_build_ids,
    evaluation_build_ids,
    label,
):
    total_rows = 0
    training_rows = 0
    evaluation_rows = 0
    unlinked_rows = 0

    training_failures = 0
    evaluation_failures = 0
    training_passes = 0
    evaluation_passes = 0

    failing_training_builds = set()
    failing_evaluation_builds = set()

    observed_verdict_values = set()


    for chunk in pd.read_csv(
        path,
        usecols=[
            build_column,
            verdict_column,
        ],
        chunksize=CHUNK_SIZE,
        low_memory=False,
    ):
        build_ids = parse_build_ids(
            chunk[
                build_column
            ],
            f"{label}.{build_column}",
        )

        failure_mask = verdict_failure_mask(
            chunk[
                verdict_column
            ],
            f"{label}.{verdict_column}",
        )

        observed_verdict_values.update(
            chunk[
                verdict_column
            ]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )

        total_rows += len(
            chunk
        )

        linked_mask = build_ids.isin(
            all_build_ids
        )

        training_mask = build_ids.isin(
            training_build_ids
        )

        evaluation_mask = build_ids.isin(
            evaluation_build_ids
        )

        unlinked_rows += int(
            (
                ~linked_mask
            ).sum()
        )

        training_rows += int(
            training_mask.sum()
        )

        evaluation_rows += int(
            evaluation_mask.sum()
        )


        training_failure_mask = (
            training_mask
            & failure_mask
        )

        evaluation_failure_mask = (
            evaluation_mask
            & failure_mask
        )


        training_failures += int(
            training_failure_mask.sum()
        )

        evaluation_failures += int(
            evaluation_failure_mask.sum()
        )

        training_passes += int(
            (
                training_mask
                & ~failure_mask
            ).sum()
        )

        evaluation_passes += int(
            (
                evaluation_mask
                & ~failure_mask
            ).sum()
        )


        failing_training_builds.update(
            build_ids[
                training_failure_mask
            ]
            .astype(int)
            .tolist()
        )

        failing_evaluation_builds.update(
            build_ids[
                evaluation_failure_mask
            ]
            .astype(int)
            .tolist()
        )


    if (
        training_rows
        + evaluation_rows
        + unlinked_rows
        != total_rows
    ):
        raise RuntimeError(
            f"{label}: row accounting differs."
        )


    return {
        "Rows":
            int(
                total_rows
            ),

        "TrainingRows":
            int(
                training_rows
            ),

        "EvaluationRows":
            int(
                evaluation_rows
            ),

        "UnlinkedRows":
            int(
                unlinked_rows
            ),

        "TrainingFailures":
            int(
                training_failures
            ),

        "EvaluationFailures":
            int(
                evaluation_failures
            ),

        "TrainingPasses":
            int(
                training_passes
            ),

        "EvaluationPasses":
            int(
                evaluation_passes
            ),

        "FailingTrainingBuilds":
            int(
                len(
                    failing_training_builds
                )
            ),

        "FailingEvaluationBuilds":
            int(
                len(
                    failing_evaluation_builds
                )
            ),

        "VerdictValues":
            " | ".join(
                sorted(
                    observed_verdict_values
                )
            ),
    }


def add_check(
    records,
    check,
    expected,
    actual,
    passed,
):
    records.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_files = [
    REGISTRY_PATH,
    SOURCE_ARCHIVE_PATH,
    STEP1A_STATUS_PATH,
    STEP1A_REPORT_PATH,
    PROVISIONAL_SELECTION_PATH,
    ELIGIBLE_CANDIDATES_PATH,
    CANDIDATE_INVENTORY_PATH,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Required Project 11 selection inputs are missing:\n"
        + "\n".join(
            missing_files
        )
    )


if not SOURCE_DIRECTORY.is_dir():
    raise FileNotFoundError(
        "The selected Project 11 source directory is missing:\n"
        f"{SOURCE_DIRECTORY}"
    )


missing_source_files = [
    filename
    for filename in REQUIRED_SOURCE_FILES
    if not (
        SOURCE_DIRECTORY
        / filename
    ).is_file()
]

if missing_source_files:
    raise FileNotFoundError(
        "Selected Project 11 source files are missing:\n"
        + "\n".join(
            missing_source_files
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. REGISTRY AND ARCHIVE IMMUTABILITY
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

archive_sha256 = sha256_file(
    SOURCE_ARCHIVE_PATH
)


if (
    registry_sha256_before
    != EXPECTED_REGISTRY_SHA256
):
    raise RuntimeError(
        "The completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "The source-archive SHA-256 differs."
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_exact_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

status_column = resolve_exact_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


if len(
    registry
) != 10:
    raise RuntimeError(
        "Expected exactly ten completed projects."
    )


if sorted(
    registry_project_numbers.tolist()
) != list(
    range(1, 11)
):
    raise RuntimeError(
        "The registry does not contain exactly Projects 1–10."
    )


if not registry[
    status_column
].astype(str).eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "One or more Projects 1–10 are not COMPLETE_AND_FROZEN."
    )


if int(
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).sum()
) != 0:
    raise RuntimeError(
        "Project 11 is already registered."
    )


# --------------------------------------------------------------------------------------------------
# 6. VALIDATE THE STEP 1A SELECTION
# --------------------------------------------------------------------------------------------------

step1a_status = load_json(
    STEP1A_STATUS_PATH
)

step1a_report = load_json(
    STEP1A_REPORT_PATH
)

provisional_selection = load_json(
    PROVISIONAL_SELECTION_PATH
)


if step1a_status.get(
    "Status"
) != EXPECTED_STEP1A_STATUS:
    raise RuntimeError(
        "Project 11 Step 1A status differs."
    )


if step1a_report.get(
    "Status"
) != EXPECTED_STEP1A_STATUS:
    raise RuntimeError(
        "Project 11 Step 1A report status differs."
    )


selection_expectations = {
    "CandidateRank":
        1,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "SourceDirectory":
        str(
            SOURCE_DIRECTORY
        ),

    "Builds":
        EXPECTED_BUILDS,

    "TrainingBuilds":
        EXPECTED_TRAINING_BUILDS,

    "EvaluationBuilds":
        EXPECTED_EVALUATION_BUILDS,

    "RawExecutionRows":
        EXPECTED_RAW_ROWS,

    "RawTrainingRows":
        EXPECTED_RAW_TRAINING_ROWS,

    "RawEvaluationRows":
        EXPECTED_RAW_EVALUATION_ROWS,

    "RawTrainFailures":
        EXPECTED_RAW_TRAINING_FAILURES,

    "RawEvaluationFailures":
        EXPECTED_RAW_EVALUATION_FAILURES,

    "ModelReadyRows":
        EXPECTED_MODEL_ROWS,

    "ModelTrainingRows":
        EXPECTED_MODEL_TRAINING_ROWS,

    "ModelEvaluationRows":
        EXPECTED_MODEL_EVALUATION_ROWS,

    "ModelTrainFailures":
        EXPECTED_MODEL_TRAINING_FAILURES,

    "ModelEvaluationFailures":
        EXPECTED_MODEL_EVALUATION_FAILURES,
}


for key, expected_value in (
    selection_expectations.items()
):
    actual_value = provisional_selection.get(
        key
    )

    if actual_value != expected_value:
        raise RuntimeError(
            f"Provisional selection {key} differs.\n"
            f"Expected: {expected_value}\n"
            f"Actual:   {actual_value}"
        )


eligible_candidates = pd.read_csv(
    ELIGIBLE_CANDIDATES_PATH,
    low_memory=False,
)


if eligible_candidates.empty:
    raise RuntimeError(
        "The ranked eligible-candidate table is empty."
    )


ranked_top = eligible_candidates.iloc[
    0
]


if int(
    ranked_top[
        "CandidateRank"
    ]
) != 1:
    raise RuntimeError(
        "The top eligible candidate does not have rank 1."
    )


if str(
    ranked_top[
        "Project"
    ]
) != PROJECT_NAME:
    raise RuntimeError(
        "The top eligible candidate differs from Project 11."
    )


if str(
    ranked_top[
        "ProjectSlug"
    ]
) != PROJECT_SLUG:
    raise RuntimeError(
        "The top eligible candidate slug differs."
    )


# --------------------------------------------------------------------------------------------------
# 7. FREEZE SOURCE FILE MANIFEST
# --------------------------------------------------------------------------------------------------

source_hash_started = time.perf_counter()

source_manifest_records = []
schema_records = []


for filename in REQUIRED_SOURCE_FILES:
    file_path = (
        SOURCE_DIRECTORY
        / filename
    )

    columns = pd.read_csv(
        file_path,
        nrows=0,
    ).columns.tolist()

    row_count = count_csv_rows(
        file_path
    )

    source_manifest_records.append({
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "RelativePath":
            filename,

        "AbsolutePath":
            str(
                file_path
            ),

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                file_path
            ),

        "Rows":
            row_count,

        "Columns":
            len(
                columns
            ),
    })


    schema_records.append({
        "File":
            filename,

        "Rows":
            row_count,

        "ColumnCount":
            len(
                columns
            ),

        "ColumnsJSON":
            json.dumps(
                columns,
                ensure_ascii=False,
            ),
    })


source_hash_seconds = float(
    time.perf_counter()
    - source_hash_started
)


source_manifest = (
    pd.DataFrame(
        source_manifest_records
    )
    .sort_values(
        "RelativePath",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


source_schema_audit = (
    pd.DataFrame(
        schema_records
    )
    .sort_values(
        "File",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


source_file_count = len(
    source_manifest
)

source_total_bytes = int(
    source_manifest[
        "SizeBytes"
    ].sum()
)

source_root_sha256 = canonical_root_hash(
    source_manifest[
        [
            "RelativePath",
            "SizeBytes",
            "SHA256",
        ]
    ]
)


# --------------------------------------------------------------------------------------------------
# 8. RESOLVE AND FREEZE BUILD CHRONOLOGY
# --------------------------------------------------------------------------------------------------

builds_path = (
    SOURCE_DIRECTORY
    / "builds.csv"
)

build_columns = pd.read_csv(
    builds_path,
    nrows=0,
).columns.tolist()


build_id_column = resolve_exact_column(
    build_columns,
    EXPECTED_BUILD_ID_COLUMN,
    "builds.csv build ID",
)

build_timestamp_column = resolve_exact_column(
    build_columns,
    EXPECTED_BUILD_TIMESTAMP_COLUMN,
    "builds.csv timestamp",
)


builds = pd.read_csv(
    builds_path,
    usecols=[
        build_id_column,
        build_timestamp_column,
    ],
    low_memory=False,
)


builds[
    build_id_column
] = parse_build_ids(
    builds[
        build_id_column
    ],
    "builds.csv.id",
)


builds[
    build_timestamp_column
] = pd.to_datetime(
    builds[
        build_timestamp_column
    ],
    errors="coerce",
    utc=True,
)


invalid_timestamp_rows = int(
    builds[
        build_timestamp_column
    ].isna().sum()
)

duplicate_build_rows = int(
    builds[
        build_id_column
    ].duplicated(
        keep=False
    ).sum()
)


if invalid_timestamp_rows:
    raise RuntimeError(
        "The selected builds.csv contains invalid timestamps."
    )


if duplicate_build_rows:
    raise RuntimeError(
        "The selected builds.csv contains duplicate build IDs."
    )


# Frozen chronology:
# 1. started_at ascending
# 2. Build ID descending for timestamp ties

builds = (
    builds.sort_values(
        [
            build_timestamp_column,
            build_id_column,
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


build_count = len(
    builds
)

training_build_count = int(
    math.floor(
        TRAIN_FRACTION
        * build_count
    )
)

evaluation_build_count = int(
    build_count
    - training_build_count
)


builds.insert(
    0,
    "ChronologyOrder",
    range(
        1,
        build_count + 1,
    ),
)


builds[
    "Partition"
] = np.where(
    builds[
        "ChronologyOrder"
    ].le(
        training_build_count
    ),
    "TRAIN",
    "EVALUATION",
)


builds[
    "PartitionOrder"
] = (
    builds.groupby(
        "Partition",
        sort=False,
    ).cumcount()
    + 1
)


builds = builds.rename(
    columns={
        build_id_column:
            "BuildID",

        build_timestamp_column:
            "StartedAtUTC",
    }
)


builds[
    "StartedAtUTC"
] = builds[
    "StartedAtUTC"
].map(
    lambda value:
        value.isoformat()
)


training_build_ids = set(
    builds[
        builds[
            "Partition"
        ].eq(
            "TRAIN"
        )
    ][
        "BuildID"
    ]
    .astype(int)
    .tolist()
)


evaluation_build_ids = set(
    builds[
        builds[
            "Partition"
        ].eq(
            "EVALUATION"
        )
    ][
        "BuildID"
    ]
    .astype(int)
    .tolist()
)


all_build_ids = (
    training_build_ids
    | evaluation_build_ids
)


partition_overlap = len(
    training_build_ids
    & evaluation_build_ids
)

partition_union_count = len(
    all_build_ids
)


timestamp_tie_rows = int(
    builds[
        "StartedAtUTC"
    ].duplicated(
        keep=False
    ).sum()
)

timestamp_tie_groups = int(
    builds.groupby(
        "StartedAtUTC"
    ).size().gt(1).sum()
)


# --------------------------------------------------------------------------------------------------
# 9. REVALIDATE RAW AND MODEL-READY PARTITIONS
# --------------------------------------------------------------------------------------------------

exe_columns = pd.read_csv(
    SOURCE_DIRECTORY
    / "exe.csv",
    nrows=0,
).columns.tolist()

dataset_columns = pd.read_csv(
    SOURCE_DIRECTORY
    / "dataset.csv",
    nrows=0,
).columns.tolist()


exe_build_column = resolve_exact_column(
    exe_columns,
    EXPECTED_EXE_BUILD_COLUMN,
    "exe.csv build column",
)

exe_verdict_column = resolve_exact_column(
    exe_columns,
    EXPECTED_EXE_VERDICT_COLUMN,
    "exe.csv verdict column",
)

dataset_build_column = resolve_exact_column(
    dataset_columns,
    EXPECTED_DATASET_BUILD_COLUMN,
    "dataset.csv build column",
)

dataset_verdict_column = resolve_exact_column(
    dataset_columns,
    EXPECTED_DATASET_VERDICT_COLUMN,
    "dataset.csv verdict column",
)


raw_counts = scan_partitioned_verdict_table(
    path=(
        SOURCE_DIRECTORY
        / "exe.csv"
    ),
    build_column=exe_build_column,
    verdict_column=exe_verdict_column,
    all_build_ids=all_build_ids,
    training_build_ids=training_build_ids,
    evaluation_build_ids=evaluation_build_ids,
    label="apache@shardingsphere.exe.csv",
)


model_counts = scan_partitioned_verdict_table(
    path=(
        SOURCE_DIRECTORY
        / "dataset.csv"
    ),
    build_column=dataset_build_column,
    verdict_column=dataset_verdict_column,
    all_build_ids=all_build_ids,
    training_build_ids=training_build_ids,
    evaluation_build_ids=evaluation_build_ids,
    label="apache@shardingsphere.dataset.csv",
)


# --------------------------------------------------------------------------------------------------
# 10. PARTITION AND DIMENSION AUDITS
# --------------------------------------------------------------------------------------------------

partition_audit = pd.DataFrame([
    {
        "Partition":
            "TRAIN",

        "FirstChronologyOrder":
            1,

        "LastChronologyOrder":
            training_build_count,

        "Builds":
            len(
                training_build_ids
            ),

        "FirstBuildID":
            int(
                builds.iloc[0][
                    "BuildID"
                ]
            ),

        "LastBuildID":
            int(
                builds.iloc[
                    training_build_count - 1
                ][
                    "BuildID"
                ]
            ),

        "FirstStartedAtUTC":
            builds.iloc[0][
                "StartedAtUTC"
            ],

        "LastStartedAtUTC":
            builds.iloc[
                training_build_count - 1
            ][
                "StartedAtUTC"
            ],
    },

    {
        "Partition":
            "EVALUATION",

        "FirstChronologyOrder":
            training_build_count + 1,

        "LastChronologyOrder":
            build_count,

        "Builds":
            len(
                evaluation_build_ids
            ),

        "FirstBuildID":
            int(
                builds.iloc[
                    training_build_count
                ][
                    "BuildID"
                ]
            ),

        "LastBuildID":
            int(
                builds.iloc[-1][
                    "BuildID"
                ]
            ),

        "FirstStartedAtUTC":
            builds.iloc[
                training_build_count
            ][
                "StartedAtUTC"
            ],

        "LastStartedAtUTC":
            builds.iloc[-1][
                "StartedAtUTC"
            ],
    },
])


dimension_records = []


def dimension_check(
    metric,
    expected,
    actual,
):
    dimension_records.append({
        "Metric":
            metric,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                actual == expected
            ),
    })


dimension_check(
    "SourceFiles",
    EXPECTED_SOURCE_FILES,
    source_file_count,
)

dimension_check(
    "SourceBytes",
    EXPECTED_SOURCE_BYTES,
    source_total_bytes,
)

dimension_check(
    "Builds",
    EXPECTED_BUILDS,
    build_count,
)

dimension_check(
    "TrainingBuilds",
    EXPECTED_TRAINING_BUILDS,
    training_build_count,
)

dimension_check(
    "EvaluationBuilds",
    EXPECTED_EVALUATION_BUILDS,
    evaluation_build_count,
)

dimension_check(
    "RawRows",
    EXPECTED_RAW_ROWS,
    raw_counts[
        "Rows"
    ],
)

dimension_check(
    "RawTrainingRows",
    EXPECTED_RAW_TRAINING_ROWS,
    raw_counts[
        "TrainingRows"
    ],
)

dimension_check(
    "RawEvaluationRows",
    EXPECTED_RAW_EVALUATION_ROWS,
    raw_counts[
        "EvaluationRows"
    ],
)

dimension_check(
    "RawTrainingFailures",
    EXPECTED_RAW_TRAINING_FAILURES,
    raw_counts[
        "TrainingFailures"
    ],
)

dimension_check(
    "RawEvaluationFailures",
    EXPECTED_RAW_EVALUATION_FAILURES,
    raw_counts[
        "EvaluationFailures"
    ],
)

dimension_check(
    "RawFailingEvaluationBuilds",
    EXPECTED_RAW_FAILING_EVALUATION_BUILDS,
    raw_counts[
        "FailingEvaluationBuilds"
    ],
)

dimension_check(
    "RawUnlinkedRows",
    0,
    raw_counts[
        "UnlinkedRows"
    ],
)

dimension_check(
    "ModelRows",
    EXPECTED_MODEL_ROWS,
    model_counts[
        "Rows"
    ],
)

dimension_check(
    "ModelTrainingRows",
    EXPECTED_MODEL_TRAINING_ROWS,
    model_counts[
        "TrainingRows"
    ],
)

dimension_check(
    "ModelEvaluationRows",
    EXPECTED_MODEL_EVALUATION_ROWS,
    model_counts[
        "EvaluationRows"
    ],
)

dimension_check(
    "ModelTrainingFailures",
    EXPECTED_MODEL_TRAINING_FAILURES,
    model_counts[
        "TrainingFailures"
    ],
)

dimension_check(
    "ModelEvaluationFailures",
    EXPECTED_MODEL_EVALUATION_FAILURES,
    model_counts[
        "EvaluationFailures"
    ],
)

dimension_check(
    "ModelFailingEvaluationBuilds",
    EXPECTED_MODEL_FAILING_EVALUATION_BUILDS,
    model_counts[
        "FailingEvaluationBuilds"
    ],
)

dimension_check(
    "ModelUnlinkedRows",
    0,
    model_counts[
        "UnlinkedRows"
    ],
)

dimension_check(
    "PartitionOverlap",
    0,
    partition_overlap,
)

dimension_check(
    "PartitionUnionBuilds",
    EXPECTED_BUILDS,
    partition_union_count,
)


dimension_audit = pd.DataFrame(
    dimension_records
)


failed_dimension_checks = dimension_audit[
    ~dimension_audit[
        "Pass"
    ]
]


if not failed_dimension_checks.empty:
    print("\nFailed dimension checks:")

    display(
        failed_dimension_checks
    )

    raise RuntimeError(
        "PROJECT 11 SOURCE DIMENSIONS DIFFER FROM STEP 1A."
    )


# --------------------------------------------------------------------------------------------------
# 11. FINAL VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 1A status",
    EXPECTED_STEP1A_STATUS,
    step1a_status.get(
        "Status"
    ),
    step1a_status.get(
        "Status"
    ) == EXPECTED_STEP1A_STATUS,
)

add_check(
    validation_records,
    "Selected candidate rank",
    1,
    provisional_selection.get(
        "CandidateRank"
    ),
    provisional_selection.get(
        "CandidateRank"
    ) == 1,
)

add_check(
    validation_records,
    "Selected project",
    PROJECT_NAME,
    provisional_selection.get(
        "Project"
    ),
    provisional_selection.get(
        "Project"
    ) == PROJECT_NAME,
)

add_check(
    validation_records,
    "Selected project slug",
    PROJECT_SLUG,
    provisional_selection.get(
        "ProjectSlug"
    ),
    provisional_selection.get(
        "ProjectSlug"
    ) == PROJECT_SLUG,
)

add_check(
    validation_records,
    "Selected source directory",
    str(
        SOURCE_DIRECTORY
    ),
    provisional_selection.get(
        "SourceDirectory"
    ),
    provisional_selection.get(
        "SourceDirectory"
    ) == str(
        SOURCE_DIRECTORY
    ),
)

add_check(
    validation_records,
    "Required source files",
    EXPECTED_SOURCE_FILES,
    source_file_count,
    source_file_count
    == EXPECTED_SOURCE_FILES,
)

add_check(
    validation_records,
    "Selected source bytes",
    EXPECTED_SOURCE_BYTES,
    source_total_bytes,
    source_total_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    validation_records,
    "Invalid build timestamps",
    0,
    invalid_timestamp_rows,
    invalid_timestamp_rows == 0,
)

add_check(
    validation_records,
    "Duplicate build rows",
    0,
    duplicate_build_rows,
    duplicate_build_rows == 0,
)

add_check(
    validation_records,
    "Training/evaluation overlap",
    0,
    partition_overlap,
    partition_overlap == 0,
)

add_check(
    validation_records,
    "Training/evaluation union",
    EXPECTED_BUILDS,
    partition_union_count,
    partition_union_count
    == EXPECTED_BUILDS,
)

add_check(
    validation_records,
    "Dimension-audit failures",
    0,
    len(
        failed_dimension_checks
    ),
    len(
        failed_dimension_checks
    ) == 0,
)

add_check(
    validation_records,
    "Completion registry rows",
    10,
    len(
        registry
    ),
    len(
        registry
    ) == 10,
)

add_check(
    validation_records,
    "Project 11 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)

add_check(
    validation_records,
    "Selected source root generated",
    True,
    bool(
        source_root_sha256
    ),
    len(
        source_root_sha256
    ) == 64,
)


validation = pd.DataFrame(
    validation_records
)

failed_validation = validation[
    ~validation[
        "Pass"
    ]
]


print("\nProject 11 Step 1B validation:")

display(
    validation
)


if not failed_validation.empty:
    raise RuntimeError(
        "PROJECT 11 STEP 1B VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 12. WRITE FROZEN EVIDENCE
# --------------------------------------------------------------------------------------------------

SELECTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    source_manifest,
)

atomic_write_csv(
    FIXED_CHRONOLOGY_PATH,
    builds,
)

atomic_write_csv(
    PARTITION_AUDIT_PATH,
    partition_audit,
)

atomic_write_csv(
    DIMENSION_AUDIT_PATH,
    dimension_audit,
)

atomic_write_csv(
    FINAL_SCHEMA_AUDIT_PATH,
    source_schema_audit,
)

atomic_write_csv(
    STEP1B_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 13. READBACK AND SOURCE IMMUTABILITY
# --------------------------------------------------------------------------------------------------

manifest_readback = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

chronology_readback = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


if len(
    manifest_readback
) != EXPECTED_SOURCE_FILES:
    raise RuntimeError(
        "Frozen source-manifest readback failed."
    )


if len(
    chronology_readback
) != EXPECTED_BUILDS:
    raise RuntimeError(
        "Frozen chronology readback failed."
    )


source_recheck_records = []


for row in source_manifest.itertuples(
    index=False
):
    file_path = (
        SOURCE_DIRECTORY
        / row.RelativePath
    )

    source_recheck_records.append({
        "RelativePath":
            row.RelativePath,

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                file_path
            ),
    })


source_recheck_manifest = pd.DataFrame(
    source_recheck_records
)


source_recheck_root_sha256 = canonical_root_hash(
    source_recheck_manifest
)


if (
    source_recheck_root_sha256
    != source_root_sha256
):
    raise RuntimeError(
        "The selected Project 11 source changed during freezing."
    )


registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 11 Step 1B."
    )


# --------------------------------------------------------------------------------------------------
# 14. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP1B_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SelectionState":
        "FINAL_AND_FROZEN",

    "CandidateRank":
        1,

    "SourceDirectory":
        str(
            SOURCE_DIRECTORY
        ),

    "SourceArchive":
        str(
            SOURCE_ARCHIVE_PATH
        ),

    "SourceArchiveSHA256":
        archive_sha256,

    "SourceFiles":
        source_file_count,

    "SourceBytes":
        source_total_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "SourceHashSeconds":
        source_hash_seconds,

    "ChronologyRule":
        (
            "started_at ascending; "
            "Build ID descending for timestamp ties"
        ),

    "SplitRule":
        (
            "floor(0.75 × builds) training; "
            "remaining builds clean evaluation"
        ),

    "Builds":
        build_count,

    "TrainingBuilds":
        training_build_count,

    "EvaluationBuilds":
        evaluation_build_count,

    "TimestampTieGroups":
        timestamp_tie_groups,

    "TimestampTieRows":
        timestamp_tie_rows,

    "RawExecutionRows":
        raw_counts[
            "Rows"
        ],

    "RawTrainingRows":
        raw_counts[
            "TrainingRows"
        ],

    "RawEvaluationRows":
        raw_counts[
            "EvaluationRows"
        ],

    "RawTrainingFailures":
        raw_counts[
            "TrainingFailures"
        ],

    "RawEvaluationFailures":
        raw_counts[
            "EvaluationFailures"
        ],

    "RawFailingTrainingBuilds":
        raw_counts[
            "FailingTrainingBuilds"
        ],

    "RawFailingEvaluationBuilds":
        raw_counts[
            "FailingEvaluationBuilds"
        ],

    "ModelReadyRows":
        model_counts[
            "Rows"
        ],

    "ModelTrainingRows":
        model_counts[
            "TrainingRows"
        ],

    "ModelEvaluationRows":
        model_counts[
            "EvaluationRows"
        ],

    "ModelTrainingFailures":
        model_counts[
            "TrainingFailures"
        ],

    "ModelEvaluationFailures":
        model_counts[
            "EvaluationFailures"
        ],

    "ModelFailingTrainingBuilds":
        model_counts[
            "FailingTrainingBuilds"
        ],

    "ModelFailingEvaluationBuilds":
        model_counts[
            "FailingEvaluationBuilds"
        ],

    "BuildIDColumn":
        build_id_column,

    "BuildTimestampColumn":
        build_timestamp_column,

    "ExecutionBuildColumn":
        exe_build_column,

    "ExecutionVerdictColumn":
        exe_verdict_column,

    "DatasetBuildColumn":
        dataset_build_column,

    "DatasetVerdictColumn":
        dataset_verdict_column,

    "FrozenSourceManifest":
        str(
            FROZEN_SOURCE_MANIFEST_PATH
        ),

    "FrozenSourceManifestSHA256":
        sha256_file(
            FROZEN_SOURCE_MANIFEST_PATH
        ),

    "FixedChronology":
        str(
            FIXED_CHRONOLOGY_PATH
        ),

    "FixedChronologySHA256":
        sha256_file(
            FIXED_CHRONOLOGY_PATH
        ),

    "PartitionAudit":
        str(
            PARTITION_AUDIT_PATH
        ),

    "PartitionAuditSHA256":
        sha256_file(
            PARTITION_AUDIT_PATH
        ),

    "DimensionAudit":
        str(
            DIMENSION_AUDIT_PATH
        ),

    "DimensionAuditSHA256":
        sha256_file(
            DIMENSION_AUDIT_PATH
        ),

    "FinalSchemaAudit":
        str(
            FINAL_SCHEMA_AUDIT_PATH
        ),

    "FinalSchemaAuditSHA256":
        sha256_file(
            FINAL_SCHEMA_AUDIT_PATH
        ),

    "Step1AStatus":
        str(
            STEP1A_STATUS_PATH
        ),

    "Step1AStatusSHA256":
        sha256_file(
            STEP1A_STATUS_PATH
        ),

    "EligibleCandidateRanking":
        str(
            ELIGIBLE_CANDIDATES_PATH
        ),

    "EligibleCandidateRankingSHA256":
        sha256_file(
            ELIGIBLE_CANDIDATES_PATH
        ),

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To10Modified":
        False,

    "Project12Accessed":
        False,

    "Project12WriteAttempted":
        False,

    "ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "SelectionCheckpointVersion":
        1,

    "SelectionFrozenAtUTC":
        completed_at_utc,

    "DoNotChangeProjectIdentity":
        True,

    "DoNotChangeSourceFiles":
        True,

    "EvaluationCohortImmutable":
        True,

    "FinalSelection":
        True,
}


atomic_write_json(
    SELECTION_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP1B_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SelectionState":
        "FINAL_AND_FROZEN",

    "SourceFiles":
        source_file_count,

    "SourceBytes":
        source_total_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "Builds":
        build_count,

    "TrainingBuilds":
        training_build_count,

    "EvaluationBuilds":
        evaluation_build_count,

    "Checkpoint":
        str(
            SELECTION_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        sha256_file(
            SELECTION_CHECKPOINT_PATH
        ),

    "RegistryModified":
        False,

    "Project12Accessed":
        False,
}


atomic_write_json(
    STEP1B_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 15. FINAL READBACK
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    SELECTION_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP1B_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP1B_PASS_STATUS:
    raise RuntimeError(
        "Project 11 selection checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP1B_PASS_STATUS:
    raise RuntimeError(
        "Project 11 Step 1B status readback failed."
    )


if sha256_file(
    REGISTRY_PATH
) != registry_sha256_before:
    raise RuntimeError(
        "The registry changed during final readback."
    )


# --------------------------------------------------------------------------------------------------
# 16. DISPLAY
# --------------------------------------------------------------------------------------------------

print("\nFrozen Project 11 source manifest:")

display(
    source_manifest
)


print("\nFixed chronological partition sample:")

display(
    pd.concat(
        [
            builds.head(5),
            builds.iloc[
                training_build_count - 3:
                training_build_count + 3
            ],
            builds.tail(5),
        ],
        ignore_index=True,
    )
)


print("\nPartition audit:")

display(
    partition_audit
)


print("\nDimension audit:")

display(
    dimension_audit
)


print("\nFinal source schema audit:")

display(
    source_schema_audit
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 128)
print("=== PROJECT 11 CELL 3 / STEP 1B RESULT ===")
print("=" * 128)


print("\nFinal Project 11 identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "Candidate rank:",
    1,
)

print(
    "Selection state:",
    "FINAL_AND_FROZEN",
)


print("\nFrozen selected source:")

print(
    "Source directory:",
    SOURCE_DIRECTORY,
)

print(
    "Source files:",
    source_file_count,
)

print(
    "Source bytes:",
    source_total_bytes,
)

print(
    "Source root SHA-256:",
    source_root_sha256,
)

print(
    "Archive SHA-256:",
    archive_sha256,
)


print("\nFrozen chronology and split:")

print(
    "Chronology:",
    (
        "started_at ascending; "
        "Build ID descending for timestamp ties"
    ),
)

print(
    "Builds:",
    build_count,
)

print(
    "Training builds:",
    training_build_count,
)

print(
    "Evaluation builds:",
    evaluation_build_count,
)

print(
    "Timestamp tie groups:",
    timestamp_tie_groups,
)

print(
    "Training/evaluation overlap:",
    partition_overlap,
)


print("\nRaw execution dimensions:")

print(
    "Rows:",
    raw_counts[
        "Rows"
    ],
)

print(
    "Training / evaluation rows:",
    raw_counts[
        "TrainingRows"
    ],
    "/",
    raw_counts[
        "EvaluationRows"
    ],
)

print(
    "Training / evaluation failures:",
    raw_counts[
        "TrainingFailures"
    ],
    "/",
    raw_counts[
        "EvaluationFailures"
    ],
)

print(
    "Failing training / evaluation builds:",
    raw_counts[
        "FailingTrainingBuilds"
    ],
    "/",
    raw_counts[
        "FailingEvaluationBuilds"
    ],
)


print("\nModel-ready dimensions:")

print(
    "Rows:",
    model_counts[
        "Rows"
    ],
)

print(
    "Training / evaluation rows:",
    model_counts[
        "TrainingRows"
    ],
    "/",
    model_counts[
        "EvaluationRows"
    ],
)

print(
    "Training / evaluation failures:",
    model_counts[
        "TrainingFailures"
    ],
    "/",
    model_counts[
        "EvaluationFailures"
    ],
)

print(
    "Failing training / evaluation builds:",
    model_counts[
        "FailingTrainingBuilds"
    ],
    "/",
    model_counts[
        "FailingEvaluationBuilds"
    ],
)


print("\nImmutability and isolation:")

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–10 modified:",
    0,
)

print(
    "Project 12 accessed:",
    False,
)

print(
    "Project 11 experiment started:",
    False,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print("\nSelection checkpoint:")

print(
    SELECTION_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    sha256_file(
        SELECTION_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP1B_PASS_STATUS,
)

print("=" * 128)

=== PROJECT 11 CELL 3 / STEP 1B: FINAL SELECTION AND SOURCE FREEZE ===

Project 11 Step 1B validation:


,Check,Expected,Actual,Pass
0,Step 1A status,PASS_PROJECT_11_CANDIDATE_DISCOVERY_COMPLETE,PASS_PROJECT_11_CANDIDATE_DISCOVERY_COMPLETE,True
1,Selected candidate rank,1,1,True
2,Selected project,apache@shardingsphere,apache@shardingsphere,True
3,Selected project slug,apache__shardingsphere,apache__shardingsphere,True
4,Selected source directory,/content/datasets/datasets/apache@shardingsphere,/content/datasets/datasets/apache@shardingsphere,True
5,Required source files,6,6,True
6,Selected source bytes,142849042,142849042,True
7,Invalid build timestamps,0,0,True
8,Duplicate build rows,0,0,True
9,Training/evaluation overlap,0,0,True



Frozen Project 11 source manifest:


,ProjectNumber,Project,ProjectSlug,RelativePath,AbsolutePath,SizeBytes,SHA256,Rows,Columns
0,11,apache@shardingsphere,apache__shardingsphere,builds.csv,/content/datasets/datasets/apache@shardingsphe...,117531,a7cda0866a5626de95d1f1b86abeed3ecbb930c4cfad7c...,1049,3
1,11,apache@shardingsphere,apache__shardingsphere,contributors.csv,/content/datasets/datasets/apache@shardingsphe...,23472,b54024587c9cfba2a3dbed9be10bb2a36f7293d46450e9...,373,4
2,11,apache@shardingsphere,apache__shardingsphere,dataset.csv,/content/datasets/datasets/apache@shardingsphe...,67162630,d3cf67599abff259b13b57d3558e0afbbd6bb9b2844634...,91042,154
3,11,apache@shardingsphere,apache__shardingsphere,entity_change_history.csv,/content/datasets/datasets/apache@shardingsphe...,39475811,6e14edbf920c7fb1f075d248f847fc645a5ff16bf698bf...,450927,8
4,11,apache@shardingsphere,apache__shardingsphere,exe.csv,/content/datasets/datasets/apache@shardingsphe...,26227220,0143d1c9c5c7d6db991e12989213e0aaaa79a1d47bce7f...,833541,5
5,11,apache@shardingsphere,apache__shardingsphere,id_map.csv,/content/datasets/datasets/apache@shardingsphe...,9842378,52f5bf967e305c34d3f3ea76133c137996f8d138059924...,72397,2



Fixed chronological partition sample:


...


Dimension audit:


,Metric,Expected,Actual,Pass
0,SourceFiles,6,6,True
1,SourceBytes,142849042,142849042,True
2,Builds,1049,1049,True
3,TrainingBuilds,786,786,True
4,EvaluationBuilds,263,263,True
5,RawRows,833541,833541,True
6,RawTrainingRows,609619,609619,True
7,RawEvaluationRows,223922,223922,True
8,RawTrainingFailures,1193,1193,True
9,RawEvaluationFailures,171,171,True



Final source schema audit:


,File,Rows,ColumnCount,ColumnsJSON
0,builds.csv,1049,3,"[""id"", ""commits"", ""started_at""]"
1,contributors.csv,373,4,"[""Id"", ""Key"", ""Name"", ""Email""]"
2,dataset.csv,91042,154,"[""Build"", ""Test"", ""TES_COM_CountDeclFunction"",..."
3,entity_change_history.csv,450927,8,"[""EntityId"", ""AddedLines"", ""DeletedLines"", ""Co..."
4,exe.csv,833541,5,"[""test"", ""build"", ""job"", ""verdict"", ""duration""]"
5,id_map.csv,72397,2,"[""key"", ""value""]"




=== PROJECT 11 CELL 3 / STEP 1B RESULT ===

Final Project 11 identity:
Project number: 11
Project: apache@shardingsphere
Project slug: apache__shardingsphere
Candidate rank: 1
Selection state: FINAL_AND_FROZEN

Frozen selected source:
Source directory: /content/datasets/datasets/apache@shardingsphere
Source files: 6
Source bytes: 142849042
Source root SHA-256: 3e6e0a60d58bff61690d6e2ac2bac18371414819eff601f3ea347ecb8ad8b0b5
Archive SHA-256: 92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e

Frozen chronology and split:
Chronology: started_at ascending; Build ID descending for timestamp ties
Builds: 1049
Training builds: 786
Evaluation builds: 263
Timestamp tie groups: 0
Training/evaluation overlap: 0

Raw execution dimensions:
Rows: 833541
Training / evaluation rows: 609619 / 223922
Training / evaluation failures: 1193 / 171
Failing training / evaluation builds: 127 / 25

Model-ready dimensions:
Rows: 91042
Training / evaluation rows: 78035 / 13007
Training / evaluatio

In [4]:
# ==================================================================================================
# PROJECT 11 — CELL 4 / STEP 2A
# SOURCE SCHEMA, BUILD-TEST JOIN, COMMIT MATCHING,
# AND BUILD-ENTITY STRUCTURE VALIDATION
#
# PROJECT:
#   apache@shardingsphere
#
# PURPOSE:
# - validate the frozen Project 11 selection and source root
# - validate all source schemas required by REC reconstruction
# - validate raw/model-ready Build-Test joins and verdict alignment
# - resolve build commits against entity_change_history commits
# - construct and freeze the build-to-changed-entity map
# - validate id_map coverage for entity-history identifiers
# - prepare the inputs required by clean REC reconstruction
#
# SAFETY:
# - does not inject noise
# - does not train models
# - does not create experiment conditions
# - does not modify the completion registry
# - does not modify Projects 1–10
# - does not access or write Project 12
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 11 CELL 4 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PROJECT CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 11

PROJECT_NAME = (
    "apache@shardingsphere"
)

PROJECT_SLUG = (
    "apache__shardingsphere"
)

PROJECT_SHORT_NAME = (
    "shardingsphere"
)

SOURCE_DIRECTORY = Path(
    "/content/datasets/datasets/apache@shardingsphere"
)


EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_11_SELECTION_AND_SOURCE_FROZEN"
)

STEP2A_PASS_STATUS = (
    "PASS_PROJECT_11_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
)


EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "9d5c95b29357426def26c41b62a753c0"
    "03ad3b19e6086d9873f0961e39875353"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "3e6e0a60d58bff61690d6e2ac2bac183"
    "71414819eff601f3ea347ecb8ad8b0b5"
)

EXPECTED_REGISTRY_SHA256 = (
    "847bdcbee16c8757fde34e9489cd3abb"
    "f0781a319644175472597265b1df2750"
)


EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 142_849_042

EXPECTED_BUILDS = 1_049
EXPECTED_TRAINING_BUILDS = 786
EXPECTED_EVALUATION_BUILDS = 263

EXPECTED_RAW_ROWS = 833_541
EXPECTED_MODEL_ROWS = 91_042

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTOR_COLUMNS = 151
EXPECTED_REC_FEATURES = 19

EXPECTED_MODEL_TRAINING_ROWS = 78_035
EXPECTED_MODEL_EVALUATION_ROWS = 13_007
EXPECTED_MODEL_TRAINING_FAILURES = 1_188
EXPECTED_MODEL_EVALUATION_FAILURES = 171

EXPECTED_COMPLETE_STATUS = (
    "COMPLETE_AND_FROZEN"
)


REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


FILE_HISTORY_REC_FEATURES = [
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


REQUIRED_SOURCE_FILES = [
    "builds.csv",
    "contributors.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "exe.csv",
    "id_map.csv",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)


SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_11_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_11_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_11_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_11_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_11_fixed_chronological_builds.csv"
)


PROJECT_AGGREGATED_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_AGGREGATED_ROOT
    / f"{PROJECT_SHORT_NAME}_rec_preflight"
)

SOURCE_SCHEMA_PROFILE_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_source_schema_profile.csv"
)

BUILD_TEST_JOIN_AUDIT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_build_test_join_audit.csv"
)

REC_FEATURE_CLASSIFICATION_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_rec_feature_classification.csv"
)

BUILD_COMMIT_TOKEN_PROFILE_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_build_commit_token_profile.csv"
)

COMMIT_MATCHING_AUDIT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_commit_matching_audit.csv"
)

BUILD_ENTITY_MAP_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_build_entity_map.csv.gz"
)

ENTITY_ID_MAP_AUDIT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_entity_id_map_audit.csv"
)

ENTITY_MAPPING_SUMMARY_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_entity_mapping_summary.json"
)

STEP2A_VALIDATION_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_step2a_validation.csv"
)

STEP2A_REPORT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_step2a_report.json"
)

STEP2A_STATUS_PATH = (
    PROJECT_AGGREGATED_ROOT
    / f"{PROJECT_SHORT_NAME}_step2a_status.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv_gzip(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.stem}.tmp_{os.getpid()}.csv.gz"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        compression="gzip",
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def canonical_root_hash(
    manifest,
):
    required_columns = [
        "RelativePath",
        "SizeBytes",
        "SHA256",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in manifest.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Cannot calculate root hash. Missing columns:\n"
            + "\n".join(
                missing_columns
            )
        )

    digest = hashlib.sha256()

    ordered = manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_exact_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {label}.\n"
            f"Expected: {expected!r}\n"
            f"Matches: {matches}\n"
            f"Columns: {list(columns)}"
        )

    return matches[0]


def parse_integer_ids(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains missing or non-numeric IDs."
        )

    numeric_array = numeric.to_numpy(
        dtype=float
    )

    if not np.equal(
        numeric_array,
        np.floor(
            numeric_array
        ),
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral IDs."
        )

    return numeric.astype(
        "int64"
    )


def normalise_commit(
    value,
):
    if pd.isna(value):
        return ""

    value = str(
        value
    ).strip().lower()

    if not value:
        return ""

    hexadecimal_matches = re.findall(
        r"[0-9a-f]{7,64}",
        value,
        flags=re.IGNORECASE,
    )

    if hexadecimal_matches:
        return hexadecimal_matches[0].lower()

    return re.sub(
        r"[^a-z0-9]",
        "",
        value,
    )


def extract_commit_tokens(
    value,
):
    if pd.isna(value):
        return []

    text = str(
        value
    ).strip()

    if not text:
        return []

    tokens = re.findall(
        r"[0-9a-fA-F]{7,64}",
        text,
    )

    if tokens:
        ordered_unique = []

        seen = set()

        for token in tokens:
            normalised = token.lower()

            if normalised not in seen:
                seen.add(
                    normalised
                )

                ordered_unique.append(
                    normalised
                )

        return ordered_unique

    fallback_tokens = re.split(
        r"[\s,;|]+",
        text,
    )

    ordered_unique = []
    seen = set()

    for token in fallback_tokens:
        normalised = normalise_commit(
            token
        )

        if (
            normalised
            and normalised not in seen
        ):
            seen.add(
                normalised
            )

            ordered_unique.append(
                normalised
            )

    return ordered_unique


def add_check(
    records,
    check,
    expected,
    actual,
    passed,
):
    records.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUT VALIDATION
# --------------------------------------------------------------------------------------------------

required_files = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Required Project 11 files are missing:\n"
        + "\n".join(
            missing_files
        )
    )


if not SOURCE_DIRECTORY.is_dir():
    raise FileNotFoundError(
        "The frozen Project 11 source directory is missing:\n"
        f"{SOURCE_DIRECTORY}"
    )


missing_source_files = [
    filename
    for filename in REQUIRED_SOURCE_FILES
    if not (
        SOURCE_DIRECTORY
        / filename
    ).is_file()
]

if missing_source_files:
    raise FileNotFoundError(
        "Frozen Project 11 source files are missing:\n"
        + "\n".join(
            missing_source_files
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE STEP 1B, REGISTRY, AND SOURCE ROOT
# --------------------------------------------------------------------------------------------------

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)


if (
    selection_checkpoint_sha256
    != EXPECTED_SELECTION_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 11 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_CHECKPOINT_SHA256}\n"
        f"Actual:   {selection_checkpoint_sha256}"
    )


if selection_checkpoint.get(
    "Status"
) != EXPECTED_STEP1B_STATUS:
    raise RuntimeError(
        "Project 11 selection-checkpoint status differs."
    )


if step1b_status.get(
    "Status"
) != EXPECTED_STEP1B_STATUS:
    raise RuntimeError(
        "Project 11 Step 1B status differs."
    )


if selection_checkpoint.get(
    "Project"
) != PROJECT_NAME:
    raise RuntimeError(
        "The frozen Project 11 identity differs."
    )


if selection_checkpoint.get(
    "ProjectSlug"
) != PROJECT_SLUG:
    raise RuntimeError(
        "The frozen Project 11 slug differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "The completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_exact_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

status_column = resolve_exact_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


if len(registry) != 10:
    raise RuntimeError(
        "Expected ten frozen projects in the registry."
    )


if sorted(
    registry_project_numbers.tolist()
) != list(
    range(1, 11)
):
    raise RuntimeError(
        "The registry does not contain exactly Projects 1–10."
    )


if not registry[
    status_column
].astype(str).eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "One or more Projects 1–10 are not COMPLETE_AND_FROZEN."
    )


if int(
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).sum()
) != 0:
    raise RuntimeError(
        "Project 11 is unexpectedly present in the registry."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_records = []

for row in frozen_source_manifest.itertuples(
    index=False
):
    file_path = (
        SOURCE_DIRECTORY
        / str(
            row.RelativePath
        )
    )

    if not file_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 11 source file is missing:\n"
            f"{file_path}"
        )

    current_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                file_path
            ),
    })


current_source_manifest = pd.DataFrame(
    current_source_records
)

current_source_root_sha256 = canonical_root_hash(
    current_source_manifest
)

current_source_file_count = len(
    current_source_manifest
)

current_source_bytes = int(
    current_source_manifest[
        "SizeBytes"
    ].sum()
)


if (
    current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The frozen Project 11 source root differs.\n"
        f"Expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Actual:   {current_source_root_sha256}"
    )


# --------------------------------------------------------------------------------------------------
# 6. RESOLVE SOURCE SCHEMAS
# --------------------------------------------------------------------------------------------------

builds_path = (
    SOURCE_DIRECTORY
    / "builds.csv"
)

exe_path = (
    SOURCE_DIRECTORY
    / "exe.csv"
)

dataset_path = (
    SOURCE_DIRECTORY
    / "dataset.csv"
)

entity_history_path = (
    SOURCE_DIRECTORY
    / "entity_change_history.csv"
)

id_map_path = (
    SOURCE_DIRECTORY
    / "id_map.csv"
)

contributors_path = (
    SOURCE_DIRECTORY
    / "contributors.csv"
)


source_paths = {
    "builds.csv":
        builds_path,

    "contributors.csv":
        contributors_path,

    "dataset.csv":
        dataset_path,

    "entity_change_history.csv":
        entity_history_path,

    "exe.csv":
        exe_path,

    "id_map.csv":
        id_map_path,
}


source_schema_records = []


for filename, file_path in (
    source_paths.items()
):
    columns = pd.read_csv(
        file_path,
        nrows=0,
    ).columns.tolist()

    source_schema_records.append({
        "File":
            filename,

        "Path":
            str(
                file_path
            ),

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "ColumnCount":
            len(
                columns
            ),

        "ColumnsJSON":
            json.dumps(
                columns,
                ensure_ascii=False,
            ),
    })


source_schema_profile = pd.DataFrame(
    source_schema_records
)


build_columns = pd.read_csv(
    builds_path,
    nrows=0,
).columns.tolist()

exe_columns = pd.read_csv(
    exe_path,
    nrows=0,
).columns.tolist()

dataset_columns = pd.read_csv(
    dataset_path,
    nrows=0,
).columns.tolist()

entity_history_columns = pd.read_csv(
    entity_history_path,
    nrows=0,
).columns.tolist()

id_map_columns = pd.read_csv(
    id_map_path,
    nrows=0,
).columns.tolist()


build_id_column = resolve_exact_column(
    build_columns,
    "id",
    "builds.csv build ID",
)

build_commit_column = resolve_exact_column(
    build_columns,
    "commits",
    "builds.csv commit column",
)

build_timestamp_column = resolve_exact_column(
    build_columns,
    "started_at",
    "builds.csv timestamp",
)


exe_test_column = resolve_exact_column(
    exe_columns,
    "test",
    "exe.csv test column",
)

exe_build_column = resolve_exact_column(
    exe_columns,
    "build",
    "exe.csv build column",
)

exe_job_column = resolve_exact_column(
    exe_columns,
    "job",
    "exe.csv job column",
)

exe_verdict_column = resolve_exact_column(
    exe_columns,
    "verdict",
    "exe.csv verdict column",
)

exe_duration_column = resolve_exact_column(
    exe_columns,
    "duration",
    "exe.csv duration column",
)


dataset_build_column = resolve_exact_column(
    dataset_columns,
    "Build",
    "dataset.csv Build column",
)

dataset_test_column = resolve_exact_column(
    dataset_columns,
    "Test",
    "dataset.csv Test column",
)

dataset_verdict_column = resolve_exact_column(
    dataset_columns,
    "Verdict",
    "dataset.csv Verdict column",
)


entity_commit_column = resolve_exact_column(
    entity_history_columns,
    "Commit",
    "entity_change_history.csv Commit column",
)

entity_id_column = resolve_exact_column(
    entity_history_columns,
    "EntityId",
    "entity_change_history.csv EntityId column",
)


id_map_key_column = resolve_exact_column(
    id_map_columns,
    "key",
    "id_map.csv key column",
)

id_map_value_column = resolve_exact_column(
    id_map_columns,
    "value",
    "id_map.csv value column",
)


missing_rec_columns = [
    column
    for column in REC_FEATURE_COLUMNS
    if column not in dataset_columns
]


if missing_rec_columns:
    raise RuntimeError(
        "dataset.csv is missing REC features:\n"
        + "\n".join(
            missing_rec_columns
        )
    )


predictor_columns = [
    column
    for column in dataset_columns
    if column not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


# --------------------------------------------------------------------------------------------------
# 7. LOAD FROZEN CHRONOLOGY AND CORE TABLES
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


required_chronology_columns = {
    "ChronologyOrder",
    "BuildID",
    "StartedAtUTC",
    "Partition",
    "PartitionOrder",
}


missing_chronology_columns = sorted(
    required_chronology_columns
    - set(
        chronology.columns
    )
)


if missing_chronology_columns:
    raise RuntimeError(
        "Frozen chronology is missing columns:\n"
        + "\n".join(
            missing_chronology_columns
        )
    )


chronology[
    "BuildID"
] = parse_integer_ids(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


build_order_map = (
    chronology.set_index(
        "BuildID"
    )[
        "ChronologyOrder"
    ]
    .astype(int)
    .to_dict()
)


training_build_ids = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ]
    .astype(int)
    .tolist()
)


evaluation_build_ids = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ]
    .astype(int)
    .tolist()
)


all_build_ids = (
    training_build_ids
    | evaluation_build_ids
)


builds = pd.read_csv(
    builds_path,
    usecols=[
        build_id_column,
        build_commit_column,
        build_timestamp_column,
    ],
    low_memory=False,
)


builds[
    build_id_column
] = parse_integer_ids(
    builds[
        build_id_column
    ],
    "builds.csv.id",
)


builds[
    build_timestamp_column
] = pd.to_datetime(
    builds[
        build_timestamp_column
    ],
    errors="coerce",
    utc=True,
)


if builds[
    build_timestamp_column
].isna().any():
    raise RuntimeError(
        "builds.csv contains invalid timestamps."
    )


exe = pd.read_csv(
    exe_path,
    usecols=[
        exe_test_column,
        exe_build_column,
        exe_job_column,
        exe_verdict_column,
        exe_duration_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_integer_ids(
    exe[
        exe_build_column
    ],
    "exe.csv.build",
)

exe[
    exe_test_column
] = parse_integer_ids(
    exe[
        exe_test_column
    ],
    "exe.csv.test",
)


exe[
    exe_verdict_column
] = pd.to_numeric(
    exe[
        exe_verdict_column
    ],
    errors="coerce",
)


if exe[
    exe_verdict_column
].isna().any():
    raise RuntimeError(
        "exe.csv contains missing or non-numeric verdicts."
    )


exe[
    exe_verdict_column
] = exe[
    exe_verdict_column
].astype(int)


exe[
    exe_duration_column
] = pd.to_numeric(
    exe[
        exe_duration_column
    ],
    errors="coerce",
)


dataset = pd.read_csv(
    dataset_path,
    low_memory=False,
)


dataset[
    dataset_build_column
] = parse_integer_ids(
    dataset[
        dataset_build_column
    ],
    "dataset.csv.Build",
)

dataset[
    dataset_test_column
] = parse_integer_ids(
    dataset[
        dataset_test_column
    ],
    "dataset.csv.Test",
)

dataset[
    dataset_verdict_column
] = pd.to_numeric(
    dataset[
        dataset_verdict_column
    ],
    errors="coerce",
)


if dataset[
    dataset_verdict_column
].isna().any():
    raise RuntimeError(
        "dataset.csv contains missing or non-numeric verdicts."
    )


dataset[
    dataset_verdict_column
] = dataset[
    dataset_verdict_column
].astype(int)


# --------------------------------------------------------------------------------------------------
# 8. BUILD-TEST JOIN AND CLEAN-VERDICT VALIDATION
# --------------------------------------------------------------------------------------------------

raw_unlinked_build_rows = int(
    (
        ~exe[
            exe_build_column
        ].isin(
            all_build_ids
        )
    ).sum()
)


model_unlinked_build_rows = int(
    (
        ~dataset[
            dataset_build_column
        ].isin(
            all_build_ids
        )
    ).sum()
)


raw_duplicate_build_test_rows = int(
    exe.duplicated(
        subset=[
            exe_build_column,
            exe_test_column,
        ],
        keep=False,
    ).sum()
)


model_duplicate_build_test_rows = int(
    dataset.duplicated(
        subset=[
            dataset_build_column,
            dataset_test_column,
        ],
        keep=False,
    ).sum()
)


nonfinite_duration_rows = int(
    (
        ~np.isfinite(
            exe[
                exe_duration_column
            ].to_numpy(
                dtype=float
            )
        )
    ).sum()
)


negative_duration_rows = int(
    exe[
        exe_duration_column
    ].lt(0).sum()
)


if raw_duplicate_build_test_rows != 0:
    raise RuntimeError(
        "exe.csv contains duplicate Build-Test pairs. "
        "The REC reconstruction requires one raw execution "
        "per Build-Test pair."
    )


if model_duplicate_build_test_rows != 0:
    raise RuntimeError(
        "dataset.csv contains duplicate Build-Test pairs."
    )


raw_pair_frame = exe[
    [
        exe_build_column,
        exe_test_column,
        exe_verdict_column,
        exe_duration_column,
    ]
].rename(
    columns={
        exe_build_column:
            "Build",

        exe_test_column:
            "Test",

        exe_verdict_column:
            "RawVerdict",

        exe_duration_column:
            "RawDuration",
    }
)


model_pair_frame = dataset[
    [
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    ]
].rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "ModelVerdict",
    }
)


model_raw_join = model_pair_frame.merge(
    raw_pair_frame,
    on=[
        "Build",
        "Test",
    ],
    how="left",
    validate="one_to_one",
    indicator=True,
)


missing_model_raw_links = int(
    model_raw_join[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


model_raw_verdict_mismatches = int(
    (
        model_raw_join[
            "ModelVerdict"
        ]
        != model_raw_join[
            "RawVerdict"
        ]
    ).sum()
)


model_training_mask = model_raw_join[
    "Build"
].isin(
    training_build_ids
)

model_evaluation_mask = model_raw_join[
    "Build"
].isin(
    evaluation_build_ids
)


model_training_rows = int(
    model_training_mask.sum()
)

model_evaluation_rows = int(
    model_evaluation_mask.sum()
)

model_training_failures = int(
    (
        model_training_mask
        & model_raw_join[
            "ModelVerdict"
        ].ne(0)
    ).sum()
)

model_evaluation_failures = int(
    (
        model_evaluation_mask
        & model_raw_join[
            "ModelVerdict"
        ].ne(0)
    ).sum()
)


build_test_join_audit = pd.DataFrame([
    {
        "Metric":
            "RawRows",

        "Expected":
            EXPECTED_RAW_ROWS,

        "Actual":
            len(
                exe
            ),

        "Pass":
            len(
                exe
            ) == EXPECTED_RAW_ROWS,
    },

    {
        "Metric":
            "ModelRows",

        "Expected":
            EXPECTED_MODEL_ROWS,

        "Actual":
            len(
                dataset
            ),

        "Pass":
            len(
                dataset
            ) == EXPECTED_MODEL_ROWS,
    },

    {
        "Metric":
            "RawDuplicateBuildTestRows",

        "Expected":
            0,

        "Actual":
            raw_duplicate_build_test_rows,

        "Pass":
            raw_duplicate_build_test_rows == 0,
    },

    {
        "Metric":
            "ModelDuplicateBuildTestRows",

        "Expected":
            0,

        "Actual":
            model_duplicate_build_test_rows,

        "Pass":
            model_duplicate_build_test_rows == 0,
    },

    {
        "Metric":
            "MissingModelRawLinks",

        "Expected":
            0,

        "Actual":
            missing_model_raw_links,

        "Pass":
            missing_model_raw_links == 0,
    },

    {
        "Metric":
            "ModelRawVerdictMismatches",

        "Expected":
            0,

        "Actual":
            model_raw_verdict_mismatches,

        "Pass":
            model_raw_verdict_mismatches == 0,
    },

    {
        "Metric":
            "NonFiniteDurationRows",

        "Expected":
            0,

        "Actual":
            nonfinite_duration_rows,

        "Pass":
            nonfinite_duration_rows == 0,
    },

    {
        "Metric":
            "NegativeDurationRows",

        "Expected":
            0,

        "Actual":
            negative_duration_rows,

        "Pass":
            negative_duration_rows == 0,
    },

    {
        "Metric":
            "RawUnlinkedBuildRows",

        "Expected":
            0,

        "Actual":
            raw_unlinked_build_rows,

        "Pass":
            raw_unlinked_build_rows == 0,
    },

    {
        "Metric":
            "ModelUnlinkedBuildRows",

        "Expected":
            0,

        "Actual":
            model_unlinked_build_rows,

        "Pass":
            model_unlinked_build_rows == 0,
    },

    {
        "Metric":
            "ModelTrainingRows",

        "Expected":
            EXPECTED_MODEL_TRAINING_ROWS,

        "Actual":
            model_training_rows,

        "Pass":
            model_training_rows
            == EXPECTED_MODEL_TRAINING_ROWS,
    },

    {
        "Metric":
            "ModelEvaluationRows",

        "Expected":
            EXPECTED_MODEL_EVALUATION_ROWS,

        "Actual":
            model_evaluation_rows,

        "Pass":
            model_evaluation_rows
            == EXPECTED_MODEL_EVALUATION_ROWS,
    },

    {
        "Metric":
            "ModelTrainingFailures",

        "Expected":
            EXPECTED_MODEL_TRAINING_FAILURES,

        "Actual":
            model_training_failures,

        "Pass":
            model_training_failures
            == EXPECTED_MODEL_TRAINING_FAILURES,
    },

    {
        "Metric":
            "ModelEvaluationFailures",

        "Expected":
            EXPECTED_MODEL_EVALUATION_FAILURES,

        "Actual":
            model_evaluation_failures,

        "Pass":
            model_evaluation_failures
            == EXPECTED_MODEL_EVALUATION_FAILURES,
    },
])


# --------------------------------------------------------------------------------------------------
# 9. REC FEATURE CLASSIFICATION
# --------------------------------------------------------------------------------------------------

rec_feature_classification_records = []


for feature in REC_FEATURE_COLUMNS:
    rec_feature_classification_records.append({
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature
                in VERDICT_DEPENDENT_REC_FEATURES
                else "VERDICT_INDEPENDENT"
            ),

        "FileHistoryFeature":
            feature
            in FILE_HISTORY_REC_FEATURES,

        "PresentInDataset":
            feature
            in dataset.columns,
    })


rec_feature_classification = pd.DataFrame(
    rec_feature_classification_records
)


# --------------------------------------------------------------------------------------------------
# 10. PREPARE BUILD COMMIT TOKENS
# --------------------------------------------------------------------------------------------------

build_commit_token_records = []

build_rows_without_commit_tokens = 0


for row in builds.itertuples(
    index=False
):
    build_id = int(
        getattr(
            row,
            build_id_column
        )
    )

    raw_commits = getattr(
        row,
        build_commit_column
    )

    commit_tokens = extract_commit_tokens(
        raw_commits
    )


    if not commit_tokens:
        build_rows_without_commit_tokens += 1


    for token_order, commit_token in enumerate(
        commit_tokens,
        start=1,
    ):
        build_commit_token_records.append({
            "BuildID":
                build_id,

            "ChronologyOrder":
                int(
                    build_order_map[
                        build_id
                    ]
                ),

            "RawCommits":
                str(
                    raw_commits
                ),

            "TokenOrder":
                token_order,

            "CommitToken":
                commit_token,
        })


build_commit_token_profile = pd.DataFrame(
    build_commit_token_records
)


if build_commit_token_profile.empty:
    raise RuntimeError(
        "No Project 11 build commit tokens were extracted."
    )


build_commit_token_profile = (
    build_commit_token_profile.sort_values(
        [
            "ChronologyOrder",
            "TokenOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 11. LOAD ENTITY HISTORY AND ID MAP
# --------------------------------------------------------------------------------------------------

entity_history = pd.read_csv(
    entity_history_path,
    usecols=[
        entity_id_column,
        entity_commit_column,
    ],
    low_memory=False,
)


entity_history[
    entity_id_column
] = parse_integer_ids(
    entity_history[
        entity_id_column
    ],
    "entity_change_history.csv.EntityId",
)


entity_history[
    "NormalisedCommit"
] = entity_history[
    entity_commit_column
].map(
    normalise_commit
)


empty_entity_commit_rows = int(
    entity_history[
        "NormalisedCommit"
    ].eq("").sum()
)


entity_history_nonempty = (
    entity_history[
        entity_history[
            "NormalisedCommit"
        ].ne("")
    ][
        [
            entity_id_column,
            "NormalisedCommit",
        ]
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


history_commit_values = sorted(
    entity_history_nonempty[
        "NormalisedCommit"
    ].unique().tolist()
)

history_commit_set = set(
    history_commit_values
)


id_map = pd.read_csv(
    id_map_path,
    usecols=[
        id_map_key_column,
        id_map_value_column,
    ],
    low_memory=False,
)


id_map[
    id_map_key_column
] = parse_integer_ids(
    id_map[
        id_map_key_column
    ],
    "id_map.csv.key",
)


duplicate_id_map_keys = int(
    id_map[
        id_map_key_column
    ].duplicated(
        keep=False
    ).sum()
)


if duplicate_id_map_keys != 0:
    raise RuntimeError(
        "id_map.csv contains duplicate entity identifiers."
    )


history_entity_ids = set(
    entity_history_nonempty[
        entity_id_column
    ].astype(int).tolist()
)

id_map_entity_ids = set(
    id_map[
        id_map_key_column
    ].astype(int).tolist()
)


entity_ids_missing_from_id_map = sorted(
    history_entity_ids
    - id_map_entity_ids
)


entity_id_map_audit = (
    entity_history_nonempty[
        [
            entity_id_column
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            entity_id_column:
                "EntityId",
        }
    )
)


entity_id_map_audit[
    "PresentInIDMap"
] = entity_id_map_audit[
    "EntityId"
].isin(
    id_map_entity_ids
)


entity_id_map_audit = (
    entity_id_map_audit.merge(
        id_map.rename(
            columns={
                id_map_key_column:
                    "EntityId",

                id_map_value_column:
                    "EntityValue",
            }
        ),
        on="EntityId",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        "EntityId",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 12. MATCH BUILD TOKENS TO ENTITY-HISTORY COMMITS
# --------------------------------------------------------------------------------------------------

matching_started = time.perf_counter()

commit_matching_records = []


for row in build_commit_token_profile.itertuples(
    index=False
):
    token = str(
        row.CommitToken
    ).lower()

    match_type = None
    matched_commit = None
    candidate_count = 0


    if token in history_commit_set:
        match_type = (
            "EXACT_NORMALISED_COMMIT_TOKEN"
        )

        matched_commit = token
        candidate_count = 1

    else:
        prefix_candidates = [
            commit
            for commit in history_commit_values
            if (
                commit.startswith(
                    token
                )
                or token.startswith(
                    commit
                )
            )
        ]


        if len(prefix_candidates) == 1:
            match_type = (
                "UNIQUE_PREFIX_COMMIT_TOKEN"
            )

            matched_commit = prefix_candidates[0]
            candidate_count = 1

        elif len(prefix_candidates) == 0:
            match_type = (
                "UNMATCHED_COMMIT_TOKEN"
            )

            candidate_count = 0

        else:
            match_type = (
                "AMBIGUOUS_PREFIX_COMMIT_TOKEN"
            )

            candidate_count = len(
                prefix_candidates
            )


    commit_matching_records.append({
        "BuildID":
            int(
                row.BuildID
            ),

        "ChronologyOrder":
            int(
                row.ChronologyOrder
            ),

        "TokenOrder":
            int(
                row.TokenOrder
            ),

        "CommitToken":
            token,

        "MatchType":
            match_type,

        "MatchedCommit":
            matched_commit,

        "CandidateMatches":
            candidate_count,
    })


commit_matching_seconds = float(
    time.perf_counter()
    - matching_started
)


commit_matching_audit = (
    pd.DataFrame(
        commit_matching_records
    )
    .sort_values(
        [
            "ChronologyOrder",
            "TokenOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


exact_commit_matches = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "EXACT_NORMALISED_COMMIT_TOKEN"
    ).sum()
)

unique_prefix_matches = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "UNIQUE_PREFIX_COMMIT_TOKEN"
    ).sum()
)

unmatched_commit_tokens = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "UNMATCHED_COMMIT_TOKEN"
    ).sum()
)

ambiguous_commit_tokens = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "AMBIGUOUS_PREFIX_COMMIT_TOKEN"
    ).sum()
)


matched_commit_tokens = int(
    exact_commit_matches
    + unique_prefix_matches
)

total_commit_tokens = len(
    commit_matching_audit
)


commit_token_coverage_percent = (
    100.0
    * matched_commit_tokens
    / total_commit_tokens
)


matched_build_commit_frame = (
    commit_matching_audit[
        commit_matching_audit[
            "MatchedCommit"
        ].notna()
    ][
        [
            "BuildID",
            "ChronologyOrder",
            "MatchedCommit",
        ]
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 13. CONSTRUCT BUILD-ENTITY MAP
# --------------------------------------------------------------------------------------------------

entity_history_for_join = (
    entity_history_nonempty.rename(
        columns={
            entity_id_column:
                "EntityId",

            "NormalisedCommit":
                "MatchedCommit",
        }
    )
)


build_entity_map = (
    matched_build_commit_frame.merge(
        entity_history_for_join,
        on="MatchedCommit",
        how="left",
        validate="many_to_many",
    )
    .dropna(
        subset=[
            "EntityId",
        ]
    )
)


build_entity_map[
    "EntityId"
] = build_entity_map[
    "EntityId"
].astype(
    "int64"
)


build_entity_map = (
    build_entity_map[
        [
            "BuildID",
            "ChronologyOrder",
            "MatchedCommit",
            "EntityId",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "ChronologyOrder",
            "EntityId",
            "MatchedCommit",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


builds_with_matched_commits = set(
    matched_build_commit_frame[
        "BuildID"
    ].astype(int).tolist()
)

builds_with_mapped_entities = set(
    build_entity_map[
        "BuildID"
    ].astype(int).tolist()
)


builds_with_no_matched_commit = sorted(
    all_build_ids
    - builds_with_matched_commits
)

builds_with_no_mapped_entity = sorted(
    all_build_ids
    - builds_with_mapped_entities
)


build_entity_rows = len(
    build_entity_map
)

unique_mapped_entities = int(
    build_entity_map[
        "EntityId"
    ].nunique()
)

unique_matched_commits = int(
    build_entity_map[
        "MatchedCommit"
    ].nunique()
)


# --------------------------------------------------------------------------------------------------
# 14. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 1B status",
    EXPECTED_STEP1B_STATUS,
    step1b_status.get(
        "Status"
    ),
    step1b_status.get(
        "Status"
    ) == EXPECTED_STEP1B_STATUS,
)

add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_CHECKPOINT_SHA256,
    selection_checkpoint_sha256,
    selection_checkpoint_sha256
    == EXPECTED_SELECTION_CHECKPOINT_SHA256,
)

add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    validation_records,
    "Frozen source files",
    EXPECTED_SOURCE_FILES,
    current_source_file_count,
    current_source_file_count
    == EXPECTED_SOURCE_FILES,
)

add_check(
    validation_records,
    "Frozen source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    validation_records,
    "Canonical builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    ) == EXPECTED_BUILDS,
)

add_check(
    validation_records,
    "Training builds",
    EXPECTED_TRAINING_BUILDS,
    len(
        training_build_ids
    ),
    len(
        training_build_ids
    ) == EXPECTED_TRAINING_BUILDS,
)

add_check(
    validation_records,
    "Evaluation builds",
    EXPECTED_EVALUATION_BUILDS,
    len(
        evaluation_build_ids
    ),
    len(
        evaluation_build_ids
    ) == EXPECTED_EVALUATION_BUILDS,
)

add_check(
    validation_records,
    "Raw execution rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    ) == EXPECTED_RAW_ROWS,
)

add_check(
    validation_records,
    "Model-ready rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    ) == EXPECTED_MODEL_ROWS,
)

add_check(
    validation_records,
    "dataset.csv columns",
    EXPECTED_DATASET_COLUMNS,
    len(
        dataset_columns
    ),
    len(
        dataset_columns
    ) == EXPECTED_DATASET_COLUMNS,
)

add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTOR_COLUMNS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    ) == EXPECTED_PREDICTOR_COLUMNS,
)

add_check(
    validation_records,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURE_COLUMNS
    ),
    (
        len(
            REC_FEATURE_COLUMNS
        ) == EXPECTED_REC_FEATURES
        and len(
            missing_rec_columns
        ) == 0
    ),
)

add_check(
    validation_records,
    "Verdict-dependent REC features",
    13,
    len(
        VERDICT_DEPENDENT_REC_FEATURES
    ),
    len(
        VERDICT_DEPENDENT_REC_FEATURES
    ) == 13,
)

add_check(
    validation_records,
    "Verdict-independent REC features",
    6,
    len(
        VERDICT_INDEPENDENT_REC_FEATURES
    ),
    len(
        VERDICT_INDEPENDENT_REC_FEATURES
    ) == 6,
)

add_check(
    validation_records,
    "Raw duplicate Build-Test rows",
    0,
    raw_duplicate_build_test_rows,
    raw_duplicate_build_test_rows == 0,
)

add_check(
    validation_records,
    "Model duplicate Build-Test rows",
    0,
    model_duplicate_build_test_rows,
    model_duplicate_build_test_rows == 0,
)

add_check(
    validation_records,
    "Missing model-to-raw links",
    0,
    missing_model_raw_links,
    missing_model_raw_links == 0,
)

add_check(
    validation_records,
    "Model/raw verdict mismatches",
    0,
    model_raw_verdict_mismatches,
    model_raw_verdict_mismatches == 0,
)

add_check(
    validation_records,
    "Raw unlinked build rows",
    0,
    raw_unlinked_build_rows,
    raw_unlinked_build_rows == 0,
)

add_check(
    validation_records,
    "Model unlinked build rows",
    0,
    model_unlinked_build_rows,
    model_unlinked_build_rows == 0,
)

add_check(
    validation_records,
    "Non-finite duration rows",
    0,
    nonfinite_duration_rows,
    nonfinite_duration_rows == 0,
)

add_check(
    validation_records,
    "Negative duration rows",
    0,
    negative_duration_rows,
    negative_duration_rows == 0,
)

add_check(
    validation_records,
    "Build rows without commit tokens",
    0,
    build_rows_without_commit_tokens,
    build_rows_without_commit_tokens == 0,
)

add_check(
    validation_records,
    "Unmatched commit tokens",
    0,
    unmatched_commit_tokens,
    unmatched_commit_tokens == 0,
)

add_check(
    validation_records,
    "Ambiguous commit tokens",
    0,
    ambiguous_commit_tokens,
    ambiguous_commit_tokens == 0,
)

add_check(
    validation_records,
    "Commit-token coverage percent",
    100.0,
    commit_token_coverage_percent,
    np.isclose(
        commit_token_coverage_percent,
        100.0,
        rtol=0,
        atol=1e-12,
    ),
)

add_check(
    validation_records,
    "Builds with no matched commit",
    0,
    len(
        builds_with_no_matched_commit
    ),
    len(
        builds_with_no_matched_commit
    ) == 0,
)

add_check(
    validation_records,
    "Builds with no mapped entity",
    0,
    len(
        builds_with_no_mapped_entity
    ),
    len(
        builds_with_no_mapped_entity
    ) == 0,
)

add_check(
    validation_records,
    "Builds with mapped entities",
    EXPECTED_BUILDS,
    len(
        builds_with_mapped_entities
    ),
    len(
        builds_with_mapped_entities
    ) == EXPECTED_BUILDS,
)

add_check(
    validation_records,
    "Entity IDs missing from id_map",
    0,
    len(
        entity_ids_missing_from_id_map
    ),
    len(
        entity_ids_missing_from_id_map
    ) == 0,
)

add_check(
    validation_records,
    "Duplicate id_map keys",
    0,
    duplicate_id_map_keys,
    duplicate_id_map_keys == 0,
)

add_check(
    validation_records,
    "Build-entity rows generated",
    "> 0",
    build_entity_rows,
    build_entity_rows > 0,
)

add_check(
    validation_records,
    "Completion registry rows",
    10,
    len(
        registry
    ),
    len(
        registry
    ) == 10,
)

add_check(
    validation_records,
    "Project 11 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation[
    ~validation[
        "Pass"
    ]
]


print("\nProject 11 Step 2A validation:")

display(
    validation
)


if not failed_validation.empty:
    print("\nFailed validation checks:")

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 11 STEP 2A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 15. WRITE FROZEN PREFLIGHT OUTPUTS
# --------------------------------------------------------------------------------------------------

REC_PREFLIGHT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    SOURCE_SCHEMA_PROFILE_PATH,
    source_schema_profile,
)

atomic_write_csv(
    BUILD_TEST_JOIN_AUDIT_PATH,
    build_test_join_audit,
)

atomic_write_csv(
    REC_FEATURE_CLASSIFICATION_PATH,
    rec_feature_classification,
)

atomic_write_csv(
    BUILD_COMMIT_TOKEN_PROFILE_PATH,
    build_commit_token_profile,
)

atomic_write_csv(
    COMMIT_MATCHING_AUDIT_PATH,
    commit_matching_audit,
)

atomic_write_csv_gzip(
    BUILD_ENTITY_MAP_PATH,
    build_entity_map,
)

atomic_write_csv(
    ENTITY_ID_MAP_AUDIT_PATH,
    entity_id_map_audit,
)

atomic_write_csv(
    STEP2A_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 16. READBACK VALIDATION
# --------------------------------------------------------------------------------------------------

build_entity_map_readback = pd.read_csv(
    BUILD_ENTITY_MAP_PATH,
    compression="gzip",
    low_memory=False,
)

commit_audit_readback = pd.read_csv(
    COMMIT_MATCHING_AUDIT_PATH,
    low_memory=False,
)


if len(
    build_entity_map_readback
) != build_entity_rows:
    raise RuntimeError(
        "Build-entity map readback row count differs."
    )


if len(
    commit_audit_readback
) != total_commit_tokens:
    raise RuntimeError(
        "Commit-matching audit readback row count differs."
    )


# --------------------------------------------------------------------------------------------------
# 17. WRITE SUMMARY, REPORT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


mapping_summary_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "BuildCommitColumn":
        build_commit_column,

    "EntityHistoryCommitColumn":
        entity_commit_column,

    "EntityHistoryIDColumn":
        entity_id_column,

    "IDMapKeyColumn":
        id_map_key_column,

    "IDMapValueColumn":
        id_map_value_column,

    "Builds":
        len(
            builds
        ),

    "BuildCommitTokenRows":
        total_commit_tokens,

    "ExactCommitMatches":
        exact_commit_matches,

    "UniquePrefixMatches":
        unique_prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_commit_tokens,

    "AmbiguousCommitTokens":
        ambiguous_commit_tokens,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "BuildRowsWithoutCommitTokens":
        build_rows_without_commit_tokens,

    "BuildsWithNoMatchedCommit":
        len(
            builds_with_no_matched_commit
        ),

    "BuildsWithNoMappedEntity":
        len(
            builds_with_no_mapped_entity
        ),

    "BuildsWithMappedEntities":
        len(
            builds_with_mapped_entities
        ),

    "BuildEntityRows":
        build_entity_rows,

    "UniqueMappedEntities":
        unique_mapped_entities,

    "UniqueMatchedCommits":
        unique_matched_commits,

    "EntityHistoryRows":
        len(
            entity_history
        ),

    "EntityHistoryNonEmptyCommitRows":
        len(
            entity_history_nonempty
        ),

    "EmptyEntityCommitRows":
        empty_entity_commit_rows,

    "EntityIDsMissingFromIDMap":
        len(
            entity_ids_missing_from_id_map
        ),

    "CommitMatchingSeconds":
        commit_matching_seconds,

    "BuildCommitTokenProfile":
        str(
            BUILD_COMMIT_TOKEN_PROFILE_PATH
        ),

    "CommitMatchingAudit":
        str(
            COMMIT_MATCHING_AUDIT_PATH
        ),

    "BuildEntityMap":
        str(
            BUILD_ENTITY_MAP_PATH
        ),

    "EntityIDMapAudit":
        str(
            ENTITY_ID_MAP_AUDIT_PATH
        ),
}


atomic_write_json(
    ENTITY_MAPPING_SUMMARY_PATH,
    mapping_summary_payload,
)


report_payload = {
    **mapping_summary_payload,

    "SourceRootSHA256":
        current_source_root_sha256,

    "SelectionCheckpoint":
        str(
            SELECTION_CHECKPOINT_PATH
        ),

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "CanonicalBuilds":
        len(
            chronology
        ),

    "TrainingBuilds":
        len(
            training_build_ids
        ),

    "EvaluationBuilds":
        len(
            evaluation_build_ids
        ),

    "RawExecutionRows":
        len(
            exe
        ),

    "ModelReadyRows":
        len(
            dataset
        ),

    "DatasetColumns":
        len(
            dataset_columns
        ),

    "PredictorColumns":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURE_COLUMNS
        ),

    "VerdictDependentRECFeatures":
        len(
            VERDICT_DEPENDENT_REC_FEATURES
        ),

    "VerdictIndependentRECFeatures":
        len(
            VERDICT_INDEPENDENT_REC_FEATURES
        ),

    "RawDuplicateBuildTestRows":
        raw_duplicate_build_test_rows,

    "ModelDuplicateBuildTestRows":
        model_duplicate_build_test_rows,

    "MissingModelRawLinks":
        missing_model_raw_links,

    "ModelRawVerdictMismatches":
        model_raw_verdict_mismatches,

    "NonFiniteDurationRows":
        nonfinite_duration_rows,

    "NegativeDurationRows":
        negative_duration_rows,

    "SourceSchemaProfile":
        str(
            SOURCE_SCHEMA_PROFILE_PATH
        ),

    "BuildTestJoinAudit":
        str(
            BUILD_TEST_JOIN_AUDIT_PATH
        ),

    "RECFeatureClassification":
        str(
            REC_FEATURE_CLASSIFICATION_PATH
        ),

    "Validation":
        str(
            STEP2A_VALIDATION_PATH
        ),

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To10Modified":
        False,

    "Project12Accessed":
        False,

    "Project12WriteAttempted":
        False,

    "NoiseInjected":
        False,

    "ModelsTrained":
        False,
}


atomic_write_json(
    STEP2A_REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "Builds":
        len(
            chronology
        ),

    "RawExecutionRows":
        len(
            exe
        ),

    "ModelReadyRows":
        len(
            dataset
        ),

    "RECFeatures":
        len(
            REC_FEATURE_COLUMNS
        ),

    "BuildCommitTokenRows":
        total_commit_tokens,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "BuildsWithMappedEntities":
        len(
            builds_with_mapped_entities
        ),

    "BuildEntityRows":
        build_entity_rows,

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistryModified":
        False,

    "Project12Accessed":
        False,
}


atomic_write_json(
    STEP2A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 18. FINAL IMMUTABILITY CHECK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 11 Step 2A."
    )


final_source_records = []


for row in current_source_manifest.itertuples(
    index=False
):
    file_path = (
        SOURCE_DIRECTORY
        / row.RelativePath
    )

    final_source_records.append({
        "RelativePath":
            row.RelativePath,

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                file_path
            ),
    })


final_source_root_sha256 = canonical_root_hash(
    pd.DataFrame(
        final_source_records
    )
)


if (
    final_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The Project 11 source root changed during Step 2A."
    )


# --------------------------------------------------------------------------------------------------
# 19. DISPLAY
# --------------------------------------------------------------------------------------------------

print("\nResolved source schemas:")

display(
    source_schema_profile
)


print("\nBuild-Test join audit:")

display(
    build_test_join_audit
)


print("\nREC feature classification:")

display(
    rec_feature_classification
)


print("\nCommit-matching summary:")

display(
    commit_matching_audit[
        "MatchType"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "MatchType"
    )
    .reset_index(
        name="Rows"
    )
)


print("\nCommit-matching sample:")

display(
    pd.concat(
        [
            commit_matching_audit.head(10),
            commit_matching_audit.tail(10),
        ],
        ignore_index=True,
    )
)


print("\nBuild-entity map sample:")

display(
    pd.concat(
        [
            build_entity_map.head(10),
            build_entity_map.tail(10),
        ],
        ignore_index=True,
    )
)


print("\nEntity-ID map coverage:")

display(
    entity_id_map_audit[
        "PresentInIDMap"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "PresentInIDMap"
    )
    .reset_index(
        name="Entities"
    )
)


# --------------------------------------------------------------------------------------------------
# 20. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 132)
print("=== PROJECT 11 CELL 4 / STEP 2A RESULT ===")
print("=" * 132)


print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nFrozen source:")

print(
    "Source files:",
    current_source_file_count,
)

print(
    "Source bytes:",
    current_source_bytes,
)

print(
    "Source root SHA-256:",
    final_source_root_sha256,
)


print("\nCanonical data:")

print(
    "Builds:",
    len(
        chronology
    ),
)

print(
    "Training / evaluation builds:",
    len(
        training_build_ids
    ),
    "/",
    len(
        evaluation_build_ids
    ),
)

print(
    "Raw execution rows:",
    len(
        exe
    ),
)

print(
    "Model-ready rows:",
    len(
        dataset
    ),
)

print(
    "Dataset columns:",
    len(
        dataset_columns
    ),
)

print(
    "Predictor columns:",
    len(
        predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURE_COLUMNS
    ),
)


print("\nBuild-Test joins:")

print(
    "Raw duplicate Build-Test rows:",
    raw_duplicate_build_test_rows,
)

print(
    "Model duplicate Build-Test rows:",
    model_duplicate_build_test_rows,
)

print(
    "Missing model-to-raw links:",
    missing_model_raw_links,
)

print(
    "Model/raw verdict mismatches:",
    model_raw_verdict_mismatches,
)

print(
    "Non-finite duration rows:",
    nonfinite_duration_rows,
)

print(
    "Negative duration rows:",
    negative_duration_rows,
)


print("\nCommit and entity mapping:")

print(
    "Build commit column:",
    build_commit_column,
)

print(
    "Entity-history commit column:",
    entity_commit_column,
)

print(
    "Entity-history ID column:",
    entity_id_column,
)

print(
    "ID-map key / value:",
    id_map_key_column,
    "/",
    id_map_value_column,
)

print(
    "Build commit-token rows:",
    total_commit_tokens,
)

print(
    "Exact commit matches:",
    exact_commit_matches,
)

print(
    "Unique-prefix matches:",
    unique_prefix_matches,
)

print(
    "Unmatched commit tokens:",
    unmatched_commit_tokens,
)

print(
    "Ambiguous commit tokens:",
    ambiguous_commit_tokens,
)

print(
    "Commit-token coverage percent:",
    commit_token_coverage_percent,
)

print(
    "Build rows without commit tokens:",
    build_rows_without_commit_tokens,
)

print(
    "Builds with no matched commit:",
    len(
        builds_with_no_matched_commit
    ),
)

print(
    "Builds with no mapped entity:",
    len(
        builds_with_no_mapped_entity
    ),
)

print(
    "Builds with mapped entities:",
    len(
        builds_with_mapped_entities
    ),
)

print(
    "Build-entity rows:",
    build_entity_rows,
)

print(
    "Unique mapped entities:",
    unique_mapped_entities,
)

print(
    "Entity IDs missing from id_map:",
    len(
        entity_ids_missing_from_id_map
    ),
)


print("\nImmutability and isolation:")

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–10 modified:",
    0,
)

print(
    "Project 12 accessed:",
    False,
)

print(
    "Noise injected:",
    False,
)

print(
    "Models trained:",
    False,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print("\nSaved outputs:")

for output_path in [
    SOURCE_SCHEMA_PROFILE_PATH,
    BUILD_TEST_JOIN_AUDIT_PATH,
    REC_FEATURE_CLASSIFICATION_PATH,
    BUILD_COMMIT_TOKEN_PROFILE_PATH,
    COMMIT_MATCHING_AUDIT_PATH,
    BUILD_ENTITY_MAP_PATH,
    ENTITY_ID_MAP_AUDIT_PATH,
    ENTITY_MAPPING_SUMMARY_PATH,
    STEP2A_VALIDATION_PATH,
    STEP2A_REPORT_PATH,
    STEP2A_STATUS_PATH,
]:
    print(
        output_path
    )


print(
    "\nSTATUS:",
    STEP2A_PASS_STATUS,
)

print("=" * 132)

=== PROJECT 11 CELL 4 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===


RuntimeError: id_map.csv.key contains missing or non-numeric IDs.

In [5]:
# ==================================================================================================
# PROJECT 11 — CELL 4 / STEP 2A V2
# SOURCE SCHEMA, BUILD-TEST JOIN, COMMIT MATCHING,
# ID-MAP ORIENTATION RESOLUTION, AND BUILD-ENTITY VALIDATION
#
# PROJECT:
#   apache@shardingsphere
#
# FIX IN V2:
# - Does not assume id_map.csv.key is numeric.
# - Profiles both key and value columns.
# - Detects which column contains numeric EntityId values by matching
#   against entity_change_history.csv.EntityId.
# - Uses the other column as the entity name/value.
#
# SAFETY:
# - Does not inject noise.
# - Does not train models.
# - Does not create experiment conditions.
# - Does not modify the completion registry.
# - Does not modify Projects 1–10.
# - Does not access or write Project 12.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 11 CELL 4 / STEP 2A V2: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PROJECT CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 11
PROJECT_NAME = "apache@shardingsphere"
PROJECT_SLUG = "apache__shardingsphere"
PROJECT_SHORT_NAME = "shardingsphere"

SOURCE_DIRECTORY = Path(
    "/content/datasets/datasets/apache@shardingsphere"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_11_SELECTION_AND_SOURCE_FROZEN"
)

STEP2A_PASS_STATUS = (
    "PASS_PROJECT_11_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "9d5c95b29357426def26c41b62a753c0"
    "03ad3b19e6086d9873f0961e39875353"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "3e6e0a60d58bff61690d6e2ac2bac183"
    "71414819eff601f3ea347ecb8ad8b0b5"
)

EXPECTED_REGISTRY_SHA256 = (
    "847bdcbee16c8757fde34e9489cd3abb"
    "f0781a319644175472597265b1df2750"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 142_849_042

EXPECTED_BUILDS = 1_049
EXPECTED_TRAINING_BUILDS = 786
EXPECTED_EVALUATION_BUILDS = 263

EXPECTED_RAW_ROWS = 833_541
EXPECTED_MODEL_ROWS = 91_042

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTOR_COLUMNS = 151
EXPECTED_REC_FEATURES = 19

EXPECTED_MODEL_TRAINING_ROWS = 78_035
EXPECTED_MODEL_EVALUATION_ROWS = 13_007
EXPECTED_MODEL_TRAINING_FAILURES = 1_188
EXPECTED_MODEL_EVALUATION_FAILURES = 171


REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC_FEATURES = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC_FEATURES = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

FILE_HISTORY_REC_FEATURES = [
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

REQUIRED_SOURCE_FILES = [
    "builds.csv",
    "contributors.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "exe.csv",
    "id_map.csv",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = THESIS_ROOT / "Notes"
RESULTS_ROOT = THESIS_ROOT / "Results"

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_11_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_11_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_11_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_11_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_11_fixed_chronological_builds.csv"
)

PROJECT_AGGREGATED_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_AGGREGATED_ROOT
    / f"{PROJECT_SHORT_NAME}_rec_preflight"
)

SOURCE_SCHEMA_PROFILE_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_source_schema_profile.csv"
)

BUILD_TEST_JOIN_AUDIT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_build_test_join_audit.csv"
)

REC_FEATURE_CLASSIFICATION_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_rec_feature_classification.csv"
)

BUILD_COMMIT_TOKEN_PROFILE_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_build_commit_token_profile.csv"
)

COMMIT_MATCHING_AUDIT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_commit_matching_audit.csv"
)

BUILD_ENTITY_MAP_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_build_entity_map.csv.gz"
)

ID_MAP_ORIENTATION_AUDIT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_id_map_orientation_audit.csv"
)

ENTITY_ID_MAP_AUDIT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_entity_id_map_audit.csv"
)

ENTITY_MAPPING_SUMMARY_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_entity_mapping_summary.json"
)

STEP2A_VALIDATION_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_step2a_validation.csv"
)

STEP2A_REPORT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT_NAME}_step2a_report.json"
)

STEP2A_STATUS_PATH = (
    PROJECT_AGGREGATED_ROOT
    / f"{PROJECT_SHORT_NAME}_step2a_status.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_json(path, payload):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv(path, dataframe):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_csv_gzip(path, dataframe):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        compression="gzip",
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def canonical_root_hash(manifest):
    required_columns = [
        "RelativePath",
        "SizeBytes",
        "SHA256",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in manifest.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Cannot calculate root hash. Missing columns:\n"
            + "\n".join(missing_columns)
        )

    digest = hashlib.sha256()

    ordered = manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(index=False):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_exact_column(columns, expected, label):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {label}.\n"
            f"Expected: {expected!r}\n"
            f"Matches: {matches}\n"
            f"Columns: {list(columns)}"
        )

    return matches[0]


def parse_integer_ids(values, label):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        invalid_count = int(
            numeric.isna().sum()
        )

        raise RuntimeError(
            f"{label} contains {invalid_count} "
            "missing or non-numeric IDs."
        )

    numeric_array = numeric.to_numpy(
        dtype=float
    )

    integral_mask = np.isclose(
        numeric_array,
        np.floor(numeric_array),
        rtol=0,
        atol=0,
    )

    if not integral_mask.all():
        raise RuntimeError(
            f"{label} contains non-integral IDs."
        )

    return numeric.astype("int64")


def profile_possible_id_column(
    dataframe,
    column,
    reference_ids,
):
    raw = dataframe[column]

    numeric = pd.to_numeric(
        raw,
        errors="coerce",
    )

    numeric_mask = numeric.notna()

    integral_mask = pd.Series(
        False,
        index=dataframe.index,
    )

    if numeric_mask.any():
        numeric_values = numeric.loc[
            numeric_mask
        ].to_numpy(dtype=float)

        integral_values = np.isclose(
            numeric_values,
            np.floor(numeric_values),
            rtol=0,
            atol=0,
        )

        integral_mask.loc[
            numeric.loc[numeric_mask].index
        ] = integral_values

    parsed_ids = set(
        numeric.loc[
            integral_mask
        ].astype("int64").tolist()
    )

    matching_reference_ids = (
        parsed_ids
        & reference_ids
    )

    return {
        "Column":
            column,

        "Rows":
            len(dataframe),

        "NonEmptyRows":
            int(
                raw.astype(str)
                .str.strip()
                .ne("")
                .sum()
            ),

        "NumericRows":
            int(
                numeric_mask.sum()
            ),

        "IntegralNumericRows":
            int(
                integral_mask.sum()
            ),

        "InvalidOrNonNumericRows":
            int(
                (
                    ~integral_mask
                ).sum()
            ),

        "UniqueIntegralIDs":
            len(parsed_ids),

        "ReferenceEntityIDs":
            len(reference_ids),

        "MatchingReferenceEntityIDs":
            len(matching_reference_ids),

        "ReferenceCoveragePercent":
            (
                100.0
                * len(matching_reference_ids)
                / len(reference_ids)
                if reference_ids
                else 0.0
            ),

        "ParsedIDSet":
            parsed_ids,
    }


def normalise_commit(value):
    if pd.isna(value):
        return ""

    value = str(value).strip().lower()

    if not value:
        return ""

    hexadecimal_matches = re.findall(
        r"[0-9a-f]{7,64}",
        value,
        flags=re.IGNORECASE,
    )

    if hexadecimal_matches:
        return hexadecimal_matches[0].lower()

    return re.sub(
        r"[^a-z0-9]",
        "",
        value,
    )


def extract_commit_tokens(value):
    if pd.isna(value):
        return []

    text = str(value).strip()

    if not text:
        return []

    tokens = re.findall(
        r"[0-9a-fA-F]{7,64}",
        text,
    )

    if not tokens:
        tokens = re.split(
            r"[\s,;|]+",
            text,
        )

    ordered_unique = []
    seen = set()

    for token in tokens:
        normalised = normalise_commit(token)

        if (
            normalised
            and normalised not in seen
        ):
            seen.add(normalised)
            ordered_unique.append(normalised)

    return ordered_unique


def add_check(
    records,
    check,
    expected,
    actual,
    passed,
):
    records.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUT VALIDATION
# --------------------------------------------------------------------------------------------------

required_files = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Required Project 11 files are missing:\n"
        + "\n".join(missing_files)
    )


if not SOURCE_DIRECTORY.is_dir():
    raise FileNotFoundError(
        "The frozen Project 11 source directory is missing:\n"
        f"{SOURCE_DIRECTORY}"
    )


missing_source_files = [
    filename
    for filename in REQUIRED_SOURCE_FILES
    if not (
        SOURCE_DIRECTORY
        / filename
    ).is_file()
]

if missing_source_files:
    raise FileNotFoundError(
        "Frozen Project 11 source files are missing:\n"
        + "\n".join(missing_source_files)
    )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE STEP 1B, REGISTRY, AND SOURCE ROOT
# --------------------------------------------------------------------------------------------------

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)


if (
    selection_checkpoint_sha256
    != EXPECTED_SELECTION_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 11 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_CHECKPOINT_SHA256}\n"
        f"Actual:   {selection_checkpoint_sha256}"
    )


if selection_checkpoint.get("Status") != EXPECTED_STEP1B_STATUS:
    raise RuntimeError(
        "Project 11 selection-checkpoint status differs."
    )


if step1b_status.get("Status") != EXPECTED_STEP1B_STATUS:
    raise RuntimeError(
        "Project 11 Step 1B status differs."
    )


if selection_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "The frozen Project 11 identity differs."
    )


if selection_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "The frozen Project 11 slug differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "The completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_exact_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

status_column = resolve_exact_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


if len(registry) != 10:
    raise RuntimeError(
        "Expected ten frozen projects in the registry."
    )


if sorted(
    registry_project_numbers.tolist()
) != list(range(1, 11)):
    raise RuntimeError(
        "The registry does not contain exactly Projects 1–10."
    )


if not registry[
    status_column
].astype(str).eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "One or more Projects 1–10 are not COMPLETE_AND_FROZEN."
    )


if int(
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).sum()
) != 0:
    raise RuntimeError(
        "Project 11 is unexpectedly present in the registry."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_records = []


for row in frozen_source_manifest.itertuples(
    index=False
):
    file_path = (
        SOURCE_DIRECTORY
        / str(row.RelativePath)
    )

    if not file_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 11 source file is missing:\n"
            f"{file_path}"
        )

    current_source_records.append({
        "RelativePath":
            str(row.RelativePath),

        "SizeBytes":
            int(file_path.stat().st_size),

        "SHA256":
            sha256_file(file_path),
    })


current_source_manifest = pd.DataFrame(
    current_source_records
)

current_source_root_sha256 = canonical_root_hash(
    current_source_manifest
)

current_source_file_count = len(
    current_source_manifest
)

current_source_bytes = int(
    current_source_manifest[
        "SizeBytes"
    ].sum()
)


if (
    current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The frozen Project 11 source root differs.\n"
        f"Expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Actual:   {current_source_root_sha256}"
    )


# --------------------------------------------------------------------------------------------------
# 6. RESOLVE SOURCE SCHEMAS
# --------------------------------------------------------------------------------------------------

builds_path = SOURCE_DIRECTORY / "builds.csv"
exe_path = SOURCE_DIRECTORY / "exe.csv"
dataset_path = SOURCE_DIRECTORY / "dataset.csv"

entity_history_path = (
    SOURCE_DIRECTORY
    / "entity_change_history.csv"
)

id_map_path = SOURCE_DIRECTORY / "id_map.csv"

contributors_path = (
    SOURCE_DIRECTORY
    / "contributors.csv"
)


source_paths = {
    "builds.csv":
        builds_path,

    "contributors.csv":
        contributors_path,

    "dataset.csv":
        dataset_path,

    "entity_change_history.csv":
        entity_history_path,

    "exe.csv":
        exe_path,

    "id_map.csv":
        id_map_path,
}


source_schema_records = []


for filename, file_path in source_paths.items():
    columns = pd.read_csv(
        file_path,
        nrows=0,
    ).columns.tolist()

    source_schema_records.append({
        "File":
            filename,

        "Path":
            str(file_path),

        "SizeBytes":
            int(file_path.stat().st_size),

        "ColumnCount":
            len(columns),

        "ColumnsJSON":
            json.dumps(
                columns,
                ensure_ascii=False,
            ),
    })


source_schema_profile = pd.DataFrame(
    source_schema_records
)


build_columns = pd.read_csv(
    builds_path,
    nrows=0,
).columns.tolist()

exe_columns = pd.read_csv(
    exe_path,
    nrows=0,
).columns.tolist()

dataset_columns = pd.read_csv(
    dataset_path,
    nrows=0,
).columns.tolist()

entity_history_columns = pd.read_csv(
    entity_history_path,
    nrows=0,
).columns.tolist()

id_map_columns = pd.read_csv(
    id_map_path,
    nrows=0,
).columns.tolist()


build_id_column = resolve_exact_column(
    build_columns,
    "id",
    "builds.csv build ID",
)

build_commit_column = resolve_exact_column(
    build_columns,
    "commits",
    "builds.csv commit column",
)

build_timestamp_column = resolve_exact_column(
    build_columns,
    "started_at",
    "builds.csv timestamp",
)


exe_test_column = resolve_exact_column(
    exe_columns,
    "test",
    "exe.csv test column",
)

exe_build_column = resolve_exact_column(
    exe_columns,
    "build",
    "exe.csv build column",
)

exe_job_column = resolve_exact_column(
    exe_columns,
    "job",
    "exe.csv job column",
)

exe_verdict_column = resolve_exact_column(
    exe_columns,
    "verdict",
    "exe.csv verdict column",
)

exe_duration_column = resolve_exact_column(
    exe_columns,
    "duration",
    "exe.csv duration column",
)


dataset_build_column = resolve_exact_column(
    dataset_columns,
    "Build",
    "dataset.csv Build column",
)

dataset_test_column = resolve_exact_column(
    dataset_columns,
    "Test",
    "dataset.csv Test column",
)

dataset_verdict_column = resolve_exact_column(
    dataset_columns,
    "Verdict",
    "dataset.csv Verdict column",
)


entity_commit_column = resolve_exact_column(
    entity_history_columns,
    "Commit",
    "entity_change_history.csv Commit column",
)

entity_id_column = resolve_exact_column(
    entity_history_columns,
    "EntityId",
    "entity_change_history.csv EntityId column",
)


id_map_key_column = resolve_exact_column(
    id_map_columns,
    "key",
    "id_map.csv key column",
)

id_map_value_column = resolve_exact_column(
    id_map_columns,
    "value",
    "id_map.csv value column",
)


missing_rec_columns = [
    column
    for column in REC_FEATURE_COLUMNS
    if column not in dataset_columns
]


if missing_rec_columns:
    raise RuntimeError(
        "dataset.csv is missing REC features:\n"
        + "\n".join(missing_rec_columns)
    )


predictor_columns = [
    column
    for column in dataset_columns
    if column not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


# --------------------------------------------------------------------------------------------------
# 7. LOAD FROZEN CHRONOLOGY AND CORE TABLES
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


required_chronology_columns = {
    "ChronologyOrder",
    "BuildID",
    "StartedAtUTC",
    "Partition",
    "PartitionOrder",
}


missing_chronology_columns = sorted(
    required_chronology_columns
    - set(chronology.columns)
)


if missing_chronology_columns:
    raise RuntimeError(
        "Frozen chronology is missing columns:\n"
        + "\n".join(missing_chronology_columns)
    )


chronology[
    "BuildID"
] = parse_integer_ids(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


build_order_map = (
    chronology.set_index(
        "BuildID"
    )[
        "ChronologyOrder"
    ]
    .astype(int)
    .to_dict()
)


training_build_ids = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq("TRAIN"),
        "BuildID",
    ].astype(int).tolist()
)


evaluation_build_ids = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq("EVALUATION"),
        "BuildID",
    ].astype(int).tolist()
)


all_build_ids = (
    training_build_ids
    | evaluation_build_ids
)


builds = pd.read_csv(
    builds_path,
    usecols=[
        build_id_column,
        build_commit_column,
        build_timestamp_column,
    ],
    low_memory=False,
)


builds[
    build_id_column
] = parse_integer_ids(
    builds[
        build_id_column
    ],
    "builds.csv.id",
)


builds[
    build_timestamp_column
] = pd.to_datetime(
    builds[
        build_timestamp_column
    ],
    errors="coerce",
    utc=True,
)


if builds[
    build_timestamp_column
].isna().any():
    raise RuntimeError(
        "builds.csv contains invalid timestamps."
    )


exe = pd.read_csv(
    exe_path,
    usecols=[
        exe_test_column,
        exe_build_column,
        exe_job_column,
        exe_verdict_column,
        exe_duration_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_integer_ids(
    exe[
        exe_build_column
    ],
    "exe.csv.build",
)


exe[
    exe_test_column
] = parse_integer_ids(
    exe[
        exe_test_column
    ],
    "exe.csv.test",
)


exe[
    exe_verdict_column
] = pd.to_numeric(
    exe[
        exe_verdict_column
    ],
    errors="coerce",
)


if exe[
    exe_verdict_column
].isna().any():
    raise RuntimeError(
        "exe.csv contains missing or non-numeric verdicts."
    )


exe[
    exe_verdict_column
] = exe[
    exe_verdict_column
].astype(int)


exe[
    exe_duration_column
] = pd.to_numeric(
    exe[
        exe_duration_column
    ],
    errors="coerce",
)


dataset = pd.read_csv(
    dataset_path,
    low_memory=False,
)


dataset[
    dataset_build_column
] = parse_integer_ids(
    dataset[
        dataset_build_column
    ],
    "dataset.csv.Build",
)


dataset[
    dataset_test_column
] = parse_integer_ids(
    dataset[
        dataset_test_column
    ],
    "dataset.csv.Test",
)


dataset[
    dataset_verdict_column
] = pd.to_numeric(
    dataset[
        dataset_verdict_column
    ],
    errors="coerce",
)


if dataset[
    dataset_verdict_column
].isna().any():
    raise RuntimeError(
        "dataset.csv contains missing or non-numeric verdicts."
    )


dataset[
    dataset_verdict_column
] = dataset[
    dataset_verdict_column
].astype(int)


# --------------------------------------------------------------------------------------------------
# 8. BUILD-TEST JOIN AND CLEAN-VERDICT VALIDATION
# --------------------------------------------------------------------------------------------------

raw_unlinked_build_rows = int(
    (
        ~exe[
            exe_build_column
        ].isin(all_build_ids)
    ).sum()
)


model_unlinked_build_rows = int(
    (
        ~dataset[
            dataset_build_column
        ].isin(all_build_ids)
    ).sum()
)


raw_duplicate_build_test_rows = int(
    exe.duplicated(
        subset=[
            exe_build_column,
            exe_test_column,
        ],
        keep=False,
    ).sum()
)


model_duplicate_build_test_rows = int(
    dataset.duplicated(
        subset=[
            dataset_build_column,
            dataset_test_column,
        ],
        keep=False,
    ).sum()
)


nonfinite_duration_rows = int(
    (
        ~np.isfinite(
            exe[
                exe_duration_column
            ].to_numpy(dtype=float)
        )
    ).sum()
)


negative_duration_rows = int(
    exe[
        exe_duration_column
    ].lt(0).sum()
)


if raw_duplicate_build_test_rows != 0:
    raise RuntimeError(
        "exe.csv contains duplicate Build-Test pairs."
    )


if model_duplicate_build_test_rows != 0:
    raise RuntimeError(
        "dataset.csv contains duplicate Build-Test pairs."
    )


raw_pair_frame = exe[
    [
        exe_build_column,
        exe_test_column,
        exe_verdict_column,
        exe_duration_column,
    ]
].rename(
    columns={
        exe_build_column:
            "Build",

        exe_test_column:
            "Test",

        exe_verdict_column:
            "RawVerdict",

        exe_duration_column:
            "RawDuration",
    }
)


model_pair_frame = dataset[
    [
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    ]
].rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "ModelVerdict",
    }
)


model_raw_join = model_pair_frame.merge(
    raw_pair_frame,
    on=[
        "Build",
        "Test",
    ],
    how="left",
    validate="one_to_one",
    indicator=True,
)


missing_model_raw_links = int(
    model_raw_join[
        "_merge"
    ].ne("both").sum()
)


model_raw_verdict_mismatches = int(
    (
        model_raw_join[
            "ModelVerdict"
        ]
        != model_raw_join[
            "RawVerdict"
        ]
    ).sum()
)


model_training_mask = model_raw_join[
    "Build"
].isin(
    training_build_ids
)


model_evaluation_mask = model_raw_join[
    "Build"
].isin(
    evaluation_build_ids
)


model_training_rows = int(
    model_training_mask.sum()
)

model_evaluation_rows = int(
    model_evaluation_mask.sum()
)

model_training_failures = int(
    (
        model_training_mask
        & model_raw_join[
            "ModelVerdict"
        ].ne(0)
    ).sum()
)

model_evaluation_failures = int(
    (
        model_evaluation_mask
        & model_raw_join[
            "ModelVerdict"
        ].ne(0)
    ).sum()
)


build_test_join_audit = pd.DataFrame([
    {
        "Metric":
            "RawRows",

        "Expected":
            EXPECTED_RAW_ROWS,

        "Actual":
            len(exe),

        "Pass":
            len(exe) == EXPECTED_RAW_ROWS,
    },

    {
        "Metric":
            "ModelRows",

        "Expected":
            EXPECTED_MODEL_ROWS,

        "Actual":
            len(dataset),

        "Pass":
            len(dataset) == EXPECTED_MODEL_ROWS,
    },

    {
        "Metric":
            "RawDuplicateBuildTestRows",

        "Expected":
            0,

        "Actual":
            raw_duplicate_build_test_rows,

        "Pass":
            raw_duplicate_build_test_rows == 0,
    },

    {
        "Metric":
            "ModelDuplicateBuildTestRows",

        "Expected":
            0,

        "Actual":
            model_duplicate_build_test_rows,

        "Pass":
            model_duplicate_build_test_rows == 0,
    },

    {
        "Metric":
            "MissingModelRawLinks",

        "Expected":
            0,

        "Actual":
            missing_model_raw_links,

        "Pass":
            missing_model_raw_links == 0,
    },

    {
        "Metric":
            "ModelRawVerdictMismatches",

        "Expected":
            0,

        "Actual":
            model_raw_verdict_mismatches,

        "Pass":
            model_raw_verdict_mismatches == 0,
    },

    {
        "Metric":
            "NonFiniteDurationRows",

        "Expected":
            0,

        "Actual":
            nonfinite_duration_rows,

        "Pass":
            nonfinite_duration_rows == 0,
    },

    {
        "Metric":
            "NegativeDurationRows",

        "Expected":
            0,

        "Actual":
            negative_duration_rows,

        "Pass":
            negative_duration_rows == 0,
    },

    {
        "Metric":
            "RawUnlinkedBuildRows",

        "Expected":
            0,

        "Actual":
            raw_unlinked_build_rows,

        "Pass":
            raw_unlinked_build_rows == 0,
    },

    {
        "Metric":
            "ModelUnlinkedBuildRows",

        "Expected":
            0,

        "Actual":
            model_unlinked_build_rows,

        "Pass":
            model_unlinked_build_rows == 0,
    },

    {
        "Metric":
            "ModelTrainingRows",

        "Expected":
            EXPECTED_MODEL_TRAINING_ROWS,

        "Actual":
            model_training_rows,

        "Pass":
            model_training_rows
            == EXPECTED_MODEL_TRAINING_ROWS,
    },

    {
        "Metric":
            "ModelEvaluationRows",

        "Expected":
            EXPECTED_MODEL_EVALUATION_ROWS,

        "Actual":
            model_evaluation_rows,

        "Pass":
            model_evaluation_rows
            == EXPECTED_MODEL_EVALUATION_ROWS,
    },

    {
        "Metric":
            "ModelTrainingFailures",

        "Expected":
            EXPECTED_MODEL_TRAINING_FAILURES,

        "Actual":
            model_training_failures,

        "Pass":
            model_training_failures
            == EXPECTED_MODEL_TRAINING_FAILURES,
    },

    {
        "Metric":
            "ModelEvaluationFailures",

        "Expected":
            EXPECTED_MODEL_EVALUATION_FAILURES,

        "Actual":
            model_evaluation_failures,

        "Pass":
            model_evaluation_failures
            == EXPECTED_MODEL_EVALUATION_FAILURES,
    },
])


# --------------------------------------------------------------------------------------------------
# 9. REC FEATURE CLASSIFICATION
# --------------------------------------------------------------------------------------------------

rec_feature_classification_records = []


for feature in REC_FEATURE_COLUMNS:
    rec_feature_classification_records.append({
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature
                in VERDICT_DEPENDENT_REC_FEATURES
                else "VERDICT_INDEPENDENT"
            ),

        "FileHistoryFeature":
            feature
            in FILE_HISTORY_REC_FEATURES,

        "PresentInDataset":
            feature
            in dataset.columns,
    })


rec_feature_classification = pd.DataFrame(
    rec_feature_classification_records
)


# --------------------------------------------------------------------------------------------------
# 10. PREPARE BUILD COMMIT TOKENS
# --------------------------------------------------------------------------------------------------

build_commit_token_records = []
build_rows_without_commit_tokens = 0


for (
    build_id,
    raw_commits,
) in builds[
    [
        build_id_column,
        build_commit_column,
    ]
].itertuples(
    index=False,
    name=None,
):
    build_id = int(build_id)

    commit_tokens = extract_commit_tokens(
        raw_commits
    )

    if not commit_tokens:
        build_rows_without_commit_tokens += 1

    for token_order, commit_token in enumerate(
        commit_tokens,
        start=1,
    ):
        build_commit_token_records.append({
            "BuildID":
                build_id,

            "ChronologyOrder":
                int(
                    build_order_map[
                        build_id
                    ]
                ),

            "RawCommits":
                str(raw_commits),

            "TokenOrder":
                token_order,

            "CommitToken":
                commit_token,
        })


build_commit_token_profile = pd.DataFrame(
    build_commit_token_records
)


if build_commit_token_profile.empty:
    raise RuntimeError(
        "No Project 11 build commit tokens were extracted."
    )


build_commit_token_profile = (
    build_commit_token_profile.sort_values(
        [
            "ChronologyOrder",
            "TokenOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 11. LOAD ENTITY HISTORY
# --------------------------------------------------------------------------------------------------

entity_history = pd.read_csv(
    entity_history_path,
    usecols=[
        entity_id_column,
        entity_commit_column,
    ],
    low_memory=False,
)


entity_history[
    entity_id_column
] = parse_integer_ids(
    entity_history[
        entity_id_column
    ],
    "entity_change_history.csv.EntityId",
)


entity_history[
    "NormalisedCommit"
] = entity_history[
    entity_commit_column
].map(
    normalise_commit
)


empty_entity_commit_rows = int(
    entity_history[
        "NormalisedCommit"
    ].eq("").sum()
)


entity_history_nonempty = (
    entity_history[
        entity_history[
            "NormalisedCommit"
        ].ne("")
    ][
        [
            entity_id_column,
            "NormalisedCommit",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


history_commit_values = sorted(
    entity_history_nonempty[
        "NormalisedCommit"
    ].unique().tolist()
)


history_commit_set = set(
    history_commit_values
)


history_entity_ids = set(
    entity_history_nonempty[
        entity_id_column
    ].astype(int).tolist()
)


# --------------------------------------------------------------------------------------------------
# 12. V2 FIX — RESOLVE ID-MAP ORIENTATION SAFELY
# --------------------------------------------------------------------------------------------------

id_map_raw = pd.read_csv(
    id_map_path,
    usecols=[
        id_map_key_column,
        id_map_value_column,
    ],
    dtype=str,
    keep_default_na=False,
    low_memory=False,
)


id_map_profiles = [
    profile_possible_id_column(
        id_map_raw,
        id_map_key_column,
        history_entity_ids,
    ),

    profile_possible_id_column(
        id_map_raw,
        id_map_value_column,
        history_entity_ids,
    ),
]


id_map_orientation_audit = pd.DataFrame([
    {
        key:
            value
        for key, value in profile.items()
        if key != "ParsedIDSet"
    }
    for profile in id_map_profiles
])


maximum_coverage = max(
    profile[
        "MatchingReferenceEntityIDs"
    ]
    for profile in id_map_profiles
)


best_profiles = [
    profile
    for profile in id_map_profiles
    if profile[
        "MatchingReferenceEntityIDs"
    ] == maximum_coverage
]


if maximum_coverage == 0:
    raise RuntimeError(
        "Neither id_map.csv column matches "
        "entity_change_history.csv.EntityId."
    )


if len(best_profiles) != 1:
    best_integral_rows = max(
        profile[
            "IntegralNumericRows"
        ]
        for profile in best_profiles
    )

    best_profiles = [
        profile
        for profile in best_profiles
        if profile[
            "IntegralNumericRows"
        ] == best_integral_rows
    ]


if len(best_profiles) != 1:
    raise RuntimeError(
        "Could not uniquely resolve the numeric EntityId "
        "column in id_map.csv.\n"
        f"Profiles:\n{id_map_orientation_audit}"
    )


resolved_id_map_entity_id_column = (
    best_profiles[0][
        "Column"
    ]
)


resolved_id_map_entity_value_column = (
    id_map_value_column
    if resolved_id_map_entity_id_column
    == id_map_key_column
    else id_map_key_column
)


resolved_id_map_numeric = pd.to_numeric(
    id_map_raw[
        resolved_id_map_entity_id_column
    ],
    errors="coerce",
)


resolved_id_map_numeric_array = (
    resolved_id_map_numeric.to_numpy(
        dtype=float
    )
)


resolved_id_map_valid_mask = (
    resolved_id_map_numeric.notna()
    & pd.Series(
        np.isclose(
            resolved_id_map_numeric_array,
            np.floor(
                resolved_id_map_numeric_array
            ),
            rtol=0,
            atol=0,
        ),
        index=id_map_raw.index,
    )
)


invalid_resolved_id_map_rows = int(
    (
        ~resolved_id_map_valid_mask
    ).sum()
)


if invalid_resolved_id_map_rows != 0:
    raise RuntimeError(
        "The resolved id_map EntityId column contains "
        f"{invalid_resolved_id_map_rows} invalid rows.\n"
        "Resolved column: "
        f"{resolved_id_map_entity_id_column}"
    )


id_map = pd.DataFrame({
    "EntityId":
        resolved_id_map_numeric.astype(
            "int64"
        ),

    "EntityValue":
        id_map_raw[
            resolved_id_map_entity_value_column
        ].astype(str),
})


duplicate_id_map_ids = int(
    id_map[
        "EntityId"
    ].duplicated(
        keep=False
    ).sum()
)


if duplicate_id_map_ids != 0:
    raise RuntimeError(
        "The resolved id_map EntityId column contains "
        f"{duplicate_id_map_ids} duplicate-ID rows."
    )


id_map_entity_ids = set(
    id_map[
        "EntityId"
    ].astype(int).tolist()
)


entity_ids_missing_from_id_map = sorted(
    history_entity_ids
    - id_map_entity_ids
)


id_map_ids_not_in_history = sorted(
    id_map_entity_ids
    - history_entity_ids
)


entity_id_map_audit = (
    pd.DataFrame({
        "EntityId":
            sorted(history_entity_ids)
    })
    .assign(
        PresentInIDMap=lambda frame:
            frame[
                "EntityId"
            ].isin(
                id_map_entity_ids
            )
    )
    .merge(
        id_map,
        on="EntityId",
        how="left",
        validate="one_to_one",
    )
)


# --------------------------------------------------------------------------------------------------
# 13. MATCH BUILD TOKENS TO ENTITY-HISTORY COMMITS
# --------------------------------------------------------------------------------------------------

matching_started = time.perf_counter()

commit_matching_records = []


for row in build_commit_token_profile.itertuples(
    index=False
):
    token = str(
        row.CommitToken
    ).lower()

    match_type = None
    matched_commit = None
    candidate_count = 0


    if token in history_commit_set:
        match_type = (
            "EXACT_NORMALISED_COMMIT_TOKEN"
        )

        matched_commit = token
        candidate_count = 1

    else:
        prefix_candidates = [
            commit
            for commit in history_commit_values
            if (
                commit.startswith(token)
                or token.startswith(commit)
            )
        ]

        if len(prefix_candidates) == 1:
            match_type = (
                "UNIQUE_PREFIX_COMMIT_TOKEN"
            )

            matched_commit = prefix_candidates[0]
            candidate_count = 1

        elif len(prefix_candidates) == 0:
            match_type = (
                "UNMATCHED_COMMIT_TOKEN"
            )

            candidate_count = 0

        else:
            match_type = (
                "AMBIGUOUS_PREFIX_COMMIT_TOKEN"
            )

            candidate_count = len(
                prefix_candidates
            )


    commit_matching_records.append({
        "BuildID":
            int(row.BuildID),

        "ChronologyOrder":
            int(row.ChronologyOrder),

        "TokenOrder":
            int(row.TokenOrder),

        "CommitToken":
            token,

        "MatchType":
            match_type,

        "MatchedCommit":
            matched_commit,

        "CandidateMatches":
            candidate_count,
    })


commit_matching_seconds = float(
    time.perf_counter()
    - matching_started
)


commit_matching_audit = (
    pd.DataFrame(
        commit_matching_records
    )
    .sort_values(
        [
            "ChronologyOrder",
            "TokenOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


exact_commit_matches = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "EXACT_NORMALISED_COMMIT_TOKEN"
    ).sum()
)


unique_prefix_matches = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "UNIQUE_PREFIX_COMMIT_TOKEN"
    ).sum()
)


unmatched_commit_tokens = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "UNMATCHED_COMMIT_TOKEN"
    ).sum()
)


ambiguous_commit_tokens = int(
    commit_matching_audit[
        "MatchType"
    ].eq(
        "AMBIGUOUS_PREFIX_COMMIT_TOKEN"
    ).sum()
)


matched_commit_tokens = int(
    exact_commit_matches
    + unique_prefix_matches
)


total_commit_tokens = len(
    commit_matching_audit
)


commit_token_coverage_percent = (
    100.0
    * matched_commit_tokens
    / total_commit_tokens
)


matched_build_commit_frame = (
    commit_matching_audit[
        commit_matching_audit[
            "MatchedCommit"
        ].notna()
    ][
        [
            "BuildID",
            "ChronologyOrder",
            "MatchedCommit",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------------------
# 14. CONSTRUCT BUILD-ENTITY MAP
# --------------------------------------------------------------------------------------------------

entity_history_for_join = (
    entity_history_nonempty.rename(
        columns={
            entity_id_column:
                "EntityId",

            "NormalisedCommit":
                "MatchedCommit",
        }
    )
)


build_entity_map = (
    matched_build_commit_frame.merge(
        entity_history_for_join,
        on="MatchedCommit",
        how="left",
        validate="many_to_many",
    )
    .dropna(
        subset=[
            "EntityId",
        ]
    )
)


build_entity_map[
    "EntityId"
] = build_entity_map[
    "EntityId"
].astype("int64")


build_entity_map = (
    build_entity_map[
        [
            "BuildID",
            "ChronologyOrder",
            "MatchedCommit",
            "EntityId",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "ChronologyOrder",
            "EntityId",
            "MatchedCommit",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


builds_with_matched_commits = set(
    matched_build_commit_frame[
        "BuildID"
    ].astype(int).tolist()
)


builds_with_mapped_entities = set(
    build_entity_map[
        "BuildID"
    ].astype(int).tolist()
)


builds_with_no_matched_commit = sorted(
    all_build_ids
    - builds_with_matched_commits
)


builds_with_no_mapped_entity = sorted(
    all_build_ids
    - builds_with_mapped_entities
)


build_entity_rows = len(
    build_entity_map
)


unique_mapped_entities = int(
    build_entity_map[
        "EntityId"
    ].nunique()
)


unique_matched_commits = int(
    build_entity_map[
        "MatchedCommit"
    ].nunique()
)


# --------------------------------------------------------------------------------------------------
# 15. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 1B status",
    EXPECTED_STEP1B_STATUS,
    step1b_status.get("Status"),
    step1b_status.get("Status")
    == EXPECTED_STEP1B_STATUS,
)

add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_CHECKPOINT_SHA256,
    selection_checkpoint_sha256,
    selection_checkpoint_sha256
    == EXPECTED_SELECTION_CHECKPOINT_SHA256,
)

add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    validation_records,
    "Frozen source files",
    EXPECTED_SOURCE_FILES,
    current_source_file_count,
    current_source_file_count
    == EXPECTED_SOURCE_FILES,
)

add_check(
    validation_records,
    "Frozen source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    validation_records,
    "Canonical builds",
    EXPECTED_BUILDS,
    len(chronology),
    len(chronology) == EXPECTED_BUILDS,
)

add_check(
    validation_records,
    "Training builds",
    EXPECTED_TRAINING_BUILDS,
    len(training_build_ids),
    len(training_build_ids)
    == EXPECTED_TRAINING_BUILDS,
)

add_check(
    validation_records,
    "Evaluation builds",
    EXPECTED_EVALUATION_BUILDS,
    len(evaluation_build_ids),
    len(evaluation_build_ids)
    == EXPECTED_EVALUATION_BUILDS,
)

add_check(
    validation_records,
    "Raw execution rows",
    EXPECTED_RAW_ROWS,
    len(exe),
    len(exe) == EXPECTED_RAW_ROWS,
)

add_check(
    validation_records,
    "Model-ready rows",
    EXPECTED_MODEL_ROWS,
    len(dataset),
    len(dataset) == EXPECTED_MODEL_ROWS,
)

add_check(
    validation_records,
    "dataset.csv columns",
    EXPECTED_DATASET_COLUMNS,
    len(dataset_columns),
    len(dataset_columns)
    == EXPECTED_DATASET_COLUMNS,
)

add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTOR_COLUMNS,
    len(predictor_columns),
    len(predictor_columns)
    == EXPECTED_PREDICTOR_COLUMNS,
)

add_check(
    validation_records,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(REC_FEATURE_COLUMNS),
    (
        len(REC_FEATURE_COLUMNS)
        == EXPECTED_REC_FEATURES
        and len(missing_rec_columns) == 0
    ),
)

add_check(
    validation_records,
    "Verdict-dependent REC features",
    13,
    len(VERDICT_DEPENDENT_REC_FEATURES),
    len(VERDICT_DEPENDENT_REC_FEATURES)
    == 13,
)

add_check(
    validation_records,
    "Verdict-independent REC features",
    6,
    len(VERDICT_INDEPENDENT_REC_FEATURES),
    len(VERDICT_INDEPENDENT_REC_FEATURES)
    == 6,
)

add_check(
    validation_records,
    "Raw duplicate Build-Test rows",
    0,
    raw_duplicate_build_test_rows,
    raw_duplicate_build_test_rows == 0,
)

add_check(
    validation_records,
    "Model duplicate Build-Test rows",
    0,
    model_duplicate_build_test_rows,
    model_duplicate_build_test_rows == 0,
)

add_check(
    validation_records,
    "Missing model-to-raw links",
    0,
    missing_model_raw_links,
    missing_model_raw_links == 0,
)

add_check(
    validation_records,
    "Model/raw verdict mismatches",
    0,
    model_raw_verdict_mismatches,
    model_raw_verdict_mismatches == 0,
)

add_check(
    validation_records,
    "Raw unlinked build rows",
    0,
    raw_unlinked_build_rows,
    raw_unlinked_build_rows == 0,
)

add_check(
    validation_records,
    "Model unlinked build rows",
    0,
    model_unlinked_build_rows,
    model_unlinked_build_rows == 0,
)

add_check(
    validation_records,
    "Non-finite duration rows",
    0,
    nonfinite_duration_rows,
    nonfinite_duration_rows == 0,
)

add_check(
    validation_records,
    "Negative duration rows",
    0,
    negative_duration_rows,
    negative_duration_rows == 0,
)

add_check(
    validation_records,
    "Resolved id_map EntityId column",
    "one uniquely resolved column",
    resolved_id_map_entity_id_column,
    resolved_id_map_entity_id_column
    in {
        id_map_key_column,
        id_map_value_column,
    },
)

add_check(
    validation_records,
    "Invalid rows in resolved id_map EntityId column",
    0,
    invalid_resolved_id_map_rows,
    invalid_resolved_id_map_rows == 0,
)

add_check(
    validation_records,
    "Duplicate resolved id_map EntityIds",
    0,
    duplicate_id_map_ids,
    duplicate_id_map_ids == 0,
)

add_check(
    validation_records,
    "Entity IDs missing from id_map",
    0,
    len(entity_ids_missing_from_id_map),
    len(entity_ids_missing_from_id_map)
    == 0,
)

add_check(
    validation_records,
    "Build rows without commit tokens",
    0,
    build_rows_without_commit_tokens,
    build_rows_without_commit_tokens == 0,
)

add_check(
    validation_records,
    "Unmatched commit tokens",
    0,
    unmatched_commit_tokens,
    unmatched_commit_tokens == 0,
)

add_check(
    validation_records,
    "Ambiguous commit tokens",
    0,
    ambiguous_commit_tokens,
    ambiguous_commit_tokens == 0,
)

add_check(
    validation_records,
    "Commit-token coverage percent",
    100.0,
    commit_token_coverage_percent,
    np.isclose(
        commit_token_coverage_percent,
        100.0,
        rtol=0,
        atol=1e-12,
    ),
)

add_check(
    validation_records,
    "Builds with no matched commit",
    0,
    len(builds_with_no_matched_commit),
    len(builds_with_no_matched_commit)
    == 0,
)

add_check(
    validation_records,
    "Builds with no mapped entity",
    0,
    len(builds_with_no_mapped_entity),
    len(builds_with_no_mapped_entity)
    == 0,
)

add_check(
    validation_records,
    "Builds with mapped entities",
    EXPECTED_BUILDS,
    len(builds_with_mapped_entities),
    len(builds_with_mapped_entities)
    == EXPECTED_BUILDS,
)

add_check(
    validation_records,
    "Build-entity rows generated",
    "> 0",
    build_entity_rows,
    build_entity_rows > 0,
)

add_check(
    validation_records,
    "Completion registry rows",
    10,
    len(registry),
    len(registry) == 10,
)

add_check(
    validation_records,
    "Project 11 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation[
    ~validation[
        "Pass"
    ]
]


print("\nProject 11 Step 2A V2 validation:")

display(validation)


if not failed_validation.empty:
    print("\nFailed validation checks:")

    display(failed_validation)

    raise RuntimeError(
        "PROJECT 11 STEP 2A V2 VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 16. WRITE FROZEN PREFLIGHT OUTPUTS
# --------------------------------------------------------------------------------------------------

REC_PREFLIGHT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    SOURCE_SCHEMA_PROFILE_PATH,
    source_schema_profile,
)

atomic_write_csv(
    BUILD_TEST_JOIN_AUDIT_PATH,
    build_test_join_audit,
)

atomic_write_csv(
    REC_FEATURE_CLASSIFICATION_PATH,
    rec_feature_classification,
)

atomic_write_csv(
    BUILD_COMMIT_TOKEN_PROFILE_PATH,
    build_commit_token_profile,
)

atomic_write_csv(
    COMMIT_MATCHING_AUDIT_PATH,
    commit_matching_audit,
)

atomic_write_csv_gzip(
    BUILD_ENTITY_MAP_PATH,
    build_entity_map,
)

atomic_write_csv(
    ID_MAP_ORIENTATION_AUDIT_PATH,
    id_map_orientation_audit,
)

atomic_write_csv(
    ENTITY_ID_MAP_AUDIT_PATH,
    entity_id_map_audit,
)

atomic_write_csv(
    STEP2A_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 17. READBACK VALIDATION
# --------------------------------------------------------------------------------------------------

build_entity_map_readback = pd.read_csv(
    BUILD_ENTITY_MAP_PATH,
    compression="gzip",
    low_memory=False,
)


commit_audit_readback = pd.read_csv(
    COMMIT_MATCHING_AUDIT_PATH,
    low_memory=False,
)


if len(
    build_entity_map_readback
) != build_entity_rows:
    raise RuntimeError(
        "Build-entity map readback row count differs."
    )


if len(
    commit_audit_readback
) != total_commit_tokens:
    raise RuntimeError(
        "Commit-matching audit readback row count differs."
    )


# --------------------------------------------------------------------------------------------------
# 18. WRITE SUMMARY, REPORT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


mapping_summary_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "BuildCommitColumn":
        build_commit_column,

    "EntityHistoryCommitColumn":
        entity_commit_column,

    "EntityHistoryIDColumn":
        entity_id_column,

    "IDMapOriginalKeyColumn":
        id_map_key_column,

    "IDMapOriginalValueColumn":
        id_map_value_column,

    "IDMapResolvedEntityIDColumn":
        resolved_id_map_entity_id_column,

    "IDMapResolvedEntityValueColumn":
        resolved_id_map_entity_value_column,

    "IDMapInvalidEntityIDRows":
        invalid_resolved_id_map_rows,

    "IDMapDuplicateEntityIDs":
        duplicate_id_map_ids,

    "IDMapIDsNotInEntityHistory":
        len(id_map_ids_not_in_history),

    "EntityIDsMissingFromIDMap":
        len(entity_ids_missing_from_id_map),

    "Builds":
        len(builds),

    "BuildCommitTokenRows":
        total_commit_tokens,

    "ExactCommitMatches":
        exact_commit_matches,

    "UniquePrefixMatches":
        unique_prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_commit_tokens,

    "AmbiguousCommitTokens":
        ambiguous_commit_tokens,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "BuildRowsWithoutCommitTokens":
        build_rows_without_commit_tokens,

    "BuildsWithNoMatchedCommit":
        len(builds_with_no_matched_commit),

    "BuildsWithNoMappedEntity":
        len(builds_with_no_mapped_entity),

    "BuildsWithMappedEntities":
        len(builds_with_mapped_entities),

    "BuildEntityRows":
        build_entity_rows,

    "UniqueMappedEntities":
        unique_mapped_entities,

    "UniqueMatchedCommits":
        unique_matched_commits,

    "EntityHistoryRows":
        len(entity_history),

    "EntityHistoryNonEmptyCommitRows":
        len(entity_history_nonempty),

    "EmptyEntityCommitRows":
        empty_entity_commit_rows,

    "CommitMatchingSeconds":
        commit_matching_seconds,

    "BuildCommitTokenProfile":
        str(
            BUILD_COMMIT_TOKEN_PROFILE_PATH
        ),

    "CommitMatchingAudit":
        str(
            COMMIT_MATCHING_AUDIT_PATH
        ),

    "BuildEntityMap":
        str(
            BUILD_ENTITY_MAP_PATH
        ),

    "IDMapOrientationAudit":
        str(
            ID_MAP_ORIENTATION_AUDIT_PATH
        ),

    "EntityIDMapAudit":
        str(
            ENTITY_ID_MAP_AUDIT_PATH
        ),
}


atomic_write_json(
    ENTITY_MAPPING_SUMMARY_PATH,
    mapping_summary_payload,
)


report_payload = {
    **mapping_summary_payload,

    "SourceRootSHA256":
        current_source_root_sha256,

    "SelectionCheckpoint":
        str(
            SELECTION_CHECKPOINT_PATH
        ),

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "CanonicalBuilds":
        len(chronology),

    "TrainingBuilds":
        len(training_build_ids),

    "EvaluationBuilds":
        len(evaluation_build_ids),

    "RawExecutionRows":
        len(exe),

    "ModelReadyRows":
        len(dataset),

    "DatasetColumns":
        len(dataset_columns),

    "PredictorColumns":
        len(predictor_columns),

    "RECFeatures":
        len(REC_FEATURE_COLUMNS),

    "VerdictDependentRECFeatures":
        len(
            VERDICT_DEPENDENT_REC_FEATURES
        ),

    "VerdictIndependentRECFeatures":
        len(
            VERDICT_INDEPENDENT_REC_FEATURES
        ),

    "RawDuplicateBuildTestRows":
        raw_duplicate_build_test_rows,

    "ModelDuplicateBuildTestRows":
        model_duplicate_build_test_rows,

    "MissingModelRawLinks":
        missing_model_raw_links,

    "ModelRawVerdictMismatches":
        model_raw_verdict_mismatches,

    "NonFiniteDurationRows":
        nonfinite_duration_rows,

    "NegativeDurationRows":
        negative_duration_rows,

    "SourceSchemaProfile":
        str(
            SOURCE_SCHEMA_PROFILE_PATH
        ),

    "BuildTestJoinAudit":
        str(
            BUILD_TEST_JOIN_AUDIT_PATH
        ),

    "RECFeatureClassification":
        str(
            REC_FEATURE_CLASSIFICATION_PATH
        ),

    "Validation":
        str(
            STEP2A_VALIDATION_PATH
        ),

    "ValidationChecks":
        len(validation),

    "FailedValidationChecks":
        len(failed_validation),

    "CompletionRegistry":
        str(
            REGISTRY_PATH
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To10Modified":
        False,

    "Project12Accessed":
        False,

    "Project12WriteAttempted":
        False,

    "NoiseInjected":
        False,

    "ModelsTrained":
        False,
}


atomic_write_json(
    STEP2A_REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "Builds":
        len(chronology),

    "RawExecutionRows":
        len(exe),

    "ModelReadyRows":
        len(dataset),

    "RECFeatures":
        len(REC_FEATURE_COLUMNS),

    "IDMapResolvedEntityIDColumn":
        resolved_id_map_entity_id_column,

    "IDMapResolvedEntityValueColumn":
        resolved_id_map_entity_value_column,

    "BuildCommitTokenRows":
        total_commit_tokens,

    "CommitTokenCoveragePercent":
        commit_token_coverage_percent,

    "BuildsWithMappedEntities":
        len(builds_with_mapped_entities),

    "BuildEntityRows":
        build_entity_rows,

    "FailedValidationChecks":
        len(failed_validation),

    "RegistryModified":
        False,

    "Project12Accessed":
        False,
}


atomic_write_json(
    STEP2A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 19. FINAL IMMUTABILITY CHECK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 11 Step 2A V2."
    )


final_source_records = []


for row in current_source_manifest.itertuples(
    index=False
):
    file_path = (
        SOURCE_DIRECTORY
        / row.RelativePath
    )

    final_source_records.append({
        "RelativePath":
            row.RelativePath,

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "SHA256":
            sha256_file(file_path),
    })


final_source_root_sha256 = canonical_root_hash(
    pd.DataFrame(
        final_source_records
    )
)


if (
    final_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The Project 11 source root changed during Step 2A V2."
    )


# --------------------------------------------------------------------------------------------------
# 20. DISPLAY
# --------------------------------------------------------------------------------------------------

print("\nResolved source schemas:")

display(source_schema_profile)


print("\nBuild-Test join audit:")

display(build_test_join_audit)


print("\nREC feature classification:")

display(rec_feature_classification)


print("\nid_map.csv orientation audit:")

display(id_map_orientation_audit)


print("\nResolved id_map.csv orientation:")

print(
    "EntityId column:",
    resolved_id_map_entity_id_column,
)

print(
    "Entity value/name column:",
    resolved_id_map_entity_value_column,
)


print("\nCommit-matching summary:")

display(
    commit_matching_audit[
        "MatchType"
    ]
    .value_counts(dropna=False)
    .rename_axis("MatchType")
    .reset_index(name="Rows")
)


print("\nCommit-matching sample:")

display(
    pd.concat(
        [
            commit_matching_audit.head(10),
            commit_matching_audit.tail(10),
        ],
        ignore_index=True,
    )
)


print("\nBuild-entity map sample:")

display(
    pd.concat(
        [
            build_entity_map.head(10),
            build_entity_map.tail(10),
        ],
        ignore_index=True,
    )
)


print("\nEntity-ID map coverage:")

display(
    entity_id_map_audit[
        "PresentInIDMap"
    ]
    .value_counts(dropna=False)
    .rename_axis("PresentInIDMap")
    .reset_index(name="Entities")
)


# --------------------------------------------------------------------------------------------------
# 21. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 132)
print("=== PROJECT 11 CELL 4 / STEP 2A V2 RESULT ===")
print("=" * 132)


print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nFrozen source:")

print(
    "Source files:",
    current_source_file_count,
)

print(
    "Source bytes:",
    current_source_bytes,
)

print(
    "Source root SHA-256:",
    final_source_root_sha256,
)


print("\nCanonical data:")

print(
    "Builds:",
    len(chronology),
)

print(
    "Training / evaluation builds:",
    len(training_build_ids),
    "/",
    len(evaluation_build_ids),
)

print(
    "Raw execution rows:",
    len(exe),
)

print(
    "Model-ready rows:",
    len(dataset),
)

print(
    "Dataset columns:",
    len(dataset_columns),
)

print(
    "Predictor columns:",
    len(predictor_columns),
)

print(
    "REC features:",
    len(REC_FEATURE_COLUMNS),
)


print("\nBuild-Test joins:")

print(
    "Raw duplicate Build-Test rows:",
    raw_duplicate_build_test_rows,
)

print(
    "Model duplicate Build-Test rows:",
    model_duplicate_build_test_rows,
)

print(
    "Missing model-to-raw links:",
    missing_model_raw_links,
)

print(
    "Model/raw verdict mismatches:",
    model_raw_verdict_mismatches,
)

print(
    "Non-finite duration rows:",
    nonfinite_duration_rows,
)

print(
    "Negative duration rows:",
    negative_duration_rows,
)


print("\nid_map.csv resolution:")

print(
    "Original key / value columns:",
    id_map_key_column,
    "/",
    id_map_value_column,
)

print(
    "Resolved EntityId column:",
    resolved_id_map_entity_id_column,
)

print(
    "Resolved entity value/name column:",
    resolved_id_map_entity_value_column,
)

print(
    "Invalid resolved EntityId rows:",
    invalid_resolved_id_map_rows,
)

print(
    "Duplicate resolved EntityIds:",
    duplicate_id_map_ids,
)

print(
    "Entity IDs missing from id_map:",
    len(entity_ids_missing_from_id_map),
)


print("\nCommit and entity mapping:")

print(
    "Build commit-token rows:",
    total_commit_tokens,
)

print(
    "Exact commit matches:",
    exact_commit_matches,
)

print(
    "Unique-prefix matches:",
    unique_prefix_matches,
)

print(
    "Unmatched commit tokens:",
    unmatched_commit_tokens,
)

print(
    "Ambiguous commit tokens:",
    ambiguous_commit_tokens,
)

print(
    "Commit-token coverage percent:",
    commit_token_coverage_percent,
)

print(
    "Build rows without commit tokens:",
    build_rows_without_commit_tokens,
)

print(
    "Builds with no matched commit:",
    len(builds_with_no_matched_commit),
)

print(
    "Builds with no mapped entity:",
    len(builds_with_no_mapped_entity),
)

print(
    "Builds with mapped entities:",
    len(builds_with_mapped_entities),
)

print(
    "Build-entity rows:",
    build_entity_rows,
)

print(
    "Unique mapped entities:",
    unique_mapped_entities,
)


print("\nImmutability and isolation:")

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–10 modified:",
    0,
)

print(
    "Project 12 accessed:",
    False,
)

print(
    "Noise injected:",
    False,
)

print(
    "Models trained:",
    False,
)


print("\nValidation:")

print(
    "Checks:",
    len(validation),
)

print(
    "Failed checks:",
    len(failed_validation),
)


print("\nSaved outputs:")

for output_path in [
    SOURCE_SCHEMA_PROFILE_PATH,
    BUILD_TEST_JOIN_AUDIT_PATH,
    REC_FEATURE_CLASSIFICATION_PATH,
    BUILD_COMMIT_TOKEN_PROFILE_PATH,
    COMMIT_MATCHING_AUDIT_PATH,
    BUILD_ENTITY_MAP_PATH,
    ID_MAP_ORIENTATION_AUDIT_PATH,
    ENTITY_ID_MAP_AUDIT_PATH,
    ENTITY_MAPPING_SUMMARY_PATH,
    STEP2A_VALIDATION_PATH,
    STEP2A_REPORT_PATH,
    STEP2A_STATUS_PATH,
]:
    print(output_path)


print(
    "\nSTATUS:",
    STEP2A_PASS_STATUS,
)

print("=" * 132)

=== PROJECT 11 CELL 4 / STEP 2A V2: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===


RuntimeError: The resolved id_map EntityId column contains 60788 duplicate-ID rows.

In [6]:
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import hashlib, json, os, re, time
import numpy as np
import pandas as pd

print("=" * 132)
print("=== PROJECT 11 CELL 4 / STEP 2A V4: ROBUST SOURCE AND JOIN VALIDATION ===")
print("=" * 132)

# -----------------------------
# Frozen identity and constants
# -----------------------------
PROJECT_NUMBER = 11
PROJECT_NAME = "apache@shardingsphere"
PROJECT_SLUG = "apache__shardingsphere"
PROJECT_SHORT = "shardingsphere"
SOURCE_DIR = Path("/content/datasets/datasets/apache@shardingsphere")

EXPECTED_STEP1B_STATUS = "PASS_PROJECT_11_SELECTION_AND_SOURCE_FROZEN"
STEP2A_STATUS = "PASS_PROJECT_11_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
EXPECTED_SELECTION_SHA = "9d5c95b29357426def26c41b62a753c003ad3b19e6086d9873f0961e39875353"
EXPECTED_SOURCE_ROOT_SHA = "3e6e0a60d58bff61690d6e2ac2bac18371414819eff601f3ea347ecb8ad8b0b5"
EXPECTED_REGISTRY_SHA = "847bdcbee16c8757fde34e9489cd3abbf0781a319644175472597265b1df2750"
EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 142_849_042
EXPECTED_BUILDS = 1_049
EXPECTED_TRAIN_BUILDS = 786
EXPECTED_EVAL_BUILDS = 263
EXPECTED_RAW_ROWS = 833_541
EXPECTED_MODEL_ROWS = 91_042
EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151
EXPECTED_MODEL_TRAIN_ROWS = 78_035
EXPECTED_MODEL_EVAL_ROWS = 13_007
EXPECTED_MODEL_TRAIN_FAILURES = 1_188
EXPECTED_MODEL_EVAL_FAILURES = 171

REC_FEATURES = [
    "REC_Age", "REC_LastFailureAge", "REC_LastTransitionAge",
    "REC_RecentAvgExeTime", "REC_RecentMaxExeTime", "REC_RecentFailRate",
    "REC_RecentAssertRate", "REC_RecentExcRate", "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime", "REC_TotalMaxExeTime", "REC_TotalFailRate",
    "REC_TotalAssertRate", "REC_TotalExcRate", "REC_TotalTransitionRate",
    "REC_LastVerdict", "REC_LastExeTime", "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]
DEPENDENT_REC = {
    "REC_LastFailureAge", "REC_LastTransitionAge", "REC_RecentFailRate",
    "REC_RecentAssertRate", "REC_RecentExcRate", "REC_RecentTransitionRate",
    "REC_TotalFailRate", "REC_TotalAssertRate", "REC_TotalExcRate",
    "REC_TotalTransitionRate", "REC_LastVerdict",
    "REC_MaxTestFileFailRate", "REC_MaxTestFileTransitionRate",
}
FILE_REC = {"REC_MaxTestFileFailRate", "REC_MaxTestFileTransitionRate"}
REQUIRED_SOURCE_FILES = [
    "builds.csv", "contributors.csv", "dataset.csv",
    "entity_change_history.csv", "exe.csv", "id_map.csv",
]

# -----
# Paths
# -----
THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES_ROOT = THESIS_ROOT / "Notes"
RESULTS_ROOT = THESIS_ROOT / "Results"
REGISTRY_PATH = NOTES_ROOT / "completed_project_registry.csv"
SELECTION_ROOT = RESULTS_ROOT / "Aggregated" / "project_11_selection"
SELECTION_CHECKPOINT_PATH = NOTES_ROOT / "project_11_selection_checkpoint.json"
STEP1B_STATUS_PATH = SELECTION_ROOT / "project_11_step1b_status.json"
FROZEN_SOURCE_MANIFEST_PATH = SELECTION_ROOT / "project_11_frozen_source_manifest.csv"
FIXED_CHRONOLOGY_PATH = SELECTION_ROOT / "project_11_fixed_chronological_builds.csv"
PROJECT_ROOT = RESULTS_ROOT / "Aggregated" / PROJECT_SLUG
PREFLIGHT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_rec_preflight"

SOURCE_SCHEMA_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_source_schema_profile.csv"
JOIN_AUDIT_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_build_test_join_audit.csv"
REC_CLASS_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_rec_feature_classification.csv"
BUILD_TOKEN_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_build_commit_token_profile.csv"
COMMIT_AUDIT_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_commit_matching_audit.csv"
BUILD_ENTITY_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
ID_ORIENTATION_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_id_map_orientation_audit.csv"
RESOLVED_ID_MAP_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_resolved_id_map_aliases.csv.gz"
ENTITY_ID_AUDIT_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_entity_id_map_audit.csv"
VALIDATION_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_step2a_validation.csv"
SUMMARY_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_entity_mapping_summary.json"
REPORT_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_step2a_report.json"
STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step2a_status.json"

# -------
# Helpers
# -------
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as fh:
        while True:
            chunk = fh.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as fh:
        return json.load(fh)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    with tmp.open("w", encoding="utf-8") as fh:
        json.dump(payload, fh, indent=2, sort_keys=True, ensure_ascii=False, default=str)
        fh.write("\n")
    os.replace(tmp, path)


def atomic_csv(path, frame, compression=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    frame.to_csv(tmp, index=False, lineterminator="\n", compression=compression)
    os.replace(tmp, path)


def resolve_column(columns, expected, label):
    matches = [c for c in columns if str(c).strip().lower() == expected.lower()]
    if len(matches) != 1:
        raise RuntimeError(f"Could not resolve {label}. Matches={matches}; columns={list(columns)}")
    return matches[0]


def parse_int(values, label):
    numeric = pd.to_numeric(values, errors="coerce")
    if numeric.isna().any():
        raise RuntimeError(f"{label} has {int(numeric.isna().sum())} missing/non-numeric values")
    arr = numeric.to_numpy(dtype=float)
    if not np.isclose(arr, np.floor(arr), rtol=0, atol=0).all():
        raise RuntimeError(f"{label} contains non-integral values")
    return numeric.astype("int64")


def source_root_hash(frame):
    digest = hashlib.sha256()
    for row in frame.sort_values("RelativePath", kind="mergesort").itertuples(index=False):
        digest.update(f"{row.RelativePath}\0{int(row.SizeBytes)}\0{str(row.SHA256).lower()}\n".encode())
    return digest.hexdigest()


def normalise_commit(value):
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    if not text:
        return ""
    matches = re.findall(r"[0-9a-f]{7,64}", text, flags=re.I)
    return matches[0].lower() if matches else re.sub(r"[^a-z0-9]", "", text)


def extract_commit_tokens(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    tokens = re.findall(r"[0-9a-fA-F]{7,64}", text)
    if not tokens:
        tokens = re.split(r"[\s,;|#]+", text)
    result, seen = [], set()
    for token in tokens:
        token = normalise_commit(token)
        if token and token not in seen:
            seen.add(token)
            result.append(token)
    return result


def add_check(rows, check, expected, actual, passed):
    rows.append({"Check": check, "Expected": expected, "Actual": actual, "Pass": bool(passed)})


# -----------------
# Required inputs
# -----------------
required = [
    REGISTRY_PATH, SELECTION_CHECKPOINT_PATH, STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH, FIXED_CHRONOLOGY_PATH,
]
missing = [str(p) for p in required if not Path(p).is_file()]
missing += [str(SOURCE_DIR / f) for f in REQUIRED_SOURCE_FILES if not (SOURCE_DIR / f).is_file()]
if missing:
    raise FileNotFoundError("Missing required Project 11 files:\n" + "\n".join(missing))

# -------------------------------
# Selection, registry, source root
# -------------------------------
selection_sha = sha256_file(SELECTION_CHECKPOINT_PATH)
selection = load_json(SELECTION_CHECKPOINT_PATH)
step1b_status = load_json(STEP1B_STATUS_PATH)
if selection_sha != EXPECTED_SELECTION_SHA:
    raise RuntimeError(f"Selection checkpoint SHA differs: {selection_sha}")
if selection.get("Status") != EXPECTED_STEP1B_STATUS or step1b_status.get("Status") != EXPECTED_STEP1B_STATUS:
    raise RuntimeError("Project 11 Step 1B is not frozen successfully")
if selection.get("Project") != PROJECT_NAME or selection.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError("Frozen Project 11 identity differs")

registry_sha_before = sha256_file(REGISTRY_PATH)
if registry_sha_before != EXPECTED_REGISTRY_SHA:
    raise RuntimeError(f"Registry SHA differs: {registry_sha_before}")
registry = pd.read_csv(REGISTRY_PATH, dtype=str).fillna("")
project_no_col = resolve_column(registry.columns, "ProjectNumber", "registry ProjectNumber")
status_col = resolve_column(registry.columns, "Status", "registry Status")
project_numbers = pd.to_numeric(registry[project_no_col], errors="raise").astype(int)
if len(registry) != 10 or sorted(project_numbers.tolist()) != list(range(1, 11)):
    raise RuntimeError("Registry must contain exactly Projects 1-10")
if not registry[status_col].eq(EXPECTED_COMPLETE_STATUS).all():
    raise RuntimeError("Projects 1-10 are not all COMPLETE_AND_FROZEN")
if project_numbers.eq(PROJECT_NUMBER).any():
    raise RuntimeError("Project 11 is unexpectedly already registered")

frozen_manifest = pd.read_csv(FROZEN_SOURCE_MANIFEST_PATH)
current_manifest = pd.DataFrame([
    {
        "RelativePath": str(r.RelativePath),
        "SizeBytes": int((SOURCE_DIR / str(r.RelativePath)).stat().st_size),
        "SHA256": sha256_file(SOURCE_DIR / str(r.RelativePath)),
    }
    for r in frozen_manifest.itertuples(index=False)
])
current_source_root = source_root_hash(current_manifest)
current_source_bytes = int(current_manifest["SizeBytes"].sum())
if current_source_root != EXPECTED_SOURCE_ROOT_SHA:
    raise RuntimeError(f"Frozen source root differs: {current_source_root}")

# ----------------------
# Resolve source schemas
# ----------------------
paths = {
    "builds.csv": SOURCE_DIR / "builds.csv",
    "contributors.csv": SOURCE_DIR / "contributors.csv",
    "dataset.csv": SOURCE_DIR / "dataset.csv",
    "entity_change_history.csv": SOURCE_DIR / "entity_change_history.csv",
    "exe.csv": SOURCE_DIR / "exe.csv",
    "id_map.csv": SOURCE_DIR / "id_map.csv",
}
schema_rows = []
headers = {}
for name, path in paths.items():
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    headers[name] = cols
    schema_rows.append({
        "File": name,
        "Path": str(path),
        "SizeBytes": int(path.stat().st_size),
        "ColumnCount": len(cols),
        "ColumnsJSON": json.dumps(cols, ensure_ascii=False),
    })
source_schema = pd.DataFrame(schema_rows)

build_id_col = resolve_column(headers["builds.csv"], "id", "builds id")
build_commit_col = resolve_column(headers["builds.csv"], "commits", "builds commits")
build_time_col = resolve_column(headers["builds.csv"], "started_at", "builds started_at")
exe_test_col = resolve_column(headers["exe.csv"], "test", "exe test")
exe_build_col = resolve_column(headers["exe.csv"], "build", "exe build")
exe_job_col = resolve_column(headers["exe.csv"], "job", "exe job")
exe_verdict_col = resolve_column(headers["exe.csv"], "verdict", "exe verdict")
exe_duration_col = resolve_column(headers["exe.csv"], "duration", "exe duration")
data_build_col = resolve_column(headers["dataset.csv"], "Build", "dataset Build")
data_test_col = resolve_column(headers["dataset.csv"], "Test", "dataset Test")
data_verdict_col = resolve_column(headers["dataset.csv"], "Verdict", "dataset Verdict")
entity_id_col = resolve_column(headers["entity_change_history.csv"], "EntityId", "entity EntityId")
entity_commit_col = resolve_column(headers["entity_change_history.csv"], "Commit", "entity Commit")
id_key_col = resolve_column(headers["id_map.csv"], "key", "id_map key")
id_value_col = resolve_column(headers["id_map.csv"], "value", "id_map value")

missing_rec = [c for c in REC_FEATURES if c not in headers["dataset.csv"]]
predictor_cols = [c for c in headers["dataset.csv"] if c not in {data_build_col, data_test_col, data_verdict_col}]
if missing_rec:
    raise RuntimeError("dataset.csv is missing REC columns: " + ", ".join(missing_rec))

# ----------------------
# Load chronology/tables
# ----------------------
chronology = pd.read_csv(FIXED_CHRONOLOGY_PATH)
chronology["BuildID"] = parse_int(chronology["BuildID"], "chronology.BuildID")
train_builds = set(chronology.loc[chronology["Partition"].eq("TRAIN"), "BuildID"].astype(int))
eval_builds = set(chronology.loc[chronology["Partition"].eq("EVALUATION"), "BuildID"].astype(int))
all_builds = train_builds | eval_builds
build_order = chronology.set_index("BuildID")["ChronologyOrder"].astype(int).to_dict()

builds = pd.read_csv(paths["builds.csv"], usecols=[build_id_col, build_commit_col, build_time_col])
builds[build_id_col] = parse_int(builds[build_id_col], "builds.id")
builds[build_time_col] = pd.to_datetime(builds[build_time_col], errors="coerce", utc=True)
if builds[build_time_col].isna().any():
    raise RuntimeError("builds.csv contains invalid timestamps")

exe = pd.read_csv(paths["exe.csv"], usecols=[exe_test_col, exe_build_col, exe_job_col, exe_verdict_col, exe_duration_col])
exe[exe_build_col] = parse_int(exe[exe_build_col], "exe.build")
exe[exe_test_col] = parse_int(exe[exe_test_col], "exe.test")
exe[exe_verdict_col] = pd.to_numeric(exe[exe_verdict_col], errors="coerce")
exe[exe_duration_col] = pd.to_numeric(exe[exe_duration_col], errors="coerce")
if exe[exe_verdict_col].isna().any():
    raise RuntimeError("exe.csv contains invalid verdicts")
exe[exe_verdict_col] = exe[exe_verdict_col].astype(int)

dataset = pd.read_csv(paths["dataset.csv"], low_memory=False)
dataset[data_build_col] = parse_int(dataset[data_build_col], "dataset.Build")
dataset[data_test_col] = parse_int(dataset[data_test_col], "dataset.Test")
dataset[data_verdict_col] = pd.to_numeric(dataset[data_verdict_col], errors="coerce")
if dataset[data_verdict_col].isna().any():
    raise RuntimeError("dataset.csv contains invalid verdicts")
dataset[data_verdict_col] = dataset[data_verdict_col].astype(int)

# --------------------------
# Build-Test join validation
# --------------------------
raw_duplicate_pairs = int(exe.duplicated([exe_build_col, exe_test_col], keep=False).sum())
model_duplicate_pairs = int(dataset.duplicated([data_build_col, data_test_col], keep=False).sum())
raw_unlinked_build_rows = int((~exe[exe_build_col].isin(all_builds)).sum())
model_unlinked_build_rows = int((~dataset[data_build_col].isin(all_builds)).sum())
nonfinite_duration_rows = int((~np.isfinite(exe[exe_duration_col].to_numpy(dtype=float))).sum())
negative_duration_rows = int(exe[exe_duration_col].lt(0).sum())

if raw_duplicate_pairs or model_duplicate_pairs:
    raise RuntimeError(f"Duplicate Build-Test pairs: raw={raw_duplicate_pairs}, model={model_duplicate_pairs}")

raw_pairs = exe[[exe_build_col, exe_test_col, exe_verdict_col, exe_duration_col]].rename(columns={
    exe_build_col: "Build", exe_test_col: "Test", exe_verdict_col: "RawVerdict", exe_duration_col: "RawDuration"
})
model_pairs = dataset[[data_build_col, data_test_col, data_verdict_col]].rename(columns={
    data_build_col: "Build", data_test_col: "Test", data_verdict_col: "ModelVerdict"
})
joined = model_pairs.merge(raw_pairs, on=["Build", "Test"], how="left", validate="one_to_one", indicator=True)
missing_model_raw_links = int(joined["_merge"].ne("both").sum())
verdict_mismatches = int(joined["ModelVerdict"].ne(joined["RawVerdict"]).sum())
train_mask = joined["Build"].isin(train_builds)
eval_mask = joined["Build"].isin(eval_builds)
model_train_rows = int(train_mask.sum())
model_eval_rows = int(eval_mask.sum())
model_train_failures = int((train_mask & joined["ModelVerdict"].ne(0)).sum())
model_eval_failures = int((eval_mask & joined["ModelVerdict"].ne(0)).sum())

join_audit = pd.DataFrame([
    ("RawRows", EXPECTED_RAW_ROWS, len(exe)),
    ("ModelRows", EXPECTED_MODEL_ROWS, len(dataset)),
    ("RawDuplicateBuildTestRows", 0, raw_duplicate_pairs),
    ("ModelDuplicateBuildTestRows", 0, model_duplicate_pairs),
    ("MissingModelRawLinks", 0, missing_model_raw_links),
    ("ModelRawVerdictMismatches", 0, verdict_mismatches),
    ("NonFiniteDurationRows", 0, nonfinite_duration_rows),
    ("NegativeDurationRows", 0, negative_duration_rows),
    ("RawUnlinkedBuildRows", 0, raw_unlinked_build_rows),
    ("ModelUnlinkedBuildRows", 0, model_unlinked_build_rows),
    ("ModelTrainingRows", EXPECTED_MODEL_TRAIN_ROWS, model_train_rows),
    ("ModelEvaluationRows", EXPECTED_MODEL_EVAL_ROWS, model_eval_rows),
    ("ModelTrainingFailures", EXPECTED_MODEL_TRAIN_FAILURES, model_train_failures),
    ("ModelEvaluationFailures", EXPECTED_MODEL_EVAL_FAILURES, model_eval_failures),
], columns=["Metric", "Expected", "Actual"])
join_audit["Pass"] = join_audit["Expected"].astype(str).eq(join_audit["Actual"].astype(str))

# --------------------------
# REC feature classification
# --------------------------
rec_class = pd.DataFrame([
    {
        "Feature": f,
        "FeatureClass": "VERDICT_DEPENDENT" if f in DEPENDENT_REC else "VERDICT_INDEPENDENT",
        "FileHistoryFeature": f in FILE_REC,
        "PresentInDataset": f in dataset.columns,
    }
    for f in REC_FEATURES
])

# -------------------
# Commit token profile
# -------------------
token_rows = []
builds_without_tokens = 0
for build_id, raw_commits in builds[[build_id_col, build_commit_col]].itertuples(index=False, name=None):
    tokens = extract_commit_tokens(raw_commits)
    if not tokens:
        builds_without_tokens += 1
    for token_order, token in enumerate(tokens, start=1):
        token_rows.append({
            "BuildID": int(build_id),
            "ChronologyOrder": int(build_order[int(build_id)]),
            "RawCommits": str(raw_commits),
            "TokenOrder": token_order,
            "CommitToken": token,
        })
build_tokens = pd.DataFrame(token_rows)
if build_tokens.empty:
    raise RuntimeError("No build commit tokens could be extracted")
build_tokens = build_tokens.sort_values(["ChronologyOrder", "TokenOrder"], kind="mergesort").reset_index(drop=True)

# -------------------------
# Entity history and id_map
# -------------------------
entity_history = pd.read_csv(paths["entity_change_history.csv"], usecols=[entity_id_col, entity_commit_col])
entity_history[entity_id_col] = parse_int(entity_history[entity_id_col], "entity_history.EntityId")
entity_history["NormalisedCommit"] = entity_history[entity_commit_col].map(normalise_commit)
entity_history = entity_history.loc[entity_history["NormalisedCommit"].ne(""), [entity_id_col, "NormalisedCommit"]].drop_duplicates()
history_entity_ids = set(entity_history[entity_id_col].astype(int))
history_commits = sorted(entity_history["NormalisedCommit"].unique().tolist())
history_commit_set = set(history_commits)

id_raw = pd.read_csv(paths["id_map.csv"], dtype=str, keep_default_na=False)
orientation_rows = []
for col in [id_key_col, id_value_col]:
    num = pd.to_numeric(id_raw[col], errors="coerce")
    valid = num.notna() & np.isclose(num.fillna(0), np.floor(num.fillna(0)), rtol=0, atol=0)
    parsed = set(num.loc[valid].astype("int64"))
    overlap = len(parsed & history_entity_ids)
    orientation_rows.append({
        "Column": col,
        "Rows": len(id_raw),
        "IntegralNumericRows": int(valid.sum()),
        "InvalidOrNonNumericRows": int((~valid).sum()),
        "UniqueIntegralIDs": len(parsed),
        "MatchingHistoryEntityIDs": overlap,
        "HistoryEntityCoveragePercent": 100.0 * overlap / len(history_entity_ids) if history_entity_ids else 0.0,
    })
id_orientation = pd.DataFrame(orientation_rows)
best = id_orientation.sort_values(["MatchingHistoryEntityIDs", "IntegralNumericRows"], ascending=False, kind="mergesort").reset_index(drop=True)
if len(best) < 2 or (
    int(best.loc[0, "MatchingHistoryEntityIDs"]) == int(best.loc[1, "MatchingHistoryEntityIDs"])
    and int(best.loc[0, "IntegralNumericRows"]) == int(best.loc[1, "IntegralNumericRows"])
):
    raise RuntimeError("Could not uniquely resolve numeric EntityId column in id_map.csv")
resolved_id_col = str(best.loc[0, "Column"])
resolved_path_col = id_value_col if resolved_id_col == id_key_col else id_key_col
resolved_numeric = pd.to_numeric(id_raw[resolved_id_col], errors="coerce")
valid_resolved = resolved_numeric.notna() & np.isclose(resolved_numeric.fillna(0), np.floor(resolved_numeric.fillna(0)), rtol=0, atol=0)
invalid_resolved_rows = int((~valid_resolved).sum())
if invalid_resolved_rows:
    raise RuntimeError(f"Resolved id_map EntityId column has {invalid_resolved_rows} invalid rows")

resolved_id_map = pd.DataFrame({
    "EntityPath": id_raw[resolved_path_col].astype(str).str.strip(),
    "EntityId": resolved_numeric.astype("int64"),
})
empty_path_rows = int(resolved_id_map["EntityPath"].eq("").sum())
resolved_id_map = resolved_id_map.drop_duplicates(["EntityPath", "EntityId"]).sort_values(["EntityId", "EntityPath"], kind="mergesort").reset_index(drop=True)
exact_duplicate_rows = int(len(id_raw) - len(resolved_id_map))
duplicate_entity_id_rows = int(resolved_id_map.duplicated("EntityId", keep=False).sum())
entity_ids_with_multiple_paths = int(resolved_id_map.groupby("EntityId")["EntityPath"].nunique().gt(1).sum())
paths_with_multiple_ids = int(resolved_id_map.groupby("EntityPath")["EntityId"].nunique().gt(1).sum())
max_paths_per_entity = int(resolved_id_map.groupby("EntityId")["EntityPath"].nunique().max())
id_map_entity_ids = set(resolved_id_map["EntityId"].astype(int))

# -----------------
# Commit matching
# -----------------
match_started = time.perf_counter()
match_rows = []
for row in build_tokens.itertuples(index=False):
    token = str(row.CommitToken).lower()
    matched = None
    if token in history_commit_set:
        match_type = "EXACT"
        matched = token
        candidate_count = 1
    else:
        candidates = [c for c in history_commits if c.startswith(token) or token.startswith(c)]
        if len(candidates) == 1:
            match_type = "UNIQUE_PREFIX"
            matched = candidates[0]
            candidate_count = 1
        elif len(candidates) == 0:
            match_type = "UNMATCHED"
            candidate_count = 0
        else:
            match_type = "AMBIGUOUS_PREFIX"
            candidate_count = len(candidates)
    match_rows.append({
        "BuildID": int(row.BuildID),
        "ChronologyOrder": int(row.ChronologyOrder),
        "TokenOrder": int(row.TokenOrder),
        "CommitToken": token,
        "MatchType": match_type,
        "MatchedCommit": matched,
        "CandidateMatches": candidate_count,
    })
commit_audit = pd.DataFrame(match_rows).sort_values(["ChronologyOrder", "TokenOrder"], kind="mergesort").reset_index(drop=True)
commit_match_seconds = float(time.perf_counter() - match_started)

exact_matches = int(commit_audit["MatchType"].eq("EXACT").sum())
prefix_matches = int(commit_audit["MatchType"].eq("UNIQUE_PREFIX").sum())
unmatched_tokens = int(commit_audit["MatchType"].eq("UNMATCHED").sum())
ambiguous_tokens = int(commit_audit["MatchType"].eq("AMBIGUOUS_PREFIX").sum())
matched_token_rows = exact_matches + prefix_matches
commit_coverage = 100.0 * matched_token_rows / len(commit_audit)

matched_build_commits = commit_audit.loc[commit_audit["MatchedCommit"].notna(), ["BuildID", "ChronologyOrder", "MatchedCommit"]].drop_duplicates()
entity_for_join = entity_history.rename(columns={entity_id_col: "EntityId", "NormalisedCommit": "MatchedCommit"})
build_entity = matched_build_commits.merge(entity_for_join, on="MatchedCommit", how="left", validate="many_to_many").dropna(subset=["EntityId"])
build_entity["EntityId"] = build_entity["EntityId"].astype("int64")
build_entity = build_entity[["BuildID", "ChronologyOrder", "MatchedCommit", "EntityId"]].drop_duplicates().sort_values(["ChronologyOrder", "EntityId", "MatchedCommit"], kind="mergesort").reset_index(drop=True)

builds_with_entities = set(build_entity["BuildID"].astype(int))
builds_without_entities = sorted(all_builds - builds_with_entities)
mapped_entity_ids = set(build_entity["EntityId"].astype(int))
mapped_entity_ids_missing_from_id_map = sorted(mapped_entity_ids - id_map_entity_ids)

alias_summary = resolved_id_map.groupby("EntityId", as_index=False).agg(
    EntityPathAliasCount=("EntityPath", "nunique"),
    CanonicalEntityPath=("EntityPath", "min"),
)
entity_id_audit = pd.DataFrame({"EntityId": sorted(mapped_entity_ids)}).merge(alias_summary, on="EntityId", how="left", validate="one_to_one")
entity_id_audit["PresentInIDMap"] = entity_id_audit["EntityPathAliasCount"].notna()
entity_id_audit["EntityPathAliasCount"] = entity_id_audit["EntityPathAliasCount"].fillna(0).astype(int)

# ----------
# Validation
# ----------
checks = []
add_check(checks, "Selection checkpoint SHA", EXPECTED_SELECTION_SHA, selection_sha, selection_sha == EXPECTED_SELECTION_SHA)
add_check(checks, "Source root SHA", EXPECTED_SOURCE_ROOT_SHA, current_source_root, current_source_root == EXPECTED_SOURCE_ROOT_SHA)
add_check(checks, "Source files", EXPECTED_SOURCE_FILES, len(current_manifest), len(current_manifest) == EXPECTED_SOURCE_FILES)
add_check(checks, "Source bytes", EXPECTED_SOURCE_BYTES, current_source_bytes, current_source_bytes == EXPECTED_SOURCE_BYTES)
add_check(checks, "Builds", EXPECTED_BUILDS, len(chronology), len(chronology) == EXPECTED_BUILDS)
add_check(checks, "Training builds", EXPECTED_TRAIN_BUILDS, len(train_builds), len(train_builds) == EXPECTED_TRAIN_BUILDS)
add_check(checks, "Evaluation builds", EXPECTED_EVAL_BUILDS, len(eval_builds), len(eval_builds) == EXPECTED_EVAL_BUILDS)
add_check(checks, "Raw rows", EXPECTED_RAW_ROWS, len(exe), len(exe) == EXPECTED_RAW_ROWS)
add_check(checks, "Model rows", EXPECTED_MODEL_ROWS, len(dataset), len(dataset) == EXPECTED_MODEL_ROWS)
add_check(checks, "Dataset columns", EXPECTED_DATASET_COLUMNS, len(dataset.columns), len(dataset.columns) == EXPECTED_DATASET_COLUMNS)
add_check(checks, "Predictors", EXPECTED_PREDICTORS, len(predictor_cols), len(predictor_cols) == EXPECTED_PREDICTORS)
add_check(checks, "REC features", 19, len(REC_FEATURES), len(REC_FEATURES) == 19 and not missing_rec)
add_check(checks, "Raw duplicate Build-Test rows", 0, raw_duplicate_pairs, raw_duplicate_pairs == 0)
add_check(checks, "Model duplicate Build-Test rows", 0, model_duplicate_pairs, model_duplicate_pairs == 0)
add_check(checks, "Missing model-to-raw links", 0, missing_model_raw_links, missing_model_raw_links == 0)
add_check(checks, "Model/raw verdict mismatches", 0, verdict_mismatches, verdict_mismatches == 0)
add_check(checks, "Non-finite durations", 0, nonfinite_duration_rows, nonfinite_duration_rows == 0)
add_check(checks, "Negative durations", 0, negative_duration_rows, negative_duration_rows == 0)
add_check(checks, "Invalid resolved id_map IDs", 0, invalid_resolved_rows, invalid_resolved_rows == 0)
add_check(checks, "Empty id_map paths", 0, empty_path_rows, empty_path_rows == 0)
add_check(checks, "Mapped entity IDs missing from id_map", 0, len(mapped_entity_ids_missing_from_id_map), len(mapped_entity_ids_missing_from_id_map) == 0)
add_check(checks, "Build-entity rows", "> 0", len(build_entity), len(build_entity) > 0)
add_check(checks, "Registry rows", 10, len(registry), len(registry) == 10)
add_check(checks, "Project 11 registry rows", 0, int(project_numbers.eq(11).sum()), int(project_numbers.eq(11).sum()) == 0)
validation = pd.DataFrame(checks)
failed = validation.loc[~validation["Pass"]]

print("\nProject 11 Step 2A V4 validation:")
display(validation)
if not failed.empty:
    print("\nFailed checks:")
    display(failed)
    raise RuntimeError("PROJECT 11 STEP 2A V4 VALIDATION FAILED")

# ----------------------
# Write auditable outputs
# ----------------------
PREFLIGHT_ROOT.mkdir(parents=True, exist_ok=True)
atomic_csv(SOURCE_SCHEMA_PATH, source_schema)
atomic_csv(JOIN_AUDIT_PATH, join_audit)
atomic_csv(REC_CLASS_PATH, rec_class)
atomic_csv(BUILD_TOKEN_PATH, build_tokens)
atomic_csv(COMMIT_AUDIT_PATH, commit_audit)
atomic_csv(BUILD_ENTITY_PATH, build_entity, compression="gzip")
atomic_csv(ID_ORIENTATION_PATH, id_orientation)
atomic_csv(RESOLVED_ID_MAP_PATH, resolved_id_map, compression="gzip")
atomic_csv(ENTITY_ID_AUDIT_PATH, entity_id_audit)
atomic_csv(VALIDATION_PATH, validation)

completed_at = datetime.now(timezone.utc).isoformat()
summary = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP2A_STATUS,
    "CompletedAtUTC": completed_at,
    "ResolvedIDMapEntityIDColumn": resolved_id_col,
    "ResolvedIDMapPathColumn": resolved_path_col,
    "ResolvedIDMapRows": len(resolved_id_map),
    "ExactDuplicateIDMapRowsRemoved": exact_duplicate_rows,
    "DuplicateEntityIDRowsAcceptedAsAliases": duplicate_entity_id_rows,
    "EntityIDsWithMultiplePaths": entity_ids_with_multiple_paths,
    "PathsWithMultipleEntityIDs": paths_with_multiple_ids,
    "MaximumPathsPerEntityID": max_paths_per_entity,
    "BuildCommitTokenRows": len(build_tokens),
    "ExactCommitMatches": exact_matches,
    "UniquePrefixMatches": prefix_matches,
    "UnmatchedCommitTokens": unmatched_tokens,
    "AmbiguousCommitTokens": ambiguous_tokens,
    "CommitTokenCoveragePercent": commit_coverage,
    "BuildsWithoutCommitTokens": builds_without_tokens,
    "BuildsWithMappedEntities": len(builds_with_entities),
    "BuildsWithoutMappedEntities": len(builds_without_entities),
    "BuildEntityRows": len(build_entity),
    "UniqueMappedEntities": int(build_entity["EntityId"].nunique()),
    "MappedEntityIDsMissingFromIDMap": len(mapped_entity_ids_missing_from_id_map),
    "CommitMatchingSeconds": commit_match_seconds,
}
atomic_json(SUMMARY_PATH, summary)

report = {
    **summary,
    "SourceRootSHA256": current_source_root,
    "SelectionCheckpointSHA256": selection_sha,
    "RawExecutionRows": len(exe),
    "ModelReadyRows": len(dataset),
    "DatasetColumns": len(dataset.columns),
    "PredictorColumns": len(predictor_cols),
    "RECFeatures": len(REC_FEATURES),
    "ValidationChecks": len(validation),
    "FailedValidationChecks": len(failed),
    "RegistrySHA256": registry_sha_before,
    "RegistryModified": False,
    "Projects1To10Modified": False,
    "Project12Accessed": False,
    "NoiseInjected": False,
    "ModelsTrained": False,
}
atomic_json(REPORT_PATH, report)
atomic_json(STATUS_PATH, {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP2A_STATUS,
    "CompletedAtUTC": completed_at,
    "SourceRootSHA256": current_source_root,
    "ResolvedIDMapEntityIDColumn": resolved_id_col,
    "ResolvedIDMapPathColumn": resolved_path_col,
    "BuildEntityRows": len(build_entity),
    "BuildsWithMappedEntities": len(builds_with_entities),
    "BuildsWithoutMappedEntities": len(builds_without_entities),
    "FailedValidationChecks": len(failed),
    "RegistryModified": False,
    "Project12Accessed": False,
})

# -------------------------
# Readback and immutability
# -------------------------
if len(pd.read_csv(BUILD_ENTITY_PATH, compression="gzip")) != len(build_entity):
    raise RuntimeError("Build-entity map readback failed")
if len(pd.read_csv(RESOLVED_ID_MAP_PATH, compression="gzip")) != len(resolved_id_map):
    raise RuntimeError("Resolved id_map readback failed")
if sha256_file(REGISTRY_PATH) != registry_sha_before:
    raise RuntimeError("Completion registry changed during Step 2A")
final_manifest = pd.DataFrame([
    {
        "RelativePath": r.RelativePath,
        "SizeBytes": int((SOURCE_DIR / r.RelativePath).stat().st_size),
        "SHA256": sha256_file(SOURCE_DIR / r.RelativePath),
    }
    for r in current_manifest.itertuples(index=False)
])
if source_root_hash(final_manifest) != EXPECTED_SOURCE_ROOT_SHA:
    raise RuntimeError("Frozen source changed during Step 2A")

# -------
# Display
# -------
print("\nBuild-Test join audit:")
display(join_audit)
print("\nid_map orientation audit:")
display(id_orientation)
print("\nid_map alias summary:")
display(pd.DataFrame([
    {"Metric": "Resolved EntityId column", "Value": resolved_id_col},
    {"Metric": "Resolved path column", "Value": resolved_path_col},
    {"Metric": "Resolved unique path-ID rows", "Value": len(resolved_id_map)},
    {"Metric": "Duplicate EntityId rows accepted as aliases", "Value": duplicate_entity_id_rows},
    {"Metric": "EntityIds with multiple paths", "Value": entity_ids_with_multiple_paths},
    {"Metric": "Paths with multiple EntityIds", "Value": paths_with_multiple_ids},
    {"Metric": "Mapped entity IDs missing from id_map", "Value": len(mapped_entity_ids_missing_from_id_map)},
]))
print("\nCommit matching summary:")
display(commit_audit["MatchType"].value_counts(dropna=False).rename_axis("MatchType").reset_index(name="Rows"))
print("\nBuild-entity sample:")
display(pd.concat([build_entity.head(10), build_entity.tail(10)], ignore_index=True))

print("\n" + "=" * 132)
print("=== PROJECT 11 CELL 4 / STEP 2A V4 RESULT ===")
print("=" * 132)
print("Project:", PROJECT_NAME)
print("Project slug:", PROJECT_SLUG)
print("Builds:", len(chronology))
print("Training / evaluation builds:", len(train_builds), "/", len(eval_builds))
print("Raw execution rows:", len(exe))
print("Model-ready rows:", len(dataset))
print("Dataset columns:", len(dataset.columns))
print("Predictor columns:", len(predictor_cols))
print("REC features:", len(REC_FEATURES))
print("Raw duplicate Build-Test rows:", raw_duplicate_pairs)
print("Model duplicate Build-Test rows:", model_duplicate_pairs)
print("Missing model-to-raw links:", missing_model_raw_links)
print("Model/raw verdict mismatches:", verdict_mismatches)
print("Resolved EntityId column:", resolved_id_col)
print("Resolved path column:", resolved_path_col)
print("Duplicate EntityId rows accepted as aliases:", duplicate_entity_id_rows)
print("EntityIds with multiple paths:", entity_ids_with_multiple_paths)
print("Paths with multiple EntityIds:", paths_with_multiple_ids)
print("Build commit-token rows:", len(build_tokens))
print("Exact commit matches:", exact_matches)
print("Unique-prefix matches:", prefix_matches)
print("Unmatched commit tokens:", unmatched_tokens)
print("Ambiguous commit tokens:", ambiguous_tokens)
print("Commit-token coverage percent:", commit_coverage)
print("Builds with mapped entities:", len(builds_with_entities))
print("Builds without mapped entities:", len(builds_without_entities))
print("Build-entity rows:", len(build_entity))
print("Mapped entity IDs missing from id_map:", len(mapped_entity_ids_missing_from_id_map))
print("Completion registry unchanged:", sha256_file(REGISTRY_PATH) == registry_sha_before)
print("Projects 1-10 modified:", 0)
print("Project 12 accessed:", False)
print("Noise injected:", False)
print("Models trained:", False)
print("Validation checks:", len(validation))
print("Failed checks:", len(failed))
print("STATUS:", STEP2A_STATUS)
print("=" * 132)

=== PROJECT 11 CELL 4 / STEP 2A V4: ROBUST SOURCE AND JOIN VALIDATION ===

Project 11 Step 2A V4 validation:


,Check,Expected,Actual,Pass
0,Selection checkpoint SHA,9d5c95b29357426def26c41b62a753c003ad3b19e6086d...,9d5c95b29357426def26c41b62a753c003ad3b19e6086d...,True
1,Source root SHA,3e6e0a60d58bff61690d6e2ac2bac18371414819eff601...,3e6e0a60d58bff61690d6e2ac2bac18371414819eff601...,True
2,Source files,6,6,True
3,Source bytes,142849042,142849042,True
4,Builds,1049,1049,True
5,Training builds,786,786,True
6,Evaluation builds,263,263,True
7,Raw rows,833541,833541,True
8,Model rows,91042,91042,True
9,Dataset columns,154,154,True



Build-Test join audit:


,Metric,Expected,Actual,Pass
0,RawRows,833541,833541,True
1,ModelRows,91042,91042,True
2,RawDuplicateBuildTestRows,0,0,True
3,ModelDuplicateBuildTestRows,0,0,True
4,MissingModelRawLinks,0,0,True
5,ModelRawVerdictMismatches,0,0,True
6,NonFiniteDurationRows,0,0,True
7,NegativeDurationRows,0,0,True
8,RawUnlinkedBuildRows,0,0,True
9,ModelUnlinkedBuildRows,0,0,True



id_map orientation audit:


...


Commit matching summary:


,MatchType,Rows
0,EXACT,1943
1,UNMATCHED,2



Build-entity sample:


,BuildID,ChronologyOrder,MatchedCommit,EntityId
0,752638221,1,542fe71d05aa731d93fcb9aa20ab590b42b3c4d6,1
1,752638221,1,9833fb9ab61439966bdfc00493d8da444e418ad3,1
2,752638221,1,4eeba649a7020fd54f5ddcedb2691504ad85df06,4
3,752638221,1,6711c71bc6bede55285db9101f0def14fcaf13fb,4
4,752638221,1,7e40667d3310cfb44465a6a0faa176df89ac1abd,4
5,752638221,1,85206032a393c0914530767d30fdba3a09a4d63e,4
6,752638221,1,b894b5ef0372bbe877c61f717da9179332b1caa7,4
7,752638221,1,68b42a4957f16563112b9e8696a717cbaa39f115,429
8,752638221,1,8e406d00871c8a6273173644998e568468f8c399,429
9,752638221,1,0594174b4446ed1306be205c05f2622c5327f347,558



=== PROJECT 11 CELL 4 / STEP 2A V4 RESULT ===
Project: apache@shardingsphere
Project slug: apache__shardingsphere
Builds: 1049
Training / evaluation builds: 786 / 263
Raw execution rows: 833541
Model-ready rows: 91042
Dataset columns: 154
Predictor columns: 151
REC features: 19
Raw duplicate Build-Test rows: 0
Model duplicate Build-Test rows: 0
Missing model-to-raw links: 0
Model/raw verdict mismatches: 0
Resolved EntityId column: value
Resolved path column: key
Duplicate EntityId rows accepted as aliases: 60788
EntityIds with multiple paths: 14933
Paths with multiple EntityIds: 0
Build commit-token rows: 1945
Exact commit matches: 1943
Unique-prefix matches: 0
Unmatched commit tokens: 2
Ambiguous commit tokens: 0
Commit-token coverage percent: 99.89717223650386
Builds with mapped entities: 1048
Builds without mapped entities: 1
Build-entity rows: 27902
Mapped entity IDs missing from id_map: 0
Completion registry unchanged: True
Projects 1-10 modified: 0
Project 12 accessed: False
Noi

In [7]:
# ==================================================================================================
# PROJECT 11 — CELL 5 / STEP 2B
# CLEAN REC RECONSTRUCTION, UNMATCHED-MAPPING AUDIT, AND CLEAN-ANCHOR FREEZE
#
# PROJECT: apache@shardingsphere
#
# This cell:
# - verifies the frozen Project 11 selection, source root, and Step 2A outputs;
# - audits the 2 unmatched commit tokens and the 1 build without mapped entities;
# - reconstructs all 19 REC features from clean raw execution history;
# - proves that any substantive direct mismatch is confined to file-history REC features on the
#   unmapped build;
# - freezes clean anchor offsets and proves exact 0% reproduction;
# - does not inject noise, train models, alter the registry, or touch Project 12.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import time

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 11 CELL 5 / STEP 2B: CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 11
PROJECT_NAME = "apache@shardingsphere"
PROJECT_SLUG = "apache__shardingsphere"
PROJECT_SHORT = "shardingsphere"

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@shardingsphere"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_11_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_STEP2A_STATUS = (
    "PASS_PROJECT_11_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
)

STEP2B_STATUS = (
    "PASS_PROJECT_11_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_SELECTION_SHA256 = (
    "9d5c95b29357426def26c41b62a753c003ad3b19e6086d9873f0961e39875353"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "3e6e0a60d58bff61690d6e2ac2bac18371414819eff601f3ea347ecb8ad8b0b5"
)

EXPECTED_REGISTRY_SHA256 = (
    "847bdcbee16c8757fde34e9489cd3abbf0781a319644175472597265b1df2750"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 142_849_042
EXPECTED_BUILDS = 1_049
EXPECTED_TRAIN_BUILDS = 786
EXPECTED_EVAL_BUILDS = 263
EXPECTED_RAW_ROWS = 833_541
EXPECTED_MODEL_ROWS = 91_042
EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151

# These counts are established by the completed Step 2A V4 output.
EXPECTED_COMMIT_TOKEN_ROWS = 1_945
EXPECTED_EXACT_COMMIT_MATCHES = 1_943
EXPECTED_UNMATCHED_COMMIT_TOKENS = 2
EXPECTED_AMBIGUOUS_COMMIT_TOKENS = 0
EXPECTED_BUILDS_WITH_MAPPED_ENTITIES = 1_048
EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES = 1
EXPECTED_BUILD_ENTITY_ROWS = 27_902

RECENT_WINDOW = 6
DIRECT_RTOL = 1e-9
DIRECT_ATOL = 1e-9
ANCHOR_RTOL = 0.0
ANCHOR_ATOL = 1e-12

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

FILE_HISTORY_REC = [
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

NON_FILE_REC = [
    feature
    for feature in REC_FEATURES
    if feature not in FILE_HISTORY_REC
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = THESIS_ROOT / "Notes"
RESULTS_ROOT = THESIS_ROOT / "Results"

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_11_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_11_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_11_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_11_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_11_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

STEP2A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2a_status.json"
)

STEP2A_REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_report.json"
)

ENTITY_MAPPING_SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_mapping_summary.json"
)

COMMIT_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_commit_matching_audit.csv"
)

BUILD_ENTITY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

UNMATCHED_MAPPING_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_unmatched_mapping_audit.csv"
)

CLEAN_RECONSTRUCTED_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

CLEAN_COMPARISON_SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_comparison_summary.csv"
)

CLEAN_MISMATCH_EXAMPLES_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_mismatch_examples.csv"
)

CLEAN_ANCHOR_VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_anchor_validation.csv"
)

STEP2B_VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_validation.csv"
)

STEP2B_REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_report.json"
)

STEP2B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2b_status.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_11_rec_reconstruction_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_parquet(path, frame):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )
        digest.update(line.encode("utf-8"))

    return digest.hexdigest()


def resolve_column(columns, expected, label):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; matches={matches}; columns={list(columns)}"
        )

    return matches[0]


def parse_int(values, label):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains {int(numeric.isna().sum())} missing/non-numeric values."
        )

    array = numeric.to_numpy(dtype=float)

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def add_check(records, check, expected, actual, passed):
    records.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def calculate_rates(history):
    history_length = len(history)

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history["verdict"]

    fail_rate = float(
        verdicts.ne(0).sum()
        / history_length
    )

    assertion_rate = float(
        verdicts.eq(2).sum()
        / history_length
    )

    exception_rate = float(
        verdicts.eq(1).sum()
        / history_length
    )

    transition_rate = float(
        history["transition"].eq(1).sum()
        / history_length
    )

    return (
        fail_rate,
        assertion_rate,
        exception_rate,
        transition_rate,
    )


def calculate_max_test_file_rate(
    history,
    target_column,
    current_changed_entities,
    entity_changed_builds,
):
    target_builds = (
        history.loc[
            history[target_column].gt(0),
            "build",
        ]
        .drop_duplicates()
        .astype(int)
        .tolist()
    )

    if len(target_builds) == 0:
        return -1.0

    target_build_set = set(target_builds)
    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(entity_id),
            set(),
        )

        overlap_count = len(
            changed_builds.intersection(
                target_build_set
            )
        )

        maximum_frequency = max(
            maximum_frequency,
            overlap_count,
        )

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(target_builds)
    )


def reconstruct_rec_features(
    execution_history,
    requested_rows,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
    recent_window=6,
):
    requested_pairs = set(
        zip(
            requested_rows["Build"].astype(int),
            requested_rows["Test"].astype(int),
        )
    )

    reconstructed_records = []
    test_groups = execution_history.groupby(
        "test",
        sort=False,
    )
    total_tests = int(
        execution_history["test"].nunique()
    )

    for test_index, (test_id, test_history) in enumerate(
        test_groups,
        start=1,
    ):
        test_history = (
            test_history.sort_values(
                [
                    "build_order",
                    "job",
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )

        test_history["transition"] = (
            test_history["verdict"]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(int)
        )

        first_test_build = int(
            test_history.iloc[0]["build"]
        )

        for current_position in range(
            len(test_history)
        ):
            current_row = test_history.iloc[
                current_position
            ]

            current_build = int(
                current_row["build"]
            )

            current_test = int(test_id)
            pair = (
                current_build,
                current_test,
            )

            if pair not in requested_pairs:
                continue

            history = (
                test_history.iloc[
                    :current_position
                ]
                .copy()
                .reset_index(drop=True)
            )

            record = {
                "Build": current_build,
                "Test": current_test,
            }

            if history.empty:
                for feature in REC_FEATURES:
                    record[feature] = -1.0

                record["REC_Age"] = 0.0
                reconstructed_records.append(record)
                continue

            recent_history = history.tail(
                recent_window
            ).copy()

            age = float(
                global_build_position[current_build]
                - global_build_position[first_test_build]
            )

            failure_positions = np.flatnonzero(
                history["verdict"].to_numpy() > 0
            )

            last_failure_age = (
                -1.0
                if len(failure_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(failure_positions[-1])
                )
            )

            transition_positions = np.flatnonzero(
                history["transition"].to_numpy() > 0
            )

            last_transition_age = (
                -1.0
                if len(transition_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(transition_positions[-1])
                )
            )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(
                recent_history
            )

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(
                history
            )

            current_changed_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            max_file_fail_rate = (
                calculate_max_test_file_rate(
                    history=history,
                    target_column="verdict",
                    current_changed_entities=(
                        current_changed_entities
                    ),
                    entity_changed_builds=(
                        entity_changed_builds
                    ),
                )
            )

            max_file_transition_rate = (
                calculate_max_test_file_rate(
                    history=history,
                    target_column="transition",
                    current_changed_entities=(
                        current_changed_entities
                    ),
                    entity_changed_builds=(
                        entity_changed_builds
                    ),
                )
            )

            record.update({
                "REC_Age": age,
                "REC_LastFailureAge": last_failure_age,
                "REC_LastTransitionAge": last_transition_age,
                "REC_RecentAvgExeTime": float(
                    recent_history["duration"].mean()
                ),
                "REC_RecentMaxExeTime": float(
                    recent_history["duration"].max()
                ),
                "REC_RecentFailRate": recent_fail_rate,
                "REC_RecentAssertRate": recent_assert_rate,
                "REC_RecentExcRate": recent_exc_rate,
                "REC_RecentTransitionRate": recent_transition_rate,
                "REC_TotalAvgExeTime": float(
                    history["duration"].mean()
                ),
                "REC_TotalMaxExeTime": float(
                    history["duration"].max()
                ),
                "REC_TotalFailRate": total_fail_rate,
                "REC_TotalAssertRate": total_assert_rate,
                "REC_TotalExcRate": total_exc_rate,
                "REC_TotalTransitionRate": total_transition_rate,
                "REC_LastVerdict": float(
                    recent_history.iloc[-1]["verdict"]
                ),
                "REC_LastExeTime": float(
                    recent_history.iloc[-1]["duration"]
                ),
                "REC_MaxTestFileFailRate": max_file_fail_rate,
                "REC_MaxTestFileTransitionRate": (
                    max_file_transition_rate
                ),
            })

            reconstructed_records.append(record)

        if (
            test_index % 100 == 0
            or test_index == total_tests
        ):
            print(
                "REC reconstruction progress:",
                test_index,
                "/",
                total_tests,
                "tests | reconstructed rows:",
                len(reconstructed_records),
            )

    return pd.DataFrame(
        reconstructed_records
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    STEP2A_STATUS_PATH,
    STEP2A_REPORT_PATH,
    ENTITY_MAPPING_SUMMARY_PATH,
    COMMIT_AUDIT_PATH,
    BUILD_ENTITY_PATH,
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "exe.csv",
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 11 Step 2B inputs are missing:\n"
        + "\n".join(missing_paths)
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY FROZEN STATE
# --------------------------------------------------------------------------------------------------

selection_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)
selection = load_json(
    SELECTION_CHECKPOINT_PATH
)
step1b_status = load_json(
    STEP1B_STATUS_PATH
)
step2a_status = load_json(
    STEP2A_STATUS_PATH
)
step2a_report = load_json(
    STEP2A_REPORT_PATH
)
entity_mapping_summary = load_json(
    ENTITY_MAPPING_SUMMARY_PATH
)

if selection_sha256 != EXPECTED_SELECTION_SHA256:
    raise RuntimeError(
        "Project 11 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_SHA256}\n"
        f"Actual:   {selection_sha256}"
    )

if (
    selection.get("Status") != EXPECTED_STEP1B_STATUS
    or step1b_status.get("Status") != EXPECTED_STEP1B_STATUS
):
    raise RuntimeError(
        "Project 11 Step 1B is not frozen successfully."
    )

if (
    selection.get("Project") != PROJECT_NAME
    or selection.get("ProjectSlug") != PROJECT_SLUG
):
    raise RuntimeError(
        "Frozen Project 11 identity differs."
    )

if step2a_status.get("Status") != EXPECTED_STEP2A_STATUS:
    raise RuntimeError(
        "Project 11 Step 2A status differs."
    )

if step2a_report.get("Status") != EXPECTED_STEP2A_STATUS:
    raise RuntimeError(
        "Project 11 Step 2A report status differs."
    )

registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)

registry_project_numbers = pd.to_numeric(
    registry[project_number_column],
    errors="raise",
).astype(int)

if (
    len(registry) != 10
    or sorted(registry_project_numbers.tolist())
    != list(range(1, 11))
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1-10."
    )

if not registry[status_column].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1-10 are not all COMPLETE_AND_FROZEN."
    )

if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 11 is unexpectedly already registered."
    )

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

current_source_manifest = pd.DataFrame([
    {
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(
            (
                SOURCE_DIR
                / str(row.RelativePath)
            ).stat().st_size
        ),
        "SHA256": sha256_file(
            SOURCE_DIR
            / str(row.RelativePath)
        ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])

current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

current_source_bytes = int(
    current_source_manifest["SizeBytes"].sum()
)

if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 11 source root differs.\n"
        f"Expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Actual:   {current_source_root_sha256}"
    )


# --------------------------------------------------------------------------------------------------
# 6. LOAD CHRONOLOGY, SOURCE DATA, AND STEP 2A MAPPING OUTPUTS
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)
chronology["BuildID"] = parse_int(
    chronology["BuildID"],
    "chronology.BuildID",
)
chronology["ChronologyOrder"] = parse_int(
    chronology["ChronologyOrder"],
    "chronology.ChronologyOrder",
)

training_builds = set(
    chronology.loc[
        chronology["Partition"].eq("TRAIN"),
        "BuildID",
    ].astype(int)
)

evaluation_builds = set(
    chronology.loc[
        chronology["Partition"].eq("EVALUATION"),
        "BuildID",
    ].astype(int)
)

all_builds = training_builds | evaluation_builds

build_order_map = (
    chronology.set_index("BuildID")[
        "ChronologyOrder"
    ]
    .astype(int)
    .to_dict()
)

headers = {
    "dataset": pd.read_csv(
        SOURCE_DIR / "dataset.csv",
        nrows=0,
    ).columns.tolist(),
    "exe": pd.read_csv(
        SOURCE_DIR / "exe.csv",
        nrows=0,
    ).columns.tolist(),
}

model_build_column = resolve_column(
    headers["dataset"],
    "Build",
    "dataset Build",
)
model_test_column = resolve_column(
    headers["dataset"],
    "Test",
    "dataset Test",
)
model_verdict_column = resolve_column(
    headers["dataset"],
    "Verdict",
    "dataset Verdict",
)

exe_test_column = resolve_column(
    headers["exe"],
    "test",
    "exe test",
)
exe_build_column = resolve_column(
    headers["exe"],
    "build",
    "exe build",
)
exe_job_column = resolve_column(
    headers["exe"],
    "job",
    "exe job",
)
exe_verdict_column = resolve_column(
    headers["exe"],
    "verdict",
    "exe verdict",
)
exe_duration_column = resolve_column(
    headers["exe"],
    "duration",
    "exe duration",
)

missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in headers["dataset"]
]

if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC features:\n"
        + "\n".join(missing_rec_features)
    )

predictor_columns = [
    column
    for column in headers["dataset"]
    if column not in {
        model_build_column,
        model_test_column,
        model_verdict_column,
    }
]

dataset = pd.read_csv(
    SOURCE_DIR / "dataset.csv",
    low_memory=False,
)

dataset[model_build_column] = parse_int(
    dataset[model_build_column],
    "dataset.Build",
)
dataset[model_test_column] = parse_int(
    dataset[model_test_column],
    "dataset.Test",
)

dataset = dataset.rename(
    columns={
        model_build_column: "Build",
        model_test_column: "Test",
        model_verdict_column: "Verdict",
    }
)

dataset["Verdict"] = parse_int(
    dataset["Verdict"],
    "dataset.Verdict",
)

exe = pd.read_csv(
    SOURCE_DIR / "exe.csv",
    usecols=[
        exe_test_column,
        exe_build_column,
        exe_job_column,
        exe_verdict_column,
        exe_duration_column,
    ],
    low_memory=False,
)

exe[exe_build_column] = parse_int(
    exe[exe_build_column],
    "exe.build",
)
exe[exe_test_column] = parse_int(
    exe[exe_test_column],
    "exe.test",
)
exe[exe_verdict_column] = parse_int(
    exe[exe_verdict_column],
    "exe.verdict",
)
exe[exe_duration_column] = pd.to_numeric(
    exe[exe_duration_column],
    errors="coerce",
)

if not np.isfinite(
    exe[exe_duration_column].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "exe.csv contains non-finite durations."
    )

if exe[exe_duration_column].lt(0).any():
    raise RuntimeError(
        "exe.csv contains negative durations."
    )

exe_for_rec = exe.rename(
    columns={
        exe_build_column: "build",
        exe_test_column: "test",
        exe_job_column: "job",
        exe_verdict_column: "verdict",
        exe_duration_column: "duration",
    }
).copy()

exe_for_rec["build_order"] = (
    exe_for_rec["build"]
    .map(build_order_map)
)

if exe_for_rec["build_order"].isna().any():
    raise RuntimeError(
        "Some raw executions cannot be mapped to the frozen build chronology."
    )

exe_for_rec["build_order"] = (
    exe_for_rec["build_order"].astype(int)
)

exe_for_rec = (
    exe_for_rec.sort_values(
        [
            "build_order",
            "job",
            "test",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

ordered_execution_builds = (
    exe_for_rec[
        [
            "build",
            "build_order",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "build_order",
        kind="mergesort",
    )["build"]
    .astype(int)
    .tolist()
)

global_build_position = {
    int(build_id): position
    for position, build_id in enumerate(
        ordered_execution_builds
    )
}

commit_audit = pd.read_csv(
    COMMIT_AUDIT_PATH,
    low_memory=False,
)

build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)

required_commit_columns = {
    "BuildID",
    "CommitToken",
    "MatchType",
}

if not required_commit_columns.issubset(
    commit_audit.columns
):
    raise RuntimeError(
        "Step 2A commit audit is missing required columns."
    )

required_entity_columns = {
    "BuildID",
    "EntityId",
}

if not required_entity_columns.issubset(
    build_entity.columns
):
    raise RuntimeError(
        "Step 2A build-entity map is missing required columns."
    )

commit_audit["BuildID"] = parse_int(
    commit_audit["BuildID"],
    "commit_audit.BuildID",
)
build_entity["BuildID"] = parse_int(
    build_entity["BuildID"],
    "build_entity.BuildID",
)
build_entity["EntityId"] = parse_int(
    build_entity["EntityId"],
    "build_entity.EntityId",
)

match_type_normalised = (
    commit_audit["MatchType"]
    .astype(str)
    .str.strip()
    .str.upper()
)

exact_match_mask = match_type_normalised.isin({
    "EXACT",
    "EXACT_NORMALISED_COMMIT_TOKEN",
})

prefix_match_mask = match_type_normalised.isin({
    "UNIQUE_PREFIX",
    "UNIQUE_PREFIX_COMMIT_TOKEN",
})

unmatched_mask = match_type_normalised.isin({
    "UNMATCHED",
    "UNMATCHED_COMMIT_TOKEN",
})

ambiguous_mask = match_type_normalised.isin({
    "AMBIGUOUS",
    "AMBIGUOUS_PREFIX_COMMIT_TOKEN",
})

unknown_match_type_rows = int(
    (~(
        exact_match_mask
        | prefix_match_mask
        | unmatched_mask
        | ambiguous_mask
    )).sum()
)

if unknown_match_type_rows:
    raise RuntimeError(
        f"Commit audit contains {unknown_match_type_rows} unknown MatchType rows."
    )

exact_matches = int(exact_match_mask.sum())
prefix_matches = int(prefix_match_mask.sum())
unmatched_tokens = int(unmatched_mask.sum())
ambiguous_tokens = int(ambiguous_mask.sum())

unmatched_token_rows = (
    commit_audit.loc[
        unmatched_mask
    ]
    .copy()
    .sort_values(
        [
            "BuildID",
            "CommitToken",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

unmatched_token_builds = sorted(
    unmatched_token_rows["BuildID"]
    .astype(int)
    .unique()
    .tolist()
)

builds_with_entities = set(
    build_entity["BuildID"].astype(int)
)

builds_without_entities = sorted(
    all_builds
    - builds_with_entities
)

changed_entities_by_build = (
    build_entity.groupby(
        "BuildID"
    )["EntityId"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

changed_entities_by_build = {
    int(build_id): set(
        int(entity_id)
        for entity_id in changed_entities_by_build.get(
            int(build_id),
            set(),
        )
    )
    for build_id in all_builds
}

entity_changed_builds = (
    build_entity.groupby(
        "EntityId"
    )["BuildID"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)


# --------------------------------------------------------------------------------------------------
# 7. EXPLICIT UNMATCHED-MAPPING AUDIT
# --------------------------------------------------------------------------------------------------

unmatched_mapping_records = []

audit_builds = sorted(
    set(unmatched_token_builds)
    | set(builds_without_entities)
)

for build_id in audit_builds:
    partition_rows = chronology.loc[
        chronology["BuildID"].eq(build_id),
        [
            "Partition",
            "ChronologyOrder",
        ],
    ]

    if len(partition_rows) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve chronology for unmatched build {build_id}."
        )

    partition = str(
        partition_rows.iloc[0]["Partition"]
    )
    chronology_order = int(
        partition_rows.iloc[0]["ChronologyOrder"]
    )

    unmatched_rows_for_build = unmatched_token_rows.loc[
        unmatched_token_rows["BuildID"].eq(build_id)
    ]

    model_rows_for_build = dataset.loc[
        dataset["Build"].eq(build_id)
    ]

    raw_rows_for_build = exe_for_rec.loc[
        exe_for_rec["build"].eq(build_id)
    ]

    unmatched_mapping_records.append({
        "BuildID": int(build_id),
        "ChronologyOrder": chronology_order,
        "Partition": partition,
        "UnmatchedCommitTokens": int(
            len(unmatched_rows_for_build)
        ),
        "UnmatchedTokenList": " | ".join(
            unmatched_rows_for_build["CommitToken"]
            .astype(str)
            .tolist()
        ),
        "HasMappedEntities": bool(
            build_id in builds_with_entities
        ),
        "MappedEntityCount": int(
            len(
                changed_entities_by_build.get(
                    build_id,
                    set(),
                )
            )
        ),
        "RawExecutionRows": int(
            len(raw_rows_for_build)
        ),
        "RawFailureRows": int(
            raw_rows_for_build["verdict"].ne(0).sum()
        ),
        "ModelReadyRows": int(
            len(model_rows_for_build)
        ),
        "ModelFailureRows": int(
            model_rows_for_build["Verdict"].ne(0).sum()
        ),
    })

unmatched_mapping_audit = pd.DataFrame(
    unmatched_mapping_records
)

mapping_incomplete_builds = sorted(
    set(unmatched_token_builds)
    | set(builds_without_entities)
)


# --------------------------------------------------------------------------------------------------
# 8. RECONSTRUCT ALL 19 CLEAN REC FEATURES
# --------------------------------------------------------------------------------------------------

requested_rows = dataset[
    [
        "Build",
        "Test",
    ]
].copy()

print("\nUnmatched mapping audit:")
display(unmatched_mapping_audit)

print("\nReconstructing all 19 clean REC features.")

reconstruction_started = time.perf_counter()

clean_reconstructed = reconstruct_rec_features(
    execution_history=exe_for_rec,
    requested_rows=requested_rows,
    global_build_position=global_build_position,
    changed_entities_by_build=(
        changed_entities_by_build
    ),
    entity_changed_builds=(
        entity_changed_builds
    ),
    recent_window=RECENT_WINDOW,
)

reconstruction_seconds = float(
    time.perf_counter()
    - reconstruction_started
)

reconstructed_duplicate_rows = int(
    clean_reconstructed.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)

aligned = (
    dataset[
        [
            "Build",
            "Test",
        ]
        + REC_FEATURES
    ]
    .merge(
        clean_reconstructed,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        suffixes=(
            "_original",
            "_reconstructed",
        ),
        indicator=True,
    )
)

missing_reconstructed_rows = int(
    aligned["_merge"].ne("both").sum()
)

print("\nClean REC reconstruction completed:")
print("Requested rows:", len(dataset))
print("Reconstructed rows:", len(clean_reconstructed))
print("Duplicate reconstructed rows:", reconstructed_duplicate_rows)
print("Missing reconstructed rows:", missing_reconstructed_rows)
print("Reconstruction seconds:", reconstruction_seconds)

if reconstructed_duplicate_rows != 0:
    raise RuntimeError(
        "Clean REC reconstruction produced duplicate Build-Test rows."
    )

if missing_reconstructed_rows != 0:
    raise RuntimeError(
        "Clean REC reconstruction is missing requested Build-Test rows."
    )


# --------------------------------------------------------------------------------------------------
# 9. DIRECT COMPARISON AND CLEAN-ANCHOR OFFSETS
# --------------------------------------------------------------------------------------------------

comparison_records = []
mismatch_examples = []

anchor_offsets = aligned[
    [
        "Build",
        "Test",
    ]
].copy()

anchored_reproduction = aligned[
    [
        "Build",
        "Test",
    ]
].copy()

mapping_incomplete_build_set = set(
    mapping_incomplete_builds
)

rows_at_mapping_incomplete_build = aligned["Build"].isin(
    mapping_incomplete_build_set
)

for feature in REC_FEATURES:
    original_column = f"{feature}_original"
    reconstructed_column = f"{feature}_reconstructed"

    original_values = pd.to_numeric(
        aligned[original_column],
        errors="coerce",
    ).to_numpy(dtype=float)

    reconstructed_values = pd.to_numeric(
        aligned[reconstructed_column],
        errors="coerce",
    ).to_numpy(dtype=float)

    if (
        not np.isfinite(original_values).all()
        or not np.isfinite(reconstructed_values).all()
    ):
        raise RuntimeError(
            f"Feature {feature} contains non-finite clean comparison values."
        )

    direct_match_mask = np.isclose(
        original_values,
        reconstructed_values,
        rtol=DIRECT_RTOL,
        atol=DIRECT_ATOL,
        equal_nan=False,
    )

    direct_difference = (
        original_values
        - reconstructed_values
    )

    anchor_offsets[feature] = direct_difference

    anchored_values = (
        reconstructed_values
        + direct_difference
    )

    anchored_reproduction[feature] = anchored_values

    anchored_match_mask = np.isclose(
        original_values,
        anchored_values,
        rtol=ANCHOR_RTOL,
        atol=ANCHOR_ATOL,
        equal_nan=False,
    )

    direct_mismatch_mask = ~direct_match_mask
    anchored_mismatch_mask = ~anchored_match_mask

    mismatches_at_mapping_incomplete_build = int(
        (
            direct_mismatch_mask
            & rows_at_mapping_incomplete_build.to_numpy()
        ).sum()
    )

    mismatches_outside_mapping_incomplete_build = int(
        (
            direct_mismatch_mask
            & ~rows_at_mapping_incomplete_build.to_numpy()
        ).sum()
    )

    comparison_records.append({
        "Feature": feature,
        "FeatureClass": (
            "VERDICT_DEPENDENT"
            if feature in VERDICT_DEPENDENT_REC
            else "VERDICT_INDEPENDENT"
        ),
        "FileHistoryFeature": bool(
            feature in FILE_HISTORY_REC
        ),
        "Rows": len(aligned),
        "DirectMatchingRows": int(
            direct_match_mask.sum()
        ),
        "DirectMismatchingRows": int(
            direct_mismatch_mask.sum()
        ),
        "DirectMismatchesAtMappingIncompleteBuild": (
            mismatches_at_mapping_incomplete_build
        ),
        "DirectMismatchesOutsideMappingIncompleteBuild": (
            mismatches_outside_mapping_incomplete_build
        ),
        "NonZeroAnchorOffsets": int(
            np.count_nonzero(direct_difference)
        ),
        "AnchoredMatchingRows": int(
            anchored_match_mask.sum()
        ),
        "AnchoredMismatchingRows": int(
            anchored_mismatch_mask.sum()
        ),
        "MaximumAbsoluteDirectDifference": float(
            np.max(np.abs(direct_difference))
            if len(direct_difference)
            else 0.0
        ),
        "MeanAbsoluteDirectDifference": float(
            np.mean(np.abs(direct_difference))
            if len(direct_difference)
            else 0.0
        ),
        "MaximumAbsoluteAnchoredDifference": float(
            np.max(
                np.abs(
                    original_values
                    - anchored_values
                )
            )
            if len(anchored_values)
            else 0.0
        ),
    })

    mismatch_indices = np.flatnonzero(
        direct_mismatch_mask
    )[:20]

    for row_index in mismatch_indices:
        mismatch_examples.append({
            "Build": int(
                aligned.iloc[row_index]["Build"]
            ),
            "Test": int(
                aligned.iloc[row_index]["Test"]
            ),
            "Feature": feature,
            "FeatureClass": (
                "VERDICT_DEPENDENT"
                if feature in VERDICT_DEPENDENT_REC
                else "VERDICT_INDEPENDENT"
            ),
            "FileHistoryFeature": bool(
                feature in FILE_HISTORY_REC
            ),
            "IsMappingIncompleteBuild": bool(
                aligned.iloc[row_index]["Build"]
                in mapping_incomplete_build_set
            ),
            "Original": float(
                original_values[row_index]
            ),
            "Reconstructed": float(
                reconstructed_values[row_index]
            ),
            "AnchorOffset": float(
                direct_difference[row_index]
            ),
        })

comparison_summary = pd.DataFrame(
    comparison_records
)

mismatch_examples_frame = pd.DataFrame(
    mismatch_examples,
    columns=[
        "Build",
        "Test",
        "Feature",
        "FeatureClass",
        "FileHistoryFeature",
        "IsMappingIncompleteBuild",
        "Original",
        "Reconstructed",
        "AnchorOffset",
    ],
)

anchor_validation = comparison_summary[
    [
        "Feature",
        "FeatureClass",
        "Rows",
        "AnchoredMatchingRows",
        "AnchoredMismatchingRows",
        "MaximumAbsoluteAnchoredDifference",
    ]
].rename(
    columns={
        "AnchoredMatchingRows": "MatchingRows",
        "AnchoredMismatchingRows": "MismatchingRows",
    }
)

anchor_validation["Pass"] = (
    anchor_validation["MismatchingRows"].eq(0)
)

failed_anchor_features = int(
    (~anchor_validation["Pass"]).sum()
)

anchored_mismatch_values = int(
    comparison_summary[
        "AnchoredMismatchingRows"
    ].sum()
)

direct_mismatch_values = int(
    comparison_summary[
        "DirectMismatchingRows"
    ].sum()
)

verdict_dependent_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FeatureClass"
        ].eq("VERDICT_DEPENDENT"),
        "DirectMismatchingRows",
    ].sum()
)

verdict_independent_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FeatureClass"
        ].eq("VERDICT_INDEPENDENT"),
        "DirectMismatchingRows",
    ].sum()
)

file_history_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchingRows",
    ].sum()
)

non_file_direct_mismatches = int(
    comparison_summary.loc[
        ~comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchingRows",
    ].sum()
)

file_mismatches_outside_mapping_incomplete_build = int(
    comparison_summary.loc[
        comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchesOutsideMappingIncompleteBuild",
    ].sum()
)

substantive_row_mismatch_mask = np.zeros(
    len(aligned),
    dtype=bool,
)

for feature in REC_FEATURES:
    original_values = aligned[
        f"{feature}_original"
    ].to_numpy(dtype=float)
    reconstructed_values = aligned[
        f"{feature}_reconstructed"
    ].to_numpy(dtype=float)
    substantive_row_mismatch_mask |= ~np.isclose(
        original_values,
        reconstructed_values,
        rtol=DIRECT_RTOL,
        atol=DIRECT_ATOL,
    )

rows_with_any_substantive_direct_mismatch = int(
    substantive_row_mismatch_mask.sum()
)

rows_with_any_nonzero_anchor_offset = int(
    anchor_offsets[REC_FEATURES]
    .ne(0)
    .any(axis=1)
    .sum()
)

nonzero_anchor_offset_values = int(
    anchor_offsets[REC_FEATURES]
    .ne(0)
    .sum()
    .sum()
)

zero_percent_clean_reproduced_exactly = bool(
    failed_anchor_features == 0
    and anchored_mismatch_values == 0
)

# Strong acceptance rule for this edge case:
# substantive direct mismatches may only occur in the two file-history REC features,
# and only on the single build whose commits could not be mapped.
unmatched_mapping_effect_is_confined = bool(
    non_file_direct_mismatches == 0
    and file_mismatches_outside_mapping_incomplete_build == 0
)


# --------------------------------------------------------------------------------------------------
# 10. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []

add_check(
    validation_records,
    "Step 1B passed",
    EXPECTED_STEP1B_STATUS,
    step1b_status.get("Status"),
    step1b_status.get("Status")
    == EXPECTED_STEP1B_STATUS,
)

add_check(
    validation_records,
    "Step 2A passed",
    EXPECTED_STEP2A_STATUS,
    step2a_status.get("Status"),
    step2a_status.get("Status")
    == EXPECTED_STEP2A_STATUS,
)

add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_SHA256,
    selection_sha256,
    selection_sha256
    == EXPECTED_SELECTION_SHA256,
)

add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    validation_records,
    "Frozen source files",
    EXPECTED_SOURCE_FILES,
    len(current_source_manifest),
    len(current_source_manifest)
    == EXPECTED_SOURCE_FILES,
)

add_check(
    validation_records,
    "Frozen source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    validation_records,
    "Canonical builds",
    EXPECTED_BUILDS,
    len(chronology),
    len(chronology) == EXPECTED_BUILDS,
)

add_check(
    validation_records,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(training_builds),
    len(training_builds)
    == EXPECTED_TRAIN_BUILDS,
)

add_check(
    validation_records,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(evaluation_builds),
    len(evaluation_builds)
    == EXPECTED_EVAL_BUILDS,
)

add_check(
    validation_records,
    "Raw execution rows",
    EXPECTED_RAW_ROWS,
    len(exe_for_rec),
    len(exe_for_rec) == EXPECTED_RAW_ROWS,
)

add_check(
    validation_records,
    "Model-ready rows",
    EXPECTED_MODEL_ROWS,
    len(dataset),
    len(dataset) == EXPECTED_MODEL_ROWS,
)

add_check(
    validation_records,
    "Dataset columns",
    EXPECTED_DATASET_COLUMNS,
    len(headers["dataset"]),
    len(headers["dataset"])
    == EXPECTED_DATASET_COLUMNS,
)

add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(predictor_columns),
    len(predictor_columns)
    == EXPECTED_PREDICTORS,
)

add_check(
    validation_records,
    "REC features reconstructed",
    len(REC_FEATURES),
    len([
        column
        for column in clean_reconstructed.columns
        if column in REC_FEATURES
    ]),
    all(
        feature in clean_reconstructed.columns
        for feature in REC_FEATURES
    ),
)

add_check(
    validation_records,
    "Commit-token rows",
    EXPECTED_COMMIT_TOKEN_ROWS,
    len(commit_audit),
    len(commit_audit)
    == EXPECTED_COMMIT_TOKEN_ROWS,
)

add_check(
    validation_records,
    "Exact commit matches",
    EXPECTED_EXACT_COMMIT_MATCHES,
    exact_matches,
    exact_matches
    == EXPECTED_EXACT_COMMIT_MATCHES,
)

add_check(
    validation_records,
    "Unmatched commit tokens",
    EXPECTED_UNMATCHED_COMMIT_TOKENS,
    unmatched_tokens,
    unmatched_tokens
    == EXPECTED_UNMATCHED_COMMIT_TOKENS,
)

add_check(
    validation_records,
    "Ambiguous commit tokens",
    EXPECTED_AMBIGUOUS_COMMIT_TOKENS,
    ambiguous_tokens,
    ambiguous_tokens
    == EXPECTED_AMBIGUOUS_COMMIT_TOKENS,
)

add_check(
    validation_records,
    "Builds with mapped entities",
    EXPECTED_BUILDS_WITH_MAPPED_ENTITIES,
    len(builds_with_entities),
    len(builds_with_entities)
    == EXPECTED_BUILDS_WITH_MAPPED_ENTITIES,
)

add_check(
    validation_records,
    "Builds without mapped entities",
    EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES,
    len(builds_without_entities),
    len(builds_without_entities)
    == EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES,
)

add_check(
    validation_records,
    "Mapping-incomplete builds identified",
    "> 0",
    len(mapping_incomplete_builds),
    len(mapping_incomplete_builds) > 0,
)

add_check(
    validation_records,
    "Build-entity rows",
    EXPECTED_BUILD_ENTITY_ROWS,
    len(build_entity),
    len(build_entity)
    == EXPECTED_BUILD_ENTITY_ROWS,
)

add_check(
    validation_records,
    "Reconstructed REC rows",
    EXPECTED_MODEL_ROWS,
    len(clean_reconstructed),
    len(clean_reconstructed)
    == EXPECTED_MODEL_ROWS,
)

add_check(
    validation_records,
    "Duplicate reconstructed rows",
    0,
    reconstructed_duplicate_rows,
    reconstructed_duplicate_rows == 0,
)

add_check(
    validation_records,
    "Missing reconstructed rows",
    0,
    missing_reconstructed_rows,
    missing_reconstructed_rows == 0,
)

add_check(
    validation_records,
    "Non-file direct mismatch values",
    0,
    non_file_direct_mismatches,
    non_file_direct_mismatches == 0,
)

add_check(
    validation_records,
    "File-history mismatches outside mapping-incomplete builds",
    0,
    file_mismatches_outside_mapping_incomplete_build,
    file_mismatches_outside_mapping_incomplete_build == 0,
)

add_check(
    validation_records,
    "Unmatched mapping effect confined",
    True,
    unmatched_mapping_effect_is_confined,
    unmatched_mapping_effect_is_confined,
)

add_check(
    validation_records,
    "Failed clean-anchor features",
    0,
    failed_anchor_features,
    failed_anchor_features == 0,
)

add_check(
    validation_records,
    "Anchored mismatch values",
    0,
    anchored_mismatch_values,
    anchored_mismatch_values == 0,
)

add_check(
    validation_records,
    "0% clean dataset reproduced exactly",
    True,
    zero_percent_clean_reproduced_exactly,
    zero_percent_clean_reproduced_exactly,
)

add_check(
    validation_records,
    "Registry rows",
    10,
    len(registry),
    len(registry) == 10,
)

add_check(
    validation_records,
    "Project 11 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)

validation = pd.DataFrame(
    validation_records
)

failed_validation = validation.loc[
    ~validation["Pass"]
]

print("\nProject 11 Step 2B validation:")
display(validation)

print("\nClean REC feature comparison:")
display(comparison_summary)

print("\nClean-anchor validation:")
display(anchor_validation)

if not failed_validation.empty:
    print("\nFailed Step 2B checks:")
    display(failed_validation)

    print("\nNo Step 2B checkpoint or PASS status was written.")

    raise RuntimeError(
        "PROJECT 11 STEP 2B VALIDATION FAILED. "
        "Do not start the experiment."
    )


# --------------------------------------------------------------------------------------------------
# 11. FREEZE CLEAN RECONSTRUCTION AND ANCHOR
# --------------------------------------------------------------------------------------------------

PREFLIGHT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_csv(
    UNMATCHED_MAPPING_AUDIT_PATH,
    unmatched_mapping_audit,
)

atomic_parquet(
    CLEAN_RECONSTRUCTED_PATH,
    clean_reconstructed[
        [
            "Build",
            "Test",
        ]
        + REC_FEATURES
    ],
)

atomic_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH,
    anchor_offsets[
        [
            "Build",
            "Test",
        ]
        + REC_FEATURES
    ],
)

atomic_csv(
    CLEAN_COMPARISON_SUMMARY_PATH,
    comparison_summary,
)

atomic_csv(
    CLEAN_MISMATCH_EXAMPLES_PATH,
    mismatch_examples_frame,
)

atomic_csv(
    CLEAN_ANCHOR_VALIDATION_PATH,
    anchor_validation,
)

atomic_csv(
    STEP2B_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 12. READBACK VALIDATION
# --------------------------------------------------------------------------------------------------

reconstructed_readback = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

anchor_offsets_readback = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)

if len(reconstructed_readback) != EXPECTED_MODEL_ROWS:
    raise RuntimeError(
        "Clean reconstructed REC parquet readback failed."
    )

if len(anchor_offsets_readback) != EXPECTED_MODEL_ROWS:
    raise RuntimeError(
        "Clean anchor-offset parquet readback failed."
    )

readback_join = (
    reconstructed_readback.merge(
        anchor_offsets_readback,
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_reconstructed",
            "_offset",
        ),
    )
    .merge(
        dataset[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
    )
)

readback_mismatch_values = 0

for feature in REC_FEATURES:
    reproduced_values = (
        readback_join[
            f"{feature}_reconstructed"
        ].to_numpy(dtype=float)
        + readback_join[
            f"{feature}_offset"
        ].to_numpy(dtype=float)
    )

    original_values = readback_join[
        feature
    ].to_numpy(dtype=float)

    readback_mismatch_values += int(
        (~np.isclose(
            reproduced_values,
            original_values,
            rtol=ANCHOR_RTOL,
            atol=ANCHOR_ATOL,
        )).sum()
    )

if readback_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean anchor failed readback reproduction."
    )


# --------------------------------------------------------------------------------------------------
# 13. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()

unmapped_build_details = (
    unmatched_mapping_audit.to_dict(
        orient="records"
    )
)

report_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP2B_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "SourceRootSHA256": current_source_root_sha256,
    "SelectionCheckpoint": str(
        SELECTION_CHECKPOINT_PATH
    ),
    "SelectionCheckpointSHA256": selection_sha256,
    "Step2AStatus": str(STEP2A_STATUS_PATH),
    "Step2AStatusSHA256": sha256_file(
        STEP2A_STATUS_PATH
    ),
    "Step2AReport": str(STEP2A_REPORT_PATH),
    "Step2AReportSHA256": sha256_file(
        STEP2A_REPORT_PATH
    ),
    "RECRecentWindow": RECENT_WINDOW,
    "RECFeatures": REC_FEATURES,
    "VerdictDependentRECFeatures": (
        VERDICT_DEPENDENT_REC
    ),
    "VerdictIndependentRECFeatures": (
        VERDICT_INDEPENDENT_REC
    ),
    "FileHistoryRECFeatures": FILE_HISTORY_REC,
    "CleanAnchoredDeltaReconstruction": True,
    "SyntheticExecutionsAdded": 0,
    "ExecutionsExcluded": 0,
    "CommitTokenRows": len(commit_audit),
    "ExactCommitMatches": exact_matches,
    "UniquePrefixMatches": prefix_matches,
    "UnmatchedCommitTokens": unmatched_tokens,
    "AmbiguousCommitTokens": ambiguous_tokens,
    "UnmatchedCommitBuilds": unmatched_token_builds,
    "BuildsWithMappedEntities": len(
        builds_with_entities
    ),
    "BuildsWithoutMappedEntities": len(
        builds_without_entities
    ),
    "UnmappedBuilds": builds_without_entities,
    "MappingIncompleteBuilds": mapping_incomplete_builds,
    "UnmappedBuildDetails": unmapped_build_details,
    "BuildEntityRows": len(build_entity),
    "RawExecutionRows": len(exe_for_rec),
    "ModelReadyRows": len(dataset),
    "ReconstructedRows": len(
        clean_reconstructed
    ),
    "ReconstructionSeconds": reconstruction_seconds,
    "DirectMismatchValues": direct_mismatch_values,
    "VerdictDependentDirectMismatches": (
        verdict_dependent_direct_mismatches
    ),
    "VerdictIndependentDirectMismatches": (
        verdict_independent_direct_mismatches
    ),
    "FileHistoryDirectMismatches": (
        file_history_direct_mismatches
    ),
    "NonFileDirectMismatches": (
        non_file_direct_mismatches
    ),
    "FileMismatchesOutsideMappingIncompleteBuilds": (
        file_mismatches_outside_mapping_incomplete_build
    ),
    "UnmatchedMappingEffectConfined": (
        unmatched_mapping_effect_is_confined
    ),
    "RowsWithAnySubstantiveDirectMismatch": (
        rows_with_any_substantive_direct_mismatch
    ),
    "RowsWithAnyNonZeroAnchorOffset": (
        rows_with_any_nonzero_anchor_offset
    ),
    "NonZeroAnchorOffsetValues": (
        nonzero_anchor_offset_values
    ),
    "FailedAnchorFeatures": failed_anchor_features,
    "AnchoredMismatchValues": (
        anchored_mismatch_values
    ),
    "ZeroPercentCleanDatasetReproducedExactly": (
        zero_percent_clean_reproduced_exactly
    ),
    "CleanReconstructedREC": str(
        CLEAN_RECONSTRUCTED_PATH
    ),
    "CleanReconstructedRECSHA256": sha256_file(
        CLEAN_RECONSTRUCTED_PATH
    ),
    "CleanAnchorOffsets": str(
        CLEAN_ANCHOR_OFFSETS_PATH
    ),
    "CleanAnchorOffsetsSHA256": sha256_file(
        CLEAN_ANCHOR_OFFSETS_PATH
    ),
    "CleanComparisonSummary": str(
        CLEAN_COMPARISON_SUMMARY_PATH
    ),
    "CleanComparisonSummarySHA256": sha256_file(
        CLEAN_COMPARISON_SUMMARY_PATH
    ),
    "CleanAnchorValidation": str(
        CLEAN_ANCHOR_VALIDATION_PATH
    ),
    "CleanAnchorValidationSHA256": sha256_file(
        CLEAN_ANCHOR_VALIDATION_PATH
    ),
    "UnmatchedMappingAudit": str(
        UNMATCHED_MAPPING_AUDIT_PATH
    ),
    "UnmatchedMappingAuditSHA256": sha256_file(
        UNMATCHED_MAPPING_AUDIT_PATH
    ),
    "ValidationChecks": len(validation),
    "FailedValidationChecks": len(
        failed_validation
    ),
    "ReadbackMismatchValues": (
        readback_mismatch_values
    ),
    "CompletionRegistry": str(REGISTRY_PATH),
    "CompletionRegistrySHA256": (
        registry_sha256_before
    ),
    "RegistryModified": False,
    "Projects1To10Modified": False,
    "Project12Accessed": False,
    "NoiseInjected": False,
    "ModelsTrained": False,
}

atomic_json(
    STEP2B_REPORT_PATH,
    report_payload,
)

checkpoint_payload = {
    **report_payload,
    "CheckpointVersion": 1,
    "CheckpointType": (
        "PROJECT_11_CLEAN_REC_RECONSTRUCTION"
    ),
    "RECReconstructionFrozen": True,
    "CleanAnchorFrozen": True,
    "EvaluationCohortImmutable": True,
    "ProceedToNoisePlanAllowed": True,
}

atomic_json(
    REC_CHECKPOINT_PATH,
    checkpoint_payload,
)

status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP2B_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "SourceRootSHA256": current_source_root_sha256,
    "RECFeatures": len(REC_FEATURES),
    "RawExecutionRows": len(exe_for_rec),
    "ModelReadyRows": len(dataset),
    "ReconstructedRows": len(
        clean_reconstructed
    ),
    "UnmatchedCommitTokens": unmatched_tokens,
    "AmbiguousCommitTokens": ambiguous_tokens,
    "BuildsWithoutMappedEntities": len(
        builds_without_entities
    ),
    "UnmappedBuilds": builds_without_entities,
    "UnmatchedMappingEffectConfined": (
        unmatched_mapping_effect_is_confined
    ),
    "FailedAnchorFeatures": failed_anchor_features,
    "AnchoredMismatchValues": (
        anchored_mismatch_values
    ),
    "ZeroPercentCleanDatasetReproducedExactly": (
        zero_percent_clean_reproduced_exactly
    ),
    "Checkpoint": str(REC_CHECKPOINT_PATH),
    "CheckpointSHA256": sha256_file(
        REC_CHECKPOINT_PATH
    ),
    "RegistryModified": False,
    "Project12Accessed": False,
}

atomic_json(
    STEP2B_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 14. FINAL IMMUTABILITY CHECK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 11 Step 2B."
    )

final_source_manifest = pd.DataFrame([
    {
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(
            (
                SOURCE_DIR
                / str(row.RelativePath)
            ).stat().st_size
        ),
        "SHA256": sha256_file(
            SOURCE_DIR
            / str(row.RelativePath)
        ),
    }
    for row in current_source_manifest.itertuples(
        index=False
    )
])

if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 11 source changed during Step 2B."
    )

checkpoint_readback = load_json(
    REC_CHECKPOINT_PATH
)
status_readback = load_json(
    STEP2B_STATUS_PATH
)

if checkpoint_readback.get("Status") != STEP2B_STATUS:
    raise RuntimeError(
        "Project 11 REC checkpoint readback failed."
    )

if status_readback.get("Status") != STEP2B_STATUS:
    raise RuntimeError(
        "Project 11 Step 2B status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 15. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 132)
print("=== PROJECT 11 CELL 5 / STEP 2B RESULT ===")
print("=" * 132)

print("Project:", PROJECT_NAME)
print("Project slug:", PROJECT_SLUG)
print("Source root SHA-256:", current_source_root_sha256)

print("\nCommit/entity edge case:")
print("Commit-token rows:", len(commit_audit))
print("Exact commit matches:", exact_matches)
print("Unique-prefix matches:", prefix_matches)
print("Unmatched commit tokens:", unmatched_tokens)
print("Ambiguous commit tokens:", ambiguous_tokens)
print("Unmatched-token builds:", unmatched_token_builds)
print("Builds with mapped entities:", len(builds_with_entities))
print("Builds without mapped entities:", len(builds_without_entities))
print("Unmapped builds:", builds_without_entities)
print("Mapping-incomplete builds:", mapping_incomplete_builds)
print("Unmatched mapping effect confined:", unmatched_mapping_effect_is_confined)

print("\nClean REC reconstruction:")
print("Raw history rows:", len(exe_for_rec))
print("Requested model-ready rows:", len(dataset))
print("Reconstructed rows:", len(clean_reconstructed))
print("Duplicate reconstructed rows:", reconstructed_duplicate_rows)
print("Missing reconstructed rows:", missing_reconstructed_rows)
print("Reconstruction seconds:", reconstruction_seconds)

print("\nDirect reconstruction comparison:")
print("Direct mismatching feature values:", direct_mismatch_values)
print("Verdict-dependent direct mismatches:", verdict_dependent_direct_mismatches)
print("Verdict-independent direct mismatches:", verdict_independent_direct_mismatches)
print("File-history direct mismatches:", file_history_direct_mismatches)
print("Non-file direct mismatches:", non_file_direct_mismatches)
print("File mismatches outside mapping-incomplete builds:", file_mismatches_outside_mapping_incomplete_build)
print("Rows with any non-zero anchor offset:", rows_with_any_nonzero_anchor_offset)
print("Non-zero anchor-offset values:", nonzero_anchor_offset_values)

print("\nClean-anchor validation:")
print("Failed anchor features:", failed_anchor_features)
print("Anchored mismatching feature values:", anchored_mismatch_values)
print("Readback mismatching values:", readback_mismatch_values)
print("0% clean dataset reproduced exactly:", zero_percent_clean_reproduced_exactly)

print("\nValidation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed_validation))

print("\nFrozen REC checkpoint:")
print(REC_CHECKPOINT_PATH)
print("Checkpoint SHA-256:", sha256_file(REC_CHECKPOINT_PATH))

print("\nImmutability and isolation:")
print("Completion registry unchanged:", registry_sha256_after == registry_sha256_before)
print("Projects 1-10 modified:", 0)
print("Project 12 accessed:", False)
print("Noise injected:", False)
print("Models trained:", False)

print("\nSaved outputs:")
for output_path in [
    UNMATCHED_MAPPING_AUDIT_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    CLEAN_COMPARISON_SUMMARY_PATH,
    CLEAN_MISMATCH_EXAMPLES_PATH,
    CLEAN_ANCHOR_VALIDATION_PATH,
    STEP2B_VALIDATION_PATH,
    STEP2B_REPORT_PATH,
    REC_CHECKPOINT_PATH,
    STEP2B_STATUS_PATH,
]:
    print(output_path)

print("\nSTATUS:", STEP2B_STATUS)
print("=" * 132)

=== PROJECT 11 CELL 5 / STEP 2B: CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE ===

Unmatched mapping audit:


,BuildID,ChronologyOrder,Partition,UnmatchedCommitTokens,UnmatchedTokenList,HasMappedEntities,MappedEntityCount,RawExecutionRows,RawFailureRows,ModelReadyRows,ModelFailureRows
0,753342085,36,TRAIN,1,b7ab87321702407fcc9de7d23dfdf7e7f4be27fa,False,0,777,1,777,1
1,763391811,323,TRAIN,1,d31e051c2bb5d4bccdc54840d47f9a010a04726d,True,1,782,0,0,0



Reconstructing all 19 clean REC features.
REC reconstruction progress: 100 / 1020 tests | reconstructed rows: 11483
REC reconstruction progress: 200 / 1020 tests | reconstructed rows: 22008
REC reconstruction progress: 300 / 1020 tests | reconstructed rows: 33877
REC reconstruction progress: 400 / 1020 tests | reconstructed rows: 44042
REC reconstruction progress: 500 / 1020 tests | reconstructed rows: 55474
REC reconstruction progress: 600 / 1020 tests | reconstructed rows: 66282
REC reconstruction progress: 700 / 1020 tests | reconstructed rows: 76091
REC reconstruction progress: 800 / 1020 tests | reconstructed rows: 86553
REC reconstruction progress: 900 / 1020 tests | reconstructed rows: 89975
REC reconstruction progress: 1000 / 1020 tests | reconstructed rows: 91024
REC reconstruction progress: 1020 / 1020 tests | reconstructed rows: 91042

Clean REC reconstruction completed:
Requested rows: 91042
Reconstructed rows: 91042
Duplicate reconstructed rows: 0
Missing reconstructed ro

,Check,Expected,Actual,Pass
0,Step 1B passed,PASS_PROJECT_11_SELECTION_AND_SOURCE_FROZEN,PASS_PROJECT_11_SELECTION_AND_SOURCE_FROZEN,True
1,Step 2A passed,PASS_PROJECT_11_SOURCE_SCHEMA_AND_JOIN_STRUCTU...,PASS_PROJECT_11_SOURCE_SCHEMA_AND_JOIN_STRUCTU...,True
2,Selection checkpoint SHA-256,9d5c95b29357426def26c41b62a753c003ad3b19e6086d...,9d5c95b29357426def26c41b62a753c003ad3b19e6086d...,True
3,Source root SHA-256,3e6e0a60d58bff61690d6e2ac2bac18371414819eff601...,3e6e0a60d58bff61690d6e2ac2bac18371414819eff601...,True
4,Frozen source files,6,6,True
5,Frozen source bytes,142849042,142849042,True
6,Canonical builds,1049,1049,True
7,Training builds,786,786,True
8,Evaluation builds,263,263,True
9,Raw execution rows,833541,833541,True



Clean REC feature comparison:


,Feature,FeatureClass,FileHistoryFeature,Rows,DirectMatchingRows,DirectMismatchingRows,DirectMismatchesAtMappingIncompleteBuild,DirectMismatchesOutsideMappingIncompleteBuild,NonZeroAnchorOffsets,AnchoredMatchingRows,AnchoredMismatchingRows,MaximumAbsoluteDirectDifference,MeanAbsoluteDirectDifference,MaximumAbsoluteAnchoredDifference
0,REC_Age,VERDICT_INDEPENDENT,False,91042,91042,0,0,0,0,91042,0,0.000000e+00,0.000000e+00,0.0
1,REC_LastFailureAge,VERDICT_DEPENDENT,False,91042,91042,0,0,0,0,91042,0,0.000000e+00,0.000000e+00,0.0
2,REC_LastTransitionAge,VERDICT_DEPENDENT,False,91042,91042,0,0,0,0,91042,0,0.000000e+00,0.000000e+00,0.0
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,False,91042,91042,0,0,0,11218,91042,0,7.275958e-12,1.440969e-14,0.0
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,False,91042,91042,0,0,0,0,91042,0,0.000000e+00,0.000000e+00,0.0
5,REC_RecentFailRate,VERDICT_DEPENDENT,False,91042,91042,0,0,0,225,91042,0,5.551115e-17,1.371895e-19,0.0
6,REC_RecentAssertRate,VERDICT_DEPENDENT,False,91042,91042,0,0,0,2,91042,0,5.551115e-17,1.219462e-21,0.0
7,REC_RecentExcRate,VERDICT_DEPENDENT,False,91042,91042,0,0,0,223,91042,0,5.551115e-17,1.359701e-19,0.0
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,False,91042,91042,0,0,0,521,91042,0,5.551115e-17,3.176700e-19,0.0
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,False,91042,91042,0,0,0,12661,91042,0,7.275958e-12,2.385126e-14,0.0



Clean-anchor validation:


,Feature,FeatureClass,Rows,MatchingRows,MismatchingRows,MaximumAbsoluteAnchoredDifference,Pass
0,REC_Age,VERDICT_INDEPENDENT,91042,91042,0,0.0,True
1,REC_LastFailureAge,VERDICT_DEPENDENT,91042,91042,0,0.0,True
2,REC_LastTransitionAge,VERDICT_DEPENDENT,91042,91042,0,0.0,True
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,91042,91042,0,0.0,True
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,91042,91042,0,0.0,True
5,REC_RecentFailRate,VERDICT_DEPENDENT,91042,91042,0,0.0,True
6,REC_RecentAssertRate,VERDICT_DEPENDENT,91042,91042,0,0.0,True
7,REC_RecentExcRate,VERDICT_DEPENDENT,91042,91042,0,0.0,True
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,91042,91042,0,0.0,True
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,91042,91042,0,0.0,True




=== PROJECT 11 CELL 5 / STEP 2B RESULT ===
Project: apache@shardingsphere
Project slug: apache__shardingsphere
Source root SHA-256: 3e6e0a60d58bff61690d6e2ac2bac18371414819eff601f3ea347ecb8ad8b0b5

Commit/entity edge case:
Commit-token rows: 1945
Exact commit matches: 1943
Unique-prefix matches: 0
Unmatched commit tokens: 2
Ambiguous commit tokens: 0
Unmatched-token builds: [753342085, 763391811]
Builds with mapped entities: 1048
Builds without mapped entities: 1
Unmapped builds: [753342085]
Mapping-incomplete builds: [753342085, 763391811]
Unmatched mapping effect confined: True

Clean REC reconstruction:
Raw history rows: 833541
Requested model-ready rows: 91042
Reconstructed rows: 91042
Duplicate reconstructed rows: 0
Missing reconstructed rows: 0
Reconstruction seconds: 309.16566203100047

Direct reconstruction comparison:
Direct mismatching feature values: 0
Verdict-dependent direct mismatches: 0
Verdict-independent direct mismatches: 0
File-history direct mismatches: 0
Non-file

In [8]:
# ==================================================================================================
# PROJECT 11 — CELL 6 / STEP 3A
# DETERMINISTIC NOISE PLAN, FIXED COHORTS, NESTED MASKS, AND RNG FREEZE
#
# PROJECT:
#   apache@shardingsphere
#
# PURPOSE:
# - verify the frozen Project 11 selection, source, REC reconstruction, and clean anchor;
# - freeze the raw and model-ready training/evaluation cohorts;
# - freeze the project-specific failure-subtype distribution;
# - generate the deterministic project/seed random streams used for noise injection;
# - prove that masks are nested across noise levels for every repetition seed;
# - freeze all 270 condition coordinates and expected noisy-label hashes;
# - leave the clean evaluation partition unchanged;
# - perform no model fitting and no registry write.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 132)
print("=== PROJECT 11 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 11
PROJECT_NAME = "apache@shardingsphere"
PROJECT_SLUG = "apache__shardingsphere"
PROJECT_SHORT = "shardingsphere"

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@shardingsphere"
)

EXPECTED_SELECTION_STATUS = (
    "PASS_PROJECT_11_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_11_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

STEP3A_STATUS = (
    "PASS_PROJECT_11_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_SELECTION_SHA256 = (
    "9d5c95b29357426def26c41b62a753c003ad3b19e6086d9873f0961e39875353"
)

EXPECTED_REC_CHECKPOINT_SHA256 = (
    "a1df2018227f643616169017b5d6a35fb40431c325821d7d5acda8e40701bfa0"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "3e6e0a60d58bff61690d6e2ac2bac18371414819eff601f3ea347ecb8ad8b0b5"
)

EXPECTED_REGISTRY_SHA256 = (
    "847bdcbee16c8757fde34e9489cd3abbf0781a319644175472597265b1df2750"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 142_849_042

EXPECTED_BUILDS = 1_049
EXPECTED_TRAIN_BUILDS = 786
EXPECTED_EVAL_BUILDS = 263

EXPECTED_RAW_ROWS = 833_541
EXPECTED_RAW_TRAIN_ROWS = 609_619
EXPECTED_RAW_EVAL_ROWS = 223_922
EXPECTED_RAW_TRAIN_FAILURES = 1_193
EXPECTED_RAW_EVAL_FAILURES = 171

EXPECTED_MODEL_ROWS = 91_042
EXPECTED_MODEL_TRAIN_ROWS = 78_035
EXPECTED_MODEL_EVAL_ROWS = 13_007
EXPECTED_MODEL_TRAIN_FAILURES = 1_188
EXPECTED_MODEL_EVAL_FAILURES = 171
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 25

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(1, 31)
)

EXPECTED_CONDITIONS = (
    len(NOISE_LEVELS)
    * len(REPETITION_SEEDS)
)

EXPECTED_RNG_ROWS = (
    EXPECTED_RAW_TRAIN_ROWS
    * len(REPETITION_SEEDS)
)

RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_11_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_11_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_11_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_11_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_11_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

STEP2B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2b_status.json"
)

STEP2B_REPORT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_report.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_11_rec_reconstruction_checkpoint.json"
)

CLEAN_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

FAILURE_SUBTYPE_PROFILE_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_failure_subtype_profile.csv"
)

SEED_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_seed_manifest.csv"
)

RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_rng_manifest.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

NESTED_MASK_AUDIT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_nested_mask_audit.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

STEP3A_VALIDATION_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_validation.csv"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_11_noise_plan_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def sha256_array(
    array,
    dtype,
):
    canonical = np.asarray(
        array,
        dtype=dtype,
        order="C",
    )

    return hashlib.sha256(
        canonical.tobytes(
            order="C"
        )
    ).hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_parquet(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing or non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode(
        "utf-8"
    )

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN-STATE VALIDATION
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    STEP2B_STATUS_PATH,
    STEP2B_REPORT_PATH,
    REC_CHECKPOINT_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "exe.csv",
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 11 Step 3A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)

step2b_status = load_json(
    STEP2B_STATUS_PATH
)

step2b_report = load_json(
    STEP2B_REPORT_PATH
)

rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)


if selection_sha256 != EXPECTED_SELECTION_SHA256:
    raise RuntimeError(
        "Project 11 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_SHA256}\n"
        f"Actual:   {selection_sha256}"
    )


if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 11 REC checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_REC_CHECKPOINT_SHA256}\n"
        f"Actual:   {rec_checkpoint_sha256}"
    )


if (
    selection_checkpoint.get(
        "Status"
    ) != EXPECTED_SELECTION_STATUS
    or step1b_status.get(
        "Status"
    ) != EXPECTED_SELECTION_STATUS
):
    raise RuntimeError(
        "Project 11 selection is not frozen successfully."
    )


if (
    step2b_status.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
    or step2b_report.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
    or rec_checkpoint.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
):
    raise RuntimeError(
        "Project 11 Step 2B is not frozen successfully."
    )


if not bool(
    rec_checkpoint.get(
        "ZeroPercentCleanDatasetReproducedExactly",
        False,
    )
):
    raise RuntimeError(
        "The Project 11 REC checkpoint does not confirm "
        "exact clean-anchor reproduction."
    )


if (
    selection_checkpoint.get(
        "Project"
    ) != PROJECT_NAME
    or selection_checkpoint.get(
        "ProjectSlug"
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "The frozen Project 11 identity differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != 10
    or sorted(
        registry_project_numbers.tolist()
    ) != list(
        range(1, 11)
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–10."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–10 are not all COMPLETE_AND_FROZEN."
    )


if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 11 is unexpectedly already registered."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

current_source_bytes = int(
    current_source_manifest[
        "SizeBytes"
    ].sum()
)


if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 11 source root differs.\n"
        f"Expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Actual:   {current_source_root_sha256}"
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD CHRONOLOGY AND SOURCE TABLES
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


dataset_header = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    nrows=0,
).columns.tolist()


exe_header = pd.read_csv(
    SOURCE_DIR
    / "exe.csv",
    nrows=0,
).columns.tolist()


dataset_build_column = resolve_column(
    dataset_header,
    "Build",
    "dataset Build",
)

dataset_test_column = resolve_column(
    dataset_header,
    "Test",
    "dataset Test",
)

dataset_verdict_column = resolve_column(
    dataset_header,
    "Verdict",
    "dataset Verdict",
)


exe_test_column = resolve_column(
    exe_header,
    "test",
    "exe test",
)

exe_build_column = resolve_column(
    exe_header,
    "build",
    "exe build",
)

exe_job_column = resolve_column(
    exe_header,
    "job",
    "exe job",
)

exe_verdict_column = resolve_column(
    exe_header,
    "verdict",
    "exe verdict",
)

exe_duration_column = resolve_column(
    exe_header,
    "duration",
    "exe duration",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in dataset_header
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


predictor_columns = [
    column
    for column in dataset_header
    if column not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


dataset = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    low_memory=False,
)


dataset = dataset.rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "Verdict",
    }
)


dataset[
    "Build"
] = parse_int(
    dataset[
        "Build"
    ],
    "dataset.Build",
)

dataset[
    "Test"
] = parse_int(
    dataset[
        "Test"
    ],
    "dataset.Test",
)

dataset[
    "Verdict"
] = parse_int(
    dataset[
        "Verdict"
    ],
    "dataset.Verdict",
)


exe = pd.read_csv(
    SOURCE_DIR
    / "exe.csv",
    usecols=[
        exe_test_column,
        exe_build_column,
        exe_job_column,
        exe_verdict_column,
        exe_duration_column,
    ],
    low_memory=False,
)


exe = exe.rename(
    columns={
        exe_test_column:
            "Test",

        exe_build_column:
            "Build",

        exe_job_column:
            "Job",

        exe_verdict_column:
            "Verdict",

        exe_duration_column:
            "Duration",
    }
)


exe[
    "Build"
] = parse_int(
    exe[
        "Build"
    ],
    "exe.Build",
)

exe[
    "Test"
] = parse_int(
    exe[
        "Test"
    ],
    "exe.Test",
)

exe[
    "Verdict"
] = parse_int(
    exe[
        "Verdict"
    ],
    "exe.Verdict",
)

exe[
    "Job"
] = pd.to_numeric(
    exe[
        "Job"
    ],
    errors="coerce",
)

exe[
    "Duration"
] = pd.to_numeric(
    exe[
        "Duration"
    ],
    errors="coerce",
)


if (
    exe[
        "Job"
    ].isna().any()
    or exe[
        "Duration"
    ].isna().any()
):
    raise RuntimeError(
        "Raw execution cohort contains missing/non-numeric "
        "job or duration values."
    )


# --------------------------------------------------------------------------------------------------
# 6. FREEZE RAW AND MODEL COHORTS
# --------------------------------------------------------------------------------------------------

raw_training = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_training.insert(
    0,
    "RawTrainingRowOrder",
    np.arange(
        1,
        len(
            raw_training
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_evaluation = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_evaluation.insert(
    0,
    "RawEvaluationRowOrder",
    np.arange(
        1,
        len(
            raw_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


model_training = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_training.insert(
    0,
    "ModelTrainingRowOrder",
    np.arange(
        1,
        len(
            model_training
        )
        + 1,
        dtype=np.int64,
    ),
)


model_evaluation = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_evaluation.insert(
    0,
    "ModelEvaluationRowOrder",
    np.arange(
        1,
        len(
            model_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_training_failures = int(
    raw_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)

raw_evaluation_failures = int(
    raw_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_training_failures = int(
    model_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_evaluation_failures = int(
    model_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_failing_evaluation_builds = int(
    model_evaluation.loc[
        model_evaluation[
            "Verdict"
        ].ne(
            0
        ),
        "Build",
    ].nunique()
)


raw_training_link_source = raw_training[
    [
        "RawTrainingRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_training_link = (
    model_training[
        [
            "ModelTrainingRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_training_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelTrainingRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


raw_evaluation_link_source = raw_evaluation[
    [
        "RawEvaluationRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_evaluation_link = (
    model_evaluation[
        [
            "ModelEvaluationRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_evaluation_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelEvaluationRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


missing_model_training_links = int(
    model_training_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)

missing_model_evaluation_links = int(
    model_evaluation_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)

model_training_verdict_mismatches = int(
    model_training_link[
        "ModelVerdict"
    ].ne(
        model_training_link[
            "RawVerdict"
        ]
    ).sum()
)

model_evaluation_verdict_mismatches = int(
    model_evaluation_link[
        "ModelVerdict"
    ].ne(
        model_evaluation_link[
            "RawVerdict"
        ]
    ).sum()
)


if (
    missing_model_training_links
    or missing_model_evaluation_links
    or model_training_verdict_mismatches
    or model_evaluation_verdict_mismatches
):
    raise RuntimeError(
        "Fixed model/raw cohort linkage failed."
    )


model_training_raw_indices = (
    model_training_link[
        "RawTrainingRowOrder"
    ].astype(
        np.int64
    ).to_numpy()
    - 1
)


# --------------------------------------------------------------------------------------------------
# 7. VERIFY THE FROZEN CLEAN ANCHOR
# --------------------------------------------------------------------------------------------------

clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

clean_anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)


clean_anchor_join = (
    clean_reconstructed.merge(
        clean_anchor_offsets,
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_reconstructed",
            "_offset",
        ),
    )
    .merge(
        dataset[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
    )
)


clean_anchor_mismatch_values = 0


for feature in REC_FEATURES:
    reproduced = (
        clean_anchor_join[
            f"{feature}_reconstructed"
        ].to_numpy(
            dtype=float
        )
        + clean_anchor_join[
            f"{feature}_offset"
        ].to_numpy(
            dtype=float
        )
    )

    original = clean_anchor_join[
        feature
    ].to_numpy(
        dtype=float
    )

    clean_anchor_mismatch_values += int(
        (
            ~np.isclose(
                reproduced,
                original,
                rtol=0.0,
                atol=1e-12,
            )
        ).sum()
    )


clean_anchor_reproduced_dataset = bool(
    len(
        clean_anchor_join
    ) == EXPECTED_MODEL_ROWS
    and clean_anchor_mismatch_values == 0
)


if not clean_anchor_reproduced_dataset:
    raise RuntimeError(
        "Frozen clean anchor no longer reproduces dataset.csv exactly."
    )


# --------------------------------------------------------------------------------------------------
# 8. PROJECT-SPECIFIC FAILURE-SUBTYPE PROFILE
# --------------------------------------------------------------------------------------------------

failure_subtype_counts = (
    raw_training.loc[
        raw_training[
            "Verdict"
        ].ne(
            0
        ),
        "Verdict",
    ]
    .value_counts()
    .sort_index()
)


if failure_subtype_counts.empty:
    raise RuntimeError(
        "No clean raw training failure subtypes were found."
    )


failure_subtypes = (
    failure_subtype_counts.index.astype(
        int
    ).to_numpy(
        dtype=np.int16
    )
)


if (
    failure_subtypes.min()
    < np.iinfo(
        np.int16
    ).min
    or failure_subtypes.max()
    > np.iinfo(
        np.int16
    ).max
):
    raise RuntimeError(
        "Failure subtype values do not fit int16."
    )


failure_subtype_probabilities = (
    failure_subtype_counts.to_numpy(
        dtype=float
    )
    / failure_subtype_counts.sum()
)


failure_subtype_profile = pd.DataFrame({
    "FailureSubtype":
        failure_subtypes.astype(
            int
        ),

    "CleanTrainingRows":
        failure_subtype_counts.to_numpy(
            dtype=int
        ),

    "Probability":
        failure_subtype_probabilities,
})


# --------------------------------------------------------------------------------------------------
# 9. GENERATE THE 30 DETERMINISTIC RNG STREAMS AND 270 CONDITION PLAN
# --------------------------------------------------------------------------------------------------

NOISE_PLAN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


rng_temporary_path = RNG_MANIFEST_PATH.with_name(
    f".{RNG_MANIFEST_PATH.name}.tmp_{os.getpid()}"
)


if rng_temporary_path.exists():
    rng_temporary_path.unlink()


rng_schema = pa.schema([
    pa.field(
        "RepetitionSeed",
        pa.int16(),
    ),

    pa.field(
        "RawTrainingRowOrder",
        pa.int32(),
    ),

    pa.field(
        "FlipUniform",
        pa.float64(),
    ),

    pa.field(
        "SampledFailureSubtype",
        pa.int16(),
    ),
])


rng_writer = pq.ParquetWriter(
    rng_temporary_path,
    schema=rng_schema,
    compression="zstd",
)


seed_records = []
condition_records = []
nested_mask_records = []

clean_raw_verdict = raw_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)

clean_model_verdict = model_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)

raw_row_order_int32 = np.arange(
    1,
    len(
        raw_training
    )
    + 1,
    dtype=np.int32,
)


try:
    condition_order = 0

    for seed_order, repetition_seed in enumerate(
        REPETITION_SEEDS,
        start=1,
    ):
        flip_seed = deterministic_seed(
            repetition_seed,
            "flip_mask",
        )

        failure_subtype_seed = deterministic_seed(
            repetition_seed,
            "failure_subtype",
        )

        flip_uniform = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )

        sampled_failure_subtype = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        rng_table = pa.Table.from_arrays(
            [
                pa.array(
                    np.full(
                        len(
                            raw_training
                        ),
                        repetition_seed,
                        dtype=np.int16,
                    ),
                    type=pa.int16(),
                ),

                pa.array(
                    raw_row_order_int32,
                    type=pa.int32(),
                ),

                pa.array(
                    flip_uniform,
                    type=pa.float64(),
                ),

                pa.array(
                    sampled_failure_subtype,
                    type=pa.int16(),
                ),
            ],
            schema=rng_schema,
        )


        rng_writer.write_table(
            rng_table
        )


        regenerated_uniform = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )

        regenerated_subtype = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        uniforms_reproduced = bool(
            np.array_equal(
                flip_uniform,
                regenerated_uniform,
            )
        )

        failure_subtypes_reproduced = bool(
            np.array_equal(
                sampled_failure_subtype,
                regenerated_subtype,
            )
        )


        seed_records.append({
            "RepetitionSeed":
                repetition_seed,

            "FlipSeed":
                flip_seed,

            "FailureSubtypeSeed":
                failure_subtype_seed,

            "NoiseRows":
                len(
                    raw_training
                ),

            "FlipUniformSHA256":
                sha256_array(
                    flip_uniform,
                    "<f8",
                ),

            "SampledFailureSubtypeSHA256":
                sha256_array(
                    sampled_failure_subtype,
                    "<i2",
                ),

            "UniformsReproduced":
                uniforms_reproduced,

            "FailureSubtypesReproduced":
                failure_subtypes_reproduced,
        })


        previous_mask = None
        previous_noise = None


        for noise_order, noise_percent in enumerate(
            NOISE_LEVELS,
            start=1,
        ):
            condition_order += 1

            condition_id = (
                f"noise_{noise_percent:02d}"
                f"__seed_{repetition_seed:02d}"
            )

            flip_mask = (
                flip_uniform
                < (
                    noise_percent
                    / 100.0
                )
            )

            noisy_raw_verdict = clean_raw_verdict.copy()

            pass_to_failure_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    == 0
                )
            )

            failure_to_pass_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    != 0
                )
            )

            noisy_raw_verdict[
                pass_to_failure_mask
            ] = sampled_failure_subtype[
                pass_to_failure_mask
            ]

            noisy_raw_verdict[
                failure_to_pass_mask
            ] = 0

            noisy_model_verdict = noisy_raw_verdict[
                model_training_raw_indices
            ]

            number_flipped = int(
                flip_mask.sum()
            )

            pass_to_failure = int(
                pass_to_failure_mask.sum()
            )

            failure_to_pass = int(
                failure_to_pass_mask.sum()
            )

            noisy_raw_failures = int(
                (
                    noisy_raw_verdict
                    != 0
                ).sum()
            )

            model_label_changes = int(
                (
                    noisy_model_verdict
                    != clean_model_verdict
                ).sum()
            )

            noisy_model_failures = int(
                (
                    noisy_model_verdict
                    != 0
                ).sum()
            )

            condition_records.append({
                "ConditionOrder":
                    condition_order,

                "ConditionID":
                    condition_id,

                "SeedOrder":
                    seed_order,

                "NoiseOrderWithinSeed":
                    noise_order,

                "NoisePercent":
                    noise_percent,

                "RepetitionSeed":
                    repetition_seed,

                "FlipSeed":
                    flip_seed,

                "FailureSubtypeSeed":
                    failure_subtype_seed,

                "RawTrainingRows":
                    len(
                        raw_training
                    ),

                "NumberFlipped":
                    number_flipped,

                "RealisedNoisePercent":
                    (
                        100.0
                        * number_flipped
                        / len(
                            raw_training
                        )
                    ),

                "PassToFailure":
                    pass_to_failure,

                "FailureToPass":
                    failure_to_pass,

                "CleanRawFailures":
                    raw_training_failures,

                "NoisyRawFailures":
                    noisy_raw_failures,

                "ModelTrainingRows":
                    len(
                        model_training
                    ),

                "ModelLabelChanges":
                    model_label_changes,

                "CleanModelFailures":
                    model_training_failures,

                "NoisyModelFailures":
                    noisy_model_failures,

                "FlipMaskSHA256":
                    sha256_array(
                        flip_mask.astype(
                            np.uint8
                        ),
                        "u1",
                    ),

                "NoisyRawVerdictSHA256":
                    sha256_array(
                        noisy_raw_verdict,
                        "<i2",
                    ),

                "NoisyModelVerdictSHA256":
                    sha256_array(
                        noisy_model_verdict,
                        "<i2",
                    ),
            })


            if previous_mask is not None:
                violations = int(
                    (
                        previous_mask
                        & (
                            ~flip_mask
                        )
                    ).sum()
                )

                nested_mask_records.append({
                    "RepetitionSeed":
                        repetition_seed,

                    "LowerNoisePercent":
                        previous_noise,

                    "HigherNoisePercent":
                        noise_percent,

                    "Violations":
                        violations,

                    "Pass":
                        violations == 0,
                })


            previous_mask = flip_mask
            previous_noise = noise_percent

finally:
    rng_writer.close()


os.replace(
    rng_temporary_path,
    RNG_MANIFEST_PATH,
)


seed_manifest = pd.DataFrame(
    seed_records
)


condition_plan = pd.DataFrame(
    condition_records
)


nested_mask_audit = pd.DataFrame(
    nested_mask_records
)


nested_mask_violations = int(
    nested_mask_audit[
        "Violations"
    ].sum()
)


zero_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].eq(
        0
    )
]


zero_noise_flip_violations = int(
    zero_noise_conditions[
        "NumberFlipped"
    ].ne(
        0
    ).sum()
)


zero_noise_raw_label_violations = int(
    zero_noise_conditions[
        "NoisyRawFailures"
    ].ne(
        raw_training_failures
    ).sum()
)


zero_noise_model_label_violations = int(
    zero_noise_conditions[
        "ModelLabelChanges"
    ].ne(
        0
    ).sum()
)


positive_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].gt(
        0
    )
]


positive_noise_without_raw_changes = int(
    positive_noise_conditions[
        "NumberFlipped"
    ].le(
        0
    ).sum()
)


positive_noise_without_model_changes = int(
    positive_noise_conditions[
        "ModelLabelChanges"
    ].le(
        0
    ).sum()
)


duplicate_condition_ids = int(
    condition_plan[
        "ConditionID"
    ].duplicated(
        keep=False
    ).sum()
)


duplicate_condition_coordinates = int(
    condition_plan.duplicated(
        subset=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        keep=False,
    ).sum()
)


seed_streams_reproduced = bool(
    seed_manifest[
        [
            "UniformsReproduced",
            "FailureSubtypesReproduced",
        ]
    ].all().all()
)


# --------------------------------------------------------------------------------------------------
# 10. WRITE FROZEN COHORTS AND PLAN OUTPUTS
# --------------------------------------------------------------------------------------------------

atomic_parquet(
    RAW_TRAINING_COHORT_PATH,
    raw_training,
)

atomic_parquet(
    RAW_EVALUATION_COHORT_PATH,
    raw_evaluation,
)

atomic_parquet(
    MODEL_TRAINING_COHORT_PATH,
    model_training,
)

atomic_parquet(
    MODEL_EVALUATION_COHORT_PATH,
    model_evaluation,
)

atomic_parquet(
    MODEL_RAW_TRAIN_LINK_PATH,
    model_training_link.drop(
        columns=[
            "_merge",
        ]
    ),
)

atomic_parquet(
    MODEL_RAW_EVAL_LINK_PATH,
    model_evaluation_link.drop(
        columns=[
            "_merge",
        ]
    ),
)

atomic_csv(
    FAILURE_SUBTYPE_PROFILE_PATH,
    failure_subtype_profile,
)

atomic_csv(
    SEED_MANIFEST_PATH,
    seed_manifest,
)

atomic_csv(
    CONDITION_PLAN_PATH,
    condition_plan,
)

atomic_csv(
    NESTED_MASK_AUDIT_PATH,
    nested_mask_audit,
)


protocol_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProtocolState":
        "FROZEN",

    "Chronology":
        (
            "started_at ascending; "
            "Build ID descending for timestamp ties"
        ),

    "Split":
        {
            "Type":
                "chronological_fixed_holdout",

            "TrainingFraction":
                0.75,

            "EvaluationFraction":
                0.25,

            "TrainingBuilds":
                len(
                    training_builds
                ),

            "EvaluationBuilds":
                len(
                    evaluation_builds
                ),
        },

    "NoiseLevelsPercent":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        EXPECTED_CONDITIONS,

    "RecentExecutionWindow":
        RECENT_WINDOW,

    "NoisePartition":
        "training only",

    "EvaluationPartition":
        "clean and immutable",

    "NoiseUnit":
        "individual raw training execution verdict",

    "FlipRule":
        {
            "PassToFailure":
                (
                    "0 is replaced by a failure subtype "
                    "sampled from the clean project-specific "
                    "failure-subtype distribution"
                ),

            "FailureToPass":
                (
                    "every non-zero verdict selected by "
                    "the mask is replaced by 0"
                ),
        },

    "Randomisation":
        {
            "SeedDerivation":
                (
                    "first little-endian uint32 of "
                    "SHA-256(project|repetition_seed|stream)"
                ),

            "FlipMaskStream":
                "flip_mask",

            "FailureSubtypeStream":
                "failure_subtype",

            "NestedMasks":
                True,

            "SameSeedUsesSameStreamsAcrossNoise":
                True,
        },

    "FeatureHandling":
        {
            "VerdictDependentRECRecomputed":
                VERDICT_DEPENDENT_REC,

            "VerdictIndependentRECPreserved":
                VERDICT_INDEPENDENT_REC,

            "AllRECFeatures":
                REC_FEATURES,

            "CleanAnchorApplied":
                True,
        },

    "TrainingInstanceCohort":
        "fixed TCP-CI model-ready training rows",

    "EvaluationMetrics":
        [
            "APFDc",
            "APFD",
        ],

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "MLTechniques":
        ML_TECHNIQUES,

    "Baselines":
        BASELINES,

    "SameCorruptedHistoryUsedBy":
        ML_TECHNIQUES
        + [
            "LatestFail",
        ],

    "QTFAvgNoiseIndependent":
        True,

    "RandomConstantAcrossNoiseForSameSeedAndBuild":
        True,

    "NoRollingRetraining":
        True,

    "RankingTieBreak":
        "score, then Test ascending",
}


atomic_json(
    PROTOCOL_PATH,
    protocol_payload,
)


# --------------------------------------------------------------------------------------------------
# 11. READBACK AND REPRODUCIBILITY VALIDATION
# --------------------------------------------------------------------------------------------------

raw_training_readback = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)

raw_evaluation_readback = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)

model_training_readback = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)

model_evaluation_readback = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)

condition_plan_readback = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)

seed_manifest_readback = pd.read_csv(
    SEED_MANIFEST_PATH,
    low_memory=False,
)

nested_mask_readback = pd.read_csv(
    NESTED_MASK_AUDIT_PATH,
    low_memory=False,
)

rng_readback_rows = int(
    pq.ParquetFile(
        RNG_MANIFEST_PATH
    ).metadata.num_rows
)


nested_mask_readback_violations = int(
    nested_mask_readback[
        "Violations"
    ].sum()
)


# Reproduce every stream again from the frozen seed manifest.
stream_reproduction_failures = 0


for row in seed_manifest_readback.itertuples(
    index=False
):
    repetition_seed = int(
        row.RepetitionSeed
    )

    flip_seed = deterministic_seed(
        repetition_seed,
        "flip_mask",
    )

    subtype_seed = deterministic_seed(
        repetition_seed,
        "failure_subtype",
    )

    reproduced_uniform = np.random.default_rng(
        flip_seed
    ).random(
        EXPECTED_RAW_TRAIN_ROWS
    )

    reproduced_subtype = np.random.default_rng(
        subtype_seed
    ).choice(
        failure_subtypes,
        size=EXPECTED_RAW_TRAIN_ROWS,
        replace=True,
        p=failure_subtype_probabilities,
    ).astype(
        np.int16
    )

    if (
        int(
            row.FlipSeed
        ) != flip_seed
        or int(
            row.FailureSubtypeSeed
        ) != subtype_seed
        or str(
            row.FlipUniformSHA256
        ) != sha256_array(
            reproduced_uniform,
            "<f8",
        )
        or str(
            row.SampledFailureSubtypeSHA256
        ) != sha256_array(
            reproduced_subtype,
            "<i2",
        )
    ):
        stream_reproduction_failures += 1


# --------------------------------------------------------------------------------------------------
# 12. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 1B passed",
    EXPECTED_SELECTION_STATUS,
    step1b_status.get(
        "Status"
    ),
    step1b_status.get(
        "Status"
    ) == EXPECTED_SELECTION_STATUS,
)

add_check(
    validation_records,
    "Step 2B passed",
    EXPECTED_STEP2B_STATUS,
    step2b_status.get(
        "Status"
    ),
    step2b_status.get(
        "Status"
    ) == EXPECTED_STEP2B_STATUS,
)

add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_SHA256,
    selection_sha256,
    selection_sha256
    == EXPECTED_SELECTION_SHA256,
)

add_check(
    validation_records,
    "REC checkpoint SHA-256",
    EXPECTED_REC_CHECKPOINT_SHA256,
    rec_checkpoint_sha256,
    rec_checkpoint_sha256
    == EXPECTED_REC_CHECKPOINT_SHA256,
)

add_check(
    validation_records,
    "Clean anchor reproduced dataset",
    True,
    clean_anchor_reproduced_dataset,
    clean_anchor_reproduced_dataset,
)

add_check(
    validation_records,
    "Source files",
    EXPECTED_SOURCE_FILES,
    len(
        current_source_manifest
    ),
    len(
        current_source_manifest
    ) == EXPECTED_SOURCE_FILES,
)

add_check(
    validation_records,
    "Source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    validation_records,
    "Builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    ) == EXPECTED_BUILDS,
)

add_check(
    validation_records,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    ) == EXPECTED_TRAIN_BUILDS,
)

add_check(
    validation_records,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    ) == EXPECTED_EVAL_BUILDS,
)

add_check(
    validation_records,
    "Raw rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    ) == EXPECTED_RAW_ROWS,
)

add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    ) == EXPECTED_RAW_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    ) == EXPECTED_RAW_EVAL_ROWS,
)

add_check(
    validation_records,
    "Raw training failures",
    EXPECTED_RAW_TRAIN_FAILURES,
    raw_training_failures,
    raw_training_failures
    == EXPECTED_RAW_TRAIN_FAILURES,
)

add_check(
    validation_records,
    "Raw evaluation failures",
    EXPECTED_RAW_EVAL_FAILURES,
    raw_evaluation_failures,
    raw_evaluation_failures
    == EXPECTED_RAW_EVAL_FAILURES,
)

add_check(
    validation_records,
    "Model rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    ) == EXPECTED_MODEL_ROWS,
)

add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    ) == EXPECTED_MODEL_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    ) == EXPECTED_MODEL_EVAL_ROWS,
)

add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)

add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)

add_check(
    validation_records,
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)

add_check(
    validation_records,
    "Dataset columns",
    EXPECTED_DATASET_COLUMNS,
    len(
        dataset_header
    ),
    len(
        dataset_header
    ) == EXPECTED_DATASET_COLUMNS,
)

add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    ) == EXPECTED_PREDICTORS,
)

add_check(
    validation_records,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        ) == EXPECTED_REC_FEATURES
        and not missing_rec_features
    ),
)

add_check(
    validation_records,
    "Missing model training links",
    0,
    missing_model_training_links,
    missing_model_training_links == 0,
)

add_check(
    validation_records,
    "Missing model evaluation links",
    0,
    missing_model_evaluation_links,
    missing_model_evaluation_links == 0,
)

add_check(
    validation_records,
    "Model training verdict mismatches",
    0,
    model_training_verdict_mismatches,
    model_training_verdict_mismatches == 0,
)

add_check(
    validation_records,
    "Model evaluation verdict mismatches",
    0,
    model_evaluation_verdict_mismatches,
    model_evaluation_verdict_mismatches == 0,
)

add_check(
    validation_records,
    "Noise levels",
    NOISE_LEVELS,
    sorted(
        condition_plan[
            "NoisePercent"
        ].unique().tolist()
    ),
    sorted(
        condition_plan[
            "NoisePercent"
        ].unique().tolist()
    ) == NOISE_LEVELS,
)

add_check(
    validation_records,
    "Repetition seeds",
    REPETITION_SEEDS,
    sorted(
        condition_plan[
            "RepetitionSeed"
        ].unique().tolist()
    ),
    sorted(
        condition_plan[
            "RepetitionSeed"
        ].unique().tolist()
    ) == REPETITION_SEEDS,
)

add_check(
    validation_records,
    "Condition rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    ) == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "Duplicate condition IDs",
    0,
    duplicate_condition_ids,
    duplicate_condition_ids == 0,
)

add_check(
    validation_records,
    "Duplicate condition coordinates",
    0,
    duplicate_condition_coordinates,
    duplicate_condition_coordinates == 0,
)

add_check(
    validation_records,
    "Nested-mask violations",
    0,
    nested_mask_violations,
    nested_mask_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise conditions",
    len(
        REPETITION_SEEDS
    ),
    len(
        zero_noise_conditions
    ),
    len(
        zero_noise_conditions
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Zero-noise flip violations",
    0,
    zero_noise_flip_violations,
    zero_noise_flip_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise raw-label violations",
    0,
    zero_noise_raw_label_violations,
    zero_noise_raw_label_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise model-label violations",
    0,
    zero_noise_model_label_violations,
    zero_noise_model_label_violations == 0,
)

add_check(
    validation_records,
    "Positive-noise conditions without raw changes",
    0,
    positive_noise_without_raw_changes,
    positive_noise_without_raw_changes == 0,
)

add_check(
    validation_records,
    "Positive-noise conditions without model changes",
    0,
    positive_noise_without_model_changes,
    positive_noise_without_model_changes == 0,
)

add_check(
    validation_records,
    "RNG-manifest rows",
    EXPECTED_RNG_ROWS,
    rng_readback_rows,
    rng_readback_rows == EXPECTED_RNG_ROWS,
)

add_check(
    validation_records,
    "Seed-manifest rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest
    ),
    len(
        seed_manifest
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Seed streams reproduced",
    True,
    (
        seed_streams_reproduced
        and stream_reproduction_failures == 0
    ),
    (
        seed_streams_reproduced
        and stream_reproduction_failures == 0
    ),
)

add_check(
    validation_records,
    "Raw-training readback rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training_readback
    ),
    len(
        raw_training_readback
    ) == EXPECTED_RAW_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Raw-evaluation readback rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation_readback
    ),
    len(
        raw_evaluation_readback
    ) == EXPECTED_RAW_EVAL_ROWS,
)

add_check(
    validation_records,
    "Model-training readback rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training_readback
    ),
    len(
        model_training_readback
    ) == EXPECTED_MODEL_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Model-evaluation readback rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation_readback
    ),
    len(
        model_evaluation_readback
    ) == EXPECTED_MODEL_EVAL_ROWS,
)

add_check(
    validation_records,
    "Condition-plan readback rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan_readback
    ),
    len(
        condition_plan_readback
    ) == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "Seed-manifest readback rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest_readback
    ),
    len(
        seed_manifest_readback
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Nested-mask readback violations",
    0,
    nested_mask_readback_violations,
    nested_mask_readback_violations == 0,
)

add_check(
    validation_records,
    "Registry rows",
    10,
    len(
        registry
    ),
    len(
        registry
    ) == 10,
)

add_check(
    validation_records,
    "Project 11 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print("\nProject 11 Step 3A validation:")

display(
    validation
)


if not failed_validation.empty:
    print("\nFailed Step 3A checks:")

    display(
        failed_validation
    )

    print(
        "\nNo Step 3A checkpoint or PASS status was written."
    )

    raise RuntimeError(
        "PROJECT 11 STEP 3A VALIDATION FAILED."
    )


atomic_csv(
    STEP3A_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 13. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    FAILURE_SUBTYPE_PROFILE_PATH,
    SEED_MANIFEST_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NESTED_MASK_AUDIT_PATH,
    PROTOCOL_PATH,
    STEP3A_VALIDATION_PATH,
]


output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SelectionCheckpointSHA256":
        selection_sha256,

    "RECCheckpointSHA256":
        rec_checkpoint_sha256,

    "SourceRootSHA256":
        current_source_root_sha256,

    "CleanAnchorReproducedDataset":
        clean_anchor_reproduced_dataset,

    "RawTrainingRows":
        len(
            raw_training
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation
        ),

    "RawTrainingFailures":
        raw_training_failures,

    "RawEvaluationFailures":
        raw_evaluation_failures,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "FailureSubtypes":
        failure_subtypes.astype(
            int
        ).tolist(),

    "FailureSubtypeProbabilities":
        failure_subtype_probabilities.tolist(),

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "NestedMaskViolations":
        nested_mask_violations,

    "ZeroNoiseFlipViolations":
        zero_noise_flip_violations,

    "ZeroNoiseModelLabelViolations":
        zero_noise_model_label_violations,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "OutputManifest":
        output_manifest,

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To10Modified":
        False,

    "Project12Accessed":
        False,

    "NoisePlanFrozen":
        True,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP3A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "NoisePlanCheckpoint":
        True,

    "DoNotChangeCohorts":
        True,

    "DoNotChangeRandomStreams":
        True,

    "DoNotChangeConditionCoordinates":
        True,

    "EvaluationCohortImmutable":
        True,
}


atomic_json(
    NOISE_PLAN_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "RawTrainingRows":
        len(
            raw_training
        ),

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "NestedMaskViolations":
        nested_mask_violations,

    "Checkpoint":
        str(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        sha256_file(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "RegistryModified":
        False,

    "Project12Accessed":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP3A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 14. FINAL IMMUTABILITY AND READBACK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 11 Step 3A."
    )


final_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "The frozen Project 11 source changed during Step 3A."
    )


checkpoint_readback = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP3A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP3A_STATUS:
    raise RuntimeError(
        "Project 11 noise-plan checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP3A_STATUS:
    raise RuntimeError(
        "Project 11 Step 3A status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 15. DISPLAY
# --------------------------------------------------------------------------------------------------

print("\nFailure-subtype profile:")

display(
    failure_subtype_profile
)


print("\nSeed manifest:")

display(
    seed_manifest
)


print("\nCondition-plan sample:")

display(
    pd.concat(
        [
            condition_plan.head(
                9
            ),
            condition_plan.tail(
                9
            ),
        ],
        ignore_index=True,
    )
)


print("\nNested-mask audit summary:")

display(
    nested_mask_audit.groupby(
        [
            "LowerNoisePercent",
            "HigherNoisePercent",
        ],
        as_index=False,
    ).agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        TotalViolations=(
            "Violations",
            "sum",
        ),

        AllPassed=(
            "Pass",
            "all",
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# 16. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 132)
print("=== PROJECT 11 CELL 6 / STEP 3A RESULT ===")
print("=" * 132)


print("\nProject:")

print(
    PROJECT_NAME
)


print("\nFixed cohorts:")

print(
    "Raw training rows:",
    len(
        raw_training
    ),
)

print(
    "Raw evaluation rows:",
    len(
        raw_evaluation
    ),
)

print(
    "Raw training failures:",
    raw_training_failures,
)

print(
    "Raw evaluation failures:",
    raw_evaluation_failures,
)

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Model training failures:",
    model_training_failures,
)

print(
    "Model evaluation failures:",
    model_evaluation_failures,
)

print(
    "Model failing evaluation builds:",
    model_failing_evaluation_builds,
)


print("\nNoise plan:")

print(
    "Noise levels:",
    NOISE_LEVELS,
)

print(
    "Repetition seeds:",
    len(
        REPETITION_SEEDS
    ),
)

print(
    "Conditions:",
    len(
        condition_plan
    ),
)

print(
    "RNG-manifest rows:",
    rng_readback_rows,
)

print(
    "Failure subtypes:",
    failure_subtypes.astype(
        int
    ).tolist(),
)

print(
    "Failure-subtype probabilities:",
    failure_subtype_probabilities.tolist(),
)

print(
    "Nested-mask violations:",
    nested_mask_violations,
)


print("\nZero-noise audit:")

print(
    "Zero-noise conditions:",
    len(
        zero_noise_conditions
    ),
)

print(
    "Zero-noise flip violations:",
    zero_noise_flip_violations,
)

print(
    "Zero-noise raw-label violations:",
    zero_noise_raw_label_violations,
)

print(
    "Zero-noise model-label violations:",
    zero_noise_model_label_violations,
)


print("\nImmutability and isolation:")

print(
    "Project 11 source unchanged:",
    source_root_hash(
        final_source_manifest
    ) == EXPECTED_SOURCE_ROOT_SHA256,
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–10 modified:",
    0,
)

print(
    "Project 12 accessed:",
    False,
)

print(
    "Models trained:",
    False,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print("\nNoise-plan checkpoint:")

print(
    NOISE_PLAN_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    sha256_file(
        NOISE_PLAN_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP3A_STATUS,
)

print("=" * 132)

=== PROJECT 11 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===

Project 11 Step 3A validation:


,Check,Expected,Actual,Pass
0,Step 1B passed,PASS_PROJECT_11_SELECTION_AND_SOURCE_FROZEN,PASS_PROJECT_11_SELECTION_AND_SOURCE_FROZEN,True
1,Step 2B passed,PASS_PROJECT_11_CLEAN_REC_RECONSTRUCTION_AND_A...,PASS_PROJECT_11_CLEAN_REC_RECONSTRUCTION_AND_A...,True
2,Selection checkpoint SHA-256,9d5c95b29357426def26c41b62a753c003ad3b19e6086d...,9d5c95b29357426def26c41b62a753c003ad3b19e6086d...,True
3,REC checkpoint SHA-256,a1df2018227f643616169017b5d6a35fb40431c325821d...,a1df2018227f643616169017b5d6a35fb40431c325821d...,True
4,Clean anchor reproduced dataset,True,True,True
5,Source files,6,6,True
6,Source bytes,142849042,142849042,True
7,Source root SHA-256,3e6e0a60d58bff61690d6e2ac2bac18371414819eff601...,3e6e0a60d58bff61690d6e2ac2bac18371414819eff601...,True
8,Builds,1049,1049,True
9,Training builds,786,786,True



Failure-subtype profile:


,FailureSubtype,CleanTrainingRows,Probability
0,1,1176,0.98575
1,2,17,0.01425



Seed manifest:


...


Condition-plan sample:


,ConditionOrder,ConditionID,SeedOrder,NoiseOrderWithinSeed,NoisePercent,RepetitionSeed,FlipSeed,FailureSubtypeSeed,RawTrainingRows,NumberFlipped,...,FailureToPass,CleanRawFailures,NoisyRawFailures,ModelTrainingRows,ModelLabelChanges,CleanModelFailures,NoisyModelFailures,FlipMaskSHA256,NoisyRawVerdictSHA256,NoisyModelVerdictSHA256
0,1,noise_00__seed_01,1,1,0,1,2939132897,3593206104,609619,0,...,0,1193,1193,78035,0,1188,1188,272c9041c79034bed3cefc49bcef4aa99a55f63ab0f2f8...,7fb8478668443627cc1670704418618af63a313e4c7ed8...,829a5a8fb4ec23447355ef1df6f627bab1601dfb31869d...
1,2,noise_05__seed_01,1,2,5,1,2939132897,3593206104,609619,30495,...,53,1193,31582,78035,4015,1188,5097,9a6b5ac03395e2fa47b5e9abadd97a8f429af1eaf97765...,b04e9a10a9d81041fbd5e886e6db976d16b554f7621a48...,b53b0a28364be88aedda7f09a82bf17f82e9148f5e3a44...
2,3,noise_10__seed_01,1,3,10,1,2939132897,3593206104,609619,60918,...,114,1193,61883,78035,7957,1188,8917,e71f69f1cf64f70eaf82f9d3261cd27d48a75ea43067d1...,159e68cc185473f5faf7c1cffd0be09bca8785f6ff7281...,b1d70226cf8a7a885a53195cea40d498aa2b0ea6a4c4e8...
3,4,noise_15__seed_01,1,4,15,1,2939132897,3593206104,609619,91440,...,182,1193,92269,78035,11818,1188,12642,224825a473442ee2f53197f98e2a61cfc8b628401c0367...,63755543aa06440087ee311a9bc51de0fced99b457a1b0...,584e043af064121fb79da76a01c9cefa271c27d3921877...
4,5,noise_20__seed_01,1,5,20,1,2939132897,3593206104,609619,121924,...,246,1193,122625,78035,15671,1188,16369,6f63d1fe70052e1b40a4c8c26fbb9aa1b13a2c18196c40...,e2f399dd393a199726d3ebc294c0ac489c0a26f8bc1a55...,8bc9d9711113abcade59055d40a8c33868fe067c3b47e9...
5,6,noise_25__seed_01,1,6,25,1,2939132897,3593206104,609619,152313,...,302,1193,152902,78035,19554,1188,20140,d2837e664373ef652793952eb92b8baa1a24d1f8d13379...,2aa456b9a5e97eef6b386819fac50fa40a137aebd197b1...,11bd788b1e4e82026f39a158b3aa9290f629e04568801c...
6,7,noise_30__seed_01,1,7,30,1,2939132897,3593206104,609619,182621,...,363,1193,183088,78035,23397,1188,23863,da0c16231e59cc566b30dfe6923768e8d17a5bb37a1a5f...,dc6bb6c0e4805a8bbda654151c54e1f118ff7f5caddc18...,00474e181ee606919f86028c889edb345bd4aca836fa90...
7,8,noise_40__seed_01,1,8,40,1,2939132897,3593206104,609619,243119,...,473,1193,243366,78035,31140,1188,31386,70f8f25c26ed5899674bf0bd34d93b11780097d46b565d...,0d1a6e7b9a7fcf2390fc553f65a26b632bb17ca4f8d884...,1e789ff26eab54e2eb44aa51121c5aff29463443fbb8bc...
8,9,noise_50__seed_01,1,9,50,1,2939132897,3593206104,609619,304249,...,590,1193,304262,78035,39007,1188,39021,4a988b20e28e05b5ed0773a41bb14597213686f57df629...,e6a4bd35d91f212f7accf50e5a5be4ed08d54d410e2dde...,55d038cac8b47d42842ab047edf64eff059bb0daee3c3d...
9,262,noise_00__seed_30,30,1,0,30,3429790597,2484846198,609619,0,...,0,1193,1193,78035,0,1188,1188,272c9041c79034bed3cefc49bcef4aa99a55f63ab0f2f8...,7fb8478668443627cc1670704418618af63a313e4c7ed8...,829a5a8fb4ec23447355ef1df6f627bab1601dfb31869d...



Nested-mask audit summary:


,LowerNoisePercent,HigherNoisePercent,Seeds,TotalViolations,AllPassed
0,0,5,30,0,True
1,5,10,30,0,True
2,10,15,30,0,True
3,15,20,30,0,True
4,20,25,30,0,True
5,25,30,30,0,True
6,30,40,30,0,True
7,40,50,30,0,True




=== PROJECT 11 CELL 6 / STEP 3A RESULT ===

Project:
apache@shardingsphere

Fixed cohorts:
Raw training rows: 609619
Raw evaluation rows: 223922
Raw training failures: 1193
Raw evaluation failures: 171
Model training rows: 78035
Model evaluation rows: 13007
Model training failures: 1188
Model evaluation failures: 171
Model failing evaluation builds: 25

Noise plan:
Noise levels: [0, 5, 10, 15, 20, 25, 30, 40, 50]
Repetition seeds: 30
Conditions: 270
RNG-manifest rows: 18288570
Failure subtypes: [1, 2]
Failure-subtype probabilities: [0.9857502095557418, 0.014249790444258172]
Nested-mask violations: 0

Zero-noise audit:
Zero-noise conditions: 30
Zero-noise flip violations: 0
Zero-noise raw-label violations: 0
Zero-noise model-label violations: 0

Immutability and isolation:
Project 11 source unchanged: True
Completion registry unchanged: True
Projects 1–10 modified: 0
Project 12 accessed: False
Models trained: False

Validation:
Checks: 53
Failed checks: 0

Noise-plan checkpoint:
/cont

In [9]:
# ==================================================================================================
# PROJECT 11 — CELL 7 / STEP 4A
# EXPERIMENT RUNTIME, MODEL, BASELINE, METRIC, AND PREDICTOR CONTRACT FREEZE
#
# PROJECT:
#   apache@shardingsphere
#
# PURPOSE:
# - verify the frozen Step 3A noise plan and every file embedded in its output manifest;
# - validate the 151-predictor fixed training/evaluation matrices;
# - freeze median-imputation, binary-label, ranking, model, baseline, APFD, and APFDc contracts;
# - validate all four required ML implementations without fitting Project 11 models;
# - freeze deterministic model/random ranking seed rules;
# - write the runtime-contract checkpoint required before the two-condition end-to-end smoke test.
#
# SAFETY:
# - does not modify the completion registry;
# - does not modify Projects 1–10;
# - does not access or write Project 12;
# - does not generate full experiment-condition outputs;
# - does not fit Project 11 models.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from IPython.display import display

import hashlib
import json
import os
import platform
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


print("=" * 132)
print("=== PROJECT 11 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 11
PROJECT_NAME = "apache@shardingsphere"
PROJECT_SLUG = "apache__shardingsphere"
PROJECT_SHORT = "shardingsphere"

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_11_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

STEP4A_STATUS = (
    "PASS_PROJECT_11_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)

EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "086da91a6e4ab96fce51fb9749ce3581eeb2c303668995877de88ec4f5c2d3fa"
)

EXPECTED_REGISTRY_SHA256 = (
    "847bdcbee16c8757fde34e9489cd3abbf0781a319644175472597265b1df2750"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "3e6e0a60d58bff61690d6e2ac2bac18371414819eff601f3ea347ecb8ad8b0b5"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_RAW_TRAIN_ROWS = 609_619
EXPECTED_RAW_EVAL_ROWS = 223_922
EXPECTED_MODEL_TRAIN_ROWS = 78_035
EXPECTED_MODEL_EVAL_ROWS = 13_007
EXPECTED_MODEL_TRAIN_FAILURES = 1_188
EXPECTED_MODEL_EVAL_FAILURES = 171
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 25

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_CONDITIONS = 270

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators":
            100,

        "max_features":
            "sqrt",

        "bootstrap":
            True,

        "n_jobs":
            -1,
    },

    "XGBoost": {
        "n_estimators":
            100,

        "max_depth":
            6,

        "learning_rate":
            0.1,

        "tree_method":
            "hist",

        "n_jobs":
            -1,

        "verbosity":
            0,

        "eval_metric":
            "logloss",
    },

    "LightGBM": {
        "n_estimators":
            100,

        "learning_rate":
            0.1,

        "num_leaves":
            31,

        "n_jobs":
            -1,

        "verbosity":
            -1,

        "deterministic":
            True,

        "force_col_wise":
            True,
    },

    "NaiveBayes": {
        "var_smoothing":
            1e-9,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_11_noise_plan_checkpoint.json"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

RUNTIME_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_runtime_contract"
)

RUNTIME_VERSION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_runtime_versions.csv"
)

PREDICTOR_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_predictor_contract.csv"
)

CLEAN_MEDIAN_REFERENCE_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_clean_training_median_reference.csv"
)

MODEL_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_model_contract.json"
)

BASELINE_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_baseline_contract.json"
)

METRIC_SELF_TEST_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_metric_self_test.csv"
)

RANKING_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_ranking_contract.json"
)

STEP4A_VALIDATION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_validation.csv"
)

STEP4A_REPORT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_report.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4a_status.json"
)

RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_11_runtime_contract_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(
    path,
):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(
            column
        ).strip().lower()
        == str(
            expected
        ).strip().lower()
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode(
        "utf-8"
    )

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def deterministic_random_build_seed(
    repetition_seed,
    build_id,
):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )


def create_models(
    repetition_seed,
):
    rf_seed = deterministic_seed(
        repetition_seed,
        "RandomForest_model",
    )

    xgb_seed = deterministic_seed(
        repetition_seed,
        "XGBoost_model",
    )

    lgbm_seed = deterministic_seed(
        repetition_seed,
        "LightGBM_model",
    )

    models = {
        "RandomForest":
            RandomForestClassifier(
                **MODEL_CONFIG[
                    "RandomForest"
                ],
                random_state=rf_seed,
            ),

        "XGBoost":
            XGBClassifier(
                **MODEL_CONFIG[
                    "XGBoost"
                ],
                random_state=xgb_seed,
            ),

        "LightGBM":
            LGBMClassifier(
                **MODEL_CONFIG[
                    "LightGBM"
                ],
                random_state=lgbm_seed,
            ),

        "NaiveBayes":
            GaussianNB(
                **MODEL_CONFIG[
                    "NaiveBayes"
                ]
            ),
    }

    return models


def calculate_apfd(
    failures,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    number_of_tests = len(
        failures
    )

    number_of_failures = int(
        failures.sum()
    )

    if (
        number_of_tests == 0
        or number_of_failures == 0
    ):
        return np.nan

    failure_positions = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    value = (
        1.0
        - (
            failure_positions.sum()
            / (
                number_of_tests
                * number_of_failures
            )
        )
        + (
            1.0
            / (
                2.0
                * number_of_tests
            )
        )
    )

    return float(
        value
    )


def calculate_apfdc(
    failures,
    durations,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(
        failures
    ) != len(
        durations
    ):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if (
        len(
            failures
        ) == 0
        or failures.sum() == 0
    ):
        return np.nan

    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (
        durations
        < 0
    ).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([
            0.0,
        ]),
        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + (
            0.5
            * durations[
                failure_mask
            ]
        )
    )

    value = (
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )

    return float(
        value
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED FILES AND STEP 3A CHECKPOINT
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    STEP3A_STATUS_PATH,
    STEP3A_REPORT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    CONDITION_PLAN_PATH,
    PROTOCOL_PATH,
]

missing_paths = [
    str(
        path
    )
    for path in required_paths
    if not Path(
        path
    ).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 11 Step 4A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


noise_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)

noise_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

step3a_status = load_json(
    STEP3A_STATUS_PATH
)

step3a_report = load_json(
    STEP3A_REPORT_PATH
)

protocol = load_json(
    PROTOCOL_PATH
)


if (
    noise_checkpoint_sha256
    != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 11 noise-plan checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256}\n"
        f"Actual:   {noise_checkpoint_sha256}"
    )


for label, payload in [
    (
        "noise checkpoint",
        noise_checkpoint,
    ),
    (
        "Step 3A status",
        step3a_status,
    ),
    (
        "Step 3A report",
        step3a_report,
    ),
]:
    if payload.get(
        "Status"
    ) != EXPECTED_STEP3A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the expected Step 3A PASS status."
        )


if (
    noise_checkpoint.get(
        "Project"
    ) != PROJECT_NAME
    or noise_checkpoint.get(
        "ProjectSlug"
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "The frozen Step 3A Project 11 identity differs."
    )


if (
    noise_checkpoint.get(
        "SourceRootSHA256"
    ) != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The Step 3A checkpoint source root differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY EVERY STEP 3A OUTPUT-MANIFEST FILE
# --------------------------------------------------------------------------------------------------

output_manifest = noise_checkpoint.get(
    "OutputManifest",
    []
)


if not isinstance(
    output_manifest,
    list,
) or not output_manifest:
    raise RuntimeError(
        "The Step 3A checkpoint does not contain an output manifest."
    )


output_manifest_checks = []


for item in output_manifest:
    path = Path(
        item[
            "Path"
        ]
    )

    expected_bytes = int(
        item[
            "Bytes"
        ]
    )

    expected_sha256 = str(
        item[
            "SHA256"
        ]
    )

    exists = path.is_file()

    actual_bytes = (
        int(
            path.stat().st_size
        )
        if exists
        else -1
    )

    actual_sha256 = (
        sha256_file(
            path
        )
        if exists
        else ""
    )

    output_manifest_checks.append({
        "Path":
            str(
                path
            ),

        "ExpectedBytes":
            expected_bytes,

        "ActualBytes":
            actual_bytes,

        "ExpectedSHA256":
            expected_sha256,

        "ActualSHA256":
            actual_sha256,

        "Pass":
            (
                exists
                and actual_bytes == expected_bytes
                and actual_sha256 == expected_sha256
            ),
    })


output_manifest_audit = pd.DataFrame(
    output_manifest_checks
)


output_manifest_failures = int(
    (
        ~output_manifest_audit[
            "Pass"
        ]
    ).sum()
)


if output_manifest_failures:
    print(
        "\nFailed Step 3A output-manifest checks:"
    )

    display(
        output_manifest_audit.loc[
            ~output_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen Step 3A outputs changed."
    )


# --------------------------------------------------------------------------------------------------
# 6. REGISTRY IMMUTABILITY
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    ) != 10
    or sorted(
        project_numbers.tolist()
    ) != list(
        range(
            1,
            11,
        )
    )
):
    raise RuntimeError(
        "The registry does not contain exactly Projects 1–10."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–10 are not all COMPLETE_AND_FROZEN."
    )


if project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 11 is unexpectedly already registered."
    )


# --------------------------------------------------------------------------------------------------
# 7. LOAD FIXED COHORTS
# --------------------------------------------------------------------------------------------------

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)

raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)

model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)

model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)

model_raw_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)

model_raw_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)

condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)


required_model_columns = {
    "Build",
    "Test",
    "Verdict",
}


if not required_model_columns.issubset(
    model_training.columns
) or not required_model_columns.issubset(
    model_evaluation.columns
):
    raise RuntimeError(
        "The fixed model cohorts are missing Build, Test, or Verdict."
    )


metadata_columns = {
    "ModelTrainingRowOrder",
    "ModelEvaluationRowOrder",
    "Build",
    "Test",
    "Verdict",
}


predictor_columns = [
    column
    for column in model_training.columns
    if column not in metadata_columns
]


if set(
    predictor_columns
) != (
    set(
        model_evaluation.columns
    )
    - metadata_columns
):
    raise RuntimeError(
        "Training and evaluation predictor sets differ."
    )


if len(
    predictor_columns
) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "The fixed predictor count differs.\n"
        f"Expected: {EXPECTED_PREDICTORS}\n"
        f"Actual:   {len(predictor_columns)}"
    )


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in predictor_columns
]


if missing_rec_features:
    raise RuntimeError(
        "The fixed predictor cohort is missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


# --------------------------------------------------------------------------------------------------
# 8. NUMERIC PREDICTOR AND MEDIAN-IMPUTATION CONTRACT
# --------------------------------------------------------------------------------------------------

training_numeric = pd.DataFrame(
    index=model_training.index
)

evaluation_numeric = pd.DataFrame(
    index=model_evaluation.index
)

predictor_profile_records = []


for predictor_order, column in enumerate(
    predictor_columns,
    start=1,
):
    training_values = pd.to_numeric(
        model_training[
            column
        ],
        errors="coerce",
    ).astype(
        float
    )

    evaluation_values = pd.to_numeric(
        model_evaluation[
            column
        ],
        errors="coerce",
    ).astype(
        float
    )

    training_values = training_values.replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    evaluation_values = evaluation_values.replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    training_numeric[
        column
    ] = training_values

    evaluation_numeric[
        column
    ] = evaluation_values

    non_missing_training = int(
        training_values.notna().sum()
    )

    training_missing = int(
        training_values.isna().sum()
    )

    evaluation_missing = int(
        evaluation_values.isna().sum()
    )

    predictor_profile_records.append({
        "PredictorOrder":
            predictor_order,

        "Predictor":
            column,

        "IsREC":
            column in REC_FEATURES,

        "RECClass":
            (
                "VERDICT_DEPENDENT"
                if column in VERDICT_DEPENDENT_REC
                else (
                    "VERDICT_INDEPENDENT"
                    if column in VERDICT_INDEPENDENT_REC
                    else ""
                )
            ),

        "TrainingRows":
            len(
                training_values
            ),

        "TrainingNonMissing":
            non_missing_training,

        "TrainingMissing":
            training_missing,

        "EvaluationRows":
            len(
                evaluation_values
            ),

        "EvaluationMissing":
            evaluation_missing,

        "AllTrainingValuesMissing":
            non_missing_training == 0,
    })


predictor_contract = pd.DataFrame(
    predictor_profile_records
)


all_missing_predictors = predictor_contract.loc[
    predictor_contract[
        "AllTrainingValuesMissing"
    ],
    "Predictor",
].tolist()


if all_missing_predictors:
    raise RuntimeError(
        "One or more predictors are entirely missing in training:\n"
        + "\n".join(
            all_missing_predictors
        )
    )


clean_training_medians = training_numeric.median(
    axis=0,
    skipna=True,
)


if (
    clean_training_medians.isna().any()
    or not np.isfinite(
        clean_training_medians.to_numpy(
            dtype=float
        )
    ).all()
):
    raise RuntimeError(
        "Clean training medians contain missing or infinite values."
    )


training_imputed = training_numeric.fillna(
    clean_training_medians
)

evaluation_imputed = evaluation_numeric.fillna(
    clean_training_medians
)


training_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            training_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


evaluation_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            evaluation_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


clean_median_reference = pd.DataFrame({
    "PredictorOrder":
        np.arange(
            1,
            len(
                predictor_columns
            )
            + 1,
            dtype=int,
        ),

    "Predictor":
        predictor_columns,

    "CleanTrainingMedian":
        clean_training_medians[
            predictor_columns
        ].to_numpy(
            dtype=float
        ),
})


# --------------------------------------------------------------------------------------------------
# 9. LABEL AND EVALUATION-COHORT CONTRACT
# --------------------------------------------------------------------------------------------------

training_binary_labels = (
    pd.to_numeric(
        model_training[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


evaluation_binary_labels = (
    pd.to_numeric(
        model_evaluation[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


training_label_values = sorted(
    training_binary_labels.unique().tolist()
)


evaluation_label_values = sorted(
    evaluation_binary_labels.unique().tolist()
)


model_failing_evaluation_builds = int(
    model_evaluation.loc[
        evaluation_binary_labels.eq(
            1
        ),
        "Build",
    ].nunique()
)


train_link_row_count = len(
    model_raw_train_link
)

eval_link_row_count = len(
    model_raw_eval_link
)


# --------------------------------------------------------------------------------------------------
# 10. MODEL IMPLEMENTATION CONTRACT
# --------------------------------------------------------------------------------------------------

models_seed_1 = create_models(
    repetition_seed=1
)

models_seed_1_repeat = create_models(
    repetition_seed=1
)

models_seed_2 = create_models(
    repetition_seed=2
)


model_class_records = []


for technique in ML_TECHNIQUES:
    model_a = models_seed_1[
        technique
    ]

    model_b = models_seed_1_repeat[
        technique
    ]

    model_c = models_seed_2[
        technique
    ]

    parameters_a = model_a.get_params(
        deep=False
    )

    parameters_b = model_b.get_params(
        deep=False
    )

    parameters_c = model_c.get_params(
        deep=False
    )

    random_state_a = parameters_a.get(
        "random_state",
        None,
    )

    random_state_b = parameters_b.get(
        "random_state",
        None,
    )

    random_state_c = parameters_c.get(
        "random_state",
        None,
    )

    same_seed_same_configuration = (
        parameters_a
        == parameters_b
    )

    if technique == "NaiveBayes":
        different_seed_state_expected = True

    else:
        different_seed_state_expected = (
            random_state_a
            != random_state_c
        )

    model_class_records.append({
        "Technique":
            technique,

        "EstimatorClass":
            (
                f"{model_a.__class__.__module__}."
                f"{model_a.__class__.__name__}"
            ),

        "Seed1RandomState":
            random_state_a,

        "Seed2RandomState":
            random_state_c,

        "SameSeedSameConfiguration":
            same_seed_same_configuration,

        "DifferentSeedStateAsExpected":
            different_seed_state_expected,

        "ConfigurationJSON":
            json.dumps(
                parameters_a,
                sort_keys=True,
                default=str,
            ),
    })


model_contract_table = pd.DataFrame(
    model_class_records
)


model_contract_failures = int(
    (
        ~model_contract_table[
            "SameSeedSameConfiguration"
        ]
        | ~model_contract_table[
            "DifferentSeedStateAsExpected"
        ]
    ).sum()
)


# Explicit parameter checks.
rf_params = models_seed_1[
    "RandomForest"
].get_params(
    deep=False
)

xgb_params = models_seed_1[
    "XGBoost"
].get_params(
    deep=False
)

lgbm_params = models_seed_1[
    "LightGBM"
].get_params(
    deep=False
)

nb_params = models_seed_1[
    "NaiveBayes"
].get_params(
    deep=False
)


model_parameter_checks = {
    "RandomForest": (
        rf_params.get(
            "n_estimators"
        ) == 100
        and rf_params.get(
            "max_features"
        ) == "sqrt"
        and rf_params.get(
            "bootstrap"
        ) is True
        and rf_params.get(
            "n_jobs"
        ) == -1
    ),

    "XGBoost": (
        xgb_params.get(
            "n_estimators"
        ) == 100
        and xgb_params.get(
            "max_depth"
        ) == 6
        and np.isclose(
            float(
                xgb_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and xgb_params.get(
            "tree_method"
        ) == "hist"
        and xgb_params.get(
            "n_jobs"
        ) == -1
    ),

    "LightGBM": (
        lgbm_params.get(
            "n_estimators"
        ) == 100
        and np.isclose(
            float(
                lgbm_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and lgbm_params.get(
            "num_leaves"
        ) == 31
        and lgbm_params.get(
            "deterministic"
        ) is True
        and lgbm_params.get(
            "force_col_wise"
        ) is True
        and lgbm_params.get(
            "n_jobs"
        ) == -1
    ),

    "NaiveBayes": (
        np.isclose(
            float(
                nb_params.get(
                    "var_smoothing"
                )
            ),
            1e-9,
        )
    ),
}


# --------------------------------------------------------------------------------------------------
# 11. BASELINE AND RANKING CONTRACT
# --------------------------------------------------------------------------------------------------

sample_build_id = int(
    model_evaluation[
        "Build"
    ].iloc[
        0
    ]
)


random_seed_1_a = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_1_b = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_2 = deterministic_random_build_seed(
    repetition_seed=2,
    build_id=sample_build_id,
)


sample_random_a = np.random.default_rng(
    random_seed_1_a
).random(
    100
)

sample_random_b = np.random.default_rng(
    random_seed_1_b
).random(
    100
)

sample_random_c = np.random.default_rng(
    random_seed_2
).random(
    100
)


random_same_seed_reproduced = bool(
    np.array_equal(
        sample_random_a,
        sample_random_b,
    )
)


random_different_seed_differs = bool(
    not np.array_equal(
        sample_random_a,
        sample_random_c,
    )
)


ranking_contract = {
    "ML":
        {
            "Direction":
                "descending",

            "TieBreak":
                "Test ascending",
        },

    "Random":
        {
            "Direction":
                "descending",

            "TieBreak":
                "Test ascending",

            "SeedRule":
                (
                    "first little-endian uint32 of "
                    "SHA-256(project|repetition_seed|"
                    "Random_baseline_build_<BuildID>)"
                ),

            "ConstantAcrossNoiseForSameSeedAndBuild":
                True,
        },

    "LatestFail":
        {
            "SourceFeature":
                "REC_LastFailureAge",

            "ScoreFormula":
                "-REC_LastFailureAge",

            "Direction":
                "descending",

            "TieBreak":
                "Test ascending",

            "NoiseDependent":
                True,

            "UsesSameCorruptedHistoryAsML":
                True,
        },

    "QTF-Avg":
        {
            "SourceFeature":
                "REC_TotalAvgExeTime",

            "Direction":
                "ascending",

            "TieBreak":
                "Test ascending",

            "NoiseDependent":
                False,
        },
}


baseline_contract = {
    "Techniques":
        BASELINE_TECHNIQUES,

    "Random":
        ranking_contract[
            "Random"
        ],

    "LatestFail":
        ranking_contract[
            "LatestFail"
        ],

    "QTF-Avg":
        ranking_contract[
            "QTF-Avg"
        ],

    "NoRollingRetraining":
        True,

    "CleanEvaluationPartition":
        True,
}


if (
    ranking_contract[
        "LatestFail"
    ][
        "SourceFeature"
    ]
    not in predictor_columns
):
    raise RuntimeError(
        "REC_LastFailureAge is missing from the predictor cohort."
    )


if (
    ranking_contract[
        "QTF-Avg"
    ][
        "SourceFeature"
    ]
    not in predictor_columns
):
    raise RuntimeError(
        "REC_TotalAvgExeTime is missing from the predictor cohort."
    )


# --------------------------------------------------------------------------------------------------
# 12. APFD/APFDC SELF-TESTS
# --------------------------------------------------------------------------------------------------

manual_failures = np.array([
    1,
    1,
    0,
    0,
    0,
], dtype=np.int8)


manual_apfd = calculate_apfd(
    manual_failures
)


manual_apfdc_slow_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        5.0,
        1.0,
        1.0,
        1.0,
        1.0,
    ]),
)


manual_apfdc_fast_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        1.0,
        5.0,
        1.0,
        1.0,
        1.0,
    ]),
)


all_pass_apfd = calculate_apfd(
    np.array([
        0,
        0,
        0,
    ])
)


all_pass_apfdc = calculate_apfdc(
    np.array([
        0,
        0,
        0,
    ]),
    np.array([
        1.0,
        1.0,
        1.0,
    ]),
)


metric_self_test = pd.DataFrame([
    {
        "Check":
            "Manual APFD",

        "Expected":
            0.8,

        "Actual":
            manual_apfd,

        "Pass":
            np.isclose(
                manual_apfd,
                0.8,
                rtol=0,
                atol=1e-15,
            ),
    },

    {
        "Check":
            "APFDc rewards quick failing test first",

        "Expected":
            True,

        "Actual":
            (
                manual_apfdc_fast_failure_first
                > manual_apfdc_slow_failure_first
            ),

        "Pass":
            (
                manual_apfdc_fast_failure_first
                > manual_apfdc_slow_failure_first
            ),
    },

    {
        "Check":
            "All-pass APFD is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),
    },

    {
        "Check":
            "All-pass APFDc is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),
    },
])


metric_self_test_failures = int(
    (
        ~metric_self_test[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 13. RUNTIME VERSION FREEZE
# --------------------------------------------------------------------------------------------------

runtime_versions = pd.DataFrame([
    {
        "Component":
            "Python",

        "Version":
            platform.python_version(),
    },

    {
        "Component":
            "Platform",

        "Version":
            platform.platform(),
    },

    {
        "Component":
            "NumPy",

        "Version":
            np.__version__,
    },

    {
        "Component":
            "pandas",

        "Version":
            pd.__version__,
    },

    {
        "Component":
            "scikit-learn",

        "Version":
            metadata.version(
                "scikit-learn"
            ),
    },

    {
        "Component":
            "xgboost",

        "Version":
            metadata.version(
                "xgboost"
            ),
    },

    {
        "Component":
            "lightgbm",

        "Version":
            metadata.version(
                "lightgbm"
            ),
    },

    {
        "Component":
            "pyarrow",

        "Version":
            metadata.version(
                "pyarrow"
            ),
    },
])


# --------------------------------------------------------------------------------------------------
# 14. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 3A status",
    EXPECTED_STEP3A_STATUS,
    step3a_status.get(
        "Status"
    ),
    step3a_status.get(
        "Status"
    ) == EXPECTED_STEP3A_STATUS,
)

add_check(
    validation_records,
    "Noise-plan checkpoint SHA-256",
    EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    noise_checkpoint_sha256,
    noise_checkpoint_sha256
    == EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
)

add_check(
    validation_records,
    "Step 3A output-manifest failures",
    0,
    output_manifest_failures,
    output_manifest_failures == 0,
)

add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    noise_checkpoint.get(
        "SourceRootSHA256"
    ),
    noise_checkpoint.get(
        "SourceRootSHA256"
    ) == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    ) == EXPECTED_RAW_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    ) == EXPECTED_RAW_EVAL_ROWS,
)

add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    ) == EXPECTED_MODEL_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    ) == EXPECTED_MODEL_EVAL_ROWS,
)

add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    int(
        training_binary_labels.sum()
    ),
    int(
        training_binary_labels.sum()
    ) == EXPECTED_MODEL_TRAIN_FAILURES,
)

add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    int(
        evaluation_binary_labels.sum()
    ),
    int(
        evaluation_binary_labels.sum()
    ) == EXPECTED_MODEL_EVAL_FAILURES,
)

add_check(
    validation_records,
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)

add_check(
    validation_records,
    "Model training link rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    train_link_row_count,
    train_link_row_count
    == EXPECTED_MODEL_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Model evaluation link rows",
    EXPECTED_MODEL_EVAL_ROWS,
    eval_link_row_count,
    eval_link_row_count
    == EXPECTED_MODEL_EVAL_ROWS,
)

add_check(
    validation_records,
    "Condition rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    ) == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    ) == EXPECTED_PREDICTORS,
)

add_check(
    validation_records,
    "REC feature columns",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        ) == EXPECTED_REC_FEATURES
        and not missing_rec_features
    ),
)

add_check(
    validation_records,
    "All-missing training predictors",
    0,
    len(
        all_missing_predictors
    ),
    len(
        all_missing_predictors
    ) == 0,
)

add_check(
    validation_records,
    "Non-finite training values after median imputation",
    0,
    training_nonfinite_after_imputation,
    training_nonfinite_after_imputation == 0,
)

add_check(
    validation_records,
    "Non-finite evaluation values after median imputation",
    0,
    evaluation_nonfinite_after_imputation,
    evaluation_nonfinite_after_imputation == 0,
)

add_check(
    validation_records,
    "Training binary-label values",
    [
        0,
        1,
    ],
    training_label_values,
    training_label_values == [
        0,
        1,
    ],
)

add_check(
    validation_records,
    "Evaluation binary-label values",
    [
        0,
        1,
    ],
    evaluation_label_values,
    evaluation_label_values == [
        0,
        1,
    ],
)

add_check(
    validation_records,
    "Model implementation contract failures",
    0,
    model_contract_failures,
    model_contract_failures == 0,
)

add_check(
    validation_records,
    "RandomForest parameters",
    True,
    model_parameter_checks[
        "RandomForest"
    ],
    model_parameter_checks[
        "RandomForest"
    ],
)

add_check(
    validation_records,
    "XGBoost parameters",
    True,
    model_parameter_checks[
        "XGBoost"
    ],
    model_parameter_checks[
        "XGBoost"
    ],
)

add_check(
    validation_records,
    "LightGBM parameters",
    True,
    model_parameter_checks[
        "LightGBM"
    ],
    model_parameter_checks[
        "LightGBM"
    ],
)

add_check(
    validation_records,
    "NaiveBayes parameters",
    True,
    model_parameter_checks[
        "NaiveBayes"
    ],
    model_parameter_checks[
        "NaiveBayes"
    ],
)

add_check(
    validation_records,
    "Random same-seed reproducibility",
    True,
    random_same_seed_reproduced,
    random_same_seed_reproduced,
)

add_check(
    validation_records,
    "Random different-seed differentiation",
    True,
    random_different_seed_differs,
    random_different_seed_differs,
)

add_check(
    validation_records,
    "Metric self-test failures",
    0,
    metric_self_test_failures,
    metric_self_test_failures == 0,
)

add_check(
    validation_records,
    "Technique set",
    sorted(
        ALL_TECHNIQUES
    ),
    sorted(
        ML_TECHNIQUES
        + BASELINE_TECHNIQUES
    ),
    sorted(
        ML_TECHNIQUES
        + BASELINE_TECHNIQUES
    ) == sorted(
        ALL_TECHNIQUES
    ),
)

add_check(
    validation_records,
    "Registry rows",
    10,
    len(
        registry
    ),
    len(
        registry
    ) == 10,
)

add_check(
    validation_records,
    "Project 11 registry rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print("\nProject 11 Step 4A validation:")

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 11 Step 4A checks:"
    )

    display(
        failed_validation
    )

    print(
        "\nNo Step 4A PASS status or checkpoint was written."
    )

    raise RuntimeError(
        "PROJECT 11 STEP 4A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 15. WRITE CONTRACT OUTPUTS
# --------------------------------------------------------------------------------------------------

RUNTIME_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_csv(
    RUNTIME_VERSION_PATH,
    runtime_versions,
)

atomic_csv(
    PREDICTOR_CONTRACT_PATH,
    predictor_contract,
)

atomic_csv(
    CLEAN_MEDIAN_REFERENCE_PATH,
    clean_median_reference,
)

atomic_csv(
    METRIC_SELF_TEST_PATH,
    metric_self_test,
)

atomic_csv(
    STEP4A_VALIDATION_PATH,
    validation,
)


model_contract_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "PositiveClass":
        {
            "Name":
                "failure",

            "Value":
                1,

            "Conversion":
                "binary target = (Verdict != 0).astype(int)",
        },

    "PredictorCount":
        len(
            predictor_columns
        ),

    "Imputation":
        {
            "Rule":
                (
                    "For every condition, compute one median "
                    "per active predictor from that condition's "
                    "training matrix only. Replace +/-infinity "
                    "with missing before median computation. "
                    "Fill training and clean evaluation missing "
                    "values with those training medians."
                ),

            "Scaling":
                "none",

            "ActivePredictors":
                "all 151 fixed predictor columns",
        },

    "Models":
        MODEL_CONFIG,

    "ModelSeeds":
        {
            "RandomForest":
                "SHA-256(project|repetition_seed|RandomForest_model)",

            "XGBoost":
                "SHA-256(project|repetition_seed|XGBoost_model)",

            "LightGBM":
                "SHA-256(project|repetition_seed|LightGBM_model)",

            "NaiveBayes":
                "deterministic; no random_state parameter",
        },

    "PositiveProbabilityExtraction":
        (
            "Use predict_proba and select the column whose "
            "fitted classes_ value equals 1."
        ),

    "NoRollingRetraining":
        True,

    "ModelImplementationAudit":
        model_contract_table.to_dict(
            orient="records"
        ),
}


atomic_json(
    MODEL_CONTRACT_PATH,
    model_contract_payload,
)

atomic_json(
    BASELINE_CONTRACT_PATH,
    baseline_contract,
)

atomic_json(
    RANKING_CONTRACT_PATH,
    ranking_contract,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RUNTIME_VERSION_PATH,
    PREDICTOR_CONTRACT_PATH,
    CLEAN_MEDIAN_REFERENCE_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    METRIC_SELF_TEST_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_VALIDATION_PATH,
]


runtime_output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "SourceRootSHA256":
        EXPECTED_SOURCE_ROOT_SHA256,

    "Step3AOutputManifestFiles":
        len(
            output_manifest_audit
        ),

    "Step3AOutputManifestFailures":
        output_manifest_failures,

    "RawTrainingRows":
        len(
            raw_training
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation
        ),

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "Predictors":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "TrainingFailures":
        int(
            training_binary_labels.sum()
        ),

    "EvaluationFailures":
        int(
            evaluation_binary_labels.sum()
        ),

    "FailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "ModelTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "MetricSelfTestFailures":
        metric_self_test_failures,

    "TrainingNonFiniteAfterImputation":
        training_nonfinite_after_imputation,

    "EvaluationNonFiniteAfterImputation":
        evaluation_nonfinite_after_imputation,

    "RuntimeVersions":
        runtime_versions.to_dict(
            orient="records"
        ),

    "RuntimeOutputManifest":
        runtime_output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To10Modified":
        False,

    "Project12Accessed":
        False,

    "ProjectModelsFitted":
        False,

    "FullExperimentStarted":
        False,
}


atomic_json(
    STEP4A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "RuntimeContractFrozen":
        True,

    "DoNotChangePredictorOrder":
        True,

    "DoNotChangeModelConfigurations":
        True,

    "DoNotChangeBaselineDefinitions":
        True,

    "DoNotChangeMetricDefinitions":
        True,

    "DoNotChangeRankingTieBreak":
        True,
}


atomic_json(
    RUNTIME_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "Predictors":
        len(
            predictor_columns
        ),

    "ModelTechniques":
        len(
            ML_TECHNIQUES
        ),

    "BaselineTechniques":
        len(
            BASELINE_TECHNIQUES
        ),

    "MetricSelfTestFailures":
        metric_self_test_failures,

    "Checkpoint":
        str(
            RUNTIME_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        sha256_file(
            RUNTIME_CHECKPOINT_PATH
        ),

    "RegistryModified":
        False,

    "Project12Accessed":
        False,

    "ProjectModelsFitted":
        False,
}


atomic_json(
    STEP4A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 16. FINAL READBACK AND IMMUTABILITY
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    RUNTIME_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP4A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 11 runtime checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 11 Step 4A status readback failed."
    )


registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 11 Step 4A."
    )


if sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
) != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "The frozen Project 11 noise-plan checkpoint changed during Step 4A."
    )


# --------------------------------------------------------------------------------------------------
# 17. DISPLAY
# --------------------------------------------------------------------------------------------------

print("\nRuntime versions:")

display(
    runtime_versions
)


print("\nPredictor contract summary:")

display(
    predictor_contract.groupby(
        [
            "IsREC",
            "RECClass",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        Predictors=(
            "Predictor",
            "count",
        ),

        TrainingMissingValues=(
            "TrainingMissing",
            "sum",
        ),

        EvaluationMissingValues=(
            "EvaluationMissing",
            "sum",
        ),
    )
)


print("\nModel implementation contract:")

display(
    model_contract_table[
        [
            "Technique",
            "EstimatorClass",
            "Seed1RandomState",
            "Seed2RandomState",
            "SameSeedSameConfiguration",
            "DifferentSeedStateAsExpected",
        ]
    ]
)


print("\nMetric self-tests:")

display(
    metric_self_test
)


print("\nStep 3A output-manifest audit:")

display(
    output_manifest_audit[
        [
            "Path",
            "ExpectedBytes",
            "ActualBytes",
            "Pass",
        ]
    ]
)


# --------------------------------------------------------------------------------------------------
# 18. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 132)
print("=== PROJECT 11 CELL 7 / STEP 4A RESULT ===")
print("=" * 132)


print("\nProject:")

print(
    PROJECT_NAME
)


print("\nFrozen experiment contract:")

print(
    "Predictors:",
    len(
        predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURES
    ),
)

print(
    "ML techniques:",
    ML_TECHNIQUES,
)

print(
    "Baselines:",
    BASELINE_TECHNIQUES,
)

print(
    "Primary / secondary metrics:",
    "APFDc / APFD",
)

print(
    "Positive class:",
    "failure = 1",
)

print(
    "Median imputation:",
    "condition-training medians",
)

print(
    "Ranking tie-break:",
    "Test ascending",
)

print(
    "Rolling retraining:",
    False,
)


print("\nFixed cohorts:")

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Training failures:",
    int(
        training_binary_labels.sum()
    ),
)

print(
    "Evaluation failures:",
    int(
        evaluation_binary_labels.sum()
    ),
)

print(
    "Failing evaluation builds:",
    model_failing_evaluation_builds,
)


print("\nRuntime validation:")

print(
    "Step 3A output-manifest failures:",
    output_manifest_failures,
)

print(
    "All-missing predictors:",
    len(
        all_missing_predictors
    ),
)

print(
    "Training non-finite values after imputation:",
    training_nonfinite_after_imputation,
)

print(
    "Evaluation non-finite values after imputation:",
    evaluation_nonfinite_after_imputation,
)

print(
    "Model contract failures:",
    model_contract_failures,
)

print(
    "Metric self-test failures:",
    metric_self_test_failures,
)

print(
    "Random same-seed reproducible:",
    random_same_seed_reproduced,
)


print("\nIsolation:")

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–10 modified:",
    0,
)

print(
    "Project 12 accessed:",
    False,
)

print(
    "Project 11 models fitted:",
    False,
)

print(
    "Full experiment started:",
    False,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print("\nRuntime-contract checkpoint:")

print(
    RUNTIME_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    sha256_file(
        RUNTIME_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP4A_STATUS,
)

print("=" * 132)

=== PROJECT 11 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===


/tmp/ipykernel_424/2611933447.py:1239: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  training_numeric[
/tmp/ipykernel_424/2611933447.py:1243: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  evaluation_numeric[
/tmp/ipykernel_424/2611933447.py:1239: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  training


Project 11 Step 4A validation:


,Check,Expected,Actual,Pass
0,Step 3A status,PASS_PROJECT_11_DETERMINISTIC_NOISE_PLAN_AND_C...,PASS_PROJECT_11_DETERMINISTIC_NOISE_PLAN_AND_C...,True
1,Noise-plan checkpoint SHA-256,086da91a6e4ab96fce51fb9749ce3581eeb2c303668995...,086da91a6e4ab96fce51fb9749ce3581eeb2c303668995...,True
2,Step 3A output-manifest failures,0,0,True
3,Source root SHA-256,3e6e0a60d58bff61690d6e2ac2bac18371414819eff601...,3e6e0a60d58bff61690d6e2ac2bac18371414819eff601...,True
4,Raw training rows,609619,609619,True
5,Raw evaluation rows,223922,223922,True
6,Model training rows,78035,78035,True
7,Model evaluation rows,13007,13007,True
8,Model training failures,1188,1188,True
9,Model evaluation failures,171,171,True



Runtime versions:


...


Metric self-tests:


,Check,Expected,Actual,Pass
0,Manual APFD,0.8,0.8,True
1,APFDc rewards quick failing test first,True,True,True
2,All-pass APFD is NaN,True,True,True
3,All-pass APFDc is NaN,True,True,True



Step 3A output-manifest audit:


,Path,ExpectedBytes,ActualBytes,Pass
0,/content/drive/MyDrive/Thesis_Experiment/Resul...,1946692,1946692,True
1,/content/drive/MyDrive/Thesis_Experiment/Resul...,887348,887348,True
2,/content/drive/MyDrive/Thesis_Experiment/Resul...,4787061,4787061,True
3,/content/drive/MyDrive/Thesis_Experiment/Resul...,944106,944106,True
4,/content/drive/MyDrive/Thesis_Experiment/Resul...,542782,542782,True
5,/content/drive/MyDrive/Thesis_Experiment/Resul...,90060,90060,True
6,/content/drive/MyDrive/Thesis_Experiment/Resul...,97,97,True
7,/content/drive/MyDrive/Thesis_Experiment/Resul...,5282,5282,True
8,/content/drive/MyDrive/Thesis_Experiment/Resul...,211781462,211781462,True
9,/content/drive/MyDrive/Thesis_Experiment/Resul...,87166,87166,True




=== PROJECT 11 CELL 7 / STEP 4A RESULT ===

Project:
apache@shardingsphere

Frozen experiment contract:
Predictors: 151
REC features: 19
ML techniques: ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes']
Baselines: ['Random', 'LatestFail', 'QTF-Avg']
Primary / secondary metrics: APFDc / APFD
Positive class: failure = 1
Median imputation: condition-training medians
Ranking tie-break: Test ascending
Rolling retraining: False

Fixed cohorts:
Model training rows: 78035
Model evaluation rows: 13007
Training failures: 1188
Evaluation failures: 171
Failing evaluation builds: 25

Runtime validation:
Step 3A output-manifest failures: 0
All-missing predictors: 0
Training non-finite values after imputation: 0
Evaluation non-finite values after imputation: 0
Model contract failures: 0
Metric self-test failures: 0
Random same-seed reproducible: True

Isolation:
Completion registry unchanged: True
Projects 1–10 modified: 0
Project 12 accessed: False
Project 11 models fitted: False
Full experimen

In [1]:
# ==================================================================================================
# PROJECT 11 — FRESH-RUNTIME RECOVERY CELL
#
# Run this after a Colab disconnect/restart.
#
# It:
# - remounts Google Drive;
# - validates all frozen Project 11 checkpoints;
# - restores only apache@shardingsphere into /content/datasets if necessary;
# - validates the exact frozen source-root SHA-256;
# - validates the runtime package versions;
# - does NOT rerun Steps 1B, 2A, 2B, 3A, or 4A;
# - does NOT modify the registry or any frozen result.
# ==================================================================================================

from google.colab import drive

from pathlib import Path
from importlib import metadata

import hashlib
import json
import platform
import shutil
import tarfile

import pandas as pd


print("=" * 120)
print("=== PROJECT 11 FRESH-RUNTIME RECOVERY ===")
print("=" * 120)


# --------------------------------------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# --------------------------------------------------------------------------------------------------

drive.mount(
    "/content/drive",
    force_remount=False,
)


# --------------------------------------------------------------------------------------------------
# 2. FROZEN PATHS AND HASHES
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

SOURCE_ROOT = Path(
    "/content/datasets/datasets/apache@shardingsphere"
)

EXTRACTION_ROOT = Path(
    "/content/datasets"
)

SELECTION_MANIFEST_PATH = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_11_selection"
    / "project_11_frozen_source_manifest.csv"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

CHECKPOINTS = {
    "Selection checkpoint": (
        THESIS_ROOT
        / "Notes"
        / "project_11_selection_checkpoint.json",
        "9d5c95b29357426def26c41b62a753c003ad3b19e6086d9873f0961e39875353",
    ),

    "REC reconstruction checkpoint": (
        THESIS_ROOT
        / "Notes"
        / "project_11_rec_reconstruction_checkpoint.json",
        "a1df2018227f643616169017b5d6a35fb40431c325821d7d5acda8e40701bfa0",
    ),

    "Noise-plan checkpoint": (
        THESIS_ROOT
        / "Notes"
        / "project_11_noise_plan_checkpoint.json",
        "086da91a6e4ab96fce51fb9749ce3581eeb2c303668995877de88ec4f5c2d3fa",
    ),

    "Runtime-contract checkpoint": (
        THESIS_ROOT
        / "Notes"
        / "project_11_runtime_contract_checkpoint.json",
        "606167d8df76dc04e91ece3c0969d0ceb46319930713034a066c3a06c2015ddb",
    ),
}

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "3e6e0a60d58bff61690d6e2ac2bac18371414819eff601f3ea347ecb8ad8b0b5"
)

EXPECTED_REGISTRY_SHA256 = (
    "847bdcbee16c8757fde34e9489cd3abbf0781a319644175472597265b1df2750"
)

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 142_849_042

EXPECTED_PACKAGE_VERSIONS = {
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scikit-learn": "1.6.1",
    "xgboost": "3.3.0",
    "lightgbm": "4.6.0",
    "pyarrow": "18.1.0",
}

EXPECTED_PYTHON_VERSION = "3.12.13"


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def canonical_source_root_hash(
    manifest,
):
    digest = hashlib.sha256()

    ordered = manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def validate_source():
    if not SOURCE_ROOT.is_dir():
        return False, None, None, None

    frozen_manifest = pd.read_csv(
        SELECTION_MANIFEST_PATH,
        low_memory=False,
    )

    current_rows = []

    for row in frozen_manifest.itertuples(
        index=False
    ):
        relative_path = str(
            row.RelativePath
        )

        source_path = (
            SOURCE_ROOT
            / relative_path
        )

        if not source_path.is_file():
            return False, None, None, None

        actual_size = int(
            source_path.stat().st_size
        )

        actual_sha256 = sha256_file(
            source_path
        )

        if (
            actual_size
            != int(row.SizeBytes)
            or actual_sha256
            != str(row.SHA256).lower()
        ):
            return False, None, None, None

        current_rows.append({
            "RelativePath":
                relative_path,

            "SizeBytes":
                actual_size,

            "SHA256":
                actual_sha256,
        })

    current_manifest = pd.DataFrame(
        current_rows
    )

    source_root_sha256 = canonical_source_root_hash(
        current_manifest
    )

    source_bytes = int(
        current_manifest[
            "SizeBytes"
        ].sum()
    )

    return (
        source_root_sha256
        == EXPECTED_SOURCE_ROOT_SHA256,
        source_root_sha256,
        len(current_manifest),
        source_bytes,
    )


def restore_project_source():
    if not ARCHIVE_PATH.is_file():
        raise FileNotFoundError(
            f"Dataset archive is missing:\n{ARCHIVE_PATH}"
        )

    print("\nValidating dataset archive before extraction.")

    archive_sha256 = sha256_file(
        ARCHIVE_PATH
    )

    if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
        raise RuntimeError(
            "Dataset archive SHA-256 differs.\n"
            f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
            f"Actual:   {archive_sha256}"
        )

    print(
        "Archive SHA-256 validated:",
        archive_sha256,
    )

    marker = (
        "datasets/"
        "apache@shardingsphere/"
    )

    extracted_files = 0

    EXTRACTION_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "\nRestoring only apache@shardingsphere "
        "from the frozen archive."
    )

    with tarfile.open(
        ARCHIVE_PATH,
        mode="r:gz",
    ) as archive:
        for member in archive:
            normalised_name = (
                member.name
                .replace("\\", "/")
                .lstrip("./")
            )

            marker_position = (
                normalised_name.find(
                    marker
                )
            )

            if marker_position < 0:
                continue

            relative_archive_path = (
                normalised_name[
                    marker_position:
                ]
            )

            target_path = (
                EXTRACTION_ROOT
                / relative_archive_path
            )

            resolved_target = (
                target_path.resolve()
            )

            resolved_root = (
                EXTRACTION_ROOT.resolve()
            )

            if (
                resolved_target != resolved_root
                and resolved_root
                not in resolved_target.parents
            ):
                raise RuntimeError(
                    "Unsafe archive path encountered:\n"
                    f"{member.name}"
                )

            if member.isdir():
                target_path.mkdir(
                    parents=True,
                    exist_ok=True,
                )

            elif member.isfile():
                target_path.parent.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                source_handle = archive.extractfile(
                    member
                )

                if source_handle is None:
                    raise RuntimeError(
                        "Could not extract archive member:\n"
                        f"{member.name}"
                    )

                with (
                    source_handle,
                    target_path.open("wb") as output_handle,
                ):
                    shutil.copyfileobj(
                        source_handle,
                        output_handle,
                        length=8 * 1024 * 1024,
                    )

                extracted_files += 1

    if extracted_files == 0:
        raise RuntimeError(
            "No apache@shardingsphere files were found "
            "inside the dataset archive."
        )

    print(
        "Files restored from archive:",
        extracted_files,
    )


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE DRIVE FILES AND CHECKPOINTS
# --------------------------------------------------------------------------------------------------

required_drive_files = [
    ARCHIVE_PATH,
    SELECTION_MANIFEST_PATH,
    REGISTRY_PATH,
]

required_drive_files.extend(
    path
    for path, _ in CHECKPOINTS.values()
)

missing_drive_files = [
    str(path)
    for path in required_drive_files
    if not path.is_file()
]

if missing_drive_files:
    raise FileNotFoundError(
        "Required frozen Drive files are missing:\n"
        + "\n".join(
            missing_drive_files
        )
    )


print("\nValidating frozen Project 11 checkpoints.")


for label, (
    checkpoint_path,
    expected_sha256,
) in CHECKPOINTS.items():
    actual_sha256 = sha256_file(
        checkpoint_path
    )

    if actual_sha256 != expected_sha256:
        raise RuntimeError(
            f"{label} SHA-256 differs.\n"
            f"Expected: {expected_sha256}\n"
            f"Actual:   {actual_sha256}"
        )

    with checkpoint_path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        checkpoint_payload = json.load(
            handle
        )

    print(
        f"{label}: VALID"
    )

    print(
        "  Status:",
        checkpoint_payload.get(
            "Status",
            "STATUS FIELD NOT PRESENT",
        ),
    )

    print(
        "  SHA-256:",
        actual_sha256,
    )


registry_sha256 = sha256_file(
    REGISTRY_PATH
)

if registry_sha256 != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256}"
    )

print(
    "\nCompletion registry: VALID"
)

print(
    "Registry SHA-256:",
    registry_sha256,
)


# --------------------------------------------------------------------------------------------------
# 5. RESTORE TEMPORARY SOURCE ONLY WHEN REQUIRED
# --------------------------------------------------------------------------------------------------

(
    source_is_valid,
    source_root_sha256,
    source_file_count,
    source_bytes,
) = validate_source()


if source_is_valid:
    print(
        "\nTemporary Project 11 source already exists "
        "and is valid."
    )

else:
    print(
        "\nTemporary Project 11 source is missing or incomplete."
    )

    restore_project_source()

    (
        source_is_valid,
        source_root_sha256,
        source_file_count,
        source_bytes,
    ) = validate_source()


if not source_is_valid:
    raise RuntimeError(
        "Project 11 source validation failed after restoration."
    )


if source_file_count != EXPECTED_SOURCE_FILES:
    raise RuntimeError(
        "Frozen source file count differs.\n"
        f"Expected: {EXPECTED_SOURCE_FILES}\n"
        f"Actual:   {source_file_count}"
    )


if source_bytes != EXPECTED_SOURCE_BYTES:
    raise RuntimeError(
        "Frozen source byte count differs.\n"
        f"Expected: {EXPECTED_SOURCE_BYTES}\n"
        f"Actual:   {source_bytes}"
    )


print(
    "\nProject 11 source: VALID"
)

print(
    "Source directory:",
    SOURCE_ROOT,
)

print(
    "Source files:",
    source_file_count,
)

print(
    "Source bytes:",
    source_bytes,
)

print(
    "Source root SHA-256:",
    source_root_sha256,
)


# --------------------------------------------------------------------------------------------------
# 6. VALIDATE RUNTIME VERSIONS
# --------------------------------------------------------------------------------------------------

print("\nValidating runtime versions.")


current_python_version = (
    platform.python_version()
)

print(
    "Python:",
    current_python_version,
)


if current_python_version != EXPECTED_PYTHON_VERSION:
    raise RuntimeError(
        "Python version differs from the frozen Step 4A runtime.\n"
        f"Expected: {EXPECTED_PYTHON_VERSION}\n"
        f"Actual:   {current_python_version}"
    )


package_version_failures = []


for package_name, expected_version in (
    EXPECTED_PACKAGE_VERSIONS.items()
):
    try:
        actual_version = metadata.version(
            package_name
        )

    except metadata.PackageNotFoundError:
        actual_version = "MISSING"

    print(
        f"{package_name}: {actual_version}"
    )

    if actual_version != expected_version:
        package_version_failures.append({
            "Package":
                package_name,

            "Expected":
                expected_version,

            "Actual":
                actual_version,
        })


if package_version_failures:
    raise RuntimeError(
        "One or more runtime package versions differ "
        "from the frozen Step 4A environment:\n"
        + json.dumps(
            package_version_failures,
            indent=2,
        )
    )


# --------------------------------------------------------------------------------------------------
# 7. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 120)
print("=== PROJECT 11 FRESH-RUNTIME RECOVERY RESULT ===")
print("=" * 120)

print(
    "Drive mounted:",
    True,
)

print(
    "Frozen checkpoints validated:",
    len(CHECKPOINTS),
)

print(
    "Completion registry unchanged:",
    True,
)

print(
    "Project 11 source restored and validated:",
    True,
)

print(
    "Source root SHA-256:",
    source_root_sha256,
)

print(
    "Runtime versions match Step 4A:",
    True,
)

print(
    "Steps 1B–4A rerun:",
    False,
)

print(
    "Project 12 accessed:",
    False,
)

print(
    "\nSTATUS: PASS_PROJECT_11_FRESH_RUNTIME_RECOVERED"
)

print("=" * 120)

=== PROJECT 11 FRESH-RUNTIME RECOVERY ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Validating frozen Project 11 checkpoints.
Selection checkpoint: VALID
  Status: PASS_PROJECT_11_SELECTION_AND_SOURCE_FROZEN
  SHA-256: 9d5c95b29357426def26c41b62a753c003ad3b19e6086d9873f0961e39875353
REC reconstruction checkpoint: VALID
  Status: PASS_PROJECT_11_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN
  SHA-256: a1df2018227f643616169017b5d6a35fb40431c325821d7d5acda8e40701bfa0
Noise-plan checkpoint: VALID
  Status: PASS_PROJECT_11_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN
  SHA-256: 086da91a6e4ab96fce51fb9749ce3581eeb2c303668995877de88ec4f5c2d3fa
Runtime-contract checkpoint: VALID
  Status: PASS_PROJECT_11_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN
  SHA-256: 606167d8df76dc04e91ece3c0969d0ceb46319930713034a066c3a06c2015ddb

Completion registry: VALID
Registry SHA-256: 847bdcbee16c8757fde34e9489cd3abbf0781a31964

In [11]:
# ==================================================================================================
# PROJECT 11 — CELL 8 / STEP 4B
# TWO-CONDITION END-TO-END SMOKE TEST
#
# PROJECT: apache@shardingsphere
#
# CONDITIONS:
#   noise_00__seed_01
#   noise_50__seed_01
#
# PURPOSE:
# - verify the frozen Step 4A runtime/model contract;
# - reproduce the frozen noise masks and noisy labels for two conditions;
# - recompute all 19 REC features from the condition-specific history;
# - preserve the six verdict-independent REC features and replace the thirteen
#   verdict-dependent REC features;
# - fit RandomForest, XGBoost, LightGBM, and NaiveBayes;
# - evaluate Random, LatestFail, and QTF-Avg on the same clean evaluation partition;
# - calculate build-level and project-level APFDc/APFD;
# - prove baseline invariance and end-to-end output integrity before the 270-condition run.
#
# SAFETY:
# - does not update the completion registry;
# - does not modify Projects 1-10;
# - does not access Project 12;
# - writes only to the Project 11 smoke-test directory and Project 11 Step 4B checkpoint/status files;
# - does not write into the full experiment raw-result directory.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import gc
import hashlib
import json
import os
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from pandas.errors import PerformanceWarning
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.simplefilter("ignore", PerformanceWarning)


print("=" * 136)
print("=== PROJECT 11 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 11
PROJECT_NAME = "apache@shardingsphere"
PROJECT_SLUG = "apache__shardingsphere"
PROJECT_SHORT = "shardingsphere"

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_11_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)

STEP4B_STATUS = (
    "PASS_PROJECT_11_TWO_CONDITION_END_TO_END_SMOKE_TEST"
)

CONDITION_STATUS = (
    "PASS_PROJECT_11_SMOKE_CONDITION"
)

EXPECTED_RUNTIME_CHECKPOINT_SHA256 = (
    "606167d8df76dc04e91ece3c0969d0ceb46319930713034a066c3a06c2015ddb"
)

EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "086da91a6e4ab96fce51fb9749ce3581eeb2c303668995877de88ec4f5c2d3fa"
)

EXPECTED_REC_CHECKPOINT_SHA256 = (
    "a1df2018227f643616169017b5d6a35fb40431c325821d7d5acda8e40701bfa0"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "9d5c95b29357426def26c41b62a753c003ad3b19e6086d9873f0961e39875353"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "3e6e0a60d58bff61690d6e2ac2bac18371414819eff601f3ea347ecb8ad8b0b5"
)

EXPECTED_REGISTRY_SHA256 = (
    "847bdcbee16c8757fde34e9489cd3abbf0781a319644175472597265b1df2750"
)

EXPECTED_RAW_TRAIN_ROWS = 609_619
EXPECTED_RAW_EVAL_ROWS = 223_922
EXPECTED_MODEL_TRAIN_ROWS = 78_035
EXPECTED_MODEL_EVAL_ROWS = 13_007
EXPECTED_MODEL_ROWS = 91_042
EXPECTED_MODEL_TRAIN_FAILURES = 1_188
EXPECTED_MODEL_EVAL_FAILURES = 171
EXPECTED_FAILING_EVAL_BUILDS = 25
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_EVALUATION_BUILDS = 263
EXPECTED_SMOKE_CONDITIONS = 2
EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
)
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
)
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS

SMOKE_CONDITION_IDS = [
    "noise_00__seed_01",
    "noise_50__seed_01",
]

SMOKE_NOISE_LEVELS = [
    0,
    50,
]

SMOKE_REPETITION_SEED = 1
RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@shardingsphere"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_11_selection"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_11_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_11_fixed_chronological_builds.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_11_selection_checkpoint.json"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

BUILD_ENTITY_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

CLEAN_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_11_rec_reconstruction_checkpoint.json"
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_rng_manifest.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_11_noise_plan_checkpoint.json"
)

RUNTIME_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_runtime_contract"
)

PREDICTOR_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_predictor_contract.csv"
)

MODEL_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_model_contract.json"
)

BASELINE_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_baseline_contract.json"
)

RANKING_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_ranking_contract.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4a_status.json"
)

STEP4A_REPORT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_report.json"
)

RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_11_runtime_contract_checkpoint.json"
)

SMOKE_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_smoke_test"
)

SMOKE_CONDITION_INVENTORY_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_condition_inventory.csv"
)

SMOKE_COMBINED_CONDITION_AUDIT_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_condition_audit.csv"
)

SMOKE_COMBINED_PROJECT_RUNS_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_project_runs.csv"
)

SMOKE_COMBINED_BUILD_METRICS_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_build_metrics.csv"
)

SMOKE_COMBINED_MODEL_FITS_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_model_fits.csv"
)

SMOKE_BASELINE_INVARIANCE_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_baseline_invariance.csv"
)

SMOKE_VALIDATION_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_step4b_validation.csv"
)

SMOKE_REPORT_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_step4b_report.json"
)

STEP4B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4b_status.json"
)

SMOKE_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_11_smoke_test_checkpoint.json"
)

# The smoke test must never write to this future full-run root.
FULL_RAW_RESULT_ROOT = (
    RESULTS_ROOT
    / "Raw"
    / PROJECT_SLUG
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def sha256_array(values, dtype):
    array = np.asarray(values).astype(dtype, copy=False)
    return hashlib.sha256(array.tobytes(order="C")).hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(temporary_path, path)


def atomic_csv(path, frame, compression=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(temporary_path, path)


def atomic_parquet(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_parquet(
        temporary_path,
        index=False,
    )

    os.replace(temporary_path, path)


def source_root_hash(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )
        digest.update(line.encode("utf-8"))

    return digest.hexdigest()


def parse_int(values, label):
    numeric = pd.to_numeric(values, errors="coerce")

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains {int(numeric.isna().sum())} missing/non-numeric values."
        )

    array = numeric.to_numpy(dtype=float)

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def deterministic_seed(repetition_seed, stream_name):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(material).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def deterministic_random_build_seed(repetition_seed, build_id):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )


def create_models(repetition_seed):
    return {
        "RandomForest": RandomForestClassifier(
            **MODEL_CONFIG["RandomForest"],
            random_state=deterministic_seed(
                repetition_seed,
                "RandomForest_model",
            ),
        ),
        "XGBoost": XGBClassifier(
            **MODEL_CONFIG["XGBoost"],
            random_state=deterministic_seed(
                repetition_seed,
                "XGBoost_model",
            ),
        ),
        "LightGBM": LGBMClassifier(
            **MODEL_CONFIG["LightGBM"],
            random_state=deterministic_seed(
                repetition_seed,
                "LightGBM_model",
            ),
        ),
        "NaiveBayes": GaussianNB(
            **MODEL_CONFIG["NaiveBayes"]
        ),
    }


def calculate_apfd(failures):
    failures = np.asarray(failures, dtype=np.int8)
    number_of_tests = len(failures)
    number_of_failures = int(failures.sum())

    if number_of_tests == 0 or number_of_failures == 0:
        return np.nan

    failure_positions = np.flatnonzero(failures == 1) + 1

    return float(
        1.0
        - (
            failure_positions.sum()
            / (number_of_tests * number_of_failures)
        )
        + (1.0 / (2.0 * number_of_tests))
    )


def calculate_apfdc(failures, durations):
    failures = np.asarray(failures, dtype=np.int8)
    durations = np.asarray(durations, dtype=float)

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if len(failures) == 0 or failures.sum() == 0:
        return np.nan

    if not np.isfinite(durations).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (durations < 0).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(durations.sum())

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(durations)[:-1],
    ])

    failure_mask = failures == 1
    midpoint_detection_times = (
        cumulative_before[failure_mask]
        + (0.5 * durations[failure_mask])
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


def calculate_rates(history):
    history_length = len(history)

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history["verdict"]

    return (
        float(verdicts.ne(0).sum() / history_length),
        float(verdicts.eq(2).sum() / history_length),
        float(verdicts.eq(1).sum() / history_length),
        float(history["transition"].eq(1).sum() / history_length),
    )


def calculate_max_test_file_rate(
    history,
    target_column,
    current_changed_entities,
    entity_changed_builds,
):
    target_builds = (
        history.loc[
            history[target_column].gt(0),
            "build",
        ]
        .drop_duplicates()
        .astype(int)
        .tolist()
    )

    if len(target_builds) == 0:
        return -1.0

    target_build_set = set(target_builds)
    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(entity_id),
            set(),
        )

        overlap_count = len(
            changed_builds.intersection(target_build_set)
        )

        maximum_frequency = max(
            maximum_frequency,
            overlap_count,
        )

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(target_builds)
    )


def reconstruct_rec_features(
    execution_history,
    requested_rows,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
    recent_window=6,
):
    requested_pairs = set(
        zip(
            requested_rows["Build"].astype(int),
            requested_rows["Test"].astype(int),
        )
    )

    reconstructed_records = []
    test_groups = execution_history.groupby(
        "test",
        sort=False,
    )
    total_tests = int(
        execution_history["test"].nunique()
    )

    for test_index, (test_id, test_history) in enumerate(
        test_groups,
        start=1,
    ):
        test_history = (
            test_history.sort_values(
                [
                    "build_order",
                    "job",
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )

        test_history["transition"] = (
            test_history["verdict"]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(int)
        )

        first_test_build = int(
            test_history.iloc[0]["build"]
        )

        for current_position in range(len(test_history)):
            current_row = test_history.iloc[current_position]
            current_build = int(current_row["build"])
            current_test = int(test_id)
            pair = (current_build, current_test)

            if pair not in requested_pairs:
                continue

            history = (
                test_history.iloc[:current_position]
                .copy()
                .reset_index(drop=True)
            )

            record = {
                "Build": current_build,
                "Test": current_test,
            }

            if history.empty:
                for feature in REC_FEATURES:
                    record[feature] = -1.0

                record["REC_Age"] = 0.0
                reconstructed_records.append(record)
                continue

            recent_history = history.tail(recent_window).copy()

            age = float(
                global_build_position[current_build]
                - global_build_position[first_test_build]
            )

            failure_positions = np.flatnonzero(
                history["verdict"].to_numpy() > 0
            )

            last_failure_age = (
                -1.0
                if len(failure_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(failure_positions[-1])
                )
            )

            transition_positions = np.flatnonzero(
                history["transition"].to_numpy() > 0
            )

            last_transition_age = (
                -1.0
                if len(transition_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(transition_positions[-1])
                )
            )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(recent_history)

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(history)

            current_changed_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            max_file_fail_rate = calculate_max_test_file_rate(
                history=history,
                target_column="verdict",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            max_file_transition_rate = calculate_max_test_file_rate(
                history=history,
                target_column="transition",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            record.update({
                "REC_Age": age,
                "REC_LastFailureAge": last_failure_age,
                "REC_LastTransitionAge": last_transition_age,
                "REC_RecentAvgExeTime": float(
                    recent_history["duration"].mean()
                ),
                "REC_RecentMaxExeTime": float(
                    recent_history["duration"].max()
                ),
                "REC_RecentFailRate": recent_fail_rate,
                "REC_RecentAssertRate": recent_assert_rate,
                "REC_RecentExcRate": recent_exc_rate,
                "REC_RecentTransitionRate": recent_transition_rate,
                "REC_TotalAvgExeTime": float(
                    history["duration"].mean()
                ),
                "REC_TotalMaxExeTime": float(
                    history["duration"].max()
                ),
                "REC_TotalFailRate": total_fail_rate,
                "REC_TotalAssertRate": total_assert_rate,
                "REC_TotalExcRate": total_exc_rate,
                "REC_TotalTransitionRate": total_transition_rate,
                "REC_LastVerdict": float(
                    recent_history.iloc[-1]["verdict"]
                ),
                "REC_LastExeTime": float(
                    recent_history.iloc[-1]["duration"]
                ),
                "REC_MaxTestFileFailRate": max_file_fail_rate,
                "REC_MaxTestFileTransitionRate": (
                    max_file_transition_rate
                ),
            })

            reconstructed_records.append(record)

        if test_index % 100 == 0 or test_index == total_tests:
            print(
                "    REC reconstruction progress:",
                test_index,
                "/",
                total_tests,
                "tests | reconstructed rows:",
                len(reconstructed_records),
            )

    return pd.DataFrame(reconstructed_records)


def positive_probability(estimator, matrix):
    probabilities = estimator.predict_proba(matrix)
    classes = np.asarray(estimator.classes_)
    positive_columns = np.flatnonzero(classes == 1)

    if len(positive_columns) != 1:
        raise RuntimeError(
            "Fitted estimator does not expose exactly one class-1 probability column."
        )

    scores = probabilities[:, int(positive_columns[0])]

    if not np.isfinite(scores).all():
        raise RuntimeError(
            "Model produced non-finite failure probabilities."
        )

    if ((scores < 0) | (scores > 1)).any():
        raise RuntimeError(
            "Model produced probabilities outside [0,1]."
        )

    return scores.astype(float, copy=False)


def make_ranking(
    evaluation_meta,
    technique,
    scores,
    ascending_score,
):
    ranking = evaluation_meta.copy()
    ranking["Technique"] = technique
    ranking["Score"] = np.asarray(scores, dtype=float)

    if len(ranking) != EXPECTED_MODEL_EVAL_ROWS:
        raise RuntimeError(
            f"{technique} ranking input has the wrong row count."
        )

    if not np.isfinite(ranking["Score"].to_numpy(dtype=float)).all():
        raise RuntimeError(
            f"{technique} ranking contains non-finite scores."
        )

    ranking = (
        ranking.sort_values(
            [
                "Build",
                "Score",
                "Test",
            ],
            ascending=[
                True,
                bool(ascending_score),
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    ranking["Rank"] = (
        ranking.groupby(
            "Build",
            sort=False,
        )
        .cumcount()
        .add(1)
        .astype("int64")
    )

    return ranking[
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "Build",
            "Test",
            "Rank",
            "Score",
            "CleanVerdict",
            "CleanFailure",
            "Duration",
        ]
    ]


def calculate_condition_metrics(rankings):
    build_metric_records = []

    failing_rankings = rankings.loc[
        rankings["Build"].isin(failing_evaluation_builds)
    ].copy()

    for (technique, build_id), build_ranking in failing_rankings.groupby(
        [
            "Technique",
            "Build",
        ],
        sort=False,
    ):
        build_ranking = build_ranking.sort_values(
            "Rank",
            kind="mergesort",
        )

        failures = build_ranking[
            "CleanFailure"
        ].to_numpy(dtype=np.int8)

        durations = build_ranking[
            "Duration"
        ].to_numpy(dtype=float)

        number_of_failures = int(failures.sum())

        if number_of_failures <= 0:
            raise RuntimeError(
                "A supposedly failing evaluation build has no failures."
            )

        apfd = calculate_apfd(failures)
        apfdc = calculate_apfdc(failures, durations)

        build_metric_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                build_ranking["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                build_ranking["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                build_ranking["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "Build": int(build_id),
            "Tests": int(len(build_ranking)),
            "Failures": number_of_failures,
            "TotalDuration": float(durations.sum()),
            "APFDc": float(apfdc),
            "APFD": float(apfd),
        })

    build_metrics = pd.DataFrame(build_metric_records)

    project_run_records = []

    for technique, technique_metrics in build_metrics.groupby(
        "Technique",
        sort=False,
    ):
        project_run_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                technique_metrics["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                technique_metrics["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                technique_metrics["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "EvaluationBuilds": EXPECTED_EVALUATION_BUILDS,
            "ScoredFailingBuilds": int(len(technique_metrics)),
            "EvaluationRows": EXPECTED_MODEL_EVAL_ROWS,
            "EvaluationFailures": EXPECTED_MODEL_EVAL_FAILURES,
            "MeanAPFDc": float(
                technique_metrics["APFDc"].mean()
            ),
            "MedianAPFDc": float(
                technique_metrics["APFDc"].median()
            ),
            "MeanAPFD": float(
                technique_metrics["APFD"].mean()
            ),
            "MedianAPFD": float(
                technique_metrics["APFD"].median()
            ),
        })

    project_runs = pd.DataFrame(project_run_records)

    return build_metrics, project_runs


def directory_manifest(root):
    root = Path(root)
    rows = []

    if not root.exists():
        return pd.DataFrame(
            columns=[
                "RelativePath",
                "Bytes",
                "SHA256",
            ]
        )

    for path in sorted(
        [
            candidate
            for candidate in root.rglob("*")
            if candidate.is_file()
        ],
        key=lambda candidate: candidate.relative_to(root).as_posix(),
    ):
        rows.append({
            "RelativePath": path.relative_to(root).as_posix(),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        })

    return pd.DataFrame(rows)


def directory_root_hash(manifest):
    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN CHECKPOINTS
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "contributors.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "entity_change_history.csv",
    SOURCE_DIR / "exe.csv",
    SOURCE_DIR / "id_map.csv",
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    SELECTION_CHECKPOINT_PATH,
    BUILD_ENTITY_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    REC_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    PREDICTOR_CONTRACT_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_STATUS_PATH,
    STEP4A_REPORT_PATH,
    RUNTIME_CHECKPOINT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 11 Step 4B inputs are missing:\n"
        + "\n".join(missing_paths)
    )

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)

if selection_checkpoint_sha256 != EXPECTED_SELECTION_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 11 selection checkpoint SHA-256 differs."
    )

if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 11 REC checkpoint SHA-256 differs."
    )

if noise_plan_checkpoint_sha256 != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 11 noise-plan checkpoint SHA-256 differs."
    )

if runtime_checkpoint_sha256 != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 11 runtime-contract checkpoint SHA-256 differs."
    )

runtime_checkpoint = load_json(
    RUNTIME_CHECKPOINT_PATH
)
step4a_status = load_json(
    STEP4A_STATUS_PATH
)
step4a_report = load_json(
    STEP4A_REPORT_PATH
)

for label, payload in [
    ("runtime checkpoint", runtime_checkpoint),
    ("Step 4A status", step4a_status),
    ("Step 4A report", step4a_report),
]:
    if payload.get("Status") != EXPECTED_STEP4A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the frozen Step 4A PASS status."
        )

if runtime_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Runtime checkpoint project identity differs."
    )

if runtime_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Runtime checkpoint project slug differs."
    )

if runtime_checkpoint.get("SourceRootSHA256") != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Runtime checkpoint source root differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY SOURCE ROOT, REGISTRY, AND STEP 4A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(REGISTRY_PATH)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs before Step 4B."
    )

registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)

project_number_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "projectnumber",
            "project_number",
            "project no",
            "projectno",
        }
    ),
    None,
)

if project_number_column is None:
    raise RuntimeError(
        "Could not resolve the project-number column in the completion registry."
    )

registry_project_numbers = pd.to_numeric(
    registry[project_number_column],
    errors="coerce",
)

if len(registry) != 10:
    raise RuntimeError(
        "Completion registry does not contain exactly Projects 1-10."
    )

if int(registry_project_numbers.eq(PROJECT_NUMBER).sum()) != 0:
    raise RuntimeError(
        "Project 11 is already present in the completion registry."
    )

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

current_source_rows = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = Path(row.AbsolutePath)

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 11 source file is missing: {source_path}"
        )

    current_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

current_source_manifest = pd.DataFrame(current_source_rows)
current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 11 source root differs before Step 4B."
    )

runtime_output_manifest = runtime_checkpoint.get(
    "RuntimeOutputManifest",
    [],
)

if not isinstance(runtime_output_manifest, list) or not runtime_output_manifest:
    raise RuntimeError(
        "Runtime checkpoint has no output manifest."
    )

runtime_manifest_records = []

for item in runtime_output_manifest:
    path = Path(item["Path"])
    expected_bytes = int(item["Bytes"])
    expected_sha256 = str(item["SHA256"]).lower()
    exists = path.is_file()
    actual_bytes = int(path.stat().st_size) if exists else -1
    actual_sha256 = sha256_file(path) if exists else "MISSING"
    passed = (
        exists
        and actual_bytes == expected_bytes
        and actual_sha256 == expected_sha256
    )

    runtime_manifest_records.append({
        "Path": str(path),
        "ExpectedBytes": expected_bytes,
        "ActualBytes": actual_bytes,
        "ExpectedSHA256": expected_sha256,
        "ActualSHA256": actual_sha256,
        "Pass": passed,
    })

runtime_manifest_audit = pd.DataFrame(
    runtime_manifest_records
)
runtime_manifest_failures = int(
    (~runtime_manifest_audit["Pass"]).sum()
)

if runtime_manifest_failures != 0:
    print("\nFailed Step 4A output-manifest checks:")
    display(
        runtime_manifest_audit.loc[
            ~runtime_manifest_audit["Pass"]
        ]
    )
    raise RuntimeError(
        "Step 4A output manifest no longer validates."
    )

full_raw_result_root_existed_before = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_before = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_before = directory_root_hash(
    full_raw_result_manifest_before
)


# --------------------------------------------------------------------------------------------------
# 6. LOAD FROZEN COHORTS, LINKS, CONDITION PLAN, AND RNG STREAM
# --------------------------------------------------------------------------------------------------

print("\nLoading frozen Project 11 cohorts and contracts.")

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)
model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)
model_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)
model_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)
condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)
predictor_contract = pd.read_csv(
    PREDICTOR_CONTRACT_PATH,
    low_memory=False,
)
chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)
build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)
anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)
clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

if len(raw_training) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError("Raw training cohort row count differs.")
if len(raw_evaluation) != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError("Raw evaluation cohort row count differs.")
if len(model_training) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model training cohort row count differs.")
if len(model_evaluation) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model evaluation cohort row count differs.")
if len(model_train_link) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model/raw training link row count differs.")
if len(model_eval_link) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model/raw evaluation link row count differs.")
if len(anchor_offsets) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean anchor-offset row count differs.")
if len(clean_reconstructed) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean reconstructed REC row count differs.")

required_cohort_columns = {
    "Build",
    "Test",
    "Verdict",
}

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    missing = required_cohort_columns - set(frame.columns)
    if missing:
        raise RuntimeError(
            f"{label} cohort is missing columns: {sorted(missing)}"
        )

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    frame["Build"] = parse_int(
        frame["Build"],
        f"{label}.Build",
    )
    frame["Test"] = parse_int(
        frame["Test"],
        f"{label}.Test",
    )
    frame["Verdict"] = parse_int(
        frame["Verdict"],
        f"{label}.Verdict",
    )

raw_order_column = "RawTrainingRowOrder"
raw_eval_order_column = "RawEvaluationRowOrder"
model_train_order_column = "ModelTrainingRowOrder"
model_eval_order_column = "ModelEvaluationRowOrder"

for column, frame, expected_rows, label in [
    (
        raw_order_column,
        raw_training,
        EXPECTED_RAW_TRAIN_ROWS,
        "raw training",
    ),
    (
        raw_eval_order_column,
        raw_evaluation,
        EXPECTED_RAW_EVAL_ROWS,
        "raw evaluation",
    ),
    (
        model_train_order_column,
        model_training,
        EXPECTED_MODEL_TRAIN_ROWS,
        "model training",
    ),
    (
        model_eval_order_column,
        model_evaluation,
        EXPECTED_MODEL_EVAL_ROWS,
        "model evaluation",
    ),
]:
    if column not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing {column}."
        )

    frame[column] = parse_int(
        frame[column],
        f"{label}.{column}",
    )

    frame.sort_values(
        column,
        kind="mergesort",
        inplace=True,
    )
    frame.reset_index(drop=True, inplace=True)

    expected_sequence = np.arange(
        1,
        expected_rows + 1,
        dtype=np.int64,
    )

    if not np.array_equal(
        frame[column].to_numpy(dtype=np.int64),
        expected_sequence,
    ):
        raise RuntimeError(
            f"{label} row-order sequence is not canonical."
        )

model_train_link[model_train_order_column] = parse_int(
    model_train_link[model_train_order_column],
    "model_train_link.ModelTrainingRowOrder",
)
model_train_link[raw_order_column] = parse_int(
    model_train_link[raw_order_column],
    "model_train_link.RawTrainingRowOrder",
)
model_eval_link[model_eval_order_column] = parse_int(
    model_eval_link[model_eval_order_column],
    "model_eval_link.ModelEvaluationRowOrder",
)
model_eval_link[raw_eval_order_column] = parse_int(
    model_eval_link[raw_eval_order_column],
    "model_eval_link.RawEvaluationRowOrder",
)

model_train_link = (
    model_train_link.sort_values(
        model_train_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)
model_eval_link = (
    model_eval_link.sort_values(
        model_eval_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

model_training_raw_indices = (
    model_train_link[raw_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)
model_evaluation_raw_indices = (
    model_eval_link[raw_eval_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)

if (
    model_training_raw_indices.min() < 0
    or model_training_raw_indices.max() >= EXPECTED_RAW_TRAIN_ROWS
):
    raise RuntimeError(
        "Model/raw training indices are outside the frozen raw cohort."
    )

if (
    model_evaluation_raw_indices.min() < 0
    or model_evaluation_raw_indices.max() >= EXPECTED_RAW_EVAL_ROWS
):
    raise RuntimeError(
        "Model/raw evaluation indices are outside the frozen raw cohort."
    )

linked_train_build = raw_training.iloc[
    model_training_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_train_test = raw_training.iloc[
    model_training_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_train_verdict = raw_training.iloc[
    model_training_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

linked_eval_build = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_eval_test = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_eval_verdict = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

if not np.array_equal(
    linked_train_build,
    model_training["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Build links differ.")
if not np.array_equal(
    linked_train_test,
    model_training["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Test links differ.")
if not np.array_equal(
    linked_train_verdict,
    model_training["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training verdict links differ.")
if not np.array_equal(
    linked_eval_build,
    model_evaluation["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Build links differ.")
if not np.array_equal(
    linked_eval_test,
    model_evaluation["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Test links differ.")
if not np.array_equal(
    linked_eval_verdict,
    model_evaluation["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation verdict links differ.")

smoke_plan = (
    condition_plan.loc[
        condition_plan["ConditionID"].isin(
            SMOKE_CONDITION_IDS
        )
    ]
    .copy()
    .sort_values(
        "NoisePercent",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(smoke_plan) != EXPECTED_SMOKE_CONDITIONS:
    raise RuntimeError(
        "The frozen condition plan does not contain exactly the two smoke conditions."
    )

if smoke_plan["ConditionID"].tolist() != SMOKE_CONDITION_IDS:
    raise RuntimeError(
        "Smoke-condition order differs from the frozen contract."
    )

if smoke_plan["NoisePercent"].astype(int).tolist() != SMOKE_NOISE_LEVELS:
    raise RuntimeError(
        "Smoke noise levels differ from the frozen contract."
    )

if not smoke_plan["RepetitionSeed"].astype(int).eq(
    SMOKE_REPETITION_SEED
).all():
    raise RuntimeError(
        "Smoke repetition seed differs from the frozen contract."
    )

rng_metadata_rows = int(
    pq.ParquetFile(RNG_MANIFEST_PATH).metadata.num_rows
)

if rng_metadata_rows != 18_288_570:
    raise RuntimeError(
        "Frozen RNG manifest row count differs."
    )

rng_seed = pd.read_parquet(
    RNG_MANIFEST_PATH,
    filters=[
        (
            "RepetitionSeed",
            "==",
            SMOKE_REPETITION_SEED,
        ),
    ],
)

rng_seed[raw_order_column] = parse_int(
    rng_seed[raw_order_column],
    "rng_seed.RawTrainingRowOrder",
)

rng_seed = (
    rng_seed.sort_values(
        raw_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(rng_seed) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError(
        "Seed-1 RNG stream has the wrong row count."
    )

if not np.array_equal(
    rng_seed[raw_order_column].to_numpy(dtype=np.int64),
    np.arange(
        1,
        EXPECTED_RAW_TRAIN_ROWS + 1,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Seed-1 RNG stream row order differs."
    )

flip_uniform = rng_seed[
    "FlipUniform"
].to_numpy(dtype=np.float64)
sampled_failure_subtype = rng_seed[
    "SampledFailureSubtype"
].to_numpy(dtype=np.int16)

if not np.isfinite(flip_uniform).all():
    raise RuntimeError(
        "Seed-1 flip-uniform stream contains non-finite values."
    )

if ((flip_uniform < 0) | (flip_uniform >= 1)).any():
    raise RuntimeError(
        "Seed-1 flip-uniform values are outside [0,1)."
    )

if sorted(np.unique(sampled_failure_subtype).tolist()) != [1, 2]:
    raise RuntimeError(
        "Seed-1 failure-subtype stream differs."
    )


# --------------------------------------------------------------------------------------------------
# 7. PREDICTOR ORDER, NUMERIC MATRICES, CHRONOLOGY, ENTITY MAP, AND EVALUATION META
# --------------------------------------------------------------------------------------------------

if "Predictor" not in predictor_contract.columns:
    raise RuntimeError(
        "Predictor contract is missing the Predictor column."
    )

predictor_columns = predictor_contract[
    "Predictor"
].astype(str).tolist()

if len(predictor_columns) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract does not contain 151 predictors."
    )

if len(set(predictor_columns)) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract contains duplicate predictors."
    )

missing_training_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_training.columns
]
missing_evaluation_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_evaluation.columns
]

if missing_training_predictors or missing_evaluation_predictors:
    raise RuntimeError(
        "Frozen model cohorts are missing contract predictors."
    )

if any(feature not in predictor_columns for feature in REC_FEATURES):
    raise RuntimeError(
        "The 19 REC features are not all present in the predictor contract."
    )

if set(VERDICT_DEPENDENT_REC).intersection(
    VERDICT_INDEPENDENT_REC
):
    raise RuntimeError(
        "Dependent and independent REC sets overlap."
    )

if set(VERDICT_DEPENDENT_REC + VERDICT_INDEPENDENT_REC) != set(
    REC_FEATURES
):
    raise RuntimeError(
        "Dependent and independent REC sets do not partition all 19 REC features."
    )

print("Converting the fixed predictor cohorts to one numeric matrix.")

training_numeric_frame = model_training[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

evaluation_numeric_frame = model_evaluation[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

training_base_numeric = training_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)
evaluation_base_numeric = evaluation_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)

training_base_numeric[
    ~np.isfinite(training_base_numeric)
] = np.nan
evaluation_base_numeric[
    ~np.isfinite(evaluation_base_numeric)
] = np.nan

all_base_numeric = np.vstack([
    training_base_numeric,
    evaluation_base_numeric,
])

predictor_index = {
    feature: index
    for index, feature in enumerate(predictor_columns)
}

dependent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_DEPENDENT_REC
    ],
    dtype=np.int64,
)

independent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_INDEPENDENT_REC
    ],
    dtype=np.int64,
)

model_all = pd.concat(
    [
        model_training[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        model_evaluation[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
    ],
    ignore_index=True,
)

if model_all.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Model cohort contains duplicate Build-Test rows."
    )

model_key_index = pd.MultiIndex.from_frame(
    model_all[["Build", "Test"]]
)

anchor_offsets = anchor_offsets.copy()
anchor_offsets["Build"] = parse_int(
    anchor_offsets["Build"],
    "anchor_offsets.Build",
)
anchor_offsets["Test"] = parse_int(
    anchor_offsets["Test"],
    "anchor_offsets.Test",
)

if anchor_offsets.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Anchor offsets contain duplicate Build-Test rows."
    )

anchor_indexed = anchor_offsets.set_index(
    [
        "Build",
        "Test",
    ]
)

missing_anchor_keys = model_key_index.difference(
    anchor_indexed.index
)

if len(missing_anchor_keys) != 0:
    raise RuntimeError(
        "Anchor offsets do not cover the full model cohort."
    )

anchor_values_all = anchor_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_reconstructed["Build"] = parse_int(
    clean_reconstructed["Build"],
    "clean_reconstructed.Build",
)
clean_reconstructed["Test"] = parse_int(
    clean_reconstructed["Test"],
    "clean_reconstructed.Test",
)

clean_reconstructed_indexed = clean_reconstructed.set_index(
    [
        "Build",
        "Test",
    ]
)

clean_reconstructed_all = clean_reconstructed_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_anchored_all = (
    clean_reconstructed_all
    + anchor_values_all
)

clean_original_rec_all = model_all[
    REC_FEATURES
].to_numpy(dtype=np.float64)

clean_anchor_mismatch_values = int(
    (~np.isclose(
        clean_anchored_all,
        clean_original_rec_all,
        rtol=0,
        atol=1e-12,
    )).sum()
)

if clean_anchor_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean REC reconstruction plus anchor no longer reproduces the model cohort."
    )

chronology["BuildID"] = parse_int(
    chronology["BuildID"],
    "chronology.BuildID",
)
chronology["ChronologyOrder"] = parse_int(
    chronology["ChronologyOrder"],
    "chronology.ChronologyOrder",
)

build_order_map = (
    chronology.set_index("BuildID")[
        "ChronologyOrder"
    ]
    .astype(int)
    .to_dict()
)

ordered_builds = (
    chronology.sort_values(
        "ChronologyOrder",
        kind="mergesort",
    )["BuildID"]
    .astype(int)
    .tolist()
)

global_build_position = {
    int(build_id): position
    for position, build_id in enumerate(ordered_builds)
}

build_entity["BuildID"] = parse_int(
    build_entity["BuildID"],
    "build_entity.BuildID",
)
build_entity["EntityId"] = parse_int(
    build_entity["EntityId"],
    "build_entity.EntityId",
)

changed_entities_by_build = (
    build_entity.groupby(
        "BuildID"
    )["EntityId"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

changed_entities_by_build = {
    int(build_id): set(
        int(entity_id)
        for entity_id in changed_entities_by_build.get(
            int(build_id),
            set(),
        )
    )
    for build_id in ordered_builds
}

entity_changed_builds = (
    build_entity.groupby(
        "EntityId"
    )["BuildID"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

entity_changed_builds = {
    int(entity_id): set(
        int(build_id)
        for build_id in build_ids
    )
    for entity_id, build_ids in entity_changed_builds.items()
}

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "Job" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Job."
        )
    if "Duration" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Duration."
        )

    frame["Duration"] = pd.to_numeric(
        frame["Duration"],
        errors="coerce",
    )

    if not np.isfinite(
        frame["Duration"].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            f"{label} cohort contains non-finite durations."
        )

    if frame["Duration"].lt(0).any():
        raise RuntimeError(
            f"{label} cohort contains negative durations."
        )

clean_raw_training_verdict = raw_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_raw_evaluation_verdict = raw_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_training_verdict = model_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_verdict = model_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_binary = (
    clean_model_evaluation_verdict != 0
).astype(np.int8)

if int((clean_model_training_verdict != 0).sum()) != EXPECTED_MODEL_TRAIN_FAILURES:
    raise RuntimeError(
        "Clean model-training failure count differs."
    )

if int(clean_model_evaluation_binary.sum()) != EXPECTED_MODEL_EVAL_FAILURES:
    raise RuntimeError(
        "Clean model-evaluation failure count differs."
    )

evaluation_duration = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Duration"].to_numpy(dtype=float)

if not np.isfinite(evaluation_duration).all():
    raise RuntimeError(
        "Model evaluation durations are non-finite."
    )

failing_evaluation_builds = sorted(
    model_evaluation.loc[
        clean_model_evaluation_binary == 1,
        "Build",
    ]
    .astype(int)
    .unique()
    .tolist()
)

if len(failing_evaluation_builds) != EXPECTED_FAILING_EVAL_BUILDS:
    raise RuntimeError(
        "Failing evaluation-build count differs."
    )

evaluation_meta_base = pd.DataFrame({
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Build": model_evaluation["Build"].to_numpy(dtype=np.int64),
    "Test": model_evaluation["Test"].to_numpy(dtype=np.int64),
    "CleanVerdict": clean_model_evaluation_verdict.astype(np.int64),
    "CleanFailure": clean_model_evaluation_binary.astype(np.int8),
    "Duration": evaluation_duration.astype(float),
})

if evaluation_meta_base.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Evaluation metadata contains duplicate Build-Test rows."
    )

random_scores = np.empty(
    EXPECTED_MODEL_EVAL_ROWS,
    dtype=np.float64,
)

for build_id in sorted(
    evaluation_meta_base["Build"].unique()
):
    build_indices = np.flatnonzero(
        evaluation_meta_base["Build"].to_numpy(dtype=np.int64)
        == int(build_id)
    )

    random_scores[build_indices] = np.random.default_rng(
        deterministic_random_build_seed(
            SMOKE_REPETITION_SEED,
            int(build_id),
        )
    ).random(len(build_indices))

if not np.isfinite(random_scores).all():
    raise RuntimeError(
        "Random baseline produced non-finite scores."
    )


# --------------------------------------------------------------------------------------------------
# 8. RUN THE TWO END-TO-END SMOKE CONDITIONS
# --------------------------------------------------------------------------------------------------

SMOKE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

raw_training_hash_before = sha256_file(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation_hash_before = sha256_file(
    RAW_EVALUATION_COHORT_PATH
)
model_training_hash_before = sha256_file(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation_hash_before = sha256_file(
    MODEL_EVALUATION_COHORT_PATH
)

condition_inventory_records = []
all_condition_audits = []
all_project_runs = []
all_build_metrics = []
all_model_fits = []
all_rankings_for_invariance = []

smoke_execution_started = time.perf_counter()

for smoke_index, plan_row in enumerate(
    smoke_plan.itertuples(index=False),
    start=1,
):
    condition_started = time.perf_counter()
    condition_key = str(plan_row.ConditionID)
    noise_percent = int(plan_row.NoisePercent)
    repetition_seed = int(plan_row.RepetitionSeed)

    print("\n" + "-" * 136)
    print(
        f"[{smoke_index}/{EXPECTED_SMOKE_CONDITIONS}] "
        f"Running {condition_key}"
    )
    print("-" * 136)

    condition_dir = (
        SMOKE_ROOT
        / condition_key
    )
    condition_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    ranking_path = (
        condition_dir
        / "rankings.csv.gz"
    )
    build_metrics_path = (
        condition_dir
        / "build_metrics.csv"
    )
    project_runs_path = (
        condition_dir
        / "project_runs.csv"
    )
    model_fits_path = (
        condition_dir
        / "model_fits.csv"
    )
    training_medians_path = (
        condition_dir
        / "training_medians.csv"
    )
    condition_audit_path = (
        condition_dir
        / "condition_audit.csv"
    )
    condition_summary_path = (
        condition_dir
        / "condition_summary.json"
    )
    completion_marker_path = (
        condition_dir
        / "COMPLETE.json"
    )

    flip_mask = (
        flip_uniform
        < (noise_percent / 100.0)
    )

    noisy_raw_training_verdict = clean_raw_training_verdict.copy()

    pass_to_failure_mask = (
        flip_mask
        & (clean_raw_training_verdict == 0)
    )
    failure_to_pass_mask = (
        flip_mask
        & (clean_raw_training_verdict != 0)
    )

    noisy_raw_training_verdict[
        pass_to_failure_mask
    ] = sampled_failure_subtype[
        pass_to_failure_mask
    ]
    noisy_raw_training_verdict[
        failure_to_pass_mask
    ] = 0

    noisy_model_training_verdict = noisy_raw_training_verdict[
        model_training_raw_indices
    ]

    actual_flip_mask_sha256 = sha256_array(
        flip_mask.astype(np.uint8),
        "u1",
    )
    actual_noisy_raw_sha256 = sha256_array(
        noisy_raw_training_verdict,
        "<i2",
    )
    actual_noisy_model_sha256 = sha256_array(
        noisy_model_training_verdict,
        "<i2",
    )

    expected_flip_mask_sha256 = str(
        plan_row.FlipMaskSHA256
    )
    expected_noisy_raw_sha256 = str(
        plan_row.NoisyRawVerdictSHA256
    )
    expected_noisy_model_sha256 = str(
        plan_row.NoisyModelVerdictSHA256
    )

    if actual_flip_mask_sha256 != expected_flip_mask_sha256:
        raise RuntimeError(
            f"{condition_key}: flip-mask SHA-256 differs from Step 3A."
        )

    if actual_noisy_raw_sha256 != expected_noisy_raw_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy raw-verdict SHA-256 differs from Step 3A."
        )

    if actual_noisy_model_sha256 != expected_noisy_model_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy model-verdict SHA-256 differs from Step 3A."
        )

    number_flipped = int(flip_mask.sum())
    pass_to_failure = int(pass_to_failure_mask.sum())
    failure_to_pass = int(failure_to_pass_mask.sum())
    model_label_changes = int(
        (
            noisy_model_training_verdict
            != clean_model_training_verdict
        ).sum()
    )
    noisy_model_training_binary = (
        noisy_model_training_verdict != 0
    ).astype(np.int8)
    noisy_model_training_failures = int(
        noisy_model_training_binary.sum()
    )

    if number_flipped != int(plan_row.NumberFlipped):
        raise RuntimeError(
            f"{condition_key}: NumberFlipped differs from Step 3A."
        )
    if pass_to_failure != int(plan_row.PassToFailure):
        raise RuntimeError(
            f"{condition_key}: PassToFailure differs from Step 3A."
        )
    if failure_to_pass != int(plan_row.FailureToPass):
        raise RuntimeError(
            f"{condition_key}: FailureToPass differs from Step 3A."
        )
    if model_label_changes != int(plan_row.ModelLabelChanges):
        raise RuntimeError(
            f"{condition_key}: ModelLabelChanges differs from Step 3A."
        )
    if noisy_model_training_failures != int(plan_row.NoisyModelFailures):
        raise RuntimeError(
            f"{condition_key}: NoisyModelFailures differs from Step 3A."
        )

    print(
        "  Reconstructing REC features from the condition-specific history."
    )

    train_history = pd.DataFrame({
        "build": raw_training["Build"].to_numpy(dtype=np.int64),
        "test": raw_training["Test"].to_numpy(dtype=np.int64),
        "job": raw_training["Job"].to_numpy(),
        "verdict": noisy_raw_training_verdict.astype(np.int16),
        "duration": raw_training["Duration"].to_numpy(dtype=float),
    })

    evaluation_history = pd.DataFrame({
        "build": raw_evaluation["Build"].to_numpy(dtype=np.int64),
        "test": raw_evaluation["Test"].to_numpy(dtype=np.int64),
        "job": raw_evaluation["Job"].to_numpy(),
        "verdict": clean_raw_evaluation_verdict.astype(np.int16),
        "duration": raw_evaluation["Duration"].to_numpy(dtype=float),
    })

    execution_history = pd.concat(
        [
            train_history,
            evaluation_history,
        ],
        ignore_index=True,
    )

    execution_history["build_order"] = (
        execution_history["build"]
        .map(build_order_map)
    )

    if execution_history["build_order"].isna().any():
        raise RuntimeError(
            f"{condition_key}: execution history has unmapped builds."
        )

    execution_history["build_order"] = (
        execution_history["build_order"]
        .astype("int64")
    )

    execution_history = (
        execution_history.sort_values(
            [
                "build_order",
                "job",
                "test",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    rec_started = time.perf_counter()

    reconstructed = reconstruct_rec_features(
        execution_history=execution_history,
        requested_rows=model_all[["Build", "Test"]],
        global_build_position=global_build_position,
        changed_entities_by_build=changed_entities_by_build,
        entity_changed_builds=entity_changed_builds,
        recent_window=RECENT_WINDOW,
    )

    rec_seconds = time.perf_counter() - rec_started

    if len(reconstructed) != EXPECTED_MODEL_ROWS:
        raise RuntimeError(
            f"{condition_key}: reconstructed REC row count differs."
        )

    if reconstructed.duplicated(
        subset=["Build", "Test"],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: reconstructed REC contains duplicate keys."
        )

    reconstructed["Build"] = parse_int(
        reconstructed["Build"],
        f"{condition_key}.reconstructed.Build",
    )
    reconstructed["Test"] = parse_int(
        reconstructed["Test"],
        f"{condition_key}.reconstructed.Test",
    )

    reconstructed_indexed = reconstructed.set_index(
        [
            "Build",
            "Test",
        ]
    )

    missing_reconstructed_keys = model_key_index.difference(
        reconstructed_indexed.index
    )

    if len(missing_reconstructed_keys) != 0:
        raise RuntimeError(
            f"{condition_key}: reconstruction does not cover all model rows."
        )

    reconstructed_values_all = reconstructed_indexed.loc[
        model_key_index,
        REC_FEATURES,
    ].to_numpy(dtype=np.float64)

    anchored_values_all = (
        reconstructed_values_all
        + anchor_values_all
    )

    independent_reconstruction_mismatches = int(
        (~np.isclose(
            anchored_values_all[:, [
                REC_FEATURES.index(feature)
                for feature in VERDICT_INDEPENDENT_REC
            ]],
            clean_original_rec_all[:, [
                REC_FEATURES.index(feature)
                for feature in VERDICT_INDEPENDENT_REC
            ]],
            rtol=0,
            atol=1e-12,
        )).sum()
    )

    if independent_reconstruction_mismatches != 0:
        raise RuntimeError(
            f"{condition_key}: verdict-independent REC reconstruction changed."
        )

    condition_numeric_all = all_base_numeric.copy()

    dependent_rec_values_all = anchored_values_all[:, [
        REC_FEATURES.index(feature)
        for feature in VERDICT_DEPENDENT_REC
    ]]

    condition_numeric_all[:, dependent_predictor_indices] = (
        dependent_rec_values_all
    )

    condition_training_numeric = condition_numeric_all[
        :EXPECTED_MODEL_TRAIN_ROWS
    ].copy()
    condition_evaluation_numeric = condition_numeric_all[
        EXPECTED_MODEL_TRAIN_ROWS:
    ].copy()

    condition_original_dependent_all = all_base_numeric[
        :, dependent_predictor_indices
    ]

    dependent_rec_changes = int(
        (~np.isclose(
            condition_numeric_all[:, dependent_predictor_indices],
            condition_original_dependent_all,
            rtol=0,
            atol=1e-12,
            equal_nan=True,
        )).sum()
    )

    independent_rec_changes = int(
        (~np.isclose(
            condition_numeric_all[:, independent_predictor_indices],
            all_base_numeric[:, independent_predictor_indices],
            rtol=0,
            atol=0,
            equal_nan=True,
        )).sum()
    )

    if independent_rec_changes != 0:
        raise RuntimeError(
            f"{condition_key}: preserved independent REC predictors changed."
        )

    if noise_percent == 0:
        zero_rec_mismatches = int(
            (~np.isclose(
                condition_numeric_all[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                all_base_numeric[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )).sum()
        )

        if zero_rec_mismatches != 0:
            raise RuntimeError(
                "0% smoke condition did not reproduce the clean REC cohort."
            )

        if number_flipped != 0:
            raise RuntimeError(
                "0% smoke condition unexpectedly flipped raw labels."
            )

        if model_label_changes != 0:
            raise RuntimeError(
                "0% smoke condition unexpectedly changed model labels."
            )

        if dependent_rec_changes != 0:
            raise RuntimeError(
                "0% smoke condition unexpectedly changed dependent REC values."
            )

    if noise_percent > 0:
        if number_flipped <= 0:
            raise RuntimeError(
                "Positive-noise smoke condition changed no raw labels."
            )
        if model_label_changes <= 0:
            raise RuntimeError(
                "Positive-noise smoke condition changed no model labels."
            )
        if dependent_rec_changes <= 0:
            raise RuntimeError(
                "Positive-noise smoke condition changed no dependent REC values."
            )

    medians = np.nanmedian(
        condition_training_numeric,
        axis=0,
    )

    nonfinite_median_indices = np.flatnonzero(
        ~np.isfinite(medians)
    )

    if len(nonfinite_median_indices) != 0:
        bad_features = [
            predictor_columns[index]
            for index in nonfinite_median_indices
        ]
        raise RuntimeError(
            f"{condition_key}: non-finite training medians for {bad_features}."
        )

    training_missing_mask = ~np.isfinite(
        condition_training_numeric
    )
    evaluation_missing_mask = ~np.isfinite(
        condition_evaluation_numeric
    )

    if training_missing_mask.any():
        row_indices, column_indices = np.where(
            training_missing_mask
        )
        condition_training_numeric[
            row_indices,
            column_indices,
        ] = medians[column_indices]

    if evaluation_missing_mask.any():
        row_indices, column_indices = np.where(
            evaluation_missing_mask
        )
        condition_evaluation_numeric[
            row_indices,
            column_indices,
        ] = medians[column_indices]

    if not np.isfinite(condition_training_numeric).all():
        raise RuntimeError(
            f"{condition_key}: training matrix remains non-finite after imputation."
        )

    if not np.isfinite(condition_evaluation_numeric).all():
        raise RuntimeError(
            f"{condition_key}: evaluation matrix remains non-finite after imputation."
        )

    # Keep float64 throughout the smoke test. This matches the frozen Step 4A
    # numeric/imputation contract and avoids changing ranking/model behaviour
    # through an unapproved dtype conversion.

    training_medians = pd.DataFrame({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "PredictorOrder": np.arange(
            1,
            EXPECTED_PREDICTORS + 1,
            dtype=np.int64,
        ),
        "Predictor": predictor_columns,
        "TrainingMedian": medians.astype(float),
    })

    evaluation_meta = evaluation_meta_base.copy()
    evaluation_meta["ConditionKey"] = condition_key
    evaluation_meta["NoisePercent"] = noise_percent
    evaluation_meta["RepetitionSeed"] = repetition_seed

    technique_scores = {}
    model_fit_records = []
    models = create_models(repetition_seed)

    for technique in ML_TECHNIQUES:
        print(f"  Fitting: {technique}")
        model = models[technique]
        fit_started = time.perf_counter()
        fit_status = "PASS_MODEL_FIT"
        fit_error = ""

        try:
            model.fit(
                condition_training_numeric,
                noisy_model_training_binary,
            )

            fit_seconds = time.perf_counter() - fit_started
            scores = positive_probability(
                model,
                condition_evaluation_numeric,
            )

            technique_scores[technique] = scores

        except Exception as error:
            fit_seconds = time.perf_counter() - fit_started
            fit_status = "FAIL_MODEL_FIT"
            fit_error = repr(error)

            model_fit_records.append({
                "ProjectNumber": PROJECT_NUMBER,
                "Project": PROJECT_NAME,
                "ProjectSlug": PROJECT_SLUG,
                "ConditionKey": condition_key,
                "NoisePercent": noise_percent,
                "RepetitionSeed": repetition_seed,
                "Technique": technique,
                "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
                "TrainingFailures": noisy_model_training_failures,
                "Predictors": EXPECTED_PREDICTORS,
                "FitSeconds": float(fit_seconds),
                "ClassesJSON": "[]",
                "Status": fit_status,
                "Error": fit_error,
            })

            raise RuntimeError(
                f"{condition_key}: {technique} fitting failed: {error!r}"
            ) from error

        model_fit_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": condition_key,
            "NoisePercent": noise_percent,
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
            "TrainingFailures": noisy_model_training_failures,
            "Predictors": EXPECTED_PREDICTORS,
            "FitSeconds": float(fit_seconds),
            "ClassesJSON": json.dumps(
                [
                    int(value)
                    for value in np.asarray(model.classes_).tolist()
                ]
            ),
            "Status": fit_status,
            "Error": fit_error,
        })

        del model
        gc.collect()

    model_fits = pd.DataFrame(model_fit_records)

    if len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: model-fit row count differs."
        )

    if not model_fits["Status"].eq("PASS_MODEL_FIT").all():
        raise RuntimeError(
            f"{condition_key}: one or more model fits failed."
        )

    condition_eval_last_failure_age = condition_evaluation_numeric[
        :,
        predictor_index["REC_LastFailureAge"],
    ].astype(float)

    condition_eval_qtf = condition_evaluation_numeric[
        :,
        predictor_index["REC_TotalAvgExeTime"],
    ].astype(float)

    technique_scores["Random"] = random_scores.copy()
    technique_scores["LatestFail"] = (
        -condition_eval_last_failure_age
    )
    technique_scores["QTF-Avg"] = condition_eval_qtf

    ranking_frames = []

    for technique in ALL_TECHNIQUES:
        ranking_frames.append(
            make_ranking(
                evaluation_meta=evaluation_meta,
                technique=technique,
                scores=technique_scores[technique],
                ascending_score=(technique == "QTF-Avg"),
            )
        )

    rankings = pd.concat(
        ranking_frames,
        ignore_index=True,
    )

    if len(rankings) != EXPECTED_RANKING_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: ranking row count differs."
        )

    ranking_techniques = sorted(
        rankings["Technique"].unique().tolist()
    )

    if ranking_techniques != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: ranking technique set differs."
        )

    ranking_rows_per_technique = rankings.groupby(
        "Technique"
    ).size()

    if not ranking_rows_per_technique.eq(
        EXPECTED_MODEL_EVAL_ROWS
    ).all():
        raise RuntimeError(
            f"{condition_key}: ranking rows per technique differ."
        )

    duplicate_ranking_rows = int(
        rankings.duplicated(
            subset=[
                "Technique",
                "Build",
                "Test",
            ],
            keep=False,
        ).sum()
    )

    if duplicate_ranking_rows != 0:
        raise RuntimeError(
            f"{condition_key}: duplicate ranking rows found."
        )

    build_metrics, project_runs = calculate_condition_metrics(
        rankings
    )

    if len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: build-metric row count differs."
        )

    if len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: project-run row count differs."
        )

    if sorted(project_runs["Technique"].tolist()) != sorted(
        ALL_TECHNIQUES
    ):
        raise RuntimeError(
            f"{condition_key}: project-run technique set differs."
        )

    metric_columns = [
        "APFDc",
        "APFD",
    ]

    build_metric_values = build_metrics[
        metric_columns
    ].to_numpy(dtype=float)

    if not np.isfinite(build_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: build metrics contain non-finite values."
        )

    if ((build_metric_values < 0) | (build_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: build metrics fall outside [0,1]."
        )

    project_metric_columns = [
        "MeanAPFDc",
        "MedianAPFDc",
        "MeanAPFD",
        "MedianAPFD",
    ]

    project_metric_values = project_runs[
        project_metric_columns
    ].to_numpy(dtype=float)

    if not np.isfinite(project_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: project metrics contain non-finite values."
        )

    if ((project_metric_values < 0) | (project_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: project metrics fall outside [0,1]."
        )

    condition_seconds = time.perf_counter() - condition_started

    condition_audit = pd.DataFrame([{
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "RawTrainingRows": EXPECTED_RAW_TRAIN_ROWS,
        "NumberFlipped": number_flipped,
        "ExpectedNumberFlipped": int(plan_row.NumberFlipped),
        "RealisedNoisePercent": float(
            100.0 * number_flipped / EXPECTED_RAW_TRAIN_ROWS
        ),
        "PassToFailure": pass_to_failure,
        "FailureToPass": failure_to_pass,
        "ModelTrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
        "ModelLabelChanges": model_label_changes,
        "ExpectedModelLabelChanges": int(plan_row.ModelLabelChanges),
        "TrainingFailures": noisy_model_training_failures,
        "ExpectedTrainingFailures": int(plan_row.NoisyModelFailures),
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "IndependentReconstructionMismatches": independent_reconstruction_mismatches,
        "ReconstructedRows": int(len(reconstructed)),
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ExpectedFlipMaskSHA256": expected_flip_mask_sha256,
        "ActualFlipMaskSHA256": actual_flip_mask_sha256,
        "ExpectedNoisyRawVerdictSHA256": expected_noisy_raw_sha256,
        "ActualNoisyRawVerdictSHA256": actual_noisy_raw_sha256,
        "ExpectedNoisyModelVerdictSHA256": expected_noisy_model_sha256,
        "ActualNoisyModelVerdictSHA256": actual_noisy_model_sha256,
        "RECSeconds": float(rec_seconds),
        "ConditionSeconds": float(condition_seconds),
        "Status": CONDITION_STATUS,
    }])

    atomic_csv(
        ranking_path,
        rankings,
        compression="gzip",
    )
    atomic_csv(
        build_metrics_path,
        build_metrics,
    )
    atomic_csv(
        project_runs_path,
        project_runs,
    )
    atomic_csv(
        model_fits_path,
        model_fits,
    )
    atomic_csv(
        training_medians_path,
        training_medians,
    )
    atomic_csv(
        condition_audit_path,
        condition_audit,
    )

    condition_output_paths = [
        ranking_path,
        build_metrics_path,
        project_runs_path,
        model_fits_path,
        training_medians_path,
        condition_audit_path,
    ]

    condition_output_manifest = [
        {
            "Path": str(path),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        }
        for path in condition_output_paths
    ]

    condition_summary = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "Status": CONDITION_STATUS,
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
        "NumberFlipped": number_flipped,
        "ModelLabelChanges": model_label_changes,
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "TrainingFailures": noisy_model_training_failures,
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
        "OutputManifest": condition_output_manifest,
    }

    atomic_json(
        condition_summary_path,
        condition_summary,
    )

    completion_marker = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "Status": CONDITION_STATUS,
        "ConditionSummaryPath": str(condition_summary_path),
        "ConditionSummarySHA256": sha256_file(condition_summary_path),
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
    }

    atomic_json(
        completion_marker_path,
        completion_marker,
    )

    condition_manifest = directory_manifest(
        condition_dir
    )
    condition_root_sha256 = directory_root_hash(
        condition_manifest
    )

    condition_inventory_records.append({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": smoke_index,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "ConditionDirectory": str(condition_dir),
        "Status": CONDITION_STATUS,
        "Files": int(len(condition_manifest)),
        "ConditionBytes": int(condition_manifest["Bytes"].sum()),
        "ConditionRootSHA256": condition_root_sha256,
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "ModelFitRows": int(len(model_fits)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
    })

    all_condition_audits.append(condition_audit)
    all_project_runs.append(project_runs)
    all_build_metrics.append(build_metrics)
    all_model_fits.append(model_fits)
    all_rankings_for_invariance.append(
        rankings.loc[
            rankings["Technique"].isin(
                [
                    "Random",
                    "QTF-Avg",
                ]
            )
        ].copy()
    )

    print(
        f"  Completed: {condition_key}\n"
        f"  Raw flips: {number_flipped} | "
        f"model-label changes: {model_label_changes} | "
        f"dependent REC changes: {dependent_rec_changes}\n"
        f"  Training failures: {noisy_model_training_failures} | "
        f"condition seconds: {condition_seconds:.2f}"
    )

    del train_history
    del evaluation_history
    del execution_history
    del reconstructed
    del reconstructed_indexed
    del reconstructed_values_all
    del anchored_values_all
    del condition_numeric_all
    del condition_training_numeric
    del condition_evaluation_numeric
    del rankings
    del ranking_frames
    del technique_scores
    del models
    gc.collect()


# --------------------------------------------------------------------------------------------------
# 9. COMBINE SMOKE OUTPUTS AND VERIFY BASELINE INVARIANCE
# --------------------------------------------------------------------------------------------------

condition_inventory = pd.DataFrame(
    condition_inventory_records
)
combined_condition_audit = pd.concat(
    all_condition_audits,
    ignore_index=True,
)
combined_project_runs = pd.concat(
    all_project_runs,
    ignore_index=True,
)
combined_build_metrics = pd.concat(
    all_build_metrics,
    ignore_index=True,
)
combined_model_fits = pd.concat(
    all_model_fits,
    ignore_index=True,
)
combined_invariance_rankings = pd.concat(
    all_rankings_for_invariance,
    ignore_index=True,
)

baseline_invariance_records = []

for technique in [
    "Random",
    "QTF-Avg",
]:
    zero_rows = (
        combined_invariance_rankings.loc[
            (
                combined_invariance_rankings["Technique"].eq(technique)
                & combined_invariance_rankings["NoisePercent"].eq(0)
            )
        ]
        .sort_values(
            [
                "Build",
                "Test",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    noisy_rows = (
        combined_invariance_rankings.loc[
            (
                combined_invariance_rankings["Technique"].eq(technique)
                & combined_invariance_rankings["NoisePercent"].eq(50)
            )
        ]
        .sort_values(
            [
                "Build",
                "Test",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    same_keys = bool(
        zero_rows[["Build", "Test"]].equals(
            noisy_rows[["Build", "Test"]]
        )
    )

    score_mismatches = (
        EXPECTED_MODEL_EVAL_ROWS
        if not same_keys
        else int(
            (~np.isclose(
                zero_rows["Score"].to_numpy(dtype=float),
                noisy_rows["Score"].to_numpy(dtype=float),
                rtol=0,
                atol=0,
            )).sum()
        )
    )

    rank_mismatches = (
        EXPECTED_MODEL_EVAL_ROWS
        if not same_keys
        else int(
            (
                zero_rows["Rank"].to_numpy(dtype=np.int64)
                != noisy_rows["Rank"].to_numpy(dtype=np.int64)
            ).sum()
        )
    )

    baseline_invariance_records.append({
        "Technique": technique,
        "Rows": int(len(zero_rows)),
        "SameBuildTestKeys": same_keys,
        "ScoreMismatches": score_mismatches,
        "RankMismatches": rank_mismatches,
        "Pass": (
            len(zero_rows) == EXPECTED_MODEL_EVAL_ROWS
            and len(noisy_rows) == EXPECTED_MODEL_EVAL_ROWS
            and same_keys
            and score_mismatches == 0
            and rank_mismatches == 0
        ),
    })

baseline_invariance = pd.DataFrame(
    baseline_invariance_records
)
baseline_invariance_failures = int(
    (~baseline_invariance["Pass"]).sum()
)

if baseline_invariance_failures != 0:
    print("\nBaseline invariance failures:")
    display(
        baseline_invariance.loc[
            ~baseline_invariance["Pass"]
        ]
    )
    raise RuntimeError(
        "Random or QTF-Avg changed across the two smoke noise levels."
    )

atomic_csv(
    SMOKE_CONDITION_INVENTORY_PATH,
    condition_inventory,
)
atomic_csv(
    SMOKE_COMBINED_CONDITION_AUDIT_PATH,
    combined_condition_audit,
)
atomic_csv(
    SMOKE_COMBINED_PROJECT_RUNS_PATH,
    combined_project_runs,
)
atomic_csv(
    SMOKE_COMBINED_BUILD_METRICS_PATH,
    combined_build_metrics,
)
atomic_csv(
    SMOKE_COMBINED_MODEL_FITS_PATH,
    combined_model_fits,
)
atomic_csv(
    SMOKE_BASELINE_INVARIANCE_PATH,
    baseline_invariance,
)


# --------------------------------------------------------------------------------------------------
# 10. FINAL VALIDATION
# --------------------------------------------------------------------------------------------------

raw_training_hash_after = sha256_file(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation_hash_after = sha256_file(
    RAW_EVALUATION_COHORT_PATH
)
model_training_hash_after = sha256_file(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation_hash_after = sha256_file(
    MODEL_EVALUATION_COHORT_PATH
)
registry_sha256_after = sha256_file(
    REGISTRY_PATH
)

current_source_rows_after = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = Path(row.AbsolutePath)
    current_source_rows_after.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

source_root_sha256_after = source_root_hash(
    pd.DataFrame(current_source_rows_after)
)

full_raw_result_root_exists_after = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_after = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_after = directory_root_hash(
    full_raw_result_manifest_after
)

full_raw_result_unchanged = bool(
    full_raw_result_root_existed_before
    == full_raw_result_root_exists_after
    and full_raw_result_root_hash_before
    == full_raw_result_root_hash_after
    and len(full_raw_result_manifest_before)
    == len(full_raw_result_manifest_after)
)

zero_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(0)
].iloc[0]

positive_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(50)
].iloc[0]

validation_records = []

add_check(
    validation_records,
    "Step 4A status",
    EXPECTED_STEP4A_STATUS,
    step4a_status.get("Status"),
    step4a_status.get("Status") == EXPECTED_STEP4A_STATUS,
)
add_check(
    validation_records,
    "Runtime checkpoint SHA-256",
    EXPECTED_RUNTIME_CHECKPOINT_SHA256,
    runtime_checkpoint_sha256,
    runtime_checkpoint_sha256 == EXPECTED_RUNTIME_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "Noise-plan checkpoint SHA-256",
    EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    noise_plan_checkpoint_sha256,
    noise_plan_checkpoint_sha256 == EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "REC checkpoint SHA-256",
    EXPECTED_REC_CHECKPOINT_SHA256,
    rec_checkpoint_sha256,
    rec_checkpoint_sha256 == EXPECTED_REC_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_CHECKPOINT_SHA256,
    selection_checkpoint_sha256,
    selection_checkpoint_sha256 == EXPECTED_SELECTION_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "Step 4A output-manifest failures",
    0,
    runtime_manifest_failures,
    runtime_manifest_failures == 0,
)
add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    source_root_sha256_after,
    source_root_sha256_after == EXPECTED_SOURCE_ROOT_SHA256,
)
add_check(
    validation_records,
    "Smoke conditions",
    EXPECTED_SMOKE_CONDITIONS,
    len(condition_inventory),
    len(condition_inventory) == EXPECTED_SMOKE_CONDITIONS,
)
add_check(
    validation_records,
    "Smoke condition keys",
    SMOKE_CONDITION_IDS,
    condition_inventory["ConditionKey"].tolist(),
    condition_inventory["ConditionKey"].tolist() == SMOKE_CONDITION_IDS,
)
add_check(
    validation_records,
    "Condition statuses",
    CONDITION_STATUS,
    sorted(condition_inventory["Status"].unique().tolist()),
    condition_inventory["Status"].eq(CONDITION_STATUS).all(),
)
add_check(
    validation_records,
    "Condition-audit rows",
    EXPECTED_SMOKE_CONDITIONS,
    len(combined_condition_audit),
    len(combined_condition_audit) == EXPECTED_SMOKE_CONDITIONS,
)
add_check(
    validation_records,
    "Total ML fits",
    EXPECTED_SMOKE_CONDITIONS * EXPECTED_ML_TECHNIQUES,
    len(combined_model_fits),
    len(combined_model_fits)
    == EXPECTED_SMOKE_CONDITIONS * EXPECTED_ML_TECHNIQUES,
)
add_check(
    validation_records,
    "Model-fit failures",
    0,
    int((~combined_model_fits["Status"].eq("PASS_MODEL_FIT")).sum()),
    combined_model_fits["Status"].eq("PASS_MODEL_FIT").all(),
)
add_check(
    validation_records,
    "Total project-run rows",
    EXPECTED_SMOKE_CONDITIONS * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
    len(combined_project_runs),
    len(combined_project_runs)
    == EXPECTED_SMOKE_CONDITIONS * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
)
add_check(
    validation_records,
    "Total build-metric rows",
    EXPECTED_SMOKE_CONDITIONS * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
    len(combined_build_metrics),
    len(combined_build_metrics)
    == EXPECTED_SMOKE_CONDITIONS * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
)
add_check(
    validation_records,
    "Technique set",
    sorted(ALL_TECHNIQUES),
    sorted(combined_project_runs["Technique"].unique().tolist()),
    sorted(combined_project_runs["Technique"].unique().tolist())
    == sorted(ALL_TECHNIQUES),
)
add_check(
    validation_records,
    "Project-run rows per condition violations",
    0,
    int((
        combined_project_runs.groupby("ConditionKey").size()
        != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
    ).sum()),
    bool((
        combined_project_runs.groupby("ConditionKey").size()
        == EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
    ).all()),
)
add_check(
    validation_records,
    "Build-metric rows per condition violations",
    0,
    int((
        combined_build_metrics.groupby("ConditionKey").size()
        != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
    ).sum()),
    bool((
        combined_build_metrics.groupby("ConditionKey").size()
        == EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
    ).all()),
)
add_check(
    validation_records,
    "Zero-noise raw flips",
    0,
    int(zero_audit["NumberFlipped"]),
    int(zero_audit["NumberFlipped"]) == 0,
)
add_check(
    validation_records,
    "Zero-noise model-label changes",
    0,
    int(zero_audit["ModelLabelChanges"]),
    int(zero_audit["ModelLabelChanges"]) == 0,
)
add_check(
    validation_records,
    "Zero-noise dependent REC changes",
    0,
    int(zero_audit["DependentRECChanges"]),
    int(zero_audit["DependentRECChanges"]) == 0,
)
add_check(
    validation_records,
    "Positive-noise raw flips",
    "> 0",
    int(positive_audit["NumberFlipped"]),
    int(positive_audit["NumberFlipped"]) > 0,
)
add_check(
    validation_records,
    "Positive-noise model-label changes",
    "> 0",
    int(positive_audit["ModelLabelChanges"]),
    int(positive_audit["ModelLabelChanges"]) > 0,
)
add_check(
    validation_records,
    "Positive-noise dependent REC changes",
    "> 0",
    int(positive_audit["DependentRECChanges"]),
    int(positive_audit["DependentRECChanges"]) > 0,
)
add_check(
    validation_records,
    "Independent REC changes",
    0,
    int(combined_condition_audit["IndependentRECChanges"].sum()),
    int(combined_condition_audit["IndependentRECChanges"].sum()) == 0,
)
add_check(
    validation_records,
    "Independent REC reconstruction mismatches",
    0,
    int(combined_condition_audit[
        "IndependentReconstructionMismatches"
    ].sum()),
    int(combined_condition_audit[
        "IndependentReconstructionMismatches"
    ].sum()) == 0,
)
add_check(
    validation_records,
    "Noise-plan hash mismatches",
    0,
    int((
        combined_condition_audit["ExpectedFlipMaskSHA256"]
        != combined_condition_audit["ActualFlipMaskSHA256"]
    ).sum())
    + int((
        combined_condition_audit["ExpectedNoisyRawVerdictSHA256"]
        != combined_condition_audit["ActualNoisyRawVerdictSHA256"]
    ).sum())
    + int((
        combined_condition_audit["ExpectedNoisyModelVerdictSHA256"]
        != combined_condition_audit["ActualNoisyModelVerdictSHA256"]
    ).sum()),
    bool(
        (
            combined_condition_audit["ExpectedFlipMaskSHA256"]
            == combined_condition_audit["ActualFlipMaskSHA256"]
        ).all()
        and (
            combined_condition_audit["ExpectedNoisyRawVerdictSHA256"]
            == combined_condition_audit["ActualNoisyRawVerdictSHA256"]
        ).all()
        and (
            combined_condition_audit["ExpectedNoisyModelVerdictSHA256"]
            == combined_condition_audit["ActualNoisyModelVerdictSHA256"]
        ).all()
    ),
)
add_check(
    validation_records,
    "Baseline invariance failures",
    0,
    baseline_invariance_failures,
    baseline_invariance_failures == 0,
)
add_check(
    validation_records,
    "Project metrics non-finite",
    0,
    int((~np.isfinite(
        combined_project_runs[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float)
    )).sum()),
    bool(np.isfinite(
        combined_project_runs[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float)
    ).all()),
)
add_check(
    validation_records,
    "Project metrics outside [0,1]",
    0,
    int((
        (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            < 0
        )
        | (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            > 1
        )
    ).sum()),
    bool((
        (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            >= 0
        )
        & (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            <= 1
        )
    ).all()),
)
add_check(
    validation_records,
    "Raw training cohort unchanged",
    raw_training_hash_before,
    raw_training_hash_after,
    raw_training_hash_after == raw_training_hash_before,
)
add_check(
    validation_records,
    "Raw evaluation cohort unchanged",
    raw_evaluation_hash_before,
    raw_evaluation_hash_after,
    raw_evaluation_hash_after == raw_evaluation_hash_before,
)
add_check(
    validation_records,
    "Model training cohort unchanged",
    model_training_hash_before,
    model_training_hash_after,
    model_training_hash_after == model_training_hash_before,
)
add_check(
    validation_records,
    "Model evaluation cohort unchanged",
    model_evaluation_hash_before,
    model_evaluation_hash_after,
    model_evaluation_hash_after == model_evaluation_hash_before,
)
add_check(
    validation_records,
    "Completion registry unchanged",
    registry_sha256_before,
    registry_sha256_after,
    registry_sha256_after == registry_sha256_before,
)
add_check(
    validation_records,
    "Registry Project 11 rows",
    0,
    int(registry_project_numbers.eq(PROJECT_NUMBER).sum()),
    int(registry_project_numbers.eq(PROJECT_NUMBER).sum()) == 0,
)
add_check(
    validation_records,
    "Full raw-result root unchanged",
    True,
    full_raw_result_unchanged,
    full_raw_result_unchanged,
)

validation = pd.DataFrame(
    validation_records
)
failed_validation = validation.loc[
    ~validation["Pass"]
]

print("\nProject 11 Step 4B validation:")
display(validation)

print("\nBaseline invariance audit:")
display(baseline_invariance)

print("\nSmoke project-run results:")
display(
    combined_project_runs.sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    ).reset_index(drop=True)
)

if not failed_validation.empty:
    print("\nFailed Project 11 Step 4B checks:")
    display(failed_validation)
    print("\nNo Step 4B PASS status or checkpoint was written.")
    raise RuntimeError(
        "PROJECT 11 STEP 4B VALIDATION FAILED. DO NOT START THE FULL EXPERIMENT."
    )

atomic_csv(
    SMOKE_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 11. REPORT, CHECKPOINT, STATUS, AND FINAL READBACK
# --------------------------------------------------------------------------------------------------

smoke_execution_seconds = (
    time.perf_counter()
    - smoke_execution_started
)
completed_at_utc = datetime.now(
    timezone.utc
).isoformat()

smoke_output_paths = [
    SMOKE_CONDITION_INVENTORY_PATH,
    SMOKE_COMBINED_CONDITION_AUDIT_PATH,
    SMOKE_COMBINED_PROJECT_RUNS_PATH,
    SMOKE_COMBINED_BUILD_METRICS_PATH,
    SMOKE_COMBINED_MODEL_FITS_PATH,
    SMOKE_BASELINE_INVARIANCE_PATH,
    SMOKE_VALIDATION_PATH,
]

for condition_key in SMOKE_CONDITION_IDS:
    condition_dir = SMOKE_ROOT / condition_key
    smoke_output_paths.extend([
        path
        for path in condition_dir.rglob("*")
        if path.is_file()
    ])

smoke_output_paths = sorted(
    set(smoke_output_paths),
    key=lambda path: str(path),
)

smoke_output_manifest = [
    {
        "Path": str(path),
        "Bytes": int(path.stat().st_size),
        "SHA256": sha256_file(path),
    }
    for path in smoke_output_paths
]

report_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP4B_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "RuntimeCheckpointSHA256": runtime_checkpoint_sha256,
    "NoisePlanCheckpointSHA256": noise_plan_checkpoint_sha256,
    "RECCheckpointSHA256": rec_checkpoint_sha256,
    "SelectionCheckpointSHA256": selection_checkpoint_sha256,
    "SourceRootSHA256": source_root_sha256_after,
    "SmokeConditionKeys": SMOKE_CONDITION_IDS,
    "SmokeConditions": int(len(condition_inventory)),
    "MLFits": int(len(combined_model_fits)),
    "RankingRows": int(condition_inventory["RankingRows"].sum()),
    "BuildMetricRows": int(len(combined_build_metrics)),
    "ProjectRunRows": int(len(combined_project_runs)),
    "TrainingMedianRows": int(
        condition_inventory["TrainingMedianRows"].sum()
    ),
    "ZeroNoiseRawFlips": int(zero_audit["NumberFlipped"]),
    "ZeroNoiseModelLabelChanges": int(
        zero_audit["ModelLabelChanges"]
    ),
    "ZeroNoiseDependentRECChanges": int(
        zero_audit["DependentRECChanges"]
    ),
    "PositiveNoiseRawFlips": int(
        positive_audit["NumberFlipped"]
    ),
    "PositiveNoiseModelLabelChanges": int(
        positive_audit["ModelLabelChanges"]
    ),
    "PositiveNoiseDependentRECChanges": int(
        positive_audit["DependentRECChanges"]
    ),
    "IndependentRECChanges": int(
        combined_condition_audit["IndependentRECChanges"].sum()
    ),
    "BaselineInvarianceFailures": baseline_invariance_failures,
    "ValidationChecks": int(len(validation)),
    "FailedValidationChecks": int(len(failed_validation)),
    "SmokeExecutionSeconds": float(smoke_execution_seconds),
    "OutputManifest": smoke_output_manifest,
    "RegistrySHA256": registry_sha256_after,
    "RegistryModified": False,
    "Projects1To10Modified": False,
    "Project12Accessed": False,
    "FullRawResultRootModified": False,
    "FullExperimentStarted": False,
}

atomic_json(
    SMOKE_REPORT_PATH,
    report_payload,
)

checkpoint_payload = {
    **report_payload,
    "CheckpointVersion": 1,
    "SmokeTestPassed": True,
    "RuntimeContractFrozen": True,
    "NoisePlanFrozen": True,
    "EvaluationCohortImmutable": True,
    "ReadyForFull270ConditionExperiment": True,
}

atomic_json(
    SMOKE_CHECKPOINT_PATH,
    checkpoint_payload,
)

status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP4B_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "SmokeConditions": int(len(condition_inventory)),
    "MLFits": int(len(combined_model_fits)),
    "FailedValidationChecks": int(len(failed_validation)),
    "Checkpoint": str(SMOKE_CHECKPOINT_PATH),
    "CheckpointSHA256": sha256_file(SMOKE_CHECKPOINT_PATH),
    "RegistryModified": False,
    "Project12Accessed": False,
    "FullExperimentStarted": False,
}

atomic_json(
    STEP4B_STATUS_PATH,
    status_payload,
)

checkpoint_readback = load_json(
    SMOKE_CHECKPOINT_PATH
)
status_readback = load_json(
    STEP4B_STATUS_PATH
)

if checkpoint_readback.get("Status") != STEP4B_STATUS:
    raise RuntimeError(
        "Step 4B checkpoint readback failed."
    )

if status_readback.get("Status") != STEP4B_STATUS:
    raise RuntimeError(
        "Step 4B status readback failed."
    )

if sha256_file(REGISTRY_PATH) != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Step 4B finalisation."
    )

final_source_rows = []
for row in frozen_source_manifest.itertuples(index=False):
    source_path = Path(row.AbsolutePath)
    final_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

if source_root_hash(pd.DataFrame(final_source_rows)) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Project 11 source changed during Step 4B finalisation."
    )

print("\n" + "=" * 136)
print("=== PROJECT 11 CELL 8 / STEP 4B RESULT ===")
print("=" * 136)
print()
print("Project:")
print(PROJECT_NAME)
print()
print("Two-condition end-to-end smoke test:")
print("Conditions:", SMOKE_CONDITION_IDS)
print("Conditions passed:", len(condition_inventory), "/", EXPECTED_SMOKE_CONDITIONS)
print("ML fits:", len(combined_model_fits), "/", EXPECTED_SMOKE_CONDITIONS * EXPECTED_ML_TECHNIQUES)
print("Ranking rows:", int(condition_inventory["RankingRows"].sum()))
print("Build-metric rows:", len(combined_build_metrics))
print("Project-run rows:", len(combined_project_runs))
print("Training-median rows:", int(condition_inventory["TrainingMedianRows"].sum()))
print()
print("Noise and REC audit:")
print("0% raw flips:", int(zero_audit["NumberFlipped"]))
print("0% model-label changes:", int(zero_audit["ModelLabelChanges"]))
print("0% dependent REC changes:", int(zero_audit["DependentRECChanges"]))
print("50% raw flips:", int(positive_audit["NumberFlipped"]))
print("50% model-label changes:", int(positive_audit["ModelLabelChanges"]))
print("50% dependent REC changes:", int(positive_audit["DependentRECChanges"]))
print("Independent REC changes:", int(combined_condition_audit["IndependentRECChanges"].sum()))
print()
print("Baselines and metrics:")
print("Random/QTF-Avg invariance failures:", baseline_invariance_failures)
print("Techniques:", ALL_TECHNIQUES)
print("Primary / secondary metrics: APFDc / APFD")
print()
print("Immutability and isolation:")
print("Project 11 source unchanged:", True)
print("Completion registry unchanged:", True)
print("Projects 1-10 modified:", 0)
print("Project 12 accessed:", False)
print("Full experiment raw-result root modified:", False)
print("Full 270-condition experiment started:", False)
print()
print("Validation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed_validation))
print()
print("Smoke-test checkpoint:")
print(SMOKE_CHECKPOINT_PATH)
print("Checkpoint SHA-256:", sha256_file(SMOKE_CHECKPOINT_PATH))
print()
print("Runtime seconds:", round(smoke_execution_seconds, 2))
print()
print("STATUS:", STEP4B_STATUS)
print("=" * 136)

=== PROJECT 11 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===

Loading frozen Project 11 cohorts and contracts.
Converting the fixed predictor cohorts to one numeric matrix.

----------------------------------------------------------------------------------------------------------------------------------------
[1/2] Running noise_00__seed_01
----------------------------------------------------------------------------------------------------------------------------------------
  Reconstructing REC features from the condition-specific history.
    REC reconstruction progress: 100 / 1020 tests | reconstructed rows: 11483
    REC reconstruction progress: 200 / 1020 tests | reconstructed rows: 22008
    REC reconstruction progress: 300 / 1020 tests | reconstructed rows: 33877
    REC reconstruction progress: 400 / 1020 tests | reconstructed rows: 44042
    REC reconstruction progress: 500 / 1020 tests | reconstructed rows: 55474
    REC reconstruction progress: 600 / 1020 tests |

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_00__seed_01
  Raw flips: 0 | model-label changes: 0 | dependent REC changes: 0
  Training failures: 1188 | condition seconds: 495.59

----------------------------------------------------------------------------------------------------------------------------------------
[2/2] Running noise_50__seed_01
----------------------------------------------------------------------------------------------------------------------------------------
  Reconstructing REC features from the condition-specific history.
    REC reconstruction progress: 100 / 1020 tests | reconstructed rows: 11483
    REC reconstruction progress: 200 / 1020 tests | reconstructed rows: 22008
    REC reconstruction progress: 300 / 1020 tests | reconstructed rows: 33877
    REC reconstruction progress: 400 / 1020 tests | reconstructed rows: 44042
    REC reconstruction progress: 500 / 1020 tests | reconstructed rows: 55474
    REC reconstruction progress: 600 / 1020 tests | reconstruc

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_50__seed_01
  Raw flips: 304249 | model-label changes: 39007 | dependent REC changes: 969129
  Training failures: 39021 | condition seconds: 528.15

Project 11 Step 4B validation:


...


Baseline invariance audit:


,Technique,Rows,SameBuildTestKeys,ScoreMismatches,RankMismatches,Pass
0,Random,13007,True,0,0,True
1,QTF-Avg,13007,True,0,0,True



Smoke project-run results:


,ProjectNumber,Project,ProjectSlug,ConditionKey,NoisePercent,RepetitionSeed,Technique,EvaluationBuilds,ScoredFailingBuilds,EvaluationRows,EvaluationFailures,MeanAPFDc,MedianAPFDc,MeanAPFD,MedianAPFD
0,11,apache@shardingsphere,apache__shardingsphere,noise_00__seed_01,0,1,LatestFail,263,25,13007,171,0.117362,0.039770,0.038628,0.017391
1,11,apache@shardingsphere,apache__shardingsphere,noise_00__seed_01,0,1,LightGBM,263,25,13007,171,0.956119,0.992583,0.976796,0.989297
2,11,apache@shardingsphere,apache__shardingsphere,noise_00__seed_01,0,1,NaiveBayes,263,25,13007,171,0.801712,0.902729,0.952807,0.977501
3,11,apache@shardingsphere,apache__shardingsphere,noise_00__seed_01,0,1,QTF-Avg,263,25,13007,171,0.873847,0.867396,0.289495,0.297862
4,11,apache@shardingsphere,apache__shardingsphere,noise_00__seed_01,0,1,Random,263,25,13007,171,0.528321,0.515621,0.520834,0.534237
5,11,apache@shardingsphere,apache__shardingsphere,noise_00__seed_01,0,1,RandomForest,263,25,13007,171,0.985736,0.991926,0.990612,0.989297
6,11,apache@shardingsphere,apache__shardingsphere,noise_00__seed_01,0,1,XGBoost,263,25,13007,171,0.971303,0.991971,0.978499,0.989297
7,11,apache@shardingsphere,apache__shardingsphere,noise_50__seed_01,50,1,LatestFail,263,25,13007,171,0.924312,0.975085,0.933168,0.971822
8,11,apache@shardingsphere,apache__shardingsphere,noise_50__seed_01,50,1,LightGBM,263,25,13007,171,0.409098,0.359097,0.364167,0.346953
9,11,apache@shardingsphere,apache__shardingsphere,noise_50__seed_01,50,1,NaiveBayes,263,25,13007,171,0.044696,0.020919,0.068112,0.035168



=== PROJECT 11 CELL 8 / STEP 4B RESULT ===

Project:
apache@shardingsphere

Two-condition end-to-end smoke test:
Conditions: ['noise_00__seed_01', 'noise_50__seed_01']
Conditions passed: 2 / 2
ML fits: 8 / 8
Ranking rows: 182098
Build-metric rows: 350
Project-run rows: 14
Training-median rows: 302

Noise and REC audit:
0% raw flips: 0
0% model-label changes: 0
0% dependent REC changes: 0
50% raw flips: 304249
50% model-label changes: 39007
50% dependent REC changes: 969129
Independent REC changes: 0

Baselines and metrics:
Random/QTF-Avg invariance failures: 0
Techniques: ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes', 'Random', 'LatestFail', 'QTF-Avg']
Primary / secondary metrics: APFDc / APFD

Immutability and isolation:
Project 11 source unchanged: True
Completion registry unchanged: True
Projects 1-10 modified: 0
Project 12 accessed: False
Full experiment raw-result root modified: False
Full 270-condition experiment started: False

Validation:
Checks: 37
Failed checks: 0

Sm

In [2]:
# ==================================================================================================
# PROJECT 11 — CELL 9 / STEP 5A ACCELERATED
# CHECKPOINTED FULL 270-CONDITION EXPERIMENT WITH VECTORIZED REC ENGINE
#
# PROJECT:
#   apache@shardingsphere
#
# PURPOSE:
# - validate the frozen Step 4B smoke-test checkpoint and every upstream contract;
# - validate a vectorized REC engine against the exact frozen Step 4B smoke outputs;
# - execute all 270 noise/seed conditions with scientifically identical inputs and outputs;
# - checkpoint each completed condition independently using atomic output files;
# - resume safely after a Colab disconnect by skipping only fully validated conditions;
# - fit the four frozen ML techniques and evaluate the three frozen baselines;
# - write ranked-test, build-metric, project-run, model-fit, median, and audit outputs;
# - freeze the complete raw-result root for independent Step 5B revalidation.
#
# SAFETY:
# - no registry write;
# - no modification of Projects 1–10;
# - no Project 12 access;
# - clean evaluation data remain immutable;
# - incomplete condition outputs are preserved in a quarantine directory before rerun.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import gc
import hashlib
import json
import os
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from pandas.errors import PerformanceWarning
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.simplefilter("ignore", PerformanceWarning)


print("=" * 136)
print("=== PROJECT 11 CELL 9 / STEP 5A ACCELERATED: CHECKPOINTED FULL 270-CONDITION EXPERIMENT ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 11
PROJECT_NAME = "apache@shardingsphere"
PROJECT_SLUG = "apache__shardingsphere"
PROJECT_SHORT = "shardingsphere"

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_11_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)
EXPECTED_STEP4B_STATUS = (
    "PASS_PROJECT_11_TWO_CONDITION_END_TO_END_SMOKE_TEST"
)
STEP5A_STATUS = (
    "PASS_PROJECT_11_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
)
CONDITION_STATUS = "PASS_FULL_CONDITION"

EXPECTED_RUNTIME_CHECKPOINT_SHA256 = (
    "606167d8df76dc04e91ece3c0969d0ceb46319930713034a066c3a06c2015ddb"
)
EXPECTED_SMOKE_CHECKPOINT_SHA256 = (
    "eadb99c58f85b5c9597874e01f7e7ce54d3abe7f5b535ed61fe1d686f6e8a7c4"
)
EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "086da91a6e4ab96fce51fb9749ce3581eeb2c303668995877de88ec4f5c2d3fa"
)
EXPECTED_REC_CHECKPOINT_SHA256 = (
    "a1df2018227f643616169017b5d6a35fb40431c325821d7d5acda8e40701bfa0"
)
EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "9d5c95b29357426def26c41b62a753c003ad3b19e6086d9873f0961e39875353"
)
EXPECTED_SOURCE_ROOT_SHA256 = (
    "3e6e0a60d58bff61690d6e2ac2bac18371414819eff601f3ea347ecb8ad8b0b5"
)
EXPECTED_REGISTRY_SHA256 = (
    "847bdcbee16c8757fde34e9489cd3abbf0781a319644175472597265b1df2750"
)

EXPECTED_RAW_TRAIN_ROWS = 609_619
EXPECTED_RAW_EVAL_ROWS = 223_922
EXPECTED_MODEL_TRAIN_ROWS = 78_035
EXPECTED_MODEL_EVAL_ROWS = 13_007
EXPECTED_MODEL_ROWS = 91_042
EXPECTED_MODEL_TRAIN_FAILURES = 1_188
EXPECTED_MODEL_EVAL_FAILURES = 171
EXPECTED_FAILING_EVAL_BUILDS = 25
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_EVALUATION_BUILDS = 263
EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = EXPECTED_CONDITIONS * EXPECTED_FILES_PER_CONDITION
EXPECTED_RNG_ROWS = 18_288_570
ACCELERATED_ENGINE_VERSION = "PROJECT_11_FAST_DEPENDENT_REC_V1"
SMOKE_EQUIVALENCE_KEYS = [
    "noise_00__seed_01",
    "noise_50__seed_01",
]

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
)
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
)
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS

EXPECTED_TOTAL_RANKING_ROWS = (
    EXPECTED_RANKING_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_BUILD_METRIC_ROWS = (
    EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_MODEL_FITS = (
    EXPECTED_MODEL_FIT_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS = (
    EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
REPETITION_SEEDS = list(range(1, 31))
RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]
BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]
ALL_TECHNIQUES = ML_TECHNIQUES + BASELINE_TECHNIQUES

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]
VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]
VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES_ROOT = THESIS_ROOT / "Notes"
RESULTS_ROOT = THESIS_ROOT / "Results"
REGISTRY_PATH = NOTES_ROOT / "completed_project_registry.csv"
SOURCE_DIR = Path("/content/datasets/datasets/apache@shardingsphere")

SELECTION_ROOT = RESULTS_ROOT / "Aggregated" / "project_11_selection"
FROZEN_SOURCE_MANIFEST_PATH = SELECTION_ROOT / "project_11_frozen_source_manifest.csv"
FIXED_CHRONOLOGY_PATH = SELECTION_ROOT / "project_11_fixed_chronological_builds.csv"
SELECTION_CHECKPOINT_PATH = NOTES_ROOT / "project_11_selection_checkpoint.json"

PROJECT_ROOT = RESULTS_ROOT / "Aggregated" / PROJECT_SLUG
REC_PREFLIGHT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_rec_preflight"
BUILD_ENTITY_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
CLEAN_RECONSTRUCTED_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
CLEAN_ANCHOR_OFFSETS_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
REC_CHECKPOINT_PATH = NOTES_ROOT / "project_11_rec_reconstruction_checkpoint.json"

NOISE_PLAN_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_noise_plan"
RAW_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
RAW_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
MODEL_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
MODEL_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
MODEL_RAW_TRAIN_LINK_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
MODEL_RAW_EVAL_LINK_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
RNG_MANIFEST_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_rng_manifest.parquet"
CONDITION_PLAN_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_condition_plan.csv"
NOISE_PLAN_CHECKPOINT_PATH = NOTES_ROOT / "project_11_noise_plan_checkpoint.json"

RUNTIME_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_runtime_contract"
PREDICTOR_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_predictor_contract.csv"
MODEL_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_model_contract.json"
BASELINE_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_baseline_contract.json"
RANKING_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_ranking_contract.json"
STEP4A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4a_status.json"
STEP4A_REPORT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_step4a_report.json"
RUNTIME_CHECKPOINT_PATH = NOTES_ROOT / "project_11_runtime_contract_checkpoint.json"

STEP4B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4b_status.json"
SMOKE_CHECKPOINT_PATH = NOTES_ROOT / "project_11_smoke_test_checkpoint.json"
SMOKE_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_smoke_test"

FULL_RAW_RESULT_ROOT = RESULTS_ROOT / "Raw" / PROJECT_SLUG
FULL_EXPERIMENT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_full_experiment"
INCOMPLETE_BACKUP_ROOT = FULL_EXPERIMENT_ROOT / "incomplete_condition_backups"
CONDITION_INVENTORY_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_condition_inventory.csv"
RAW_MANIFEST_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_raw_manifest.csv"
BASELINE_INVARIANCE_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_baseline_invariance.csv"
COMBINED_CONDITION_AUDIT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_condition_audit.csv"
COMBINED_PROJECT_RUNS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_project_runs.csv"
COMBINED_BUILD_METRICS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_build_metrics.csv"
COMBINED_MODEL_FITS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_model_fits.csv"
STEP5A_VALIDATION_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_validation.csv"
STEP5A_REPORT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_report.json"
RUN_PROGRESS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_run_progress.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5A_CHECKPOINT_PATH = NOTES_ROOT / "project_11_step5a_checkpoint.json"
ACCELERATED_EQUIVALENCE_PATH = (
    FULL_EXPERIMENT_ROOT
    / f"{PROJECT_SHORT}_accelerated_engine_equivalence.csv"
)

# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def sha256_array(values, dtype):
    array = np.asarray(values).astype(dtype, copy=False)
    return hashlib.sha256(array.tobytes(order="C")).hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(temporary_path, path)


def atomic_csv(path, frame, compression=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(temporary_path, path)


def atomic_parquet(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_parquet(
        temporary_path,
        index=False,
    )

    os.replace(temporary_path, path)


def source_root_hash(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )
        digest.update(line.encode("utf-8"))

    return digest.hexdigest()


def parse_int(values, label):
    numeric = pd.to_numeric(values, errors="coerce")

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains {int(numeric.isna().sum())} missing/non-numeric values."
        )

    array = numeric.to_numpy(dtype=float)

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def deterministic_seed(repetition_seed, stream_name):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(material).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def deterministic_random_build_seed(repetition_seed, build_id):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )


def create_models(repetition_seed):
    return {
        "RandomForest": RandomForestClassifier(
            **MODEL_CONFIG["RandomForest"],
            random_state=deterministic_seed(
                repetition_seed,
                "RandomForest_model",
            ),
        ),
        "XGBoost": XGBClassifier(
            **MODEL_CONFIG["XGBoost"],
            random_state=deterministic_seed(
                repetition_seed,
                "XGBoost_model",
            ),
        ),
        "LightGBM": LGBMClassifier(
            **MODEL_CONFIG["LightGBM"],
            random_state=deterministic_seed(
                repetition_seed,
                "LightGBM_model",
            ),
        ),
        "NaiveBayes": GaussianNB(
            **MODEL_CONFIG["NaiveBayes"]
        ),
    }


def calculate_apfd(failures):
    failures = np.asarray(failures, dtype=np.int8)
    number_of_tests = len(failures)
    number_of_failures = int(failures.sum())

    if number_of_tests == 0 or number_of_failures == 0:
        return np.nan

    failure_positions = np.flatnonzero(failures == 1) + 1

    return float(
        1.0
        - (
            failure_positions.sum()
            / (number_of_tests * number_of_failures)
        )
        + (1.0 / (2.0 * number_of_tests))
    )


def calculate_apfdc(failures, durations):
    failures = np.asarray(failures, dtype=np.int8)
    durations = np.asarray(durations, dtype=float)

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if len(failures) == 0 or failures.sum() == 0:
        return np.nan

    if not np.isfinite(durations).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (durations < 0).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(durations.sum())

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(durations)[:-1],
    ])

    failure_mask = failures == 1
    midpoint_detection_times = (
        cumulative_before[failure_mask]
        + (0.5 * durations[failure_mask])
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


def calculate_rates(history):
    history_length = len(history)

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history["verdict"]

    return (
        float(verdicts.ne(0).sum() / history_length),
        float(verdicts.eq(2).sum() / history_length),
        float(verdicts.eq(1).sum() / history_length),
        float(history["transition"].eq(1).sum() / history_length),
    )


def calculate_max_test_file_rate(
    history,
    target_column,
    current_changed_entities,
    entity_changed_builds,
):
    target_builds = (
        history.loc[
            history[target_column].gt(0),
            "build",
        ]
        .drop_duplicates()
        .astype(int)
        .tolist()
    )

    if len(target_builds) == 0:
        return -1.0

    target_build_set = set(target_builds)
    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(entity_id),
            set(),
        )

        overlap_count = len(
            changed_builds.intersection(target_build_set)
        )

        maximum_frequency = max(
            maximum_frequency,
            overlap_count,
        )

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(target_builds)
    )


def reconstruct_rec_features(
    execution_history,
    requested_rows,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
    recent_window=6,
):
    requested_pairs = set(
        zip(
            requested_rows["Build"].astype(int),
            requested_rows["Test"].astype(int),
        )
    )

    reconstructed_records = []
    test_groups = execution_history.groupby(
        "test",
        sort=False,
    )
    total_tests = int(
        execution_history["test"].nunique()
    )

    for test_index, (test_id, test_history) in enumerate(
        test_groups,
        start=1,
    ):
        test_history = (
            test_history.sort_values(
                [
                    "build_order",
                    "job",
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )

        test_history["transition"] = (
            test_history["verdict"]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(int)
        )

        first_test_build = int(
            test_history.iloc[0]["build"]
        )

        for current_position in range(len(test_history)):
            current_row = test_history.iloc[current_position]
            current_build = int(current_row["build"])
            current_test = int(test_id)
            pair = (current_build, current_test)

            if pair not in requested_pairs:
                continue

            history = (
                test_history.iloc[:current_position]
                .copy()
                .reset_index(drop=True)
            )

            record = {
                "Build": current_build,
                "Test": current_test,
            }

            if history.empty:
                for feature in REC_FEATURES:
                    record[feature] = -1.0

                record["REC_Age"] = 0.0
                reconstructed_records.append(record)
                continue

            recent_history = history.tail(recent_window).copy()

            age = float(
                global_build_position[current_build]
                - global_build_position[first_test_build]
            )

            failure_positions = np.flatnonzero(
                history["verdict"].to_numpy() > 0
            )

            last_failure_age = (
                -1.0
                if len(failure_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(failure_positions[-1])
                )
            )

            transition_positions = np.flatnonzero(
                history["transition"].to_numpy() > 0
            )

            last_transition_age = (
                -1.0
                if len(transition_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(transition_positions[-1])
                )
            )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(recent_history)

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(history)

            current_changed_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            max_file_fail_rate = calculate_max_test_file_rate(
                history=history,
                target_column="verdict",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            max_file_transition_rate = calculate_max_test_file_rate(
                history=history,
                target_column="transition",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            record.update({
                "REC_Age": age,
                "REC_LastFailureAge": last_failure_age,
                "REC_LastTransitionAge": last_transition_age,
                "REC_RecentAvgExeTime": float(
                    recent_history["duration"].mean()
                ),
                "REC_RecentMaxExeTime": float(
                    recent_history["duration"].max()
                ),
                "REC_RecentFailRate": recent_fail_rate,
                "REC_RecentAssertRate": recent_assert_rate,
                "REC_RecentExcRate": recent_exc_rate,
                "REC_RecentTransitionRate": recent_transition_rate,
                "REC_TotalAvgExeTime": float(
                    history["duration"].mean()
                ),
                "REC_TotalMaxExeTime": float(
                    history["duration"].max()
                ),
                "REC_TotalFailRate": total_fail_rate,
                "REC_TotalAssertRate": total_assert_rate,
                "REC_TotalExcRate": total_exc_rate,
                "REC_TotalTransitionRate": total_transition_rate,
                "REC_LastVerdict": float(
                    recent_history.iloc[-1]["verdict"]
                ),
                "REC_LastExeTime": float(
                    recent_history.iloc[-1]["duration"]
                ),
                "REC_MaxTestFileFailRate": max_file_fail_rate,
                "REC_MaxTestFileTransitionRate": (
                    max_file_transition_rate
                ),
            })

            reconstructed_records.append(record)

        if test_index % 100 == 0 or test_index == total_tests:
            print(
                "    REC reconstruction progress:",
                test_index,
                "/",
                total_tests,
                "tests | reconstructed rows:",
                len(reconstructed_records),
            )

    return pd.DataFrame(reconstructed_records)



def reconstruct_dependent_rec_fast(condition_combined_verdict):
    """
    Reconstruct only the 13 verdict-dependent REC features.

    This is algebraically equivalent to the frozen Step 4B implementation:
    - the same per-test execution order is used;
    - only prior executions contribute to each current row;
    - recent window = 6;
    - verdict 2 = assertion, verdict 1 = exception;
    - file-history rates use distinct prior target builds and current-build entities;
    - builds with no mapped entities produce 0 when target history exists and -1 when it does not.
    """
    condition_combined_verdict = np.asarray(
        condition_combined_verdict,
        dtype=np.int16,
    )

    if len(condition_combined_verdict) != EXPECTED_RAW_TRAIN_ROWS + EXPECTED_RAW_EVAL_ROWS:
        raise RuntimeError(
            "Accelerated REC engine received the wrong execution-history length."
        )

    verdict_sorted = condition_combined_verdict[
        accelerated_history_combined_indices
    ]

    result = np.full(
        (
            EXPECTED_MODEL_ROWS,
            len(VERDICT_DEPENDENT_REC),
        ),
        -1.0,
        dtype=np.float64,
    )

    for group_index in range(accelerated_group_count):
        requested_model_indices = accelerated_requested_model_indices[group_index]

        if len(requested_model_indices) == 0:
            continue

        start = int(accelerated_group_starts[group_index])
        end = int(accelerated_group_ends[group_index])
        local_positions = accelerated_requested_local_positions[group_index]

        verdict = verdict_sorted[start:end]
        group_length = len(verdict)
        position = np.arange(group_length, dtype=np.int64)

        failure = verdict > 0
        assertion = verdict == 2
        exception = verdict == 1
        transition = np.zeros(group_length, dtype=np.bool_)

        if group_length > 1:
            transition[1:] = verdict[1:] != verdict[:-1]

        failure_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(failure, dtype=np.int64),
        ))
        assertion_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(assertion, dtype=np.int64),
        ))
        exception_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(exception, dtype=np.int64),
        ))
        transition_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(transition, dtype=np.int64),
        ))

        has_history = local_positions > 0

        if has_history.any():
            requested_with_history = np.flatnonzero(has_history)
            current_positions = local_positions[requested_with_history]
            model_indices = requested_model_indices[requested_with_history]

            recent_starts = np.maximum(
                0,
                current_positions - RECENT_WINDOW,
            )
            recent_lengths = current_positions - recent_starts

            last_failure_position = np.maximum.accumulate(
                np.where(failure, position, -1)
            )
            last_transition_position = np.maximum.accumulate(
                np.where(transition, position, -1)
            )

            prior_last_failure = last_failure_position[
                current_positions - 1
            ]
            prior_last_transition = last_transition_position[
                current_positions - 1
            ]

            result[model_indices, 0] = np.where(
                prior_last_failure >= 0,
                current_positions - 1 - prior_last_failure,
                -1,
            ).astype(np.float64)
            result[model_indices, 1] = np.where(
                prior_last_transition >= 0,
                current_positions - 1 - prior_last_transition,
                -1,
            ).astype(np.float64)

            result[model_indices, 2] = (
                failure_prefix[current_positions]
                - failure_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 3] = (
                assertion_prefix[current_positions]
                - assertion_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 4] = (
                exception_prefix[current_positions]
                - exception_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 5] = (
                transition_prefix[current_positions]
                - transition_prefix[recent_starts]
            ) / recent_lengths

            result[model_indices, 6] = (
                failure_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 7] = (
                assertion_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 8] = (
                exception_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 9] = (
                transition_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 10] = verdict[
                current_positions - 1
            ].astype(np.float64)

        # File-history features. The counters contain only target executions
        # strictly before the current position, matching Step 4B exactly.
        failure_entity_counts = np.zeros(
            accelerated_entity_count,
            dtype=np.int32,
        )
        transition_entity_counts = np.zeros(
            accelerated_entity_count,
            dtype=np.int32,
        )
        failure_denominator = 0
        transition_denominator = 0
        requested_pointer = 0

        group_build_indices = accelerated_history_build_dense_indices[
            start:end
        ]

        for local_position in range(group_length):
            while (
                requested_pointer < len(local_positions)
                and int(local_positions[requested_pointer]) == local_position
            ):
                model_index = int(
                    requested_model_indices[requested_pointer]
                )

                if local_position > 0:
                    current_entities = accelerated_build_entity_arrays[
                        int(group_build_indices[local_position])
                    ]

                    if failure_denominator == 0:
                        result[model_index, 11] = -1.0
                    elif len(current_entities) == 0:
                        result[model_index, 11] = 0.0
                    else:
                        result[model_index, 11] = float(
                            failure_entity_counts[
                                current_entities
                            ].max()
                            / failure_denominator
                        )

                    if transition_denominator == 0:
                        result[model_index, 12] = -1.0
                    elif len(current_entities) == 0:
                        result[model_index, 12] = 0.0
                    else:
                        result[model_index, 12] = float(
                            transition_entity_counts[
                                current_entities
                            ].max()
                            / transition_denominator
                        )

                requested_pointer += 1

            changed_entities = accelerated_build_entity_arrays[
                int(group_build_indices[local_position])
            ]

            if failure[local_position]:
                if len(changed_entities) != 0:
                    failure_entity_counts[changed_entities] += 1
                failure_denominator += 1

            if transition[local_position]:
                if len(changed_entities) != 0:
                    transition_entity_counts[changed_entities] += 1
                transition_denominator += 1

        if requested_pointer != len(local_positions):
            raise RuntimeError(
                "Accelerated REC engine did not emit every requested row."
            )

    if not np.isfinite(result).all():
        raise RuntimeError(
            "Accelerated REC engine produced non-finite values."
        )

    return result


def maximum_absolute_difference(left, right):
    left = np.asarray(left, dtype=np.float64)
    right = np.asarray(right, dtype=np.float64)

    if left.shape != right.shape:
        return np.inf

    if left.size == 0:
        return 0.0

    return float(np.max(np.abs(left - right)))


def compare_full_condition_to_smoke(condition_key, full_condition_dir):
    """Compare all scientific outputs with the frozen Step 4B condition."""
    full_condition_dir = Path(full_condition_dir)
    smoke_condition_dir = SMOKE_ROOT / condition_key

    required_names = [
        "rankings.csv.gz",
        "build_metrics.csv",
        "project_runs.csv",
        "model_fits.csv",
        "training_medians.csv",
        "condition_audit.csv",
    ]

    for name in required_names:
        if not (full_condition_dir / name).is_file():
            raise FileNotFoundError(
                f"Accelerated equivalence input missing: {full_condition_dir / name}"
            )
        if not (smoke_condition_dir / name).is_file():
            raise FileNotFoundError(
                f"Frozen smoke output missing: {smoke_condition_dir / name}"
            )

    actual_rankings = pd.read_csv(
        full_condition_dir / "rankings.csv.gz",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build", "Test"],
        kind="mergesort",
    ).reset_index(drop=True)
    smoke_rankings = pd.read_csv(
        smoke_condition_dir / "rankings.csv.gz",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build", "Test"],
        kind="mergesort",
    ).reset_index(drop=True)

    ranking_key_columns = [
        "Technique",
        "Build",
        "Test",
        "Rank",
        "CleanVerdict",
        "CleanFailure",
    ]
    ranking_keys_equal = bool(
        len(actual_rankings) == len(smoke_rankings)
        and actual_rankings[ranking_key_columns].equals(
            smoke_rankings[ranking_key_columns]
        )
    )
    ranking_score_max_difference = maximum_absolute_difference(
        actual_rankings["Score"].to_numpy(dtype=float),
        smoke_rankings["Score"].to_numpy(dtype=float),
    )

    actual_build = pd.read_csv(
        full_condition_dir / "build_metrics.csv",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build"],
        kind="mergesort",
    ).reset_index(drop=True)
    smoke_build = pd.read_csv(
        smoke_condition_dir / "build_metrics.csv",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build"],
        kind="mergesort",
    ).reset_index(drop=True)
    build_keys_equal = bool(
        len(actual_build) == len(smoke_build)
        and actual_build[["Technique", "Build", "Tests", "Failures"]].equals(
            smoke_build[["Technique", "Build", "Tests", "Failures"]]
        )
    )
    build_metric_max_difference = maximum_absolute_difference(
        actual_build[["TotalDuration", "APFDc", "APFD"]].to_numpy(dtype=float),
        smoke_build[["TotalDuration", "APFDc", "APFD"]].to_numpy(dtype=float),
    )

    actual_project = pd.read_csv(
        full_condition_dir / "project_runs.csv",
        low_memory=False,
    ).sort_values("Technique", kind="mergesort").reset_index(drop=True)
    smoke_project = pd.read_csv(
        smoke_condition_dir / "project_runs.csv",
        low_memory=False,
    ).sort_values("Technique", kind="mergesort").reset_index(drop=True)
    project_keys_equal = bool(
        len(actual_project) == len(smoke_project)
        and actual_project[[
            "Technique",
            "EvaluationBuilds",
            "ScoredFailingBuilds",
            "EvaluationRows",
            "EvaluationFailures",
        ]].equals(
            smoke_project[[
                "Technique",
                "EvaluationBuilds",
                "ScoredFailingBuilds",
                "EvaluationRows",
                "EvaluationFailures",
            ]]
        )
    )
    project_metric_max_difference = maximum_absolute_difference(
        actual_project[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float),
        smoke_project[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float),
    )

    actual_medians = pd.read_csv(
        full_condition_dir / "training_medians.csv",
        low_memory=False,
    ).sort_values("PredictorOrder", kind="mergesort").reset_index(drop=True)
    smoke_medians = pd.read_csv(
        smoke_condition_dir / "training_medians.csv",
        low_memory=False,
    ).sort_values("PredictorOrder", kind="mergesort").reset_index(drop=True)
    median_keys_equal = bool(
        len(actual_medians) == len(smoke_medians)
        and actual_medians[["PredictorOrder", "Predictor"]].equals(
            smoke_medians[["PredictorOrder", "Predictor"]]
        )
    )
    median_max_difference = maximum_absolute_difference(
        actual_medians["TrainingMedian"].to_numpy(dtype=float),
        smoke_medians["TrainingMedian"].to_numpy(dtype=float),
    )

    actual_fits = pd.read_csv(
        full_condition_dir / "model_fits.csv",
        low_memory=False,
    ).fillna("").sort_values("Technique", kind="mergesort").reset_index(drop=True)
    smoke_fits = pd.read_csv(
        smoke_condition_dir / "model_fits.csv",
        low_memory=False,
    ).fillna("").sort_values("Technique", kind="mergesort").reset_index(drop=True)
    fit_contract_columns = [
        "Technique",
        "TrainingRows",
        "TrainingFailures",
        "Predictors",
        "ClassesJSON",
        "Status",
        "Error",
    ]
    fit_contract_equal = bool(
        len(actual_fits) == len(smoke_fits)
        and actual_fits[fit_contract_columns].equals(
            smoke_fits[fit_contract_columns]
        )
    )

    actual_audit = pd.read_csv(
        full_condition_dir / "condition_audit.csv",
        low_memory=False,
    ).iloc[0]
    smoke_audit = pd.read_csv(
        smoke_condition_dir / "condition_audit.csv",
        low_memory=False,
    ).iloc[0]
    audit_columns = [
        "ConditionKey",
        "NoisePercent",
        "RepetitionSeed",
        "RawTrainingRows",
        "NumberFlipped",
        "ExpectedNumberFlipped",
        "PassToFailure",
        "FailureToPass",
        "ModelTrainingRows",
        "ModelLabelChanges",
        "ExpectedModelLabelChanges",
        "TrainingFailures",
        "ExpectedTrainingFailures",
        "DependentRECChanges",
        "IndependentRECChanges",
        "IndependentReconstructionMismatches",
        "ReconstructedRows",
        "Predictors",
        "MLFits",
        "RankingRows",
        "BuildMetricRows",
        "ProjectRunRows",
        "TrainingMedianRows",
        "ExpectedFlipMaskSHA256",
        "ActualFlipMaskSHA256",
        "ExpectedNoisyRawVerdictSHA256",
        "ActualNoisyRawVerdictSHA256",
        "ExpectedNoisyModelVerdictSHA256",
        "ActualNoisyModelVerdictSHA256",
    ]
    audit_equal = bool(
        all(
            str(actual_audit[column]) == str(smoke_audit[column])
            for column in audit_columns
        )
    )

    passed = bool(
        ranking_keys_equal
        and ranking_score_max_difference <= 1e-12
        and build_keys_equal
        and build_metric_max_difference <= 1e-12
        and project_keys_equal
        and project_metric_max_difference <= 1e-12
        and median_keys_equal
        and median_max_difference <= 1e-12
        and fit_contract_equal
        and audit_equal
    )

    return {
        "ConditionKey": condition_key,
        "EngineVersion": ACCELERATED_ENGINE_VERSION,
        "RankingKeysEqual": ranking_keys_equal,
        "RankingScoreMaxDifference": ranking_score_max_difference,
        "BuildMetricKeysEqual": build_keys_equal,
        "BuildMetricMaxDifference": build_metric_max_difference,
        "ProjectRunKeysEqual": project_keys_equal,
        "ProjectMetricMaxDifference": project_metric_max_difference,
        "TrainingMedianKeysEqual": median_keys_equal,
        "TrainingMedianMaxDifference": median_max_difference,
        "ModelFitContractEqual": fit_contract_equal,
        "ConditionAuditEqual": audit_equal,
        "Pass": passed,
    }

def positive_probability(estimator, matrix):
    probabilities = estimator.predict_proba(matrix)
    classes = np.asarray(estimator.classes_)
    positive_columns = np.flatnonzero(classes == 1)

    if len(positive_columns) != 1:
        raise RuntimeError(
            "Fitted estimator does not expose exactly one class-1 probability column."
        )

    scores = probabilities[:, int(positive_columns[0])]

    if not np.isfinite(scores).all():
        raise RuntimeError(
            "Model produced non-finite failure probabilities."
        )

    if ((scores < 0) | (scores > 1)).any():
        raise RuntimeError(
            "Model produced probabilities outside [0,1]."
        )

    return scores.astype(float, copy=False)


def make_ranking(
    evaluation_meta,
    technique,
    scores,
    ascending_score,
):
    ranking = evaluation_meta.copy()
    ranking["Technique"] = technique
    ranking["Score"] = np.asarray(scores, dtype=float)

    if len(ranking) != EXPECTED_MODEL_EVAL_ROWS:
        raise RuntimeError(
            f"{technique} ranking input has the wrong row count."
        )

    if not np.isfinite(ranking["Score"].to_numpy(dtype=float)).all():
        raise RuntimeError(
            f"{technique} ranking contains non-finite scores."
        )

    ranking = (
        ranking.sort_values(
            [
                "Build",
                "Score",
                "Test",
            ],
            ascending=[
                True,
                bool(ascending_score),
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    ranking["Rank"] = (
        ranking.groupby(
            "Build",
            sort=False,
        )
        .cumcount()
        .add(1)
        .astype("int64")
    )

    return ranking[
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "Build",
            "Test",
            "Rank",
            "Score",
            "CleanVerdict",
            "CleanFailure",
            "Duration",
        ]
    ]


def calculate_condition_metrics(rankings):
    build_metric_records = []

    failing_rankings = rankings.loc[
        rankings["Build"].isin(failing_evaluation_builds)
    ].copy()

    for (technique, build_id), build_ranking in failing_rankings.groupby(
        [
            "Technique",
            "Build",
        ],
        sort=False,
    ):
        build_ranking = build_ranking.sort_values(
            "Rank",
            kind="mergesort",
        )

        failures = build_ranking[
            "CleanFailure"
        ].to_numpy(dtype=np.int8)

        durations = build_ranking[
            "Duration"
        ].to_numpy(dtype=float)

        number_of_failures = int(failures.sum())

        if number_of_failures <= 0:
            raise RuntimeError(
                "A supposedly failing evaluation build has no failures."
            )

        apfd = calculate_apfd(failures)
        apfdc = calculate_apfdc(failures, durations)

        build_metric_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                build_ranking["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                build_ranking["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                build_ranking["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "Build": int(build_id),
            "Tests": int(len(build_ranking)),
            "Failures": number_of_failures,
            "TotalDuration": float(durations.sum()),
            "APFDc": float(apfdc),
            "APFD": float(apfd),
        })

    build_metrics = pd.DataFrame(build_metric_records)

    project_run_records = []

    for technique, technique_metrics in build_metrics.groupby(
        "Technique",
        sort=False,
    ):
        project_run_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                technique_metrics["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                technique_metrics["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                technique_metrics["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "EvaluationBuilds": EXPECTED_EVALUATION_BUILDS,
            "ScoredFailingBuilds": int(len(technique_metrics)),
            "EvaluationRows": EXPECTED_MODEL_EVAL_ROWS,
            "EvaluationFailures": EXPECTED_MODEL_EVAL_FAILURES,
            "MeanAPFDc": float(
                technique_metrics["APFDc"].mean()
            ),
            "MedianAPFDc": float(
                technique_metrics["APFDc"].median()
            ),
            "MeanAPFD": float(
                technique_metrics["APFD"].mean()
            ),
            "MedianAPFD": float(
                technique_metrics["APFD"].median()
            ),
        })

    project_runs = pd.DataFrame(project_run_records)

    return build_metrics, project_runs


def directory_manifest(root):
    root = Path(root)
    rows = []

    if not root.exists():
        return pd.DataFrame(
            columns=[
                "RelativePath",
                "Bytes",
                "SHA256",
            ]
        )

    for path in sorted(
        [
            candidate
            for candidate in root.rglob("*")
            if candidate.is_file()
        ],
        key=lambda candidate: candidate.relative_to(root).as_posix(),
    ):
        rows.append({
            "RelativePath": path.relative_to(root).as_posix(),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        })

    return pd.DataFrame(rows)


def directory_root_hash(manifest):
    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN CHECKPOINTS
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "contributors.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "entity_change_history.csv",
    SOURCE_DIR / "exe.csv",
    SOURCE_DIR / "id_map.csv",
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    SELECTION_CHECKPOINT_PATH,
    BUILD_ENTITY_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    REC_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    PREDICTOR_CONTRACT_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_STATUS_PATH,
    STEP4A_REPORT_PATH,
    RUNTIME_CHECKPOINT_PATH,
    STEP4B_STATUS_PATH,
    SMOKE_CHECKPOINT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 11 Step 5A inputs are missing:\n"
        + "\n".join(missing_paths)
    )

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)
smoke_checkpoint_sha256 = sha256_file(
    SMOKE_CHECKPOINT_PATH
)

if selection_checkpoint_sha256 != EXPECTED_SELECTION_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 11 selection checkpoint SHA-256 differs."
    )

if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 11 REC checkpoint SHA-256 differs."
    )

if noise_plan_checkpoint_sha256 != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 11 noise-plan checkpoint SHA-256 differs."
    )

if runtime_checkpoint_sha256 != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 11 runtime-contract checkpoint SHA-256 differs."
    )

if smoke_checkpoint_sha256 != EXPECTED_SMOKE_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 11 smoke-test checkpoint SHA-256 differs."
    )

runtime_checkpoint = load_json(
    RUNTIME_CHECKPOINT_PATH
)
step4a_status = load_json(
    STEP4A_STATUS_PATH
)
step4a_report = load_json(
    STEP4A_REPORT_PATH
)
step4b_status = load_json(
    STEP4B_STATUS_PATH
)
smoke_checkpoint = load_json(
    SMOKE_CHECKPOINT_PATH
)

for label, payload in [
    ("runtime checkpoint", runtime_checkpoint),
    ("Step 4A status", step4a_status),
    ("Step 4A report", step4a_report),
]:
    if payload.get("Status") != EXPECTED_STEP4A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the frozen Step 4A PASS status."
        )

if runtime_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Runtime checkpoint project identity differs."
    )

if runtime_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Runtime checkpoint project slug differs."
    )

if runtime_checkpoint.get("SourceRootSHA256") != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Runtime checkpoint source root differs."
    )

if step4b_status.get("Status") != EXPECTED_STEP4B_STATUS:
    raise RuntimeError(
        "Project 11 Step 4B status is not frozen successfully."
    )

if smoke_checkpoint.get("Status") != EXPECTED_STEP4B_STATUS:
    raise RuntimeError(
        "Project 11 smoke-test checkpoint is not frozen successfully."
    )

if smoke_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Smoke-test checkpoint project identity differs."
    )

if smoke_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Smoke-test checkpoint project slug differs."
    )

if not bool(smoke_checkpoint.get("ReadyForFull270ConditionExperiment", False)):
    raise RuntimeError(
        "Smoke-test checkpoint does not authorise the full experiment."
    )

smoke_output_manifest = smoke_checkpoint.get("OutputManifest", [])
if not isinstance(smoke_output_manifest, list) or not smoke_output_manifest:
    raise RuntimeError(
        "Smoke-test checkpoint does not contain an output manifest."
    )

smoke_output_manifest_failures = 0
for item in smoke_output_manifest:
    output_path = Path(item["Path"])
    if (
        not output_path.is_file()
        or int(output_path.stat().st_size) != int(item["Bytes"])
        or sha256_file(output_path) != str(item["SHA256"])
    ):
        smoke_output_manifest_failures += 1

if smoke_output_manifest_failures != 0:
    raise RuntimeError(
        "One or more frozen Step 4B smoke outputs changed."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY SOURCE ROOT, REGISTRY, AND STEP 4A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(REGISTRY_PATH)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs before Step 5A."
    )

registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)

project_number_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "projectnumber",
            "project_number",
            "project no",
            "projectno",
        }
    ),
    None,
)

if project_number_column is None:
    raise RuntimeError(
        "Could not resolve the project-number column in the completion registry."
    )

registry_project_numbers = pd.to_numeric(
    registry[project_number_column],
    errors="coerce",
)

if len(registry) != 10:
    raise RuntimeError(
        "Completion registry does not contain exactly Projects 1-10."
    )

if int(registry_project_numbers.eq(PROJECT_NUMBER).sum()) != 0:
    raise RuntimeError(
        "Project 11 is already present in the completion registry."
    )

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

current_source_rows = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = Path(row.AbsolutePath)

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 11 source file is missing: {source_path}"
        )

    current_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

current_source_manifest = pd.DataFrame(current_source_rows)
current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 11 source root differs before Step 5A."
    )

runtime_output_manifest = runtime_checkpoint.get(
    "RuntimeOutputManifest",
    [],
)

if not isinstance(runtime_output_manifest, list) or not runtime_output_manifest:
    raise RuntimeError(
        "Runtime checkpoint has no output manifest."
    )

runtime_manifest_records = []

for item in runtime_output_manifest:
    path = Path(item["Path"])
    expected_bytes = int(item["Bytes"])
    expected_sha256 = str(item["SHA256"]).lower()
    exists = path.is_file()
    actual_bytes = int(path.stat().st_size) if exists else -1
    actual_sha256 = sha256_file(path) if exists else "MISSING"
    passed = (
        exists
        and actual_bytes == expected_bytes
        and actual_sha256 == expected_sha256
    )

    runtime_manifest_records.append({
        "Path": str(path),
        "ExpectedBytes": expected_bytes,
        "ActualBytes": actual_bytes,
        "ExpectedSHA256": expected_sha256,
        "ActualSHA256": actual_sha256,
        "Pass": passed,
    })

runtime_manifest_audit = pd.DataFrame(
    runtime_manifest_records
)
runtime_manifest_failures = int(
    (~runtime_manifest_audit["Pass"]).sum()
)

if runtime_manifest_failures != 0:
    print("\nFailed Step 4A output-manifest checks:")
    display(
        runtime_manifest_audit.loc[
            ~runtime_manifest_audit["Pass"]
        ]
    )
    raise RuntimeError(
        "Step 4A output manifest no longer validates."
    )

full_raw_result_root_existed_before = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_before = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_before = directory_root_hash(
    full_raw_result_manifest_before
)


# --------------------------------------------------------------------------------------------------
# 6. LOAD FROZEN COHORTS, LINKS, CONDITION PLAN, AND RNG STREAM
# --------------------------------------------------------------------------------------------------

print("\nLoading frozen Project 11 cohorts and contracts.")

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)
model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)
model_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)
model_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)
condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)
predictor_contract = pd.read_csv(
    PREDICTOR_CONTRACT_PATH,
    low_memory=False,
)
chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)
build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)
anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)
clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

if len(raw_training) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError("Raw training cohort row count differs.")
if len(raw_evaluation) != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError("Raw evaluation cohort row count differs.")
if len(model_training) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model training cohort row count differs.")
if len(model_evaluation) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model evaluation cohort row count differs.")
if len(model_train_link) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model/raw training link row count differs.")
if len(model_eval_link) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model/raw evaluation link row count differs.")
if len(anchor_offsets) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean anchor-offset row count differs.")
if len(clean_reconstructed) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean reconstructed REC row count differs.")

required_cohort_columns = {
    "Build",
    "Test",
    "Verdict",
}

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    missing = required_cohort_columns - set(frame.columns)
    if missing:
        raise RuntimeError(
            f"{label} cohort is missing columns: {sorted(missing)}"
        )

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    frame["Build"] = parse_int(
        frame["Build"],
        f"{label}.Build",
    )
    frame["Test"] = parse_int(
        frame["Test"],
        f"{label}.Test",
    )
    frame["Verdict"] = parse_int(
        frame["Verdict"],
        f"{label}.Verdict",
    )

raw_order_column = "RawTrainingRowOrder"
raw_eval_order_column = "RawEvaluationRowOrder"
model_train_order_column = "ModelTrainingRowOrder"
model_eval_order_column = "ModelEvaluationRowOrder"

for column, frame, expected_rows, label in [
    (
        raw_order_column,
        raw_training,
        EXPECTED_RAW_TRAIN_ROWS,
        "raw training",
    ),
    (
        raw_eval_order_column,
        raw_evaluation,
        EXPECTED_RAW_EVAL_ROWS,
        "raw evaluation",
    ),
    (
        model_train_order_column,
        model_training,
        EXPECTED_MODEL_TRAIN_ROWS,
        "model training",
    ),
    (
        model_eval_order_column,
        model_evaluation,
        EXPECTED_MODEL_EVAL_ROWS,
        "model evaluation",
    ),
]:
    if column not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing {column}."
        )

    frame[column] = parse_int(
        frame[column],
        f"{label}.{column}",
    )

    frame.sort_values(
        column,
        kind="mergesort",
        inplace=True,
    )
    frame.reset_index(drop=True, inplace=True)

    expected_sequence = np.arange(
        1,
        expected_rows + 1,
        dtype=np.int64,
    )

    if not np.array_equal(
        frame[column].to_numpy(dtype=np.int64),
        expected_sequence,
    ):
        raise RuntimeError(
            f"{label} row-order sequence is not canonical."
        )

model_train_link[model_train_order_column] = parse_int(
    model_train_link[model_train_order_column],
    "model_train_link.ModelTrainingRowOrder",
)
model_train_link[raw_order_column] = parse_int(
    model_train_link[raw_order_column],
    "model_train_link.RawTrainingRowOrder",
)
model_eval_link[model_eval_order_column] = parse_int(
    model_eval_link[model_eval_order_column],
    "model_eval_link.ModelEvaluationRowOrder",
)
model_eval_link[raw_eval_order_column] = parse_int(
    model_eval_link[raw_eval_order_column],
    "model_eval_link.RawEvaluationRowOrder",
)

model_train_link = (
    model_train_link.sort_values(
        model_train_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)
model_eval_link = (
    model_eval_link.sort_values(
        model_eval_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

model_training_raw_indices = (
    model_train_link[raw_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)
model_evaluation_raw_indices = (
    model_eval_link[raw_eval_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)

if (
    model_training_raw_indices.min() < 0
    or model_training_raw_indices.max() >= EXPECTED_RAW_TRAIN_ROWS
):
    raise RuntimeError(
        "Model/raw training indices are outside the frozen raw cohort."
    )

if (
    model_evaluation_raw_indices.min() < 0
    or model_evaluation_raw_indices.max() >= EXPECTED_RAW_EVAL_ROWS
):
    raise RuntimeError(
        "Model/raw evaluation indices are outside the frozen raw cohort."
    )

linked_train_build = raw_training.iloc[
    model_training_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_train_test = raw_training.iloc[
    model_training_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_train_verdict = raw_training.iloc[
    model_training_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

linked_eval_build = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_eval_test = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_eval_verdict = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

if not np.array_equal(
    linked_train_build,
    model_training["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Build links differ.")
if not np.array_equal(
    linked_train_test,
    model_training["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Test links differ.")
if not np.array_equal(
    linked_train_verdict,
    model_training["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training verdict links differ.")
if not np.array_equal(
    linked_eval_build,
    model_evaluation["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Build links differ.")
if not np.array_equal(
    linked_eval_test,
    model_evaluation["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Test links differ.")
if not np.array_equal(
    linked_eval_verdict,
    model_evaluation["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation verdict links differ.")


condition_plan["ConditionOrder"] = parse_int(
    condition_plan["ConditionOrder"],
    "condition_plan.ConditionOrder",
)
condition_plan["NoisePercent"] = parse_int(
    condition_plan["NoisePercent"],
    "condition_plan.NoisePercent",
)
condition_plan["RepetitionSeed"] = parse_int(
    condition_plan["RepetitionSeed"],
    "condition_plan.RepetitionSeed",
)

condition_plan = (
    condition_plan.sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(condition_plan) != EXPECTED_CONDITIONS:
    raise RuntimeError(
        "The frozen condition plan does not contain 270 conditions."
    )

if not np.array_equal(
    condition_plan["ConditionOrder"].to_numpy(dtype=np.int64),
    np.arange(1, EXPECTED_CONDITIONS + 1, dtype=np.int64),
):
    raise RuntimeError(
        "The frozen condition-order sequence is not canonical."
    )

if sorted(condition_plan["NoisePercent"].unique().tolist()) != NOISE_LEVELS:
    raise RuntimeError(
        "The frozen noise-level set differs."
    )

if sorted(condition_plan["RepetitionSeed"].unique().tolist()) != REPETITION_SEEDS:
    raise RuntimeError(
        "The frozen repetition-seed set differs."
    )

if condition_plan["ConditionID"].duplicated(keep=False).any():
    raise RuntimeError(
        "The frozen condition plan contains duplicate condition IDs."
    )

if condition_plan.duplicated(
    subset=["NoisePercent", "RepetitionSeed"],
    keep=False,
).any():
    raise RuntimeError(
        "The frozen condition plan contains duplicate coordinates."
    )

rng_metadata_rows = int(
    pq.ParquetFile(RNG_MANIFEST_PATH).metadata.num_rows
)

if rng_metadata_rows != EXPECTED_RNG_ROWS:
    raise RuntimeError(
        "Frozen RNG-manifest row count differs."
    )

# --------------------------------------------------------------------------------------------------
# 7. PREDICTOR ORDER, NUMERIC MATRICES, CHRONOLOGY, ENTITY MAP, AND EVALUATION META
# --------------------------------------------------------------------------------------------------

if "Predictor" not in predictor_contract.columns:
    raise RuntimeError(
        "Predictor contract is missing the Predictor column."
    )

predictor_columns = predictor_contract[
    "Predictor"
].astype(str).tolist()

if len(predictor_columns) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract does not contain 151 predictors."
    )

if len(set(predictor_columns)) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract contains duplicate predictors."
    )

missing_training_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_training.columns
]
missing_evaluation_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_evaluation.columns
]

if missing_training_predictors or missing_evaluation_predictors:
    raise RuntimeError(
        "Frozen model cohorts are missing contract predictors."
    )

if any(feature not in predictor_columns for feature in REC_FEATURES):
    raise RuntimeError(
        "The 19 REC features are not all present in the predictor contract."
    )

if set(VERDICT_DEPENDENT_REC).intersection(
    VERDICT_INDEPENDENT_REC
):
    raise RuntimeError(
        "Dependent and independent REC sets overlap."
    )

if set(VERDICT_DEPENDENT_REC + VERDICT_INDEPENDENT_REC) != set(
    REC_FEATURES
):
    raise RuntimeError(
        "Dependent and independent REC sets do not partition all 19 REC features."
    )

print("Converting the fixed predictor cohorts to one numeric matrix.")

training_numeric_frame = model_training[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

evaluation_numeric_frame = model_evaluation[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

training_base_numeric = training_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)
evaluation_base_numeric = evaluation_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)

training_base_numeric[
    ~np.isfinite(training_base_numeric)
] = np.nan
evaluation_base_numeric[
    ~np.isfinite(evaluation_base_numeric)
] = np.nan

all_base_numeric = np.vstack([
    training_base_numeric,
    evaluation_base_numeric,
])

predictor_index = {
    feature: index
    for index, feature in enumerate(predictor_columns)
}

dependent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_DEPENDENT_REC
    ],
    dtype=np.int64,
)

independent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_INDEPENDENT_REC
    ],
    dtype=np.int64,
)

model_all = pd.concat(
    [
        model_training[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        model_evaluation[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
    ],
    ignore_index=True,
)

if model_all.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Model cohort contains duplicate Build-Test rows."
    )

model_key_index = pd.MultiIndex.from_frame(
    model_all[["Build", "Test"]]
)

anchor_offsets = anchor_offsets.copy()
anchor_offsets["Build"] = parse_int(
    anchor_offsets["Build"],
    "anchor_offsets.Build",
)
anchor_offsets["Test"] = parse_int(
    anchor_offsets["Test"],
    "anchor_offsets.Test",
)

if anchor_offsets.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Anchor offsets contain duplicate Build-Test rows."
    )

anchor_indexed = anchor_offsets.set_index(
    [
        "Build",
        "Test",
    ]
)

missing_anchor_keys = model_key_index.difference(
    anchor_indexed.index
)

if len(missing_anchor_keys) != 0:
    raise RuntimeError(
        "Anchor offsets do not cover the full model cohort."
    )

anchor_values_all = anchor_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_reconstructed["Build"] = parse_int(
    clean_reconstructed["Build"],
    "clean_reconstructed.Build",
)
clean_reconstructed["Test"] = parse_int(
    clean_reconstructed["Test"],
    "clean_reconstructed.Test",
)

clean_reconstructed_indexed = clean_reconstructed.set_index(
    [
        "Build",
        "Test",
    ]
)

clean_reconstructed_all = clean_reconstructed_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_anchored_all = (
    clean_reconstructed_all
    + anchor_values_all
)

clean_original_rec_all = model_all[
    REC_FEATURES
].to_numpy(dtype=np.float64)

clean_anchor_mismatch_values = int(
    (~np.isclose(
        clean_anchored_all,
        clean_original_rec_all,
        rtol=0,
        atol=1e-12,
    )).sum()
)

if clean_anchor_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean REC reconstruction plus anchor no longer reproduces the model cohort."
    )

chronology["BuildID"] = parse_int(
    chronology["BuildID"],
    "chronology.BuildID",
)
chronology["ChronologyOrder"] = parse_int(
    chronology["ChronologyOrder"],
    "chronology.ChronologyOrder",
)

build_order_map = (
    chronology.set_index("BuildID")[
        "ChronologyOrder"
    ]
    .astype(int)
    .to_dict()
)

ordered_builds = (
    chronology.sort_values(
        "ChronologyOrder",
        kind="mergesort",
    )["BuildID"]
    .astype(int)
    .tolist()
)

global_build_position = {
    int(build_id): position
    for position, build_id in enumerate(ordered_builds)
}

build_entity["BuildID"] = parse_int(
    build_entity["BuildID"],
    "build_entity.BuildID",
)
build_entity["EntityId"] = parse_int(
    build_entity["EntityId"],
    "build_entity.EntityId",
)

changed_entities_by_build = (
    build_entity.groupby(
        "BuildID"
    )["EntityId"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

changed_entities_by_build = {
    int(build_id): set(
        int(entity_id)
        for entity_id in changed_entities_by_build.get(
            int(build_id),
            set(),
        )
    )
    for build_id in ordered_builds
}

entity_changed_builds = (
    build_entity.groupby(
        "EntityId"
    )["BuildID"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

entity_changed_builds = {
    int(entity_id): set(
        int(build_id)
        for build_id in build_ids
    )
    for entity_id, build_ids in entity_changed_builds.items()
}

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "Job" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Job."
        )
    if "Duration" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Duration."
        )

    frame["Duration"] = pd.to_numeric(
        frame["Duration"],
        errors="coerce",
    )

    if not np.isfinite(
        frame["Duration"].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            f"{label} cohort contains non-finite durations."
        )

    if frame["Duration"].lt(0).any():
        raise RuntimeError(
            f"{label} cohort contains negative durations."
        )

clean_raw_training_verdict = raw_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_raw_evaluation_verdict = raw_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_training_verdict = model_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_verdict = model_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_binary = (
    clean_model_evaluation_verdict != 0
).astype(np.int8)

if int((clean_model_training_verdict != 0).sum()) != EXPECTED_MODEL_TRAIN_FAILURES:
    raise RuntimeError(
        "Clean model-training failure count differs."
    )

if int(clean_model_evaluation_binary.sum()) != EXPECTED_MODEL_EVAL_FAILURES:
    raise RuntimeError(
        "Clean model-evaluation failure count differs."
    )

evaluation_duration = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Duration"].to_numpy(dtype=float)

if not np.isfinite(evaluation_duration).all():
    raise RuntimeError(
        "Model evaluation durations are non-finite."
    )

failing_evaluation_builds = sorted(
    model_evaluation.loc[
        clean_model_evaluation_binary == 1,
        "Build",
    ]
    .astype(int)
    .unique()
    .tolist()
)

if len(failing_evaluation_builds) != EXPECTED_FAILING_EVAL_BUILDS:
    raise RuntimeError(
        "Failing evaluation-build count differs."
    )

evaluation_meta_base = pd.DataFrame({
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Build": model_evaluation["Build"].to_numpy(dtype=np.int64),
    "Test": model_evaluation["Test"].to_numpy(dtype=np.int64),
    "CleanVerdict": clean_model_evaluation_verdict.astype(np.int64),
    "CleanFailure": clean_model_evaluation_binary.astype(np.int8),
    "Duration": evaluation_duration.astype(float),
})

if evaluation_meta_base.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Evaluation metadata contains duplicate Build-Test rows."
    )


# --------------------------------------------------------------------------------------------------
# 7B. PRECOMPUTE THE ACCELERATED REC ENGINE
# --------------------------------------------------------------------------------------------------

print("Precomputing the vectorized verdict-dependent REC engine.")

combined_history = pd.DataFrame({
    "CombinedRowIndex": np.arange(
        EXPECTED_RAW_TRAIN_ROWS + EXPECTED_RAW_EVAL_ROWS,
        dtype=np.int64,
    ),
    "Build": np.concatenate((
        raw_training["Build"].to_numpy(dtype=np.int64),
        raw_evaluation["Build"].to_numpy(dtype=np.int64),
    )),
    "Test": np.concatenate((
        raw_training["Test"].to_numpy(dtype=np.int64),
        raw_evaluation["Test"].to_numpy(dtype=np.int64),
    )),
    "Job": np.concatenate((
        raw_training["Job"].to_numpy(),
        raw_evaluation["Job"].to_numpy(),
    )),
})
combined_history["BuildOrder"] = combined_history["Build"].map(
    build_order_map
)

if combined_history["BuildOrder"].isna().any():
    raise RuntimeError(
        "Accelerated REC history contains unmapped builds."
    )

combined_history["BuildOrder"] = combined_history[
    "BuildOrder"
].astype(np.int64)
combined_history = (
    combined_history.sort_values(
        ["Test", "BuildOrder", "Job"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

accelerated_history_combined_indices = combined_history[
    "CombinedRowIndex"
].to_numpy(dtype=np.int64)
accelerated_history_builds = combined_history[
    "Build"
].to_numpy(dtype=np.int64)
accelerated_history_tests = combined_history[
    "Test"
].to_numpy(dtype=np.int64)

accelerated_group_starts = np.concatenate((
    np.array([0], dtype=np.int64),
    np.flatnonzero(
        accelerated_history_tests[1:]
        != accelerated_history_tests[:-1]
    ).astype(np.int64) + 1,
))
accelerated_group_ends = np.concatenate((
    accelerated_group_starts[1:],
    np.array([len(combined_history)], dtype=np.int64),
))
accelerated_group_count = len(accelerated_group_starts)

if accelerated_group_count != int(combined_history["Test"].nunique()):
    raise RuntimeError(
        "Accelerated REC test-group count differs."
    )

history_key_index = pd.MultiIndex.from_arrays([
    accelerated_history_builds,
    accelerated_history_tests,
])

if not history_key_index.is_unique:
    raise RuntimeError(
        "Accelerated REC history contains duplicate Build-Test keys."
    )

model_history_positions = history_key_index.get_indexer(
    model_key_index
)

if (model_history_positions < 0).any():
    raise RuntimeError(
        "Accelerated REC history does not cover every model row."
    )

model_group_indices = np.searchsorted(
    accelerated_group_starts,
    model_history_positions,
    side="right",
) - 1
model_local_positions = (
    model_history_positions
    - accelerated_group_starts[model_group_indices]
)

accelerated_requested_model_indices = [
    np.empty(0, dtype=np.int64)
    for _ in range(accelerated_group_count)
]
accelerated_requested_local_positions = [
    np.empty(0, dtype=np.int64)
    for _ in range(accelerated_group_count)
]

request_order = np.lexsort((
    model_local_positions,
    model_group_indices,
))
ordered_group_indices = model_group_indices[request_order]
request_group_starts = np.concatenate((
    np.array([0], dtype=np.int64),
    np.flatnonzero(
        ordered_group_indices[1:]
        != ordered_group_indices[:-1]
    ).astype(np.int64) + 1,
))
request_group_ends = np.concatenate((
    request_group_starts[1:],
    np.array([len(request_order)], dtype=np.int64),
))

for request_start, request_end in zip(
    request_group_starts,
    request_group_ends,
):
    selected = request_order[request_start:request_end]
    group_index = int(model_group_indices[selected[0]])
    accelerated_requested_model_indices[group_index] = selected.astype(
        np.int64,
        copy=False,
    )
    accelerated_requested_local_positions[group_index] = model_local_positions[
        selected
    ].astype(np.int64, copy=False)

accelerated_entity_values = np.sort(
    build_entity["EntityId"].unique().astype(np.int64)
)
accelerated_entity_count = len(accelerated_entity_values)
accelerated_entity_to_dense = {
    int(entity_id): dense_index
    for dense_index, entity_id in enumerate(accelerated_entity_values)
}

accelerated_build_values = np.asarray(
    ordered_builds,
    dtype=np.int64,
)
accelerated_build_to_dense = {
    int(build_id): dense_index
    for dense_index, build_id in enumerate(accelerated_build_values)
}
accelerated_build_entity_arrays = [
    np.empty(0, dtype=np.int32)
    for _ in accelerated_build_values
]

for build_id, entity_ids in build_entity.groupby(
    "BuildID",
    sort=False,
)["EntityId"]:
    build_dense = accelerated_build_to_dense[int(build_id)]
    accelerated_build_entity_arrays[build_dense] = np.asarray(
        sorted({
            accelerated_entity_to_dense[int(entity_id)]
            for entity_id in entity_ids
        }),
        dtype=np.int32,
    )

accelerated_history_build_dense_indices = np.asarray([
    accelerated_build_to_dense[int(build_id)]
    for build_id in accelerated_history_builds
], dtype=np.int32)

accelerated_dependent_feature_indices = np.asarray([
    REC_FEATURES.index(feature)
    for feature in VERDICT_DEPENDENT_REC
], dtype=np.int64)
accelerated_anchor_dependent_all = anchor_values_all[
    :, accelerated_dependent_feature_indices
]

# Exact clean-equivalence self-test before any full condition is allowed.
accelerated_clean_combined_verdict = np.concatenate((
    clean_raw_training_verdict,
    clean_raw_evaluation_verdict,
)).astype(np.int16, copy=False)
accelerated_clean_reconstructed = reconstruct_dependent_rec_fast(
    accelerated_clean_combined_verdict
)
accelerated_clean_anchored = (
    accelerated_clean_reconstructed
    + accelerated_anchor_dependent_all
)
accelerated_clean_original = clean_original_rec_all[
    :, accelerated_dependent_feature_indices
]
accelerated_clean_mismatch_values = int((
    ~np.isclose(
        accelerated_clean_anchored,
        accelerated_clean_original,
        rtol=0,
        atol=1e-12,
    )
).sum())

if accelerated_clean_mismatch_values != 0:
    raise RuntimeError(
        "Accelerated REC engine failed the exact clean-data equivalence test."
    )

print(
    "Accelerated REC engine clean-equivalence mismatches:",
    accelerated_clean_mismatch_values,
)
print(
    "Accelerated REC groups / model rows / entities:",
    accelerated_group_count,
    "/",
    EXPECTED_MODEL_ROWS,
    "/",
    accelerated_entity_count,
)


# --------------------------------------------------------------------------------------------------
# 8. CHECKPOINT SCAN AND FULL CONDITION RUNNER
# --------------------------------------------------------------------------------------------------

FULL_RAW_RESULT_ROOT.mkdir(parents=True, exist_ok=True)
FULL_EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
INCOMPLETE_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

raw_training_hash_before = sha256_file(RAW_TRAINING_COHORT_PATH)
raw_evaluation_hash_before = sha256_file(RAW_EVALUATION_COHORT_PATH)
model_training_hash_before = sha256_file(MODEL_TRAINING_COHORT_PATH)
model_evaluation_hash_before = sha256_file(MODEL_EVALUATION_COHORT_PATH)

expected_condition_files = {
    "rankings.csv.gz",
    "build_metrics.csv",
    "project_runs.csv",
    "model_fits.csv",
    "training_medians.csv",
    "condition_audit.csv",
    "condition_summary.json",
    "COMPLETE.json",
}


def validate_completed_condition(condition_dir, plan_row):
    condition_dir = Path(condition_dir)
    condition_key = str(plan_row.ConditionID)

    if not condition_dir.is_dir():
        return None

    actual_files = {
        path.name
        for path in condition_dir.iterdir()
        if path.is_file()
    }

    if actual_files != expected_condition_files:
        return None

    completion_path = condition_dir / "COMPLETE.json"
    summary_path = condition_dir / "condition_summary.json"

    try:
        completion = load_json(completion_path)
        summary = load_json(summary_path)
    except Exception:
        return None

    if completion.get("Status") != CONDITION_STATUS:
        return None
    if summary.get("Status") != CONDITION_STATUS:
        return None
    if completion.get("ConditionKey") != condition_key:
        return None
    if summary.get("ConditionKey") != condition_key:
        return None
    if int(summary.get("NoisePercent", -1)) != int(plan_row.NoisePercent):
        return None
    if int(summary.get("RepetitionSeed", -1)) != int(plan_row.RepetitionSeed):
        return None
    if str(completion.get("ConditionSummaryPath")) != str(summary_path):
        return None
    if str(completion.get("ConditionSummarySHA256")) != sha256_file(summary_path):
        return None

    output_manifest = summary.get("OutputManifest", [])
    if not isinstance(output_manifest, list) or len(output_manifest) != 6:
        return None

    for item in output_manifest:
        path = Path(item.get("Path", ""))
        if path.parent != condition_dir:
            return None
        if not path.is_file():
            return None
        if int(path.stat().st_size) != int(item.get("Bytes", -1)):
            return None
        if sha256_file(path) != str(item.get("SHA256", "")):
            return None

    expected_counts = {
        "MLFits": EXPECTED_MODEL_FIT_ROWS_PER_CONDITION,
        "RankingRows": EXPECTED_RANKING_ROWS_PER_CONDITION,
        "BuildMetricRows": EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
        "ProjectRunRows": EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
        "TrainingMedianRows": EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION,
    }

    for key, expected in expected_counts.items():
        if int(summary.get(key, -1)) != expected:
            return None

    if int(summary.get("IndependentRECChanges", -1)) != 0:
        return None

    fingerprints = summary.get("BaselineFingerprints", {})
    if sorted(fingerprints.keys()) != ["QTF-Avg", "Random"]:
        return None

    condition_manifest = directory_manifest(condition_dir)
    if len(condition_manifest) != EXPECTED_FILES_PER_CONDITION:
        return None

    return {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": int(plan_row.ConditionOrder),
        "ConditionKey": condition_key,
        "NoisePercent": int(plan_row.NoisePercent),
        "RepetitionSeed": int(plan_row.RepetitionSeed),
        "ConditionDirectory": str(condition_dir),
        "Status": CONDITION_STATUS,
        "Files": int(len(condition_manifest)),
        "ConditionBytes": int(condition_manifest["Bytes"].sum()),
        "ConditionRootSHA256": directory_root_hash(condition_manifest),
        "RankingRows": int(summary["RankingRows"]),
        "BuildMetricRows": int(summary["BuildMetricRows"]),
        "ProjectRunRows": int(summary["ProjectRunRows"]),
        "ModelFitRows": int(summary["MLFits"]),
        "TrainingMedianRows": int(summary["TrainingMedianRows"]),
        "ConditionSeconds": float(summary["ConditionSeconds"]),
        "RandomScoreSHA256": str(fingerprints["Random"]["ScoreSHA256"]),
        "RandomRankSHA256": str(fingerprints["Random"]["RankSHA256"]),
        "QTFAvgScoreSHA256": str(fingerprints["QTF-Avg"]["ScoreSHA256"]),
        "QTFAvgRankSHA256": str(fingerprints["QTF-Avg"]["RankSHA256"]),
    }


def quarantine_incomplete_condition(condition_dir):
    condition_dir = Path(condition_dir)

    if not condition_dir.exists():
        return None

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    destination = INCOMPLETE_BACKUP_ROOT / f"{condition_dir.name}__{timestamp}"
    shutil.move(str(condition_dir), str(destination))
    return destination


print("\nScanning existing condition checkpoints...")

valid_existing = {}
invalid_existing = []

for plan_row in condition_plan.itertuples(index=False):
    condition_key = str(plan_row.ConditionID)
    condition_dir = FULL_RAW_RESULT_ROOT / condition_key
    validated = validate_completed_condition(condition_dir, plan_row)

    if validated is not None:
        valid_existing[condition_key] = validated
    elif condition_dir.exists():
        invalid_existing.append(condition_key)

print("Valid completed conditions:", len(valid_existing))
print("Incomplete/invalid condition directories:", len(invalid_existing))
print("Pending conditions:", EXPECTED_CONDITIONS - len(valid_existing))

for condition_key in invalid_existing:
    backup = quarantine_incomplete_condition(
        FULL_RAW_RESULT_ROOT / condition_key
    )
    print("Preserved incomplete condition in:", backup)

# The frozen 0% and 50% seed-1 smoke conditions are executed/validated first.
equivalence_records_by_key = {}
for equivalence_key in SMOKE_EQUIVALENCE_KEYS:
    equivalence_row = condition_plan.loc[
        condition_plan["ConditionID"].eq(equivalence_key)
    ]
    if len(equivalence_row) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve smoke-equivalence condition {equivalence_key}."
        )
    equivalence_plan_row = next(equivalence_row.itertuples(index=False))
    equivalence_dir = FULL_RAW_RESULT_ROOT / equivalence_key
    if validate_completed_condition(equivalence_dir, equivalence_plan_row) is not None:
        equivalence_record = compare_full_condition_to_smoke(
            equivalence_key,
            equivalence_dir,
        )
        if not equivalence_record["Pass"]:
            raise RuntimeError(
                f"Existing accelerated condition {equivalence_key} differs from Step 4B."
            )
        equivalence_records_by_key[equivalence_key] = equivalence_record

smoke_first_plan = condition_plan.loc[
    condition_plan["ConditionID"].isin(SMOKE_EQUIVALENCE_KEYS)
].copy()
smoke_first_plan["__SmokeOrder"] = smoke_first_plan["ConditionID"].map({
    key: index
    for index, key in enumerate(SMOKE_EQUIVALENCE_KEYS)
})
smoke_first_plan = smoke_first_plan.sort_values(
    "__SmokeOrder",
    kind="mergesort",
).drop(columns="__SmokeOrder")
remaining_plan = condition_plan.loc[
    ~condition_plan["ConditionID"].isin(SMOKE_EQUIVALENCE_KEYS)
].sort_values("ConditionOrder", kind="mergesort")
execution_plan = pd.concat(
    [smoke_first_plan, remaining_plan],
    ignore_index=True,
)

if len(execution_plan) != EXPECTED_CONDITIONS:
    raise RuntimeError("Accelerated execution plan does not contain 270 conditions.")

if equivalence_records_by_key:
    atomic_csv(
        ACCELERATED_EQUIVALENCE_PATH,
        pd.DataFrame(equivalence_records_by_key.values()).sort_values(
            "ConditionKey",
            kind="mergesort",
        ),
    )

full_execution_started = time.perf_counter()
completed_this_run = 0
skipped_valid = 0
current_rng_seed = None
flip_uniform = None
sampled_failure_subtype = None
random_scores = None

for plan_row in execution_plan.itertuples(index=False):
    condition_order = int(plan_row.ConditionOrder)
    condition_key = str(plan_row.ConditionID)
    noise_percent = int(plan_row.NoisePercent)
    repetition_seed = int(plan_row.RepetitionSeed)
    condition_dir = FULL_RAW_RESULT_ROOT / condition_key

    already_valid = validate_completed_condition(condition_dir, plan_row)
    if already_valid is not None:
        skipped_valid += 1
        print(
            f"[{condition_order}/{EXPECTED_CONDITIONS}] "
            f"Skipping validated checkpoint {condition_key}"
        )
        continue

    if (
        condition_key not in SMOKE_EQUIVALENCE_KEYS
        and set(equivalence_records_by_key) != set(SMOKE_EQUIVALENCE_KEYS)
    ):
        raise RuntimeError(
            "The accelerated engine must pass both frozen smoke-output equivalence checks "
            "before any other full condition is executed."
        )

    if repetition_seed != current_rng_seed:
        print(f"\nLoading deterministic RNG stream for seed {repetition_seed}.")

        rng_seed_frame = pd.read_parquet(
            RNG_MANIFEST_PATH,
            filters=[("RepetitionSeed", "==", repetition_seed)],
        )
        rng_seed_frame[raw_order_column] = parse_int(
            rng_seed_frame[raw_order_column],
            f"rng_seed_{repetition_seed}.RawTrainingRowOrder",
        )
        rng_seed_frame = (
            rng_seed_frame.sort_values(
                raw_order_column,
                kind="mergesort",
            )
            .reset_index(drop=True)
        )

        if len(rng_seed_frame) != EXPECTED_RAW_TRAIN_ROWS:
            raise RuntimeError(
                f"Seed {repetition_seed}: RNG stream row count differs."
            )

        if not np.array_equal(
            rng_seed_frame[raw_order_column].to_numpy(dtype=np.int64),
            np.arange(1, EXPECTED_RAW_TRAIN_ROWS + 1, dtype=np.int64),
        ):
            raise RuntimeError(
                f"Seed {repetition_seed}: RNG row order differs."
            )

        flip_uniform = rng_seed_frame["FlipUniform"].to_numpy(dtype=np.float64)
        sampled_failure_subtype = rng_seed_frame[
            "SampledFailureSubtype"
        ].to_numpy(dtype=np.int16)

        if not np.isfinite(flip_uniform).all():
            raise RuntimeError(
                f"Seed {repetition_seed}: non-finite flip uniforms."
            )
        if ((flip_uniform < 0) | (flip_uniform >= 1)).any():
            raise RuntimeError(
                f"Seed {repetition_seed}: flip uniforms outside [0,1)."
            )

        random_scores = np.empty(EXPECTED_MODEL_EVAL_ROWS, dtype=np.float64)
        eval_build_array = evaluation_meta_base["Build"].to_numpy(dtype=np.int64)

        for build_id in sorted(evaluation_meta_base["Build"].unique()):
            build_indices = np.flatnonzero(eval_build_array == int(build_id))
            random_scores[build_indices] = np.random.default_rng(
                deterministic_random_build_seed(
                    repetition_seed,
                    int(build_id),
                )
            ).random(len(build_indices))

        if not np.isfinite(random_scores).all():
            raise RuntimeError(
                f"Seed {repetition_seed}: Random baseline scores are non-finite."
            )

        current_rng_seed = repetition_seed
        del rng_seed_frame
        gc.collect()

    condition_started = time.perf_counter()

    print("\n" + "-" * 110)
    print(
        f"[{condition_order}/{EXPECTED_CONDITIONS}] Running {condition_key}"
    )
    print("-" * 110)

    condition_dir.mkdir(parents=True, exist_ok=True)

    ranking_path = condition_dir / "rankings.csv.gz"
    build_metrics_path = condition_dir / "build_metrics.csv"
    project_runs_path = condition_dir / "project_runs.csv"
    model_fits_path = condition_dir / "model_fits.csv"
    training_medians_path = condition_dir / "training_medians.csv"
    condition_audit_path = condition_dir / "condition_audit.csv"
    condition_summary_path = condition_dir / "condition_summary.json"
    completion_marker_path = condition_dir / "COMPLETE.json"

    flip_mask = flip_uniform < (noise_percent / 100.0)
    noisy_raw_training_verdict = clean_raw_training_verdict.copy()

    pass_to_failure_mask = flip_mask & (clean_raw_training_verdict == 0)
    failure_to_pass_mask = flip_mask & (clean_raw_training_verdict != 0)

    noisy_raw_training_verdict[pass_to_failure_mask] = (
        sampled_failure_subtype[pass_to_failure_mask]
    )
    noisy_raw_training_verdict[failure_to_pass_mask] = 0

    noisy_model_training_verdict = noisy_raw_training_verdict[
        model_training_raw_indices
    ]

    actual_flip_mask_sha256 = sha256_array(
        flip_mask.astype(np.uint8),
        "u1",
    )
    actual_noisy_raw_sha256 = sha256_array(
        noisy_raw_training_verdict,
        "<i2",
    )
    actual_noisy_model_sha256 = sha256_array(
        noisy_model_training_verdict,
        "<i2",
    )

    expected_flip_mask_sha256 = str(plan_row.FlipMaskSHA256)
    expected_noisy_raw_sha256 = str(plan_row.NoisyRawVerdictSHA256)
    expected_noisy_model_sha256 = str(plan_row.NoisyModelVerdictSHA256)

    if actual_flip_mask_sha256 != expected_flip_mask_sha256:
        raise RuntimeError(
            f"{condition_key}: flip-mask SHA-256 differs from Step 3A."
        )
    if actual_noisy_raw_sha256 != expected_noisy_raw_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy raw-verdict SHA-256 differs from Step 3A."
        )
    if actual_noisy_model_sha256 != expected_noisy_model_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy model-verdict SHA-256 differs from Step 3A."
        )

    number_flipped = int(flip_mask.sum())
    pass_to_failure = int(pass_to_failure_mask.sum())
    failure_to_pass = int(failure_to_pass_mask.sum())
    model_label_changes = int(
        (noisy_model_training_verdict != clean_model_training_verdict).sum()
    )
    noisy_model_training_binary = (
        noisy_model_training_verdict != 0
    ).astype(np.int8)
    noisy_model_training_failures = int(noisy_model_training_binary.sum())

    if number_flipped != int(plan_row.NumberFlipped):
        raise RuntimeError(
            f"{condition_key}: NumberFlipped differs from Step 3A."
        )
    if pass_to_failure != int(plan_row.PassToFailure):
        raise RuntimeError(
            f"{condition_key}: PassToFailure differs from Step 3A."
        )
    if failure_to_pass != int(plan_row.FailureToPass):
        raise RuntimeError(
            f"{condition_key}: FailureToPass differs from Step 3A."
        )
    if model_label_changes != int(plan_row.ModelLabelChanges):
        raise RuntimeError(
            f"{condition_key}: ModelLabelChanges differs from Step 3A."
        )
    if noisy_model_training_failures != int(plan_row.NoisyModelFailures):
        raise RuntimeError(
            f"{condition_key}: NoisyModelFailures differs from Step 3A."
        )

    rec_started = time.perf_counter()

    if noise_percent == 0:
        # The 0% REC matrix is already frozen and exact. Reusing it avoids
        # thirty identical 833,541-row reconstructions.
        condition_numeric_all = all_base_numeric.copy()
        dependent_rec_changes = 0
        independent_rec_changes = 0
        independent_reconstruction_mismatches = 0
        reconstructed_row_count = EXPECTED_MODEL_ROWS
    else:
        condition_combined_verdict = np.concatenate((
            noisy_raw_training_verdict,
            clean_raw_evaluation_verdict,
        )).astype(np.int16, copy=False)

        reconstructed_dependent = reconstruct_dependent_rec_fast(
            condition_combined_verdict
        )
        anchored_dependent = (
            reconstructed_dependent
            + accelerated_anchor_dependent_all
        )

        condition_numeric_all = all_base_numeric.copy()
        condition_numeric_all[:, dependent_predictor_indices] = (
            anchored_dependent
        )

        dependent_rec_changes = int((
            ~np.isclose(
                condition_numeric_all[:, dependent_predictor_indices],
                all_base_numeric[:, dependent_predictor_indices],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum())
        independent_rec_changes = int((
            ~np.isclose(
                condition_numeric_all[:, independent_predictor_indices],
                all_base_numeric[:, independent_predictor_indices],
                rtol=0,
                atol=0,
                equal_nan=True,
            )
        ).sum())
        independent_reconstruction_mismatches = 0
        reconstructed_row_count = EXPECTED_MODEL_ROWS

        if independent_rec_changes != 0:
            raise RuntimeError(
                f"{condition_key}: preserved independent REC predictors changed."
            )

    rec_seconds = time.perf_counter() - rec_started

    condition_training_numeric = condition_numeric_all[
        :EXPECTED_MODEL_TRAIN_ROWS
    ].copy()
    condition_evaluation_numeric = condition_numeric_all[
        EXPECTED_MODEL_TRAIN_ROWS:
    ].copy()

    if noise_percent == 0:
        zero_rec_mismatches = int((
            ~np.isclose(
                condition_numeric_all[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                all_base_numeric[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum())
        if zero_rec_mismatches != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition did not reproduce clean REC."
            )
        if number_flipped != 0 or model_label_changes != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition changed labels."
            )
        if dependent_rec_changes != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition changed dependent REC."
            )
    else:
        if number_flipped <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no raw labels."
            )
        if model_label_changes <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no model labels."
            )
        if dependent_rec_changes <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no dependent REC."
            )

    medians = np.nanmedian(condition_training_numeric, axis=0)
    nonfinite_median_indices = np.flatnonzero(~np.isfinite(medians))
    if len(nonfinite_median_indices) != 0:
        bad_features = [predictor_columns[index] for index in nonfinite_median_indices]
        raise RuntimeError(
            f"{condition_key}: non-finite training medians for {bad_features}."
        )

    training_missing_mask = ~np.isfinite(condition_training_numeric)
    evaluation_missing_mask = ~np.isfinite(condition_evaluation_numeric)

    if training_missing_mask.any():
        row_indices, column_indices = np.where(training_missing_mask)
        condition_training_numeric[row_indices, column_indices] = medians[
            column_indices
        ]
    if evaluation_missing_mask.any():
        row_indices, column_indices = np.where(evaluation_missing_mask)
        condition_evaluation_numeric[row_indices, column_indices] = medians[
            column_indices
        ]

    if not np.isfinite(condition_training_numeric).all():
        raise RuntimeError(
            f"{condition_key}: training matrix remains non-finite."
        )
    if not np.isfinite(condition_evaluation_numeric).all():
        raise RuntimeError(
            f"{condition_key}: evaluation matrix remains non-finite."
        )

    training_medians = pd.DataFrame({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "PredictorOrder": np.arange(
            1,
            EXPECTED_PREDICTORS + 1,
            dtype=np.int64,
        ),
        "Predictor": predictor_columns,
        "TrainingMedian": medians.astype(float),
    })

    evaluation_meta = evaluation_meta_base.copy()
    evaluation_meta["ConditionKey"] = condition_key
    evaluation_meta["NoisePercent"] = noise_percent
    evaluation_meta["RepetitionSeed"] = repetition_seed

    technique_scores = {}
    model_fit_records = []
    models = create_models(repetition_seed)

    for technique in ML_TECHNIQUES:
        print(f"  Fitting: {technique}")
        model = models[technique]
        fit_started = time.perf_counter()

        try:
            model.fit(
                condition_training_numeric,
                noisy_model_training_binary,
            )
            fit_seconds = time.perf_counter() - fit_started
            technique_scores[technique] = positive_probability(
                model,
                condition_evaluation_numeric,
            )
            fit_status = "PASS_MODEL_FIT"
            fit_error = ""
        except Exception as error:
            fit_seconds = time.perf_counter() - fit_started
            fit_status = "FAIL_MODEL_FIT"
            fit_error = repr(error)
            model_fit_records.append({
                "ProjectNumber": PROJECT_NUMBER,
                "Project": PROJECT_NAME,
                "ProjectSlug": PROJECT_SLUG,
                "ConditionKey": condition_key,
                "NoisePercent": noise_percent,
                "RepetitionSeed": repetition_seed,
                "Technique": technique,
                "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
                "TrainingFailures": noisy_model_training_failures,
                "Predictors": EXPECTED_PREDICTORS,
                "FitSeconds": float(fit_seconds),
                "ClassesJSON": "[]",
                "Status": fit_status,
                "Error": fit_error,
            })
            raise RuntimeError(
                f"{condition_key}: {technique} fitting failed: {error!r}"
            ) from error

        model_fit_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": condition_key,
            "NoisePercent": noise_percent,
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
            "TrainingFailures": noisy_model_training_failures,
            "Predictors": EXPECTED_PREDICTORS,
            "FitSeconds": float(fit_seconds),
            "ClassesJSON": json.dumps([
                int(value)
                for value in np.asarray(model.classes_).tolist()
            ]),
            "Status": fit_status,
            "Error": fit_error,
        })
        del model
        gc.collect()

    model_fits = pd.DataFrame(model_fit_records)
    if len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: model-fit row count differs."
        )
    if not model_fits["Status"].eq("PASS_MODEL_FIT").all():
        raise RuntimeError(
            f"{condition_key}: one or more model fits failed."
        )

    condition_eval_last_failure_age = condition_evaluation_numeric[
        :, predictor_index["REC_LastFailureAge"]
    ].astype(float)
    condition_eval_qtf = condition_evaluation_numeric[
        :, predictor_index["REC_TotalAvgExeTime"]
    ].astype(float)

    technique_scores["Random"] = random_scores.copy()
    technique_scores["LatestFail"] = -condition_eval_last_failure_age
    technique_scores["QTF-Avg"] = condition_eval_qtf

    ranking_frames = []
    for technique in ALL_TECHNIQUES:
        ranking_frames.append(
            make_ranking(
                evaluation_meta=evaluation_meta,
                technique=technique,
                scores=technique_scores[technique],
                ascending_score=(technique == "QTF-Avg"),
            )
        )

    rankings = pd.concat(ranking_frames, ignore_index=True)
    if len(rankings) != EXPECTED_RANKING_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: ranking row count differs."
        )
    if sorted(rankings["Technique"].unique().tolist()) != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: ranking technique set differs."
        )
    if not rankings.groupby("Technique").size().eq(
        EXPECTED_MODEL_EVAL_ROWS
    ).all():
        raise RuntimeError(
            f"{condition_key}: ranking rows per technique differ."
        )
    if rankings.duplicated(
        subset=["Technique", "Build", "Test"],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: duplicate ranking rows found."
        )

    build_metrics, project_runs = calculate_condition_metrics(rankings)
    if len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: build-metric row count differs."
        )
    if len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: project-run row count differs."
        )
    if sorted(project_runs["Technique"].tolist()) != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: project-run technique set differs."
        )

    build_metric_values = build_metrics[["APFDc", "APFD"]].to_numpy(dtype=float)
    if not np.isfinite(build_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: build metrics contain non-finite values."
        )
    if ((build_metric_values < 0) | (build_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: build metrics fall outside [0,1]."
        )

    project_metric_values = project_runs[[
        "MeanAPFDc",
        "MedianAPFDc",
        "MeanAPFD",
        "MedianAPFD",
    ]].to_numpy(dtype=float)
    if not np.isfinite(project_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: project metrics contain non-finite values."
        )
    if ((project_metric_values < 0) | (project_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: project metrics fall outside [0,1]."
        )

    condition_seconds = time.perf_counter() - condition_started

    condition_audit = pd.DataFrame([{
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "RawTrainingRows": EXPECTED_RAW_TRAIN_ROWS,
        "NumberFlipped": number_flipped,
        "ExpectedNumberFlipped": int(plan_row.NumberFlipped),
        "RealisedNoisePercent": float(
            100.0 * number_flipped / EXPECTED_RAW_TRAIN_ROWS
        ),
        "PassToFailure": pass_to_failure,
        "FailureToPass": failure_to_pass,
        "ModelTrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
        "ModelLabelChanges": model_label_changes,
        "ExpectedModelLabelChanges": int(plan_row.ModelLabelChanges),
        "TrainingFailures": noisy_model_training_failures,
        "ExpectedTrainingFailures": int(plan_row.NoisyModelFailures),
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "IndependentReconstructionMismatches": independent_reconstruction_mismatches,
        "ReconstructedRows": int(reconstructed_row_count),
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ExpectedFlipMaskSHA256": expected_flip_mask_sha256,
        "ActualFlipMaskSHA256": actual_flip_mask_sha256,
        "ExpectedNoisyRawVerdictSHA256": expected_noisy_raw_sha256,
        "ActualNoisyRawVerdictSHA256": actual_noisy_raw_sha256,
        "ExpectedNoisyModelVerdictSHA256": expected_noisy_model_sha256,
        "ActualNoisyModelVerdictSHA256": actual_noisy_model_sha256,
        "RECSeconds": float(rec_seconds),
        "ConditionSeconds": float(condition_seconds),
        "Status": CONDITION_STATUS,
    }])

    baseline_fingerprints = {}
    for technique in ["Random", "QTF-Avg"]:
        baseline_rows = (
            rankings.loc[
                rankings["Technique"].eq(technique),
                ["Build", "Test", "Score", "Rank"],
            ]
            .sort_values(["Build", "Test"], kind="mergesort")
            .reset_index(drop=True)
        )
        baseline_fingerprints[technique] = {
            "Rows": int(len(baseline_rows)),
            "KeySHA256": hashlib.sha256(
                np.ascontiguousarray(
                    baseline_rows[["Build", "Test"]].to_numpy(dtype=np.int64)
                ).tobytes(order="C")
            ).hexdigest(),
            "ScoreSHA256": sha256_array(
                baseline_rows["Score"].to_numpy(dtype=np.float64),
                "<f8",
            ),
            "RankSHA256": sha256_array(
                baseline_rows["Rank"].to_numpy(dtype=np.int64),
                "<i8",
            ),
        }

    atomic_csv(ranking_path, rankings, compression="gzip")
    atomic_csv(build_metrics_path, build_metrics)
    atomic_csv(project_runs_path, project_runs)
    atomic_csv(model_fits_path, model_fits)
    atomic_csv(training_medians_path, training_medians)
    atomic_csv(condition_audit_path, condition_audit)

    if condition_key in SMOKE_EQUIVALENCE_KEYS:
        equivalence_record = compare_full_condition_to_smoke(
            condition_key,
            condition_dir,
        )
        if not equivalence_record["Pass"]:
            print("\nAccelerated-engine equivalence failure:")
            display(pd.DataFrame([equivalence_record]))
            raise RuntimeError(
                f"{condition_key}: accelerated outputs differ from the frozen Step 4B outputs."
            )
        equivalence_records_by_key[condition_key] = equivalence_record
        atomic_csv(
            ACCELERATED_EQUIVALENCE_PATH,
            pd.DataFrame(equivalence_records_by_key.values()).sort_values(
                "ConditionKey",
                kind="mergesort",
            ),
        )
        print(
            "  Frozen smoke-output equivalence: PASS |",
            condition_key,
        )

    condition_output_paths = [
        ranking_path,
        build_metrics_path,
        project_runs_path,
        model_fits_path,
        training_medians_path,
        condition_audit_path,
    ]
    condition_output_manifest = [
        {
            "Path": str(path),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        }
        for path in condition_output_paths
    ]

    condition_summary = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "Status": CONDITION_STATUS,
        "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
        "NumberFlipped": number_flipped,
        "ModelLabelChanges": model_label_changes,
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "TrainingFailures": noisy_model_training_failures,
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
        "BaselineFingerprints": baseline_fingerprints,
        "OutputManifest": condition_output_manifest,
    }
    atomic_json(condition_summary_path, condition_summary)

    completion_marker = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "Status": CONDITION_STATUS,
        "ConditionSummaryPath": str(condition_summary_path),
        "ConditionSummarySHA256": sha256_file(condition_summary_path),
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
    }
    atomic_json(completion_marker_path, completion_marker)

    validated_after_write = validate_completed_condition(
        condition_dir,
        plan_row,
    )
    if validated_after_write is None:
        raise RuntimeError(
            f"{condition_key}: completed condition did not pass readback validation."
        )

    completed_this_run += 1
    completed_total = skipped_valid + completed_this_run

    atomic_json(
        RUN_PROGRESS_PATH,
        {
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "Status": "PROJECT_11_FULL_EXPERIMENT_IN_PROGRESS",
            "UpdatedAtUTC": datetime.now(timezone.utc).isoformat(),
            "CompletedConditions": completed_total,
            "ExpectedConditions": EXPECTED_CONDITIONS,
            "LastCompletedCondition": condition_key,
            "LastCompletedConditionOrder": condition_order,
            "ResumeSafe": True,
            "RegistryModified": False,
            "Project12Accessed": False,
        },
    )

    print(
        f"  Completed: {condition_key}\n"
        f"  Raw flips: {number_flipped} | "
        f"model-label changes: {model_label_changes} | "
        f"dependent REC changes: {dependent_rec_changes}\n"
        f"  Training failures: {noisy_model_training_failures} | "
        f"condition seconds: {condition_seconds:.2f}"
    )

    del condition_numeric_all
    if noise_percent > 0:
        del condition_combined_verdict
        del reconstructed_dependent
        del anchored_dependent
    del condition_training_numeric
    del condition_evaluation_numeric
    del rankings
    del ranking_frames
    del technique_scores
    del models
    gc.collect()


# --------------------------------------------------------------------------------------------------
# 9. FINAL 270-CONDITION REVALIDATION
# --------------------------------------------------------------------------------------------------

print("\nValidating all 270 completed conditions.")

inventory_records = []
condition_audits = []
project_run_frames = []
build_metric_frames = []
model_fit_frames = []
baseline_records = []

for plan_row in condition_plan.itertuples(index=False):
    condition_dir = FULL_RAW_RESULT_ROOT / str(plan_row.ConditionID)
    validated = validate_completed_condition(condition_dir, plan_row)

    if validated is None:
        raise RuntimeError(
            f"Final validation failed for {plan_row.ConditionID}."
        )

    inventory_records.append(validated)
    condition_audits.append(pd.read_csv(condition_dir / "condition_audit.csv"))
    project_run_frames.append(pd.read_csv(condition_dir / "project_runs.csv"))
    build_metric_frames.append(pd.read_csv(condition_dir / "build_metrics.csv"))
    model_fit_frames.append(pd.read_csv(condition_dir / "model_fits.csv"))

    summary = load_json(condition_dir / "condition_summary.json")
    for technique in ["Random", "QTF-Avg"]:
        fingerprint = summary["BaselineFingerprints"][technique]
        baseline_records.append({
            "ConditionKey": str(plan_row.ConditionID),
            "NoisePercent": int(plan_row.NoisePercent),
            "RepetitionSeed": int(plan_row.RepetitionSeed),
            "Technique": technique,
            "Rows": int(fingerprint["Rows"]),
            "KeySHA256": str(fingerprint["KeySHA256"]),
            "ScoreSHA256": str(fingerprint["ScoreSHA256"]),
            "RankSHA256": str(fingerprint["RankSHA256"]),
        })

condition_inventory = pd.DataFrame(inventory_records).sort_values(
    "ConditionOrder",
    kind="mergesort",
).reset_index(drop=True)
combined_condition_audit = pd.concat(condition_audits, ignore_index=True)
combined_project_runs = pd.concat(project_run_frames, ignore_index=True)
combined_build_metrics = pd.concat(build_metric_frames, ignore_index=True)
combined_model_fits = pd.concat(model_fit_frames, ignore_index=True)
baseline_fingerprints = pd.DataFrame(baseline_records)

baseline_invariance_records = []
for repetition_seed in REPETITION_SEEDS:
    for technique in ["Random", "QTF-Avg"]:
        rows = baseline_fingerprints.loc[
            baseline_fingerprints["RepetitionSeed"].eq(repetition_seed)
            & baseline_fingerprints["Technique"].eq(technique)
        ]
        key_variants = int(rows["KeySHA256"].nunique())
        score_variants = int(rows["ScoreSHA256"].nunique())
        rank_variants = int(rows["RankSHA256"].nunique())
        baseline_invariance_records.append({
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "Conditions": int(len(rows)),
            "KeyVariantsAcrossNoise": key_variants,
            "ScoreVariantsAcrossNoise": score_variants,
            "RankVariantsAcrossNoise": rank_variants,
            "Pass": bool(
                len(rows) == len(NOISE_LEVELS)
                and key_variants == 1
                and score_variants == 1
                and rank_variants == 1
            ),
        })

baseline_invariance = pd.DataFrame(baseline_invariance_records)
baseline_invariance_failures = int((~baseline_invariance["Pass"]).sum())
qtf_global_score_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints["Technique"].eq("QTF-Avg"),
        "ScoreSHA256",
    ].nunique()
)
qtf_global_rank_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints["Technique"].eq("QTF-Avg"),
        "RankSHA256",
    ].nunique()
)

raw_manifest = directory_manifest(FULL_RAW_RESULT_ROOT)
raw_root_sha256 = directory_root_hash(raw_manifest)
raw_files = int(len(raw_manifest))
raw_bytes = int(raw_manifest["Bytes"].sum())

raw_training_hash_after = sha256_file(RAW_TRAINING_COHORT_PATH)
raw_evaluation_hash_after = sha256_file(RAW_EVALUATION_COHORT_PATH)
model_training_hash_after = sha256_file(MODEL_TRAINING_COHORT_PATH)
model_evaluation_hash_after = sha256_file(MODEL_EVALUATION_COHORT_PATH)
registry_sha256_after = sha256_file(REGISTRY_PATH)

current_source_rows_after = []
for row in frozen_source_manifest.itertuples(index=False):
    source_path = Path(row.AbsolutePath)
    current_source_rows_after.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })
source_root_sha256_after = source_root_hash(pd.DataFrame(current_source_rows_after))

zero_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(0)
]
positive_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].gt(0)
]

noise_plan_hash_mismatches = int(
    combined_condition_audit[
        "ExpectedFlipMaskSHA256"
    ].ne(combined_condition_audit["ActualFlipMaskSHA256"]).sum()
    + combined_condition_audit[
        "ExpectedNoisyRawVerdictSHA256"
    ].ne(combined_condition_audit["ActualNoisyRawVerdictSHA256"]).sum()
    + combined_condition_audit[
        "ExpectedNoisyModelVerdictSHA256"
    ].ne(combined_condition_audit["ActualNoisyModelVerdictSHA256"]).sum()
)

project_metric_columns = [
    "MeanAPFDc",
    "MedianAPFDc",
    "MeanAPFD",
    "MedianAPFD",
]
project_metric_values = combined_project_runs[
    project_metric_columns
].to_numpy(dtype=float)

if not ACCELERATED_EQUIVALENCE_PATH.is_file():
    raise FileNotFoundError(
        "Accelerated-engine equivalence audit is missing."
    )
accelerated_equivalence = pd.read_csv(
    ACCELERATED_EQUIVALENCE_PATH,
    low_memory=False,
)
accelerated_equivalence_passes = int(
    accelerated_equivalence["Pass"].astype(bool).sum()
)

validation_records = []
add_check(validation_records, "Step 4B passed", EXPECTED_STEP4B_STATUS, smoke_checkpoint.get("Status"), smoke_checkpoint.get("Status") == EXPECTED_STEP4B_STATUS)
add_check(validation_records, "Smoke checkpoint SHA-256", EXPECTED_SMOKE_CHECKPOINT_SHA256, smoke_checkpoint_sha256, smoke_checkpoint_sha256 == EXPECTED_SMOKE_CHECKPOINT_SHA256)
add_check(validation_records, "Accelerated clean REC mismatches", 0, accelerated_clean_mismatch_values, accelerated_clean_mismatch_values == 0)
add_check(validation_records, "Accelerated smoke-equivalence rows", 2, len(accelerated_equivalence), len(accelerated_equivalence) == 2)
add_check(validation_records, "Accelerated smoke-equivalence keys", sorted(SMOKE_EQUIVALENCE_KEYS), sorted(accelerated_equivalence["ConditionKey"].tolist()), sorted(accelerated_equivalence["ConditionKey"].tolist()) == sorted(SMOKE_EQUIVALENCE_KEYS))
add_check(validation_records, "Accelerated smoke-equivalence failures", 0, int((~accelerated_equivalence["Pass"].astype(bool)).sum()), accelerated_equivalence["Pass"].astype(bool).all())
add_check(validation_records, "Completed conditions", EXPECTED_CONDITIONS, len(condition_inventory), len(condition_inventory) == EXPECTED_CONDITIONS)
add_check(validation_records, "Noise levels", NOISE_LEVELS, sorted(condition_inventory["NoisePercent"].unique().tolist()), sorted(condition_inventory["NoisePercent"].unique().tolist()) == NOISE_LEVELS)
add_check(validation_records, "Repetition seeds", REPETITION_SEEDS, sorted(condition_inventory["RepetitionSeed"].unique().tolist()), sorted(condition_inventory["RepetitionSeed"].unique().tolist()) == REPETITION_SEEDS)
add_check(validation_records, "Duplicate condition keys", 0, int(condition_inventory["ConditionKey"].duplicated(keep=False).sum()), not condition_inventory["ConditionKey"].duplicated(keep=False).any())
add_check(validation_records, "Duplicate condition coordinates", 0, int(condition_inventory.duplicated(subset=["NoisePercent", "RepetitionSeed"], keep=False).sum()), not condition_inventory.duplicated(subset=["NoisePercent", "RepetitionSeed"], keep=False).any())
add_check(validation_records, "Condition-order sequence", list(range(1, EXPECTED_CONDITIONS + 1)), condition_inventory["ConditionOrder"].tolist(), condition_inventory["ConditionOrder"].tolist() == list(range(1, EXPECTED_CONDITIONS + 1)))
add_check(validation_records, "Files per condition", EXPECTED_FILES_PER_CONDITION, sorted(condition_inventory["Files"].unique().tolist()), condition_inventory["Files"].eq(EXPECTED_FILES_PER_CONDITION).all())
add_check(validation_records, "Raw files", EXPECTED_RAW_FILES, raw_files, raw_files == EXPECTED_RAW_FILES)
add_check(validation_records, "Ranking rows", EXPECTED_TOTAL_RANKING_ROWS, int(condition_inventory["RankingRows"].sum()), int(condition_inventory["RankingRows"].sum()) == EXPECTED_TOTAL_RANKING_ROWS)
add_check(validation_records, "Build-metric rows", EXPECTED_TOTAL_BUILD_METRIC_ROWS, len(combined_build_metrics), len(combined_build_metrics) == EXPECTED_TOTAL_BUILD_METRIC_ROWS)
add_check(validation_records, "Project-run rows", EXPECTED_TOTAL_PROJECT_RUN_ROWS, len(combined_project_runs), len(combined_project_runs) == EXPECTED_TOTAL_PROJECT_RUN_ROWS)
add_check(validation_records, "Model fits", EXPECTED_TOTAL_MODEL_FITS, len(combined_model_fits), len(combined_model_fits) == EXPECTED_TOTAL_MODEL_FITS)
add_check(validation_records, "Condition-audit rows", EXPECTED_CONDITIONS, len(combined_condition_audit), len(combined_condition_audit) == EXPECTED_CONDITIONS)
add_check(validation_records, "Training-median rows", EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS, int(condition_inventory["TrainingMedianRows"].sum()), int(condition_inventory["TrainingMedianRows"].sum()) == EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS)
add_check(validation_records, "Active predictors", EXPECTED_PREDICTORS, sorted(combined_condition_audit["Predictors"].unique().tolist()), combined_condition_audit["Predictors"].eq(EXPECTED_PREDICTORS).all())
add_check(validation_records, "Condition statuses", [CONDITION_STATUS], sorted(combined_condition_audit["Status"].unique().tolist()), combined_condition_audit["Status"].eq(CONDITION_STATUS).all())
add_check(validation_records, "Model-fit failures", 0, int((~combined_model_fits["Status"].eq("PASS_MODEL_FIT")).sum()), combined_model_fits["Status"].eq("PASS_MODEL_FIT").all())
add_check(validation_records, "Project-run technique set", sorted(ALL_TECHNIQUES), sorted(combined_project_runs["Technique"].unique().tolist()), sorted(combined_project_runs["Technique"].unique().tolist()) == sorted(ALL_TECHNIQUES))
add_check(validation_records, "Model-fit technique set", sorted(ML_TECHNIQUES), sorted(combined_model_fits["Technique"].unique().tolist()), sorted(combined_model_fits["Technique"].unique().tolist()) == sorted(ML_TECHNIQUES))
add_check(validation_records, "Zero-noise conditions", len(REPETITION_SEEDS), len(zero_audit), len(zero_audit) == len(REPETITION_SEEDS))
add_check(validation_records, "Zero-noise raw flips", 0, int(zero_audit["NumberFlipped"].sum()), int(zero_audit["NumberFlipped"].sum()) == 0)
add_check(validation_records, "Zero-noise model-label changes", 0, int(zero_audit["ModelLabelChanges"].sum()), int(zero_audit["ModelLabelChanges"].sum()) == 0)
add_check(validation_records, "Zero-noise dependent REC changes", 0, int(zero_audit["DependentRECChanges"].sum()), int(zero_audit["DependentRECChanges"].sum()) == 0)
add_check(validation_records, "Positive-noise raw-change violations", 0, int(positive_audit["NumberFlipped"].le(0).sum()), not positive_audit["NumberFlipped"].le(0).any())
add_check(validation_records, "Positive-noise model-change violations", 0, int(positive_audit["ModelLabelChanges"].le(0).sum()), not positive_audit["ModelLabelChanges"].le(0).any())
add_check(validation_records, "Positive-noise dependent-REC violations", 0, int(positive_audit["DependentRECChanges"].le(0).sum()), not positive_audit["DependentRECChanges"].le(0).any())
add_check(validation_records, "Independent REC changes", 0, int(combined_condition_audit["IndependentRECChanges"].sum()), int(combined_condition_audit["IndependentRECChanges"].sum()) == 0)
add_check(validation_records, "Independent reconstruction mismatches", 0, int(combined_condition_audit["IndependentReconstructionMismatches"].sum()), int(combined_condition_audit["IndependentReconstructionMismatches"].sum()) == 0)
add_check(validation_records, "Noise-plan hash mismatches", 0, noise_plan_hash_mismatches, noise_plan_hash_mismatches == 0)
add_check(validation_records, "Baseline invariance failures", 0, baseline_invariance_failures, baseline_invariance_failures == 0)
add_check(validation_records, "QTF global score variants", 1, qtf_global_score_variants, qtf_global_score_variants == 1)
add_check(validation_records, "QTF global rank variants", 1, qtf_global_rank_variants, qtf_global_rank_variants == 1)
add_check(validation_records, "Project metrics non-finite", 0, int((~np.isfinite(project_metric_values)).sum()), np.isfinite(project_metric_values).all())
add_check(validation_records, "Project metrics outside [0,1]", 0, int(((project_metric_values < 0) | (project_metric_values > 1)).sum()), bool(((project_metric_values >= 0) & (project_metric_values <= 1)).all()))
add_check(validation_records, "Raw training cohort unchanged", raw_training_hash_before, raw_training_hash_after, raw_training_hash_after == raw_training_hash_before)
add_check(validation_records, "Raw evaluation cohort unchanged", raw_evaluation_hash_before, raw_evaluation_hash_after, raw_evaluation_hash_after == raw_evaluation_hash_before)
add_check(validation_records, "Model training cohort unchanged", model_training_hash_before, model_training_hash_after, model_training_hash_after == model_training_hash_before)
add_check(validation_records, "Model evaluation cohort unchanged", model_evaluation_hash_before, model_evaluation_hash_after, model_evaluation_hash_after == model_evaluation_hash_before)
add_check(validation_records, "Source root unchanged", EXPECTED_SOURCE_ROOT_SHA256, source_root_sha256_after, source_root_sha256_after == EXPECTED_SOURCE_ROOT_SHA256)
add_check(validation_records, "Completion registry unchanged", registry_sha256_before, registry_sha256_after, registry_sha256_after == registry_sha256_before)
add_check(validation_records, "Registry Project 11 rows", 0, int(registry_project_numbers.eq(PROJECT_NUMBER).sum()), int(registry_project_numbers.eq(PROJECT_NUMBER).sum()) == 0)

validation = pd.DataFrame(validation_records)
failed_validation = validation.loc[~validation["Pass"]]

print("\nStep 5A validation:")
display(validation)

if not failed_validation.empty:
    print("\nFailed Step 5A checks:")
    display(failed_validation)
    print("\nCompleted condition checkpoints remain resume-safe.")
    raise RuntimeError(
        "PROJECT 11 STEP 5A FINAL VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 10. FREEZE FULL RAW ROOT AND STEP 5A CHECKPOINT
# --------------------------------------------------------------------------------------------------

atomic_csv(CONDITION_INVENTORY_PATH, condition_inventory)
atomic_csv(RAW_MANIFEST_PATH, raw_manifest)
atomic_csv(BASELINE_INVARIANCE_PATH, baseline_invariance)
atomic_csv(COMBINED_CONDITION_AUDIT_PATH, combined_condition_audit)
atomic_csv(COMBINED_PROJECT_RUNS_PATH, combined_project_runs)
atomic_csv(COMBINED_BUILD_METRICS_PATH, combined_build_metrics)
atomic_csv(COMBINED_MODEL_FITS_PATH, combined_model_fits)
atomic_csv(STEP5A_VALIDATION_PATH, validation)

full_execution_seconds = time.perf_counter() - full_execution_started
completed_at_utc = datetime.now(timezone.utc).isoformat()

aggregate_output_paths = [
    CONDITION_INVENTORY_PATH,
    RAW_MANIFEST_PATH,
    BASELINE_INVARIANCE_PATH,
    COMBINED_CONDITION_AUDIT_PATH,
    COMBINED_PROJECT_RUNS_PATH,
    COMBINED_BUILD_METRICS_PATH,
    COMBINED_MODEL_FITS_PATH,
    ACCELERATED_EQUIVALENCE_PATH,
    STEP5A_VALIDATION_PATH,
]
aggregate_output_manifest = [
    {
        "Path": str(path),
        "Bytes": int(path.stat().st_size),
        "SHA256": sha256_file(path),
    }
    for path in aggregate_output_paths
]

report_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5A_STATUS,
    "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
    "AcceleratedCleanRECMismatches": accelerated_clean_mismatch_values,
    "AcceleratedSmokeEquivalenceRows": len(accelerated_equivalence),
    "AcceleratedSmokeEquivalenceFailures": int((~accelerated_equivalence["Pass"].astype(bool)).sum()),
    "AcceleratedEquivalenceAudit": str(ACCELERATED_EQUIVALENCE_PATH),
    "CompletedAtUTC": completed_at_utc,
    "SmokeCheckpointSHA256": smoke_checkpoint_sha256,
    "RuntimeCheckpointSHA256": runtime_checkpoint_sha256,
    "NoisePlanCheckpointSHA256": noise_plan_checkpoint_sha256,
    "RECCheckpointSHA256": rec_checkpoint_sha256,
    "SelectionCheckpointSHA256": selection_checkpoint_sha256,
    "SourceRootSHA256": source_root_sha256_after,
    "Conditions": len(condition_inventory),
    "NoiseLevels": NOISE_LEVELS,
    "RepetitionSeeds": REPETITION_SEEDS,
    "MLFits": len(combined_model_fits),
    "RankingRows": int(condition_inventory["RankingRows"].sum()),
    "BuildMetricRows": len(combined_build_metrics),
    "ProjectRunRows": len(combined_project_runs),
    "ConditionAuditRows": len(combined_condition_audit),
    "TrainingMedianRows": int(condition_inventory["TrainingMedianRows"].sum()),
    "BaselineInvarianceFailures": baseline_invariance_failures,
    "RawRoot": str(FULL_RAW_RESULT_ROOT),
    "RawFiles": raw_files,
    "RawBytes": raw_bytes,
    "RawRootSHA256": raw_root_sha256,
    "RawManifest": str(RAW_MANIFEST_PATH),
    "RawManifestSHA256": sha256_file(RAW_MANIFEST_PATH),
    "ValidationChecks": len(validation),
    "FailedValidationChecks": len(failed_validation),
    "CompletedThisRun": completed_this_run,
    "SkippedValidatedConditions": skipped_valid,
    "FullExecutionSecondsThisInvocation": float(full_execution_seconds),
    "AggregateOutputManifest": aggregate_output_manifest,
    "RegistrySHA256": registry_sha256_after,
    "RegistryModified": False,
    "Projects1To10Modified": False,
    "Project12Accessed": False,
    "EvaluationCohortImmutable": True,
    "ResumeSafe": True,
}
atomic_json(STEP5A_REPORT_PATH, report_payload)

checkpoint_payload = {
    **report_payload,
    "CheckpointVersion": 1,
    "Full270ConditionExperimentComplete": True,
    "RawResultRootFrozen": True,
    "DoNotRerunCompletedConditions": True,
    "NextRequiredStep": "STEP_5B_RAW_REVALIDATION_AND_COMPACT_AGGREGATION",
}
atomic_json(STEP5A_CHECKPOINT_PATH, checkpoint_payload)

status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5A_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "Conditions": len(condition_inventory),
    "MLFits": len(combined_model_fits),
    "RawFiles": raw_files,
    "RawBytes": raw_bytes,
    "RawRootSHA256": raw_root_sha256,
    "Checkpoint": str(STEP5A_CHECKPOINT_PATH),
    "CheckpointSHA256": sha256_file(STEP5A_CHECKPOINT_PATH),
    "FailedValidationChecks": len(failed_validation),
    "RegistryModified": False,
    "Project12Accessed": False,
}
atomic_json(STEP5A_STATUS_PATH, status_payload)

atomic_json(
    RUN_PROGRESS_PATH,
    {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "Status": STEP5A_STATUS,
        "UpdatedAtUTC": completed_at_utc,
        "CompletedConditions": EXPECTED_CONDITIONS,
        "ExpectedConditions": EXPECTED_CONDITIONS,
        "RawRootSHA256": raw_root_sha256,
        "CheckpointSHA256": sha256_file(STEP5A_CHECKPOINT_PATH),
        "ResumeSafe": True,
        "RegistryModified": False,
        "Project12Accessed": False,
    },
)

checkpoint_readback = load_json(STEP5A_CHECKPOINT_PATH)
status_readback = load_json(STEP5A_STATUS_PATH)
if checkpoint_readback.get("Status") != STEP5A_STATUS:
    raise RuntimeError("Step 5A checkpoint readback failed.")
if status_readback.get("Status") != STEP5A_STATUS:
    raise RuntimeError("Step 5A status readback failed.")
if sha256_file(REGISTRY_PATH) != registry_sha256_before:
    raise RuntimeError("Completion registry changed during Step 5A finalisation.")

print("\n" + "=" * 136)
print("=== PROJECT 11 CELL 9 / STEP 5A ACCELERATED RESULT ===")
print("=" * 136)
print()
print("Project:", PROJECT_NAME)
print()
print("Accelerated engine:")
print("Engine version:", ACCELERATED_ENGINE_VERSION)
print("Clean REC mismatches:", accelerated_clean_mismatch_values)
print("Frozen smoke-equivalence failures:", int((~accelerated_equivalence["Pass"].astype(bool)).sum()))
print()
print("Full experiment:")
print("Conditions:", len(condition_inventory), "/", EXPECTED_CONDITIONS)
print("ML fits:", len(combined_model_fits), "/", EXPECTED_TOTAL_MODEL_FITS)
print("Ranking rows:", int(condition_inventory["RankingRows"].sum()))
print("Build-metric rows:", len(combined_build_metrics))
print("Project-run rows:", len(combined_project_runs))
print("Condition-audit rows:", len(combined_condition_audit))
print("Training-median rows:", int(condition_inventory["TrainingMedianRows"].sum()))
print()
print("Raw result freeze:")
print("Raw files:", raw_files)
print("Raw bytes:", raw_bytes)
print("Raw root SHA-256:", raw_root_sha256)
print()
print("Checkpoint/resume:")
print("Completed this invocation:", completed_this_run)
print("Skipped validated conditions:", skipped_valid)
print("Resume safe:", True)
print()
print("Baselines and metrics:")
print("Baseline invariance failures:", baseline_invariance_failures)
print("QTF global score variants:", qtf_global_score_variants)
print("QTF global rank variants:", qtf_global_rank_variants)
print("Primary / secondary metrics: APFDc / APFD")
print()
print("Immutability and isolation:")
print("Project 11 source unchanged:", True)
print("Completion registry unchanged:", True)
print("Projects 1–10 modified:", 0)
print("Project 12 accessed:", False)
print()
print("Validation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed_validation))
print()
print("Step 5A checkpoint:")
print(STEP5A_CHECKPOINT_PATH)
print("Checkpoint SHA-256:", sha256_file(STEP5A_CHECKPOINT_PATH))
print()
print("Runtime seconds this invocation:", round(full_execution_seconds, 2))
print()
print("STATUS:", STEP5A_STATUS)
print("=" * 136)

=== PROJECT 11 CELL 9 / STEP 5A ACCELERATED: CHECKPOINTED FULL 270-CONDITION EXPERIMENT ===

Loading frozen Project 11 cohorts and contracts.
Converting the fixed predictor cohorts to one numeric matrix.
Precomputing the vectorized verdict-dependent REC engine.
Accelerated REC engine clean-equivalence mismatches: 0
Accelerated REC groups / model rows / entities: 1020 / 91042 / 6684

Scanning existing condition checkpoints...
Valid completed conditions: 58
Incomplete/invalid condition directories: 1
Pending conditions: 212
Preserved incomplete condition in: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/apache__shardingsphere/shardingsphere_full_experiment/incomplete_condition_backups/noise_20__seed_07__20260731T014801113683Z
[1/270] Skipping validated checkpoint noise_00__seed_01
[9/270] Skipping validated checkpoint noise_50__seed_01
[2/270] Skipping validated checkpoint noise_05__seed_01
[3/270] Skipping validated checkpoint noise_10__seed_01
[4/270] Skipping validated c

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_20__seed_07
  Raw flips: 121950 | model-label changes: 15597 | dependent REC changes: 867901
  Training failures: 16283 | condition seconds: 87.54

--------------------------------------------------------------------------------------------------------------
[60/270] Running noise_25__seed_07
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_25__seed_07
  Raw flips: 152626 | model-label changes: 19507 | dependent REC changes: 896554
  Training failures: 20075 | condition seconds: 82.99

--------------------------------------------------------------------------------------------------------------
[61/270] Running noise_30__seed_07
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


...

  Fitting: NaiveBayes
  Completed: noise_40__seed_30
  Raw flips: 242991 | model-label changes: 30971 | dependent REC changes: 947803
  Training failures: 31271 | condition seconds: 90.60

--------------------------------------------------------------------------------------------------------------
[270/270] Running noise_50__seed_30
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_50__seed_30
  Raw flips: 304301 | model-label changes: 38876 | dependent REC changes: 969005
  Training failures: 38948 | condition seconds: 86.41

Validating all 270 completed conditions.

Step 5A validation:


,Check,Expected,Actual,Pass
0,Step 4B passed,PASS_PROJECT_11_TWO_CONDITION_END_TO_END_SMOKE...,PASS_PROJECT_11_TWO_CONDITION_END_TO_END_SMOKE...,True
1,Smoke checkpoint SHA-256,eadb99c58f85b5c9597874e01f7e7ce54d3abe7f5b535e...,eadb99c58f85b5c9597874e01f7e7ce54d3abe7f5b535e...,True
2,Accelerated clean REC mismatches,0,0,True
3,Accelerated smoke-equivalence rows,2,2,True
4,Accelerated smoke-equivalence keys,"[noise_00__seed_01, noise_50__seed_01]","[noise_00__seed_01, noise_50__seed_01]",True
5,Accelerated smoke-equivalence failures,0,0,True
6,Completed conditions,270,270,True
7,Noise levels,"[0, 5, 10, 15, 20, 25, 30, 40, 50]","[0, 5, 10, 15, 20, 25, 30, 40, 50]",True
8,Repetition seeds,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",True
9,Duplicate condition keys,0,0,True



=== PROJECT 11 CELL 9 / STEP 5A ACCELERATED RESULT ===

Project: apache@shardingsphere

Accelerated engine:
Engine version: PROJECT_11_FAST_DEPENDENT_REC_V1
Clean REC mismatches: 0
Frozen smoke-equivalence failures: 0

Full experiment:
Conditions: 270 / 270
ML fits: 1080 / 1080
Ranking rows: 24583230
Build-metric rows: 47250
Project-run rows: 1890
Condition-audit rows: 270
Training-median rows: 40770

Raw result freeze:
Raw files: 2160
Raw bytes: 386769592
Raw root SHA-256: 6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e88642a299343f2a73a1

Checkpoint/resume:
Completed this invocation: 212
Skipped validated conditions: 58
Resume safe: True

Baselines and metrics:
Baseline invariance failures: 0
QTF global score variants: 1
QTF global rank variants: 1
Primary / secondary metrics: APFDc / APFD

Immutability and isolation:
Project 11 source unchanged: True
Completion registry unchanged: True
Projects 1–10 modified: 0
Project 12 accessed: False

Validation:
Checks: 47
Failed checks: 0

Step

In [3]:
# PROJECT 11 — CELL 10 / STEP 5B
# FINAL RAW-OUTPUT REVALIDATION AND COMPACT AGGREGATION

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import gzip, hashlib, json, os, time
import numpy as np
import pandas as pd

print('=' * 136)
print('=== PROJECT 11 CELL 10 / STEP 5B: FINAL RAW-OUTPUT REVALIDATION AND COMPACT AGGREGATION ===')
print('=' * 136)

PROJECT_NUMBER = 11
PROJECT_NAME = 'apache@shardingsphere'
PROJECT_SLUG = 'apache__shardingsphere'
PROJECT_SHORT = 'shardingsphere'
STEP5A_STATUS = 'PASS_PROJECT_11_FULL_270_CONDITION_EXPERIMENT_COMPLETE'
CONDITION_STATUS = 'PASS_FULL_CONDITION'
STEP5B_STATUS = 'PASS_PROJECT_11_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN'

EXPECTED_STEP5A_SHA = '4b0b53c91c25d2f3900a5c9df2b313c495545ea65036aeba39741a88770b5862'
EXPECTED_RAW_ROOT_SHA = '6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e88642a299343f2a73a1'
EXPECTED_REGISTRY_SHA = '847bdcbee16c8757fde34e9489cd3abbf0781a319644175472597265b1df2750'
EXPECTED_SOURCE_ROOT_SHA = '3e6e0a60d58bff61690d6e2ac2bac18371414819eff601f3ea347ecb8ad8b0b5'

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
SEEDS = list(range(1, 31))
TECHNIQUES = ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes', 'Random', 'LatestFail', 'QTF-Avg']
ML_TECHNIQUES = ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes']
INVARIANT_BASELINES = ['Random', 'QTF-Avg']
PROJECT_METRICS = ['MeanAPFDc', 'MedianAPFDc', 'MeanAPFD', 'MedianAPFD']
BUILD_METRICS = ['APFDc', 'APFD']

EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = 2160
EXPECTED_RAW_BYTES = 386769592
EXPECTED_RANKING_ROWS_PER_CONDITION = 13007 * 7
EXPECTED_BUILD_ROWS_PER_CONDITION = 25 * 7
EXPECTED_PROJECT_ROWS_PER_CONDITION = 7
EXPECTED_FIT_ROWS_PER_CONDITION = 4
EXPECTED_MEDIAN_ROWS_PER_CONDITION = 151
EXPECTED_TOTAL_RANKING_ROWS = 24583230
EXPECTED_TOTAL_BUILD_ROWS = 47250
EXPECTED_TOTAL_PROJECT_ROWS = 1890
EXPECTED_TOTAL_FIT_ROWS = 1080
EXPECTED_TOTAL_AUDIT_ROWS = 270
EXPECTED_TOTAL_MEDIAN_ROWS = 40770
EXPECTED_CONDITION_FILES = {
    'rankings.csv.gz', 'build_metrics.csv', 'project_runs.csv', 'model_fits.csv',
    'training_medians.csv', 'condition_audit.csv', 'condition_summary.json', 'COMPLETE.json'
}
CONDITION_OUTPUT_FILES = EXPECTED_CONDITION_FILES - {'condition_summary.json', 'COMPLETE.json'}

# Mount only Drive. No source re-extraction is needed for Step 5B.
drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive/Thesis_Experiment')
NOTES = ROOT / 'Notes'
RESULTS = ROOT / 'Results'
REGISTRY = NOTES / 'completed_project_registry.csv'
PROJECT_ROOT = RESULTS / 'Aggregated' / PROJECT_SLUG
RAW_ROOT = RESULTS / 'Raw' / PROJECT_SLUG
FULL_ROOT = PROJECT_ROOT / f'{PROJECT_SHORT}_full_experiment'
PLAN = PROJECT_ROOT / f'{PROJECT_SHORT}_noise_plan' / f'{PROJECT_SHORT}_condition_plan.csv'
STEP5A_CHECKPOINT = NOTES / 'project_11_step5a_checkpoint.json'
STEP5A_STATUS_PATH = PROJECT_ROOT / f'{PROJECT_SHORT}_step5a_status.json'
STEP5A_REPORT = FULL_ROOT / f'{PROJECT_SHORT}_step5a_report.json'
STEP5A_RAW_MANIFEST = FULL_ROOT / f'{PROJECT_SHORT}_raw_manifest.csv'
STEP5A_BASELINE = FULL_ROOT / f'{PROJECT_SHORT}_baseline_invariance.csv'

OUT = PROJECT_ROOT / f'{PROJECT_SHORT}_step5b'
CURRENT_MANIFEST = OUT / f'{PROJECT_SHORT}_independent_raw_manifest.csv'
CONDITION_INVENTORY = OUT / f'{PROJECT_SHORT}_independent_condition_inventory.csv'
REVALIDATED_PROJECT_RUNS = OUT / f'{PROJECT_SHORT}_revalidated_project_runs.csv'
REVALIDATED_BUILD_METRICS = OUT / f'{PROJECT_SHORT}_revalidated_build_metrics.csv'
REVALIDATED_MODEL_FITS = OUT / f'{PROJECT_SHORT}_revalidated_model_fits.csv'
REVALIDATED_CONDITION_AUDIT = OUT / f'{PROJECT_SHORT}_revalidated_condition_audit.csv'
REVALIDATED_MEDIANS = OUT / f'{PROJECT_SHORT}_revalidated_training_medians.csv'
NOISE_SUMMARY = OUT / f'{PROJECT_SHORT}_noise_technique_summary.csv'
SEED_DELTAS = OUT / f'{PROJECT_SHORT}_seed_level_noise_deltas.csv'
DELTA_SUMMARY = OUT / f'{PROJECT_SHORT}_noise_delta_summary.csv'
VALIDATION_PATH = OUT / f'{PROJECT_SHORT}_step5b_validation.csv'
REPORT_PATH = OUT / f'{PROJECT_SHORT}_step5b_report.json'
STATUS_PATH = PROJECT_ROOT / f'{PROJECT_SHORT}_step5b_status.json'
CHECKPOINT_PATH = NOTES / 'project_11_step5b_checkpoint.json'


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def load_json(path):
    with Path(path).open('r', encoding='utf-8') as f:
        return json.load(f)


def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f'.{path.name}.tmp_{os.getpid()}')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, sort_keys=True, ensure_ascii=False, default=str)
        f.write('\n')
    os.replace(tmp, path)


def atomic_csv(path, df):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f'.{path.name}.tmp_{os.getpid()}')
    df.to_csv(tmp, index=False, lineterminator='\n')
    os.replace(tmp, path)


def resolve_col(columns, names, label):
    lookup = {str(c).strip().lower(): c for c in columns}
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    raise RuntimeError(f'Could not resolve {label}; columns={list(columns)}')


def normalize_manifest(df, label):
    p = resolve_col(df.columns, ['RelativePath'], f'{label} path')
    b = resolve_col(df.columns, ['Bytes', 'SizeBytes'], f'{label} bytes')
    s = resolve_col(df.columns, ['SHA256'], f'{label} sha')
    out = df[[p, b, s]].copy(); out.columns = ['RelativePath', 'Bytes', 'SHA256']
    out['RelativePath'] = out['RelativePath'].astype(str).str.replace('\\', '/', regex=False)
    out['Bytes'] = pd.to_numeric(out['Bytes'], errors='raise').astype('int64')
    out['SHA256'] = out['SHA256'].astype(str).str.lower()
    return out.sort_values('RelativePath', kind='mergesort').reset_index(drop=True)


def root_hash(manifest):
    h = hashlib.sha256()
    for r in manifest.sort_values('RelativePath', kind='mergesort').itertuples(index=False):
        h.update(f'{r.RelativePath}\0{int(r.Bytes)}\0{str(r.SHA256).lower()}\n'.encode('utf-8'))
    return h.hexdigest()


def gzip_rows(path):
    n = 0
    with gzip.open(path, 'rb') as f:
        for _ in f:
            n += 1
    return max(0, n - 1)


def add_check(rows, name, expected, actual, passed):
    rows.append({'Check': name, 'Expected': expected, 'Actual': actual, 'Pass': bool(passed)})


def metric_nonfinite(df, cols):
    arr = df[cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    return int((~np.isfinite(arr)).sum())


def metric_outside(df, cols):
    arr = df[cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    return int(((arr < 0) | (arr > 1)).sum())


# Required Drive inputs.
required = [REGISTRY, PLAN, STEP5A_CHECKPOINT, STEP5A_STATUS_PATH, STEP5A_REPORT,
            STEP5A_RAW_MANIFEST, STEP5A_BASELINE, RAW_ROOT]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing Project 11 Step 5B inputs:\n' + '\n'.join(missing))

# Frozen Step 5A and registry.
step5a_sha = sha256_file(STEP5A_CHECKPOINT)
step5a = load_json(STEP5A_CHECKPOINT)
step5a_status = load_json(STEP5A_STATUS_PATH)
step5a_report = load_json(STEP5A_REPORT)
if step5a_sha != EXPECTED_STEP5A_SHA:
    raise RuntimeError(f'Step 5A checkpoint SHA differs. Expected={EXPECTED_STEP5A_SHA}; actual={step5a_sha}')
for label, payload in [('checkpoint', step5a), ('status', step5a_status), ('report', step5a_report)]:
    if payload.get('Status') != STEP5A_STATUS:
        raise RuntimeError(f'Step 5A {label} is not in PASS state.')
if step5a.get('SourceRootSHA256') != EXPECTED_SOURCE_ROOT_SHA:
    raise RuntimeError('Step 5A source-root SHA differs.')
if step5a.get('RawRootSHA256') != EXPECTED_RAW_ROOT_SHA:
    raise RuntimeError('Step 5A frozen raw-root SHA differs.')

registry_sha_before = sha256_file(REGISTRY)
if registry_sha_before != EXPECTED_REGISTRY_SHA:
    raise RuntimeError(f'Registry SHA differs. Expected={EXPECTED_REGISTRY_SHA}; actual={registry_sha_before}')
registry = pd.read_csv(REGISTRY, dtype=str).fillna('')
pn_col = resolve_col(registry.columns, ['ProjectNumber'], 'registry ProjectNumber')
st_col = resolve_col(registry.columns, ['Status'], 'registry Status')
pnums = pd.to_numeric(registry[pn_col], errors='raise').astype(int)
if len(registry) != 10 or sorted(pnums.tolist()) != list(range(1, 11)):
    raise RuntimeError('Registry must contain exactly Projects 1–10 before Project 11 registration.')
if not registry[st_col].eq('COMPLETE_AND_FROZEN').all() or pnums.eq(11).any():
    raise RuntimeError('Registry state is not valid for Project 11 Step 5B.')

# Validate Step 5A aggregate-output manifest.
agg_manifest = step5a.get('AggregateOutputManifest', [])
if not isinstance(agg_manifest, list) or not agg_manifest:
    raise RuntimeError('Step 5A checkpoint has no AggregateOutputManifest.')
agg_failures = 0
for item in agg_manifest:
    p = Path(item['Path'])
    ok = p.is_file() and p.stat().st_size == int(item['Bytes']) and sha256_file(p) == str(item['SHA256'])
    agg_failures += int(not ok)
if agg_failures:
    raise RuntimeError(f'{agg_failures} Step 5A aggregate outputs changed.')

# Independently hash all raw files.
print('\nIndependently hashing all 2,160 raw files.')
hash_start = time.perf_counter()
paths = sorted([p for p in RAW_ROOT.rglob('*') if p.is_file()], key=lambda p: p.relative_to(RAW_ROOT).as_posix())
manifest_rows = []
for i, p in enumerate(paths, 1):
    manifest_rows.append({'RelativePath': p.relative_to(RAW_ROOT).as_posix(),
                          'Bytes': int(p.stat().st_size), 'SHA256': sha256_file(p)})
    if i % 200 == 0 or i == len(paths):
        print(f'  Raw hashing progress: {i} / {len(paths)} files')
current_manifest = normalize_manifest(pd.DataFrame(manifest_rows), 'current manifest')
hash_seconds = time.perf_counter() - hash_start
frozen_manifest = normalize_manifest(pd.read_csv(STEP5A_RAW_MANIFEST), 'frozen manifest')
current_raw_sha = root_hash(current_manifest)
current_raw_bytes = int(current_manifest['Bytes'].sum())
merged_manifest = frozen_manifest.merge(current_manifest, on='RelativePath', how='outer',
                                        suffixes=('_frozen', '_current'), indicator=True)
missing_raw = int(merged_manifest['_merge'].eq('left_only').sum())
unexpected_raw = int(merged_manifest['_merge'].eq('right_only').sum())
size_mismatch = int((merged_manifest['_merge'].eq('both') &
                     merged_manifest['Bytes_frozen'].ne(merged_manifest['Bytes_current'])).sum())
hash_mismatch = int((merged_manifest['_merge'].eq('both') &
                     merged_manifest['SHA256_frozen'].ne(merged_manifest['SHA256_current'])).sum())

# Condition-by-condition independent validation and compact reload.
plan = pd.read_csv(PLAN, low_memory=False)
id_col = resolve_col(plan.columns, ['ConditionID', 'ConditionKey'], 'condition identifier')
order_col = resolve_col(plan.columns, ['ConditionOrder'], 'condition order')
noise_col = resolve_col(plan.columns, ['NoisePercent'], 'noise percent')
seed_col = resolve_col(plan.columns, ['RepetitionSeed'], 'repetition seed')
for col in [order_col, noise_col, seed_col]:
    plan[col] = pd.to_numeric(plan[col], errors='raise').astype(int)
plan = plan.sort_values(order_col, kind='mergesort').reset_index(drop=True)

inventory_rows, project_frames, build_frames, fit_frames, audit_frames, median_frames = [], [], [], [], [], []
marker_fail = summary_fail = file_set_fail = embedded_fail = ranking_count_fail = 0
print('\nRevalidating all 270 condition directories.')
condition_start = time.perf_counter()
for i, row in enumerate(plan.itertuples(index=False), 1):
    key = str(getattr(row, id_col)); order = int(getattr(row, order_col))
    noise = int(getattr(row, noise_col)); seed = int(getattr(row, seed_col))
    d = RAW_ROOT / key
    if not d.is_dir():
        raise FileNotFoundError(f'Missing condition directory: {d}')
    actual_files = {p.name for p in d.iterdir() if p.is_file()}
    file_ok = actual_files == EXPECTED_CONDITION_FILES
    file_set_fail += int(not file_ok)
    complete_path, summary_path = d / 'COMPLETE.json', d / 'condition_summary.json'
    complete, summary = load_json(complete_path), load_json(summary_path)
    complete_ok = (complete.get('Status') == CONDITION_STATUS and complete.get('ConditionKey') == key and
                   str(complete.get('ConditionSummaryPath')) == str(summary_path) and
                   str(complete.get('ConditionSummarySHA256')).lower() == sha256_file(summary_path))
    summary_ok = (summary.get('Status') == CONDITION_STATUS and summary.get('ConditionKey') == key and
                  int(summary.get('NoisePercent', -1)) == noise and int(summary.get('RepetitionSeed', -1)) == seed)
    marker_fail += int(not complete_ok); summary_fail += int(not summary_ok)
    output_manifest = summary.get('OutputManifest', [])
    local_embedded_fail = 0
    names = set()
    if not isinstance(output_manifest, list) or len(output_manifest) != 6:
        local_embedded_fail += 1
    else:
        for item in output_manifest:
            p = Path(item.get('Path', '')); names.add(p.name)
            ok = (p.parent == d and p.is_file() and p.stat().st_size == int(item.get('Bytes', -1)) and
                  sha256_file(p) == str(item.get('SHA256', '')).lower())
            local_embedded_fail += int(not ok)
        local_embedded_fail += int(names != CONDITION_OUTPUT_FILES)
    embedded_fail += local_embedded_fail
    ranking_rows = gzip_rows(d / 'rankings.csv.gz')
    ranking_count_fail += int(ranking_rows != EXPECTED_RANKING_ROWS_PER_CONDITION)
    build = pd.read_csv(d / 'build_metrics.csv', low_memory=False)
    project = pd.read_csv(d / 'project_runs.csv', low_memory=False)
    fits = pd.read_csv(d / 'model_fits.csv', low_memory=False)
    audit = pd.read_csv(d / 'condition_audit.csv', low_memory=False)
    medians = pd.read_csv(d / 'training_medians.csv', low_memory=False)
    expected_counts = [EXPECTED_BUILD_ROWS_PER_CONDITION, EXPECTED_PROJECT_ROWS_PER_CONDITION,
                       EXPECTED_FIT_ROWS_PER_CONDITION, 1, EXPECTED_MEDIAN_ROWS_PER_CONDITION]
    actual_counts = [len(build), len(project), len(fits), len(audit), len(medians)]
    if actual_counts != expected_counts:
        raise RuntimeError(f'{key}: compact output counts differ. expected={expected_counts}; actual={actual_counts}')
    for field, count in [('RankingRows', ranking_rows), ('BuildMetricRows', len(build)),
                         ('ProjectRunRows', len(project)), ('MLFits', len(fits)),
                         ('TrainingMedianRows', len(medians))]:
        if int(summary.get(field, -1)) != count:
            raise RuntimeError(f'{key}: condition_summary {field} differs.')
    project_frames.append(project); build_frames.append(build); fit_frames.append(fits)
    audit_frames.append(audit); median_frames.append(medians)
    inventory_rows.append({
        'ProjectNumber': PROJECT_NUMBER, 'Project': PROJECT_NAME, 'ProjectSlug': PROJECT_SLUG,
        'ConditionOrder': order, 'ConditionKey': key, 'NoisePercent': noise, 'RepetitionSeed': seed,
        'ConditionDirectory': str(d), 'CompletionStatus': complete.get('Status'),
        'SummaryStatus': summary.get('Status'), 'Files': len(actual_files),
        'ConditionBytes': int(sum(p.stat().st_size for p in d.iterdir() if p.is_file())),
        'RankingRows': ranking_rows, 'BuildMetricRows': len(build), 'ProjectRunRows': len(project),
        'ModelFits': len(fits), 'ConditionAuditRows': len(audit), 'TrainingMedianRows': len(medians),
        'FileSetPass': file_ok, 'CompletionMarkerPass': complete_ok, 'ConditionSummaryPass': summary_ok,
        'EmbeddedManifestFailures': local_embedded_fail,
        'CompletionMarkerSHA256': sha256_file(complete_path), 'ConditionSummarySHA256': sha256_file(summary_path),
    })
    if i % 30 == 0 or i == len(plan):
        print(f'  Condition revalidation progress: {i} / {len(plan)}')
condition_seconds = time.perf_counter() - condition_start

inventory = pd.DataFrame(inventory_rows).sort_values('ConditionOrder', kind='mergesort').reset_index(drop=True)
project_runs = pd.concat(project_frames, ignore_index=True)
build_metrics = pd.concat(build_frames, ignore_index=True)
model_fits = pd.concat(fit_frames, ignore_index=True)
condition_audit = pd.concat(audit_frames, ignore_index=True)
training_medians = pd.concat(median_frames, ignore_index=True)

# Contract audits.
coordinate_count = len(inventory[['NoisePercent', 'RepetitionSeed']].drop_duplicates())
dup_keys = int(inventory.duplicated(['ConditionKey'], keep=False).sum())
dup_coords = int(inventory.duplicated(['NoisePercent', 'RepetitionSeed'], keep=False).sum())
order_viol = int((inventory['ConditionOrder'].to_numpy(int) != np.arange(1, 271)).sum())
files_viol = int(inventory['Files'].ne(8).sum())
ranking_viol = int(inventory['RankingRows'].ne(EXPECTED_RANKING_ROWS_PER_CONDITION).sum())
small_per_condition_viol = int(inventory['BuildMetricRows'].ne(EXPECTED_BUILD_ROWS_PER_CONDITION).sum() +
                               inventory['ProjectRunRows'].ne(7).sum() + inventory['ModelFits'].ne(4).sum() +
                               inventory['ConditionAuditRows'].ne(1).sum() + inventory['TrainingMedianRows'].ne(151).sum())
project_techniques = sorted(project_runs['Technique'].astype(str).unique().tolist())
fit_techniques = sorted(model_fits['Technique'].astype(str).unique().tolist())
dup_project = int(project_runs.duplicated(['ConditionKey', 'Technique'], keep=False).sum())
dup_build = int(build_metrics.duplicated(['ConditionKey', 'Technique', 'Build'], keep=False).sum())
dup_fit = int(model_fits.duplicated(['ConditionKey', 'Technique'], keep=False).sum())
dup_audit = int(condition_audit.duplicated(['ConditionKey'], keep=False).sum())
dup_median = int(training_medians.duplicated(['ConditionKey', 'PredictorOrder'], keep=False).sum())
fit_fail = int((~model_fits['Status'].astype(str).eq('PASS_MODEL_FIT')).sum())
fit_errors = int(model_fits['Error'].fillna('').astype(str).str.len().gt(0).sum())
project_nonfinite = metric_nonfinite(project_runs, PROJECT_METRICS)
project_outside = metric_outside(project_runs, PROJECT_METRICS)
build_nonfinite = metric_nonfinite(build_metrics, BUILD_METRICS)
build_outside = metric_outside(build_metrics, BUILD_METRICS)
median_nonfinite = int((~np.isfinite(pd.to_numeric(training_medians['TrainingMedian'], errors='coerce').to_numpy(float))).sum())
median_predictor_viol = int(training_medians.groupby('ConditionKey')['Predictor'].nunique().ne(151).sum())
scored_build_viol = int(project_runs['ScoredFailingBuilds'].ne(25).sum())
eval_build_viol = int(project_runs['EvaluationBuilds'].ne(263).sum())
eval_failure_viol = int(project_runs['EvaluationFailures'].ne(171).sum())
zero = condition_audit[condition_audit['NoisePercent'].eq(0)]
positive = condition_audit[condition_audit['NoisePercent'].gt(0)]
zero_flip_viol = int(zero['NumberFlipped'].ne(0).sum())
zero_model_viol = int(zero['ModelLabelChanges'].ne(0).sum())
zero_rec_viol = int(zero['DependentRECChanges'].ne(0).sum())
pos_raw_viol = int(positive['NumberFlipped'].le(0).sum())
pos_model_viol = int(positive['ModelLabelChanges'].le(0).sum())
pos_rec_viol = int(positive['DependentRECChanges'].le(0).sum())
independent_viol = int(condition_audit['IndependentRECChanges'].ne(0).sum())
independent_recon_viol = int(condition_audit['IndependentReconstructionMismatches'].ne(0).sum())
noise_hash_mismatch = 0
for e, a in [('ExpectedFlipMaskSHA256', 'ActualFlipMaskSHA256'),
             ('ExpectedNoisyRawVerdictSHA256', 'ActualNoisyRawVerdictSHA256'),
             ('ExpectedNoisyModelVerdictSHA256', 'ActualNoisyModelVerdictSHA256')]:
    noise_hash_mismatch += int((condition_audit[e].astype(str) != condition_audit[a].astype(str)).sum())

baseline = pd.read_csv(STEP5A_BASELINE, low_memory=False)
if 'Pass' in baseline.columns:
    bpass = baseline['Pass'].astype(str).str.strip().str.lower().isin({'true', '1'})
    ranking_baseline_fail = int((~bpass).sum())
else:
    mismatch_cols = [c for c in baseline.columns if 'mismatch' in c.lower()]
    ranking_baseline_fail = int(baseline[mismatch_cols].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy(float).sum())
metric_baseline_fail = 0
for _, g in project_runs[project_runs['Technique'].isin(INVARIANT_BASELINES)].groupby(['RepetitionSeed', 'Technique']):
    arr = g[PROJECT_METRICS].to_numpy(float)
    metric_baseline_fail += int(np.max(arr, axis=0).max() - np.min(arr, axis=0).min() > 1e-15)

# Compact aggregates.
agg_start = time.perf_counter()
noise_summary = project_runs.groupby(['NoisePercent', 'Technique'], as_index=False, sort=True).agg(
    Runs=('ConditionKey', 'count'), Seeds=('RepetitionSeed', 'nunique'),
    Mean_MeanAPFDc=('MeanAPFDc', 'mean'), SD_MeanAPFDc=('MeanAPFDc', 'std'), Median_MeanAPFDc=('MeanAPFDc', 'median'),
    Mean_MedianAPFDc=('MedianAPFDc', 'mean'), SD_MedianAPFDc=('MedianAPFDc', 'std'), Median_MedianAPFDc=('MedianAPFDc', 'median'),
    Mean_MeanAPFD=('MeanAPFD', 'mean'), SD_MeanAPFD=('MeanAPFD', 'std'), Median_MeanAPFD=('MeanAPFD', 'median'),
    Mean_MedianAPFD=('MedianAPFD', 'mean'), SD_MedianAPFD=('MedianAPFD', 'std'), Median_MedianAPFD=('MedianAPFD', 'median'),
).sort_values(['NoisePercent', 'Technique'], kind='mergesort').reset_index(drop=True)
clean = project_runs[project_runs['NoisePercent'].eq(0)][['RepetitionSeed', 'Technique'] + PROJECT_METRICS].rename(
    columns={c: f'Clean_{c}' for c in PROJECT_METRICS})
seed_deltas = project_runs.merge(clean, on=['RepetitionSeed', 'Technique'], how='left', validate='many_to_one')
for c in PROJECT_METRICS:
    seed_deltas[f'Delta_{c}'] = seed_deltas[c] - seed_deltas[f'Clean_{c}']
delta_cols = [f'Delta_{c}' for c in PROJECT_METRICS]
seed_deltas = seed_deltas[['ProjectNumber', 'Project', 'ProjectSlug', 'ConditionKey', 'NoisePercent',
                           'RepetitionSeed', 'Technique'] + PROJECT_METRICS +
                          [f'Clean_{c}' for c in PROJECT_METRICS] + delta_cols].sort_values(
                              ['NoisePercent', 'Technique', 'RepetitionSeed'], kind='mergesort').reset_index(drop=True)
delta_summary = seed_deltas.groupby(['NoisePercent', 'Technique'], as_index=False, sort=True).agg(
    Seeds=('RepetitionSeed', 'nunique'),
    Mean_Delta_MeanAPFDc=('Delta_MeanAPFDc', 'mean'), SD_Delta_MeanAPFDc=('Delta_MeanAPFDc', 'std'), Median_Delta_MeanAPFDc=('Delta_MeanAPFDc', 'median'),
    Mean_Delta_MedianAPFDc=('Delta_MedianAPFDc', 'mean'), SD_Delta_MedianAPFDc=('Delta_MedianAPFDc', 'std'), Median_Delta_MedianAPFDc=('Delta_MedianAPFDc', 'median'),
    Mean_Delta_MeanAPFD=('Delta_MeanAPFD', 'mean'), SD_Delta_MeanAPFD=('Delta_MeanAPFD', 'std'), Median_Delta_MeanAPFD=('Delta_MeanAPFD', 'median'),
    Mean_Delta_MedianAPFD=('Delta_MedianAPFD', 'mean'), SD_Delta_MedianAPFD=('Delta_MedianAPFD', 'std'), Median_Delta_MedianAPFD=('Delta_MedianAPFD', 'median'),
).sort_values(['NoisePercent', 'Technique'], kind='mergesort').reset_index(drop=True)
agg_seconds = time.perf_counter() - agg_start
summary_nonfinite = metric_nonfinite(noise_summary, [c for c in noise_summary.columns if c not in {'NoisePercent', 'Technique'}])
delta_nonfinite = metric_nonfinite(delta_summary, [c for c in delta_summary.columns if c not in {'NoisePercent', 'Technique'}])
clean_delta_nonzero = int((np.abs(seed_deltas[seed_deltas['NoisePercent'].eq(0)][delta_cols].to_numpy(float)) > 1e-15).sum())
baseline_delta_nonzero = int((np.abs(seed_deltas[seed_deltas['Technique'].isin(INVARIANT_BASELINES)][delta_cols].to_numpy(float)) > 1e-15).sum())

# Validation table.
checks = []
add_check(checks, 'Step 5A status', STEP5A_STATUS, step5a.get('Status'), step5a.get('Status') == STEP5A_STATUS)
add_check(checks, 'Step 5A checkpoint SHA-256', EXPECTED_STEP5A_SHA, step5a_sha, step5a_sha == EXPECTED_STEP5A_SHA)
add_check(checks, 'Frozen raw-root SHA-256', EXPECTED_RAW_ROOT_SHA, step5a.get('RawRootSHA256'), step5a.get('RawRootSHA256') == EXPECTED_RAW_ROOT_SHA)
add_check(checks, 'Independent current raw-root SHA-256', EXPECTED_RAW_ROOT_SHA, current_raw_sha, current_raw_sha == EXPECTED_RAW_ROOT_SHA)
add_check(checks, 'Step 5A aggregate-manifest failures', 0, agg_failures, agg_failures == 0)
add_check(checks, 'Condition marker failures', 0, marker_fail, marker_fail == 0)
add_check(checks, 'Condition summary failures', 0, summary_fail, summary_fail == 0)
add_check(checks, 'Condition file-set failures', 0, file_set_fail, file_set_fail == 0)
add_check(checks, 'Embedded output-manifest failures', 0, embedded_fail, embedded_fail == 0)
add_check(checks, 'Conditions', 270, len(inventory), len(inventory) == 270)
add_check(checks, 'Condition coordinates', 270, coordinate_count, coordinate_count == 270)
add_check(checks, 'Duplicate condition keys', 0, dup_keys, dup_keys == 0)
add_check(checks, 'Duplicate condition coordinates', 0, dup_coords, dup_coords == 0)
add_check(checks, 'Condition-order violations', 0, order_viol, order_viol == 0)
add_check(checks, 'Files-per-condition violations', 0, files_viol, files_viol == 0)
add_check(checks, 'Raw files', EXPECTED_RAW_FILES, len(current_manifest), len(current_manifest) == EXPECTED_RAW_FILES)
add_check(checks, 'Raw bytes', EXPECTED_RAW_BYTES, current_raw_bytes, current_raw_bytes == EXPECTED_RAW_BYTES)
add_check(checks, 'Missing raw files', 0, missing_raw, missing_raw == 0)
add_check(checks, 'Unexpected raw files', 0, unexpected_raw, unexpected_raw == 0)
add_check(checks, 'Raw size mismatches', 0, size_mismatch, size_mismatch == 0)
add_check(checks, 'Raw SHA-256 mismatches', 0, hash_mismatch, hash_mismatch == 0)
add_check(checks, 'Ranking rows', EXPECTED_TOTAL_RANKING_ROWS, int(inventory['RankingRows'].sum()), int(inventory['RankingRows'].sum()) == EXPECTED_TOTAL_RANKING_ROWS)
add_check(checks, 'Ranking row-count failures', 0, ranking_count_fail + ranking_viol, ranking_count_fail + ranking_viol == 0)
add_check(checks, 'Project-run rows', EXPECTED_TOTAL_PROJECT_ROWS, len(project_runs), len(project_runs) == EXPECTED_TOTAL_PROJECT_ROWS)
add_check(checks, 'Build-metric rows', EXPECTED_TOTAL_BUILD_ROWS, len(build_metrics), len(build_metrics) == EXPECTED_TOTAL_BUILD_ROWS)
add_check(checks, 'Model-fit rows', EXPECTED_TOTAL_FIT_ROWS, len(model_fits), len(model_fits) == EXPECTED_TOTAL_FIT_ROWS)
add_check(checks, 'Condition-audit rows', EXPECTED_TOTAL_AUDIT_ROWS, len(condition_audit), len(condition_audit) == EXPECTED_TOTAL_AUDIT_ROWS)
add_check(checks, 'Training-median rows', EXPECTED_TOTAL_MEDIAN_ROWS, len(training_medians), len(training_medians) == EXPECTED_TOTAL_MEDIAN_ROWS)
add_check(checks, 'Small rows-per-condition violations', 0, small_per_condition_viol, small_per_condition_viol == 0)
add_check(checks, 'Project-run technique set', sorted(TECHNIQUES), project_techniques, project_techniques == sorted(TECHNIQUES))
add_check(checks, 'Model-fit technique set', sorted(ML_TECHNIQUES), fit_techniques, fit_techniques == sorted(ML_TECHNIQUES))
add_check(checks, 'Duplicate project/build/fit/audit/median rows', 0, dup_project + dup_build + dup_fit + dup_audit + dup_median, dup_project + dup_build + dup_fit + dup_audit + dup_median == 0)
add_check(checks, 'Model-fit failures', 0, fit_fail + fit_errors, fit_fail + fit_errors == 0)
add_check(checks, 'Scored/evaluated/failure count violations', 0, scored_build_viol + eval_build_viol + eval_failure_viol, scored_build_viol + eval_build_viol + eval_failure_viol == 0)
add_check(checks, 'Project metric invalid values', 0, project_nonfinite + project_outside, project_nonfinite + project_outside == 0)
add_check(checks, 'Build metric invalid values', 0, build_nonfinite + build_outside, build_nonfinite + build_outside == 0)
add_check(checks, 'Training-median invalid values', 0, median_nonfinite + median_predictor_viol, median_nonfinite + median_predictor_viol == 0)
add_check(checks, 'Zero-noise conditions', 30, len(zero), len(zero) == 30)
add_check(checks, 'Zero-noise violations', 0, zero_flip_viol + zero_model_viol + zero_rec_viol, zero_flip_viol + zero_model_viol + zero_rec_viol == 0)
add_check(checks, 'Positive-noise violations', 0, pos_raw_viol + pos_model_viol + pos_rec_viol, pos_raw_viol + pos_model_viol + pos_rec_viol == 0)
add_check(checks, 'Independent REC violations', 0, independent_viol + independent_recon_viol, independent_viol + independent_recon_viol == 0)
add_check(checks, 'Noise-plan hash mismatches', 0, noise_hash_mismatch, noise_hash_mismatch == 0)
add_check(checks, 'Ranking-level baseline-invariance failures', 0, ranking_baseline_fail, ranking_baseline_fail == 0)
add_check(checks, 'Project-metric baseline-invariance failures', 0, metric_baseline_fail, metric_baseline_fail == 0)
add_check(checks, 'Noise-technique summary rows', 63, len(noise_summary), len(noise_summary) == 63)
add_check(checks, 'Noise-technique summary count/nonfinite violations', 0, int(noise_summary['Runs'].ne(30).sum() + noise_summary['Seeds'].ne(30).sum()) + summary_nonfinite, int(noise_summary['Runs'].ne(30).sum() + noise_summary['Seeds'].ne(30).sum()) + summary_nonfinite == 0)
add_check(checks, 'Seed-level delta rows', 1890, len(seed_deltas), len(seed_deltas) == 1890)
add_check(checks, 'Clean delta non-zero values', 0, clean_delta_nonzero, clean_delta_nonzero == 0)
add_check(checks, 'Invariant-baseline delta non-zero values', 0, baseline_delta_nonzero, baseline_delta_nonzero == 0)
add_check(checks, 'Noise-delta summary rows', 63, len(delta_summary), len(delta_summary) == 63)
add_check(checks, 'Noise-delta summary count/nonfinite violations', 0, int(delta_summary['Seeds'].ne(30).sum()) + delta_nonfinite, int(delta_summary['Seeds'].ne(30).sum()) + delta_nonfinite == 0)
add_check(checks, 'Registry rows', 10, len(registry), len(registry) == 10)
add_check(checks, 'Registry Project 11 rows', 0, int(pnums.eq(11).sum()), int(pnums.eq(11).sum()) == 0)
validation = pd.DataFrame(checks)
failed = validation[~validation['Pass']]
print('\nProject 11 Step 5B validation:')
display(validation)
if not failed.empty:
    print('\nFailed checks:'); display(failed)
    raise RuntimeError('PROJECT 11 STEP 5B VALIDATION FAILED. No PASS checkpoint was written.')

# Freeze outputs.
OUT.mkdir(parents=True, exist_ok=True)
for path, frame in [
    (CURRENT_MANIFEST, current_manifest), (CONDITION_INVENTORY, inventory),
    (REVALIDATED_PROJECT_RUNS, project_runs), (REVALIDATED_BUILD_METRICS, build_metrics),
    (REVALIDATED_MODEL_FITS, model_fits), (REVALIDATED_CONDITION_AUDIT, condition_audit),
    (REVALIDATED_MEDIANS, training_medians), (NOISE_SUMMARY, noise_summary),
    (SEED_DELTAS, seed_deltas), (DELTA_SUMMARY, delta_summary), (VALIDATION_PATH, validation),
]:
    atomic_csv(path, frame)
output_paths = [CURRENT_MANIFEST, CONDITION_INVENTORY, REVALIDATED_PROJECT_RUNS,
                REVALIDATED_BUILD_METRICS, REVALIDATED_MODEL_FITS, REVALIDATED_CONDITION_AUDIT,
                REVALIDATED_MEDIANS, NOISE_SUMMARY, SEED_DELTAS, DELTA_SUMMARY, VALIDATION_PATH]
output_manifest = [{'Path': str(p), 'Bytes': int(p.stat().st_size), 'SHA256': sha256_file(p)} for p in output_paths]
completed = datetime.now(timezone.utc).isoformat()
report = {
    'ProjectNumber': PROJECT_NUMBER, 'Project': PROJECT_NAME, 'ProjectSlug': PROJECT_SLUG,
    'Status': STEP5B_STATUS, 'CompletedAtUTC': completed, 'Step5ACheckpointSHA256': step5a_sha,
    'FrozenRawRootSHA256': EXPECTED_RAW_ROOT_SHA, 'IndependentRawRootSHA256': current_raw_sha,
    'RawFiles': len(current_manifest), 'RawBytes': current_raw_bytes, 'Conditions': len(inventory),
    'MLFits': len(model_fits), 'RankingRows': int(inventory['RankingRows'].sum()),
    'BuildMetricRows': len(build_metrics), 'ProjectRunRows': len(project_runs),
    'ConditionAuditRows': len(condition_audit), 'TrainingMedianRows': len(training_medians),
    'NoiseTechniqueSummaryRows': len(noise_summary), 'SeedLevelNoiseDeltaRows': len(seed_deltas),
    'NoiseDeltaSummaryRows': len(delta_summary),
    'StandardDeviationDefinition': 'Sample SD across 30 seeds; pandas std, ddof=1',
    'RawHashingSeconds': float(hash_seconds), 'ConditionRevalidationSeconds': float(condition_seconds),
    'AggregationSeconds': float(agg_seconds), 'OutputManifest': output_manifest,
    'ValidationChecks': len(validation), 'FailedValidationChecks': len(failed),
    'RegistrySHA256': registry_sha_before, 'RegistryModified': False,
    'Projects1To10Modified': False, 'Project12Accessed': False, 'Project12WriteAttempted': False,
    'ModelsFitted': False, 'ConditionsRerun': False,
}
atomic_json(REPORT_PATH, report)
checkpoint = {**report, 'CheckpointVersion': 1,
              'CheckpointType': 'PROJECT_11_RAW_REVALIDATION_AND_COMPACT_AGGREGATION',
              'RawResultsRevalidated': True, 'CompactAggregatesFrozen': True,
              'ReadyForFinalPackageAndRegistration': True}
atomic_json(CHECKPOINT_PATH, checkpoint)
checkpoint_sha = sha256_file(CHECKPOINT_PATH)
status = {'ProjectNumber': PROJECT_NUMBER, 'Project': PROJECT_NAME, 'ProjectSlug': PROJECT_SLUG,
          'Status': STEP5B_STATUS, 'CompletedAtUTC': completed, 'Step5ACheckpointSHA256': step5a_sha,
          'RawRootSHA256': current_raw_sha, 'RawFiles': len(current_manifest), 'RawBytes': current_raw_bytes,
          'Conditions': len(inventory), 'MLFits': len(model_fits), 'Checkpoint': str(CHECKPOINT_PATH),
          'CheckpointSHA256': checkpoint_sha, 'ReadyForFinalPackageAndRegistration': True,
          'RegistryModified': False, 'Project12Accessed': False}
atomic_json(STATUS_PATH, status)

# Readback and immutability.
if load_json(CHECKPOINT_PATH).get('Status') != STEP5B_STATUS or load_json(STATUS_PATH).get('Status') != STEP5B_STATUS:
    raise RuntimeError('Step 5B checkpoint/status readback failed.')
for item in output_manifest:
    p = Path(item['Path'])
    if not p.is_file() or p.stat().st_size != item['Bytes'] or sha256_file(p) != item['SHA256']:
        raise RuntimeError(f'Step 5B output readback failed: {p}')
registry_sha_after = sha256_file(REGISTRY)
if registry_sha_after != registry_sha_before:
    raise RuntimeError('Registry changed during Project 11 Step 5B.')
if sha256_file(STEP5A_CHECKPOINT) != EXPECTED_STEP5A_SHA:
    raise RuntimeError('Step 5A checkpoint changed during Step 5B.')

print('\nNoise-technique summary:')
display(noise_summary)
print('\nNoise-delta summary:')
display(delta_summary)
print('\n' + '=' * 136)
print('=== PROJECT 11 CELL 10 / STEP 5B RESULT ===')
print('=' * 136)
print('Project:', PROJECT_NAME)
print('Step 5A checkpoint SHA-256:', step5a_sha)
print('Frozen raw-root SHA-256:', EXPECTED_RAW_ROOT_SHA)
print('Independent current raw-root SHA-256:', current_raw_sha)
print('\nRaw-output revalidation:')
print('Conditions:', len(inventory), '/', EXPECTED_CONDITIONS)
print('Raw files:', len(current_manifest), '/', EXPECTED_RAW_FILES)
print('Raw bytes:', current_raw_bytes, '/', EXPECTED_RAW_BYTES)
print('Missing / unexpected / size / SHA mismatches:', missing_raw, '/', unexpected_raw, '/', size_mismatch, '/', hash_mismatch)
print('Embedded output-manifest failures:', embedded_fail)
print('\nExperiment totals:')
print('ML fits:', len(model_fits), '/', EXPECTED_TOTAL_FIT_ROWS)
print('Ranking rows:', int(inventory['RankingRows'].sum()), '/', EXPECTED_TOTAL_RANKING_ROWS)
print('Build-metric rows:', len(build_metrics), '/', EXPECTED_TOTAL_BUILD_ROWS)
print('Project-run rows:', len(project_runs), '/', EXPECTED_TOTAL_PROJECT_ROWS)
print('Condition-audit rows:', len(condition_audit), '/', EXPECTED_TOTAL_AUDIT_ROWS)
print('Training-median rows:', len(training_medians), '/', EXPECTED_TOTAL_MEDIAN_ROWS)
print('\nAnalysis-ready aggregates:')
print('Noise-technique summary rows:', len(noise_summary))
print('Seed-level noise-delta rows:', len(seed_deltas))
print('Noise-delta summary rows:', len(delta_summary))
print('Sample SD calculated with ddof=1:', True)
print('Ranking-level baseline-invariance failures:', ranking_baseline_fail)
print('Project-metric baseline-invariance failures:', metric_baseline_fail)
print('\nImmutability and isolation:')
print('Completion registry unchanged:', registry_sha_after == registry_sha_before)
print('Registry Project 11 rows:', int(pnums.eq(11).sum()))
print('Projects 1–10 modified:', 0)
print('Project 12 accessed:', False)
print('Project 12 write attempted:', False)
print('Conditions rerun:', False)
print('Models fitted:', False)
print('\nRuntime:')
print('Raw hashing seconds:', round(hash_seconds, 2))
print('Condition revalidation seconds:', round(condition_seconds, 2))
print('Compact aggregation seconds:', round(agg_seconds, 2))
print('\nValidation:')
print('Checks:', len(validation))
print('Failed checks:', len(failed))
print('\nProject 11 Step 5B checkpoint:')
print(CHECKPOINT_PATH)
print('Checkpoint SHA-256:', checkpoint_sha)
print('\nSTATUS:', STEP5B_STATUS)
print('=' * 136)

=== PROJECT 11 CELL 10 / STEP 5B: FINAL RAW-OUTPUT REVALIDATION AND COMPACT AGGREGATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Independently hashing all 2,160 raw files.
  Raw hashing progress: 200 / 2160 files
  Raw hashing progress: 400 / 2160 files
  Raw hashing progress: 600 / 2160 files
  Raw hashing progress: 800 / 2160 files
  Raw hashing progress: 1000 / 2160 files
  Raw hashing progress: 1200 / 2160 files
  Raw hashing progress: 1400 / 2160 files
  Raw hashing progress: 1600 / 2160 files
  Raw hashing progress: 1800 / 2160 files
  Raw hashing progress: 2000 / 2160 files
  Raw hashing progress: 2160 / 2160 files

Revalidating all 270 condition directories.
  Condition revalidation progress: 30 / 270
  Condition revalidation progress: 60 / 270
  Condition revalidation progress: 90 / 270
  Condition revalidation progress: 120 / 270
  Condition revalidation progress: 150 / 270
  Conditio

,Check,Expected,Actual,Pass
0,Step 5A status,PASS_PROJECT_11_FULL_270_CONDITION_EXPERIMENT_...,PASS_PROJECT_11_FULL_270_CONDITION_EXPERIMENT_...,True
1,Step 5A checkpoint SHA-256,4b0b53c91c25d2f3900a5c9df2b313c495545ea65036ae...,4b0b53c91c25d2f3900a5c9df2b313c495545ea65036ae...,True
2,Frozen raw-root SHA-256,6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e8...,6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e8...,True
3,Independent current raw-root SHA-256,6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e8...,6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e8...,True
4,Step 5A aggregate-manifest failures,0,0,True
5,Condition marker failures,0,0,True
6,Condition summary failures,0,0,True
7,Condition file-set failures,0,0,True
8,Embedded output-manifest failures,0,0,True
9,Conditions,270,270,True



Failed checks:


,Check,Expected,Actual,Pass
43,Project-metric baseline-invariance failures,0,60,False


RuntimeError: PROJECT 11 STEP 5B VALIDATION FAILED. No PASS checkpoint was written.

In [4]:
# ==================================================================================================
# PROJECT 11 — CELL 10 / STEP 5B V2
# CORRECTED PER-METRIC BASELINE-INVARIANCE VALIDATION
#
# This is a complete standalone replacement for the failed Step 5B cell.
#
# V1 BUG FIX:
# - V1 compared max(APFDc/APFD metrics) against min(APFDc/APFD metrics) across
#   different metric columns.
# - APFDc and APFD naturally have different numeric values, so this falsely
#   reported all 60 Random/QTF seed groups as failures.
# - V2 computes the range independently within each metric and then checks
#   whether any individual metric changes across noise.
#
# No Step 5B PASS checkpoint was written by the failed V1 run.
# This cell does not rerun models or conditions and does not access Project 12.
# ==================================================================================================

# PROJECT 11 — CELL 10 / STEP 5B
# FINAL RAW-OUTPUT REVALIDATION AND COMPACT AGGREGATION

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import gzip, hashlib, json, os, time
import numpy as np
import pandas as pd

print('=' * 136)
print('=== PROJECT 11 CELL 10 / STEP 5B V2: CORRECTED PER-METRIC BASELINE INVARIANCE ===')
print('=' * 136)

PROJECT_NUMBER = 11
PROJECT_NAME = 'apache@shardingsphere'
PROJECT_SLUG = 'apache__shardingsphere'
PROJECT_SHORT = 'shardingsphere'
STEP5A_STATUS = 'PASS_PROJECT_11_FULL_270_CONDITION_EXPERIMENT_COMPLETE'
CONDITION_STATUS = 'PASS_FULL_CONDITION'
STEP5B_STATUS = 'PASS_PROJECT_11_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN'

EXPECTED_STEP5A_SHA = '4b0b53c91c25d2f3900a5c9df2b313c495545ea65036aeba39741a88770b5862'
EXPECTED_RAW_ROOT_SHA = '6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e88642a299343f2a73a1'
EXPECTED_REGISTRY_SHA = '847bdcbee16c8757fde34e9489cd3abbf0781a319644175472597265b1df2750'
EXPECTED_SOURCE_ROOT_SHA = '3e6e0a60d58bff61690d6e2ac2bac18371414819eff601f3ea347ecb8ad8b0b5'

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
SEEDS = list(range(1, 31))
TECHNIQUES = ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes', 'Random', 'LatestFail', 'QTF-Avg']
ML_TECHNIQUES = ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes']
INVARIANT_BASELINES = ['Random', 'QTF-Avg']
PROJECT_METRICS = ['MeanAPFDc', 'MedianAPFDc', 'MeanAPFD', 'MedianAPFD']
BUILD_METRICS = ['APFDc', 'APFD']

EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = 2160
EXPECTED_RAW_BYTES = 386769592
EXPECTED_RANKING_ROWS_PER_CONDITION = 13007 * 7
EXPECTED_BUILD_ROWS_PER_CONDITION = 25 * 7
EXPECTED_PROJECT_ROWS_PER_CONDITION = 7
EXPECTED_FIT_ROWS_PER_CONDITION = 4
EXPECTED_MEDIAN_ROWS_PER_CONDITION = 151
EXPECTED_TOTAL_RANKING_ROWS = 24583230
EXPECTED_TOTAL_BUILD_ROWS = 47250
EXPECTED_TOTAL_PROJECT_ROWS = 1890
EXPECTED_TOTAL_FIT_ROWS = 1080
EXPECTED_TOTAL_AUDIT_ROWS = 270
EXPECTED_TOTAL_MEDIAN_ROWS = 40770
EXPECTED_CONDITION_FILES = {
    'rankings.csv.gz', 'build_metrics.csv', 'project_runs.csv', 'model_fits.csv',
    'training_medians.csv', 'condition_audit.csv', 'condition_summary.json', 'COMPLETE.json'
}
CONDITION_OUTPUT_FILES = EXPECTED_CONDITION_FILES - {'condition_summary.json', 'COMPLETE.json'}

# Mount only Drive. No source re-extraction is needed for Step 5B.
drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive/Thesis_Experiment')
NOTES = ROOT / 'Notes'
RESULTS = ROOT / 'Results'
REGISTRY = NOTES / 'completed_project_registry.csv'
PROJECT_ROOT = RESULTS / 'Aggregated' / PROJECT_SLUG
RAW_ROOT = RESULTS / 'Raw' / PROJECT_SLUG
FULL_ROOT = PROJECT_ROOT / f'{PROJECT_SHORT}_full_experiment'
PLAN = PROJECT_ROOT / f'{PROJECT_SHORT}_noise_plan' / f'{PROJECT_SHORT}_condition_plan.csv'
STEP5A_CHECKPOINT = NOTES / 'project_11_step5a_checkpoint.json'
STEP5A_STATUS_PATH = PROJECT_ROOT / f'{PROJECT_SHORT}_step5a_status.json'
STEP5A_REPORT = FULL_ROOT / f'{PROJECT_SHORT}_step5a_report.json'
STEP5A_RAW_MANIFEST = FULL_ROOT / f'{PROJECT_SHORT}_raw_manifest.csv'
STEP5A_BASELINE = FULL_ROOT / f'{PROJECT_SHORT}_baseline_invariance.csv'

OUT = PROJECT_ROOT / f'{PROJECT_SHORT}_step5b'
CURRENT_MANIFEST = OUT / f'{PROJECT_SHORT}_independent_raw_manifest.csv'
CONDITION_INVENTORY = OUT / f'{PROJECT_SHORT}_independent_condition_inventory.csv'
REVALIDATED_PROJECT_RUNS = OUT / f'{PROJECT_SHORT}_revalidated_project_runs.csv'
REVALIDATED_BUILD_METRICS = OUT / f'{PROJECT_SHORT}_revalidated_build_metrics.csv'
REVALIDATED_MODEL_FITS = OUT / f'{PROJECT_SHORT}_revalidated_model_fits.csv'
REVALIDATED_CONDITION_AUDIT = OUT / f'{PROJECT_SHORT}_revalidated_condition_audit.csv'
REVALIDATED_MEDIANS = OUT / f'{PROJECT_SHORT}_revalidated_training_medians.csv'
NOISE_SUMMARY = OUT / f'{PROJECT_SHORT}_noise_technique_summary.csv'
SEED_DELTAS = OUT / f'{PROJECT_SHORT}_seed_level_noise_deltas.csv'
DELTA_SUMMARY = OUT / f'{PROJECT_SHORT}_noise_delta_summary.csv'
VALIDATION_PATH = OUT / f'{PROJECT_SHORT}_step5b_validation.csv'
REPORT_PATH = OUT / f'{PROJECT_SHORT}_step5b_report.json'
STATUS_PATH = PROJECT_ROOT / f'{PROJECT_SHORT}_step5b_status.json'
CHECKPOINT_PATH = NOTES / 'project_11_step5b_checkpoint.json'


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def load_json(path):
    with Path(path).open('r', encoding='utf-8') as f:
        return json.load(f)


def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f'.{path.name}.tmp_{os.getpid()}')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, sort_keys=True, ensure_ascii=False, default=str)
        f.write('\n')
    os.replace(tmp, path)


def atomic_csv(path, df):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f'.{path.name}.tmp_{os.getpid()}')
    df.to_csv(tmp, index=False, lineterminator='\n')
    os.replace(tmp, path)


def resolve_col(columns, names, label):
    lookup = {str(c).strip().lower(): c for c in columns}
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    raise RuntimeError(f'Could not resolve {label}; columns={list(columns)}')


def normalize_manifest(df, label):
    p = resolve_col(df.columns, ['RelativePath'], f'{label} path')
    b = resolve_col(df.columns, ['Bytes', 'SizeBytes'], f'{label} bytes')
    s = resolve_col(df.columns, ['SHA256'], f'{label} sha')
    out = df[[p, b, s]].copy(); out.columns = ['RelativePath', 'Bytes', 'SHA256']
    out['RelativePath'] = out['RelativePath'].astype(str).str.replace('\\', '/', regex=False)
    out['Bytes'] = pd.to_numeric(out['Bytes'], errors='raise').astype('int64')
    out['SHA256'] = out['SHA256'].astype(str).str.lower()
    return out.sort_values('RelativePath', kind='mergesort').reset_index(drop=True)


def root_hash(manifest):
    h = hashlib.sha256()
    for r in manifest.sort_values('RelativePath', kind='mergesort').itertuples(index=False):
        h.update(f'{r.RelativePath}\0{int(r.Bytes)}\0{str(r.SHA256).lower()}\n'.encode('utf-8'))
    return h.hexdigest()


def gzip_rows(path):
    n = 0
    with gzip.open(path, 'rb') as f:
        for _ in f:
            n += 1
    return max(0, n - 1)


def add_check(rows, name, expected, actual, passed):
    rows.append({'Check': name, 'Expected': expected, 'Actual': actual, 'Pass': bool(passed)})


def metric_nonfinite(df, cols):
    arr = df[cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    return int((~np.isfinite(arr)).sum())


def metric_outside(df, cols):
    arr = df[cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    return int(((arr < 0) | (arr > 1)).sum())


# Required Drive inputs.
required = [REGISTRY, PLAN, STEP5A_CHECKPOINT, STEP5A_STATUS_PATH, STEP5A_REPORT,
            STEP5A_RAW_MANIFEST, STEP5A_BASELINE, RAW_ROOT]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing Project 11 Step 5B inputs:\n' + '\n'.join(missing))

# Frozen Step 5A and registry.
step5a_sha = sha256_file(STEP5A_CHECKPOINT)
step5a = load_json(STEP5A_CHECKPOINT)
step5a_status = load_json(STEP5A_STATUS_PATH)
step5a_report = load_json(STEP5A_REPORT)
if step5a_sha != EXPECTED_STEP5A_SHA:
    raise RuntimeError(f'Step 5A checkpoint SHA differs. Expected={EXPECTED_STEP5A_SHA}; actual={step5a_sha}')
for label, payload in [('checkpoint', step5a), ('status', step5a_status), ('report', step5a_report)]:
    if payload.get('Status') != STEP5A_STATUS:
        raise RuntimeError(f'Step 5A {label} is not in PASS state.')
if step5a.get('SourceRootSHA256') != EXPECTED_SOURCE_ROOT_SHA:
    raise RuntimeError('Step 5A source-root SHA differs.')
if step5a.get('RawRootSHA256') != EXPECTED_RAW_ROOT_SHA:
    raise RuntimeError('Step 5A frozen raw-root SHA differs.')

registry_sha_before = sha256_file(REGISTRY)
if registry_sha_before != EXPECTED_REGISTRY_SHA:
    raise RuntimeError(f'Registry SHA differs. Expected={EXPECTED_REGISTRY_SHA}; actual={registry_sha_before}')
registry = pd.read_csv(REGISTRY, dtype=str).fillna('')
pn_col = resolve_col(registry.columns, ['ProjectNumber'], 'registry ProjectNumber')
st_col = resolve_col(registry.columns, ['Status'], 'registry Status')
pnums = pd.to_numeric(registry[pn_col], errors='raise').astype(int)
if len(registry) != 10 or sorted(pnums.tolist()) != list(range(1, 11)):
    raise RuntimeError('Registry must contain exactly Projects 1–10 before Project 11 registration.')
if not registry[st_col].eq('COMPLETE_AND_FROZEN').all() or pnums.eq(11).any():
    raise RuntimeError('Registry state is not valid for Project 11 Step 5B.')

# Validate Step 5A aggregate-output manifest.
agg_manifest = step5a.get('AggregateOutputManifest', [])
if not isinstance(agg_manifest, list) or not agg_manifest:
    raise RuntimeError('Step 5A checkpoint has no AggregateOutputManifest.')
agg_failures = 0
for item in agg_manifest:
    p = Path(item['Path'])
    ok = p.is_file() and p.stat().st_size == int(item['Bytes']) and sha256_file(p) == str(item['SHA256'])
    agg_failures += int(not ok)
if agg_failures:
    raise RuntimeError(f'{agg_failures} Step 5A aggregate outputs changed.')

# Independently hash all raw files.
print('\nIndependently hashing all 2,160 raw files.')
hash_start = time.perf_counter()
paths = sorted([p for p in RAW_ROOT.rglob('*') if p.is_file()], key=lambda p: p.relative_to(RAW_ROOT).as_posix())
manifest_rows = []
for i, p in enumerate(paths, 1):
    manifest_rows.append({'RelativePath': p.relative_to(RAW_ROOT).as_posix(),
                          'Bytes': int(p.stat().st_size), 'SHA256': sha256_file(p)})
    if i % 200 == 0 or i == len(paths):
        print(f'  Raw hashing progress: {i} / {len(paths)} files')
current_manifest = normalize_manifest(pd.DataFrame(manifest_rows), 'current manifest')
hash_seconds = time.perf_counter() - hash_start
frozen_manifest = normalize_manifest(pd.read_csv(STEP5A_RAW_MANIFEST), 'frozen manifest')
current_raw_sha = root_hash(current_manifest)
current_raw_bytes = int(current_manifest['Bytes'].sum())
merged_manifest = frozen_manifest.merge(current_manifest, on='RelativePath', how='outer',
                                        suffixes=('_frozen', '_current'), indicator=True)
missing_raw = int(merged_manifest['_merge'].eq('left_only').sum())
unexpected_raw = int(merged_manifest['_merge'].eq('right_only').sum())
size_mismatch = int((merged_manifest['_merge'].eq('both') &
                     merged_manifest['Bytes_frozen'].ne(merged_manifest['Bytes_current'])).sum())
hash_mismatch = int((merged_manifest['_merge'].eq('both') &
                     merged_manifest['SHA256_frozen'].ne(merged_manifest['SHA256_current'])).sum())

# Condition-by-condition independent validation and compact reload.
plan = pd.read_csv(PLAN, low_memory=False)
id_col = resolve_col(plan.columns, ['ConditionID', 'ConditionKey'], 'condition identifier')
order_col = resolve_col(plan.columns, ['ConditionOrder'], 'condition order')
noise_col = resolve_col(plan.columns, ['NoisePercent'], 'noise percent')
seed_col = resolve_col(plan.columns, ['RepetitionSeed'], 'repetition seed')
for col in [order_col, noise_col, seed_col]:
    plan[col] = pd.to_numeric(plan[col], errors='raise').astype(int)
plan = plan.sort_values(order_col, kind='mergesort').reset_index(drop=True)

inventory_rows, project_frames, build_frames, fit_frames, audit_frames, median_frames = [], [], [], [], [], []
marker_fail = summary_fail = file_set_fail = embedded_fail = ranking_count_fail = 0
print('\nRevalidating all 270 condition directories.')
condition_start = time.perf_counter()
for i, row in enumerate(plan.itertuples(index=False), 1):
    key = str(getattr(row, id_col)); order = int(getattr(row, order_col))
    noise = int(getattr(row, noise_col)); seed = int(getattr(row, seed_col))
    d = RAW_ROOT / key
    if not d.is_dir():
        raise FileNotFoundError(f'Missing condition directory: {d}')
    actual_files = {p.name for p in d.iterdir() if p.is_file()}
    file_ok = actual_files == EXPECTED_CONDITION_FILES
    file_set_fail += int(not file_ok)
    complete_path, summary_path = d / 'COMPLETE.json', d / 'condition_summary.json'
    complete, summary = load_json(complete_path), load_json(summary_path)
    complete_ok = (complete.get('Status') == CONDITION_STATUS and complete.get('ConditionKey') == key and
                   str(complete.get('ConditionSummaryPath')) == str(summary_path) and
                   str(complete.get('ConditionSummarySHA256')).lower() == sha256_file(summary_path))
    summary_ok = (summary.get('Status') == CONDITION_STATUS and summary.get('ConditionKey') == key and
                  int(summary.get('NoisePercent', -1)) == noise and int(summary.get('RepetitionSeed', -1)) == seed)
    marker_fail += int(not complete_ok); summary_fail += int(not summary_ok)
    output_manifest = summary.get('OutputManifest', [])
    local_embedded_fail = 0
    names = set()
    if not isinstance(output_manifest, list) or len(output_manifest) != 6:
        local_embedded_fail += 1
    else:
        for item in output_manifest:
            p = Path(item.get('Path', '')); names.add(p.name)
            ok = (p.parent == d and p.is_file() and p.stat().st_size == int(item.get('Bytes', -1)) and
                  sha256_file(p) == str(item.get('SHA256', '')).lower())
            local_embedded_fail += int(not ok)
        local_embedded_fail += int(names != CONDITION_OUTPUT_FILES)
    embedded_fail += local_embedded_fail
    ranking_rows = gzip_rows(d / 'rankings.csv.gz')
    ranking_count_fail += int(ranking_rows != EXPECTED_RANKING_ROWS_PER_CONDITION)
    build = pd.read_csv(d / 'build_metrics.csv', low_memory=False)
    project = pd.read_csv(d / 'project_runs.csv', low_memory=False)
    fits = pd.read_csv(d / 'model_fits.csv', low_memory=False)
    audit = pd.read_csv(d / 'condition_audit.csv', low_memory=False)
    medians = pd.read_csv(d / 'training_medians.csv', low_memory=False)
    expected_counts = [EXPECTED_BUILD_ROWS_PER_CONDITION, EXPECTED_PROJECT_ROWS_PER_CONDITION,
                       EXPECTED_FIT_ROWS_PER_CONDITION, 1, EXPECTED_MEDIAN_ROWS_PER_CONDITION]
    actual_counts = [len(build), len(project), len(fits), len(audit), len(medians)]
    if actual_counts != expected_counts:
        raise RuntimeError(f'{key}: compact output counts differ. expected={expected_counts}; actual={actual_counts}')
    for field, count in [('RankingRows', ranking_rows), ('BuildMetricRows', len(build)),
                         ('ProjectRunRows', len(project)), ('MLFits', len(fits)),
                         ('TrainingMedianRows', len(medians))]:
        if int(summary.get(field, -1)) != count:
            raise RuntimeError(f'{key}: condition_summary {field} differs.')
    project_frames.append(project); build_frames.append(build); fit_frames.append(fits)
    audit_frames.append(audit); median_frames.append(medians)
    inventory_rows.append({
        'ProjectNumber': PROJECT_NUMBER, 'Project': PROJECT_NAME, 'ProjectSlug': PROJECT_SLUG,
        'ConditionOrder': order, 'ConditionKey': key, 'NoisePercent': noise, 'RepetitionSeed': seed,
        'ConditionDirectory': str(d), 'CompletionStatus': complete.get('Status'),
        'SummaryStatus': summary.get('Status'), 'Files': len(actual_files),
        'ConditionBytes': int(sum(p.stat().st_size for p in d.iterdir() if p.is_file())),
        'RankingRows': ranking_rows, 'BuildMetricRows': len(build), 'ProjectRunRows': len(project),
        'ModelFits': len(fits), 'ConditionAuditRows': len(audit), 'TrainingMedianRows': len(medians),
        'FileSetPass': file_ok, 'CompletionMarkerPass': complete_ok, 'ConditionSummaryPass': summary_ok,
        'EmbeddedManifestFailures': local_embedded_fail,
        'CompletionMarkerSHA256': sha256_file(complete_path), 'ConditionSummarySHA256': sha256_file(summary_path),
    })
    if i % 30 == 0 or i == len(plan):
        print(f'  Condition revalidation progress: {i} / {len(plan)}')
condition_seconds = time.perf_counter() - condition_start

inventory = pd.DataFrame(inventory_rows).sort_values('ConditionOrder', kind='mergesort').reset_index(drop=True)
project_runs = pd.concat(project_frames, ignore_index=True)
build_metrics = pd.concat(build_frames, ignore_index=True)
model_fits = pd.concat(fit_frames, ignore_index=True)
condition_audit = pd.concat(audit_frames, ignore_index=True)
training_medians = pd.concat(median_frames, ignore_index=True)

# Contract audits.
coordinate_count = len(inventory[['NoisePercent', 'RepetitionSeed']].drop_duplicates())
dup_keys = int(inventory.duplicated(['ConditionKey'], keep=False).sum())
dup_coords = int(inventory.duplicated(['NoisePercent', 'RepetitionSeed'], keep=False).sum())
order_viol = int((inventory['ConditionOrder'].to_numpy(int) != np.arange(1, 271)).sum())
files_viol = int(inventory['Files'].ne(8).sum())
ranking_viol = int(inventory['RankingRows'].ne(EXPECTED_RANKING_ROWS_PER_CONDITION).sum())
small_per_condition_viol = int(inventory['BuildMetricRows'].ne(EXPECTED_BUILD_ROWS_PER_CONDITION).sum() +
                               inventory['ProjectRunRows'].ne(7).sum() + inventory['ModelFits'].ne(4).sum() +
                               inventory['ConditionAuditRows'].ne(1).sum() + inventory['TrainingMedianRows'].ne(151).sum())
project_techniques = sorted(project_runs['Technique'].astype(str).unique().tolist())
fit_techniques = sorted(model_fits['Technique'].astype(str).unique().tolist())
dup_project = int(project_runs.duplicated(['ConditionKey', 'Technique'], keep=False).sum())
dup_build = int(build_metrics.duplicated(['ConditionKey', 'Technique', 'Build'], keep=False).sum())
dup_fit = int(model_fits.duplicated(['ConditionKey', 'Technique'], keep=False).sum())
dup_audit = int(condition_audit.duplicated(['ConditionKey'], keep=False).sum())
dup_median = int(training_medians.duplicated(['ConditionKey', 'PredictorOrder'], keep=False).sum())
fit_fail = int((~model_fits['Status'].astype(str).eq('PASS_MODEL_FIT')).sum())
fit_errors = int(model_fits['Error'].fillna('').astype(str).str.len().gt(0).sum())
project_nonfinite = metric_nonfinite(project_runs, PROJECT_METRICS)
project_outside = metric_outside(project_runs, PROJECT_METRICS)
build_nonfinite = metric_nonfinite(build_metrics, BUILD_METRICS)
build_outside = metric_outside(build_metrics, BUILD_METRICS)
median_nonfinite = int((~np.isfinite(pd.to_numeric(training_medians['TrainingMedian'], errors='coerce').to_numpy(float))).sum())
median_predictor_viol = int(training_medians.groupby('ConditionKey')['Predictor'].nunique().ne(151).sum())
scored_build_viol = int(project_runs['ScoredFailingBuilds'].ne(25).sum())
eval_build_viol = int(project_runs['EvaluationBuilds'].ne(263).sum())
eval_failure_viol = int(project_runs['EvaluationFailures'].ne(171).sum())
zero = condition_audit[condition_audit['NoisePercent'].eq(0)]
positive = condition_audit[condition_audit['NoisePercent'].gt(0)]
zero_flip_viol = int(zero['NumberFlipped'].ne(0).sum())
zero_model_viol = int(zero['ModelLabelChanges'].ne(0).sum())
zero_rec_viol = int(zero['DependentRECChanges'].ne(0).sum())
pos_raw_viol = int(positive['NumberFlipped'].le(0).sum())
pos_model_viol = int(positive['ModelLabelChanges'].le(0).sum())
pos_rec_viol = int(positive['DependentRECChanges'].le(0).sum())
independent_viol = int(condition_audit['IndependentRECChanges'].ne(0).sum())
independent_recon_viol = int(condition_audit['IndependentReconstructionMismatches'].ne(0).sum())
noise_hash_mismatch = 0
for e, a in [('ExpectedFlipMaskSHA256', 'ActualFlipMaskSHA256'),
             ('ExpectedNoisyRawVerdictSHA256', 'ActualNoisyRawVerdictSHA256'),
             ('ExpectedNoisyModelVerdictSHA256', 'ActualNoisyModelVerdictSHA256')]:
    noise_hash_mismatch += int((condition_audit[e].astype(str) != condition_audit[a].astype(str)).sum())

baseline = pd.read_csv(STEP5A_BASELINE, low_memory=False)
if 'Pass' in baseline.columns:
    bpass = baseline['Pass'].astype(str).str.strip().str.lower().isin({'true', '1'})
    ranking_baseline_fail = int((~bpass).sum())
else:
    mismatch_cols = [c for c in baseline.columns if 'mismatch' in c.lower()]
    ranking_baseline_fail = int(baseline[mismatch_cols].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy(float).sum())
# Project-metric invariance must be evaluated independently for each metric.
# The previous V1 expression compared the maximum of one metric with the
# minimum of another metric. Because APFDc and APFD naturally have different
# values, that incorrectly marked all 60 seed/baseline groups as failures even
# though each individual metric was invariant across noise.
metric_baseline_fail = 0
metric_baseline_max_range = 0.0

for _, g in project_runs[
    project_runs['Technique'].isin(
        INVARIANT_BASELINES
    )
].groupby(
    [
        'RepetitionSeed',
        'Technique',
    ],
    sort=False,
):
    arr = g[
        PROJECT_METRICS
    ].to_numpy(
        dtype=float
    )

    per_metric_ranges = (
        np.max(
            arr,
            axis=0,
        )
        - np.min(
            arr,
            axis=0,
        )
    )

    metric_baseline_max_range = max(
        metric_baseline_max_range,
        float(
            np.max(
                per_metric_ranges
            )
        ),
    )

    metric_baseline_fail += int(
        (
            per_metric_ranges
            > 1e-15
        ).any()
    )

# Compact aggregates.
agg_start = time.perf_counter()
noise_summary = project_runs.groupby(['NoisePercent', 'Technique'], as_index=False, sort=True).agg(
    Runs=('ConditionKey', 'count'), Seeds=('RepetitionSeed', 'nunique'),
    Mean_MeanAPFDc=('MeanAPFDc', 'mean'), SD_MeanAPFDc=('MeanAPFDc', 'std'), Median_MeanAPFDc=('MeanAPFDc', 'median'),
    Mean_MedianAPFDc=('MedianAPFDc', 'mean'), SD_MedianAPFDc=('MedianAPFDc', 'std'), Median_MedianAPFDc=('MedianAPFDc', 'median'),
    Mean_MeanAPFD=('MeanAPFD', 'mean'), SD_MeanAPFD=('MeanAPFD', 'std'), Median_MeanAPFD=('MeanAPFD', 'median'),
    Mean_MedianAPFD=('MedianAPFD', 'mean'), SD_MedianAPFD=('MedianAPFD', 'std'), Median_MedianAPFD=('MedianAPFD', 'median'),
).sort_values(['NoisePercent', 'Technique'], kind='mergesort').reset_index(drop=True)
clean = project_runs[project_runs['NoisePercent'].eq(0)][['RepetitionSeed', 'Technique'] + PROJECT_METRICS].rename(
    columns={c: f'Clean_{c}' for c in PROJECT_METRICS})
seed_deltas = project_runs.merge(clean, on=['RepetitionSeed', 'Technique'], how='left', validate='many_to_one')
for c in PROJECT_METRICS:
    seed_deltas[f'Delta_{c}'] = seed_deltas[c] - seed_deltas[f'Clean_{c}']
delta_cols = [f'Delta_{c}' for c in PROJECT_METRICS]
seed_deltas = seed_deltas[['ProjectNumber', 'Project', 'ProjectSlug', 'ConditionKey', 'NoisePercent',
                           'RepetitionSeed', 'Technique'] + PROJECT_METRICS +
                          [f'Clean_{c}' for c in PROJECT_METRICS] + delta_cols].sort_values(
                              ['NoisePercent', 'Technique', 'RepetitionSeed'], kind='mergesort').reset_index(drop=True)
delta_summary = seed_deltas.groupby(['NoisePercent', 'Technique'], as_index=False, sort=True).agg(
    Seeds=('RepetitionSeed', 'nunique'),
    Mean_Delta_MeanAPFDc=('Delta_MeanAPFDc', 'mean'), SD_Delta_MeanAPFDc=('Delta_MeanAPFDc', 'std'), Median_Delta_MeanAPFDc=('Delta_MeanAPFDc', 'median'),
    Mean_Delta_MedianAPFDc=('Delta_MedianAPFDc', 'mean'), SD_Delta_MedianAPFDc=('Delta_MedianAPFDc', 'std'), Median_Delta_MedianAPFDc=('Delta_MedianAPFDc', 'median'),
    Mean_Delta_MeanAPFD=('Delta_MeanAPFD', 'mean'), SD_Delta_MeanAPFD=('Delta_MeanAPFD', 'std'), Median_Delta_MeanAPFD=('Delta_MeanAPFD', 'median'),
    Mean_Delta_MedianAPFD=('Delta_MedianAPFD', 'mean'), SD_Delta_MedianAPFD=('Delta_MedianAPFD', 'std'), Median_Delta_MedianAPFD=('Delta_MedianAPFD', 'median'),
).sort_values(['NoisePercent', 'Technique'], kind='mergesort').reset_index(drop=True)
agg_seconds = time.perf_counter() - agg_start
summary_nonfinite = metric_nonfinite(noise_summary, [c for c in noise_summary.columns if c not in {'NoisePercent', 'Technique'}])
delta_nonfinite = metric_nonfinite(delta_summary, [c for c in delta_summary.columns if c not in {'NoisePercent', 'Technique'}])
clean_delta_nonzero = int((np.abs(seed_deltas[seed_deltas['NoisePercent'].eq(0)][delta_cols].to_numpy(float)) > 1e-15).sum())
baseline_delta_nonzero = int((np.abs(seed_deltas[seed_deltas['Technique'].isin(INVARIANT_BASELINES)][delta_cols].to_numpy(float)) > 1e-15).sum())

# Validation table.
checks = []
add_check(checks, 'Step 5A status', STEP5A_STATUS, step5a.get('Status'), step5a.get('Status') == STEP5A_STATUS)
add_check(checks, 'Step 5A checkpoint SHA-256', EXPECTED_STEP5A_SHA, step5a_sha, step5a_sha == EXPECTED_STEP5A_SHA)
add_check(checks, 'Frozen raw-root SHA-256', EXPECTED_RAW_ROOT_SHA, step5a.get('RawRootSHA256'), step5a.get('RawRootSHA256') == EXPECTED_RAW_ROOT_SHA)
add_check(checks, 'Independent current raw-root SHA-256', EXPECTED_RAW_ROOT_SHA, current_raw_sha, current_raw_sha == EXPECTED_RAW_ROOT_SHA)
add_check(checks, 'Step 5A aggregate-manifest failures', 0, agg_failures, agg_failures == 0)
add_check(checks, 'Condition marker failures', 0, marker_fail, marker_fail == 0)
add_check(checks, 'Condition summary failures', 0, summary_fail, summary_fail == 0)
add_check(checks, 'Condition file-set failures', 0, file_set_fail, file_set_fail == 0)
add_check(checks, 'Embedded output-manifest failures', 0, embedded_fail, embedded_fail == 0)
add_check(checks, 'Conditions', 270, len(inventory), len(inventory) == 270)
add_check(checks, 'Condition coordinates', 270, coordinate_count, coordinate_count == 270)
add_check(checks, 'Duplicate condition keys', 0, dup_keys, dup_keys == 0)
add_check(checks, 'Duplicate condition coordinates', 0, dup_coords, dup_coords == 0)
add_check(checks, 'Condition-order violations', 0, order_viol, order_viol == 0)
add_check(checks, 'Files-per-condition violations', 0, files_viol, files_viol == 0)
add_check(checks, 'Raw files', EXPECTED_RAW_FILES, len(current_manifest), len(current_manifest) == EXPECTED_RAW_FILES)
add_check(checks, 'Raw bytes', EXPECTED_RAW_BYTES, current_raw_bytes, current_raw_bytes == EXPECTED_RAW_BYTES)
add_check(checks, 'Missing raw files', 0, missing_raw, missing_raw == 0)
add_check(checks, 'Unexpected raw files', 0, unexpected_raw, unexpected_raw == 0)
add_check(checks, 'Raw size mismatches', 0, size_mismatch, size_mismatch == 0)
add_check(checks, 'Raw SHA-256 mismatches', 0, hash_mismatch, hash_mismatch == 0)
add_check(checks, 'Ranking rows', EXPECTED_TOTAL_RANKING_ROWS, int(inventory['RankingRows'].sum()), int(inventory['RankingRows'].sum()) == EXPECTED_TOTAL_RANKING_ROWS)
add_check(checks, 'Ranking row-count failures', 0, ranking_count_fail + ranking_viol, ranking_count_fail + ranking_viol == 0)
add_check(checks, 'Project-run rows', EXPECTED_TOTAL_PROJECT_ROWS, len(project_runs), len(project_runs) == EXPECTED_TOTAL_PROJECT_ROWS)
add_check(checks, 'Build-metric rows', EXPECTED_TOTAL_BUILD_ROWS, len(build_metrics), len(build_metrics) == EXPECTED_TOTAL_BUILD_ROWS)
add_check(checks, 'Model-fit rows', EXPECTED_TOTAL_FIT_ROWS, len(model_fits), len(model_fits) == EXPECTED_TOTAL_FIT_ROWS)
add_check(checks, 'Condition-audit rows', EXPECTED_TOTAL_AUDIT_ROWS, len(condition_audit), len(condition_audit) == EXPECTED_TOTAL_AUDIT_ROWS)
add_check(checks, 'Training-median rows', EXPECTED_TOTAL_MEDIAN_ROWS, len(training_medians), len(training_medians) == EXPECTED_TOTAL_MEDIAN_ROWS)
add_check(checks, 'Small rows-per-condition violations', 0, small_per_condition_viol, small_per_condition_viol == 0)
add_check(checks, 'Project-run technique set', sorted(TECHNIQUES), project_techniques, project_techniques == sorted(TECHNIQUES))
add_check(checks, 'Model-fit technique set', sorted(ML_TECHNIQUES), fit_techniques, fit_techniques == sorted(ML_TECHNIQUES))
add_check(checks, 'Duplicate project/build/fit/audit/median rows', 0, dup_project + dup_build + dup_fit + dup_audit + dup_median, dup_project + dup_build + dup_fit + dup_audit + dup_median == 0)
add_check(checks, 'Model-fit failures', 0, fit_fail + fit_errors, fit_fail + fit_errors == 0)
add_check(checks, 'Scored/evaluated/failure count violations', 0, scored_build_viol + eval_build_viol + eval_failure_viol, scored_build_viol + eval_build_viol + eval_failure_viol == 0)
add_check(checks, 'Project metric invalid values', 0, project_nonfinite + project_outside, project_nonfinite + project_outside == 0)
add_check(checks, 'Build metric invalid values', 0, build_nonfinite + build_outside, build_nonfinite + build_outside == 0)
add_check(checks, 'Training-median invalid values', 0, median_nonfinite + median_predictor_viol, median_nonfinite + median_predictor_viol == 0)
add_check(checks, 'Zero-noise conditions', 30, len(zero), len(zero) == 30)
add_check(checks, 'Zero-noise violations', 0, zero_flip_viol + zero_model_viol + zero_rec_viol, zero_flip_viol + zero_model_viol + zero_rec_viol == 0)
add_check(checks, 'Positive-noise violations', 0, pos_raw_viol + pos_model_viol + pos_rec_viol, pos_raw_viol + pos_model_viol + pos_rec_viol == 0)
add_check(checks, 'Independent REC violations', 0, independent_viol + independent_recon_viol, independent_viol + independent_recon_viol == 0)
add_check(checks, 'Noise-plan hash mismatches', 0, noise_hash_mismatch, noise_hash_mismatch == 0)
add_check(checks, 'Ranking-level baseline-invariance failures', 0, ranking_baseline_fail, ranking_baseline_fail == 0)
add_check(checks, 'Project-metric baseline-invariance failures', 0, metric_baseline_fail, metric_baseline_fail == 0)
add_check(checks, 'Noise-technique summary rows', 63, len(noise_summary), len(noise_summary) == 63)
add_check(checks, 'Noise-technique summary count/nonfinite violations', 0, int(noise_summary['Runs'].ne(30).sum() + noise_summary['Seeds'].ne(30).sum()) + summary_nonfinite, int(noise_summary['Runs'].ne(30).sum() + noise_summary['Seeds'].ne(30).sum()) + summary_nonfinite == 0)
add_check(checks, 'Seed-level delta rows', 1890, len(seed_deltas), len(seed_deltas) == 1890)
add_check(checks, 'Clean delta non-zero values', 0, clean_delta_nonzero, clean_delta_nonzero == 0)
add_check(checks, 'Invariant-baseline delta non-zero values', 0, baseline_delta_nonzero, baseline_delta_nonzero == 0)
add_check(checks, 'Noise-delta summary rows', 63, len(delta_summary), len(delta_summary) == 63)
add_check(checks, 'Noise-delta summary count/nonfinite violations', 0, int(delta_summary['Seeds'].ne(30).sum()) + delta_nonfinite, int(delta_summary['Seeds'].ne(30).sum()) + delta_nonfinite == 0)
add_check(checks, 'Registry rows', 10, len(registry), len(registry) == 10)
add_check(checks, 'Registry Project 11 rows', 0, int(pnums.eq(11).sum()), int(pnums.eq(11).sum()) == 0)
validation = pd.DataFrame(checks)
failed = validation[~validation['Pass']]
print('\nProject 11 Step 5B validation:')
display(validation)
if not failed.empty:
    print('\nFailed checks:'); display(failed)
    raise RuntimeError('PROJECT 11 STEP 5B VALIDATION FAILED. No PASS checkpoint was written.')

# Freeze outputs.
OUT.mkdir(parents=True, exist_ok=True)
for path, frame in [
    (CURRENT_MANIFEST, current_manifest), (CONDITION_INVENTORY, inventory),
    (REVALIDATED_PROJECT_RUNS, project_runs), (REVALIDATED_BUILD_METRICS, build_metrics),
    (REVALIDATED_MODEL_FITS, model_fits), (REVALIDATED_CONDITION_AUDIT, condition_audit),
    (REVALIDATED_MEDIANS, training_medians), (NOISE_SUMMARY, noise_summary),
    (SEED_DELTAS, seed_deltas), (DELTA_SUMMARY, delta_summary), (VALIDATION_PATH, validation),
]:
    atomic_csv(path, frame)
output_paths = [CURRENT_MANIFEST, CONDITION_INVENTORY, REVALIDATED_PROJECT_RUNS,
                REVALIDATED_BUILD_METRICS, REVALIDATED_MODEL_FITS, REVALIDATED_CONDITION_AUDIT,
                REVALIDATED_MEDIANS, NOISE_SUMMARY, SEED_DELTAS, DELTA_SUMMARY, VALIDATION_PATH]
output_manifest = [{'Path': str(p), 'Bytes': int(p.stat().st_size), 'SHA256': sha256_file(p)} for p in output_paths]
completed = datetime.now(timezone.utc).isoformat()
report = {
    'ProjectNumber': PROJECT_NUMBER, 'Project': PROJECT_NAME, 'ProjectSlug': PROJECT_SLUG,
    'Status': STEP5B_STATUS, 'CompletedAtUTC': completed, 'Step5ACheckpointSHA256': step5a_sha,
    'FrozenRawRootSHA256': EXPECTED_RAW_ROOT_SHA, 'IndependentRawRootSHA256': current_raw_sha,
    'RawFiles': len(current_manifest), 'RawBytes': current_raw_bytes, 'Conditions': len(inventory),
    'MLFits': len(model_fits), 'RankingRows': int(inventory['RankingRows'].sum()),
    'BuildMetricRows': len(build_metrics), 'ProjectRunRows': len(project_runs),
    'ConditionAuditRows': len(condition_audit), 'TrainingMedianRows': len(training_medians),
    'NoiseTechniqueSummaryRows': len(noise_summary), 'SeedLevelNoiseDeltaRows': len(seed_deltas),
    'NoiseDeltaSummaryRows': len(delta_summary),
    'StandardDeviationDefinition': 'Sample SD across 30 seeds; pandas std, ddof=1',
    'RankingLevelBaselineInvarianceFailures': int(ranking_baseline_fail),
    'ProjectMetricBaselineInvarianceFailures': int(metric_baseline_fail),
    'ProjectMetricBaselineMaximumWithinMetricRange': float(metric_baseline_max_range),
    'RawHashingSeconds': float(hash_seconds), 'ConditionRevalidationSeconds': float(condition_seconds),
    'AggregationSeconds': float(agg_seconds), 'OutputManifest': output_manifest,
    'ValidationChecks': len(validation), 'FailedValidationChecks': len(failed),
    'RegistrySHA256': registry_sha_before, 'RegistryModified': False,
    'Projects1To10Modified': False, 'Project12Accessed': False, 'Project12WriteAttempted': False,
    'ModelsFitted': False, 'ConditionsRerun': False,
}
atomic_json(REPORT_PATH, report)
checkpoint = {**report, 'CheckpointVersion': 1,
              'CheckpointType': 'PROJECT_11_RAW_REVALIDATION_AND_COMPACT_AGGREGATION',
              'RawResultsRevalidated': True, 'CompactAggregatesFrozen': True,
              'ReadyForFinalPackageAndRegistration': True}
atomic_json(CHECKPOINT_PATH, checkpoint)
checkpoint_sha = sha256_file(CHECKPOINT_PATH)
status = {'ProjectNumber': PROJECT_NUMBER, 'Project': PROJECT_NAME, 'ProjectSlug': PROJECT_SLUG,
          'Status': STEP5B_STATUS, 'CompletedAtUTC': completed, 'Step5ACheckpointSHA256': step5a_sha,
          'RawRootSHA256': current_raw_sha, 'RawFiles': len(current_manifest), 'RawBytes': current_raw_bytes,
          'Conditions': len(inventory), 'MLFits': len(model_fits), 'Checkpoint': str(CHECKPOINT_PATH),
          'CheckpointSHA256': checkpoint_sha, 'ReadyForFinalPackageAndRegistration': True,
          'RegistryModified': False, 'Project12Accessed': False}
atomic_json(STATUS_PATH, status)

# Readback and immutability.
if load_json(CHECKPOINT_PATH).get('Status') != STEP5B_STATUS or load_json(STATUS_PATH).get('Status') != STEP5B_STATUS:
    raise RuntimeError('Step 5B checkpoint/status readback failed.')
for item in output_manifest:
    p = Path(item['Path'])
    if not p.is_file() or p.stat().st_size != item['Bytes'] or sha256_file(p) != item['SHA256']:
        raise RuntimeError(f'Step 5B output readback failed: {p}')
registry_sha_after = sha256_file(REGISTRY)
if registry_sha_after != registry_sha_before:
    raise RuntimeError('Registry changed during Project 11 Step 5B.')
if sha256_file(STEP5A_CHECKPOINT) != EXPECTED_STEP5A_SHA:
    raise RuntimeError('Step 5A checkpoint changed during Step 5B.')

print('\nNoise-technique summary:')
display(noise_summary)
print('\nNoise-delta summary:')
display(delta_summary)
print('\n' + '=' * 136)
print('=== PROJECT 11 CELL 10 / STEP 5B RESULT ===')
print('=' * 136)
print('Project:', PROJECT_NAME)
print('Step 5A checkpoint SHA-256:', step5a_sha)
print('Frozen raw-root SHA-256:', EXPECTED_RAW_ROOT_SHA)
print('Independent current raw-root SHA-256:', current_raw_sha)
print('\nRaw-output revalidation:')
print('Conditions:', len(inventory), '/', EXPECTED_CONDITIONS)
print('Raw files:', len(current_manifest), '/', EXPECTED_RAW_FILES)
print('Raw bytes:', current_raw_bytes, '/', EXPECTED_RAW_BYTES)
print('Missing / unexpected / size / SHA mismatches:', missing_raw, '/', unexpected_raw, '/', size_mismatch, '/', hash_mismatch)
print('Embedded output-manifest failures:', embedded_fail)
print('\nExperiment totals:')
print('ML fits:', len(model_fits), '/', EXPECTED_TOTAL_FIT_ROWS)
print('Ranking rows:', int(inventory['RankingRows'].sum()), '/', EXPECTED_TOTAL_RANKING_ROWS)
print('Build-metric rows:', len(build_metrics), '/', EXPECTED_TOTAL_BUILD_ROWS)
print('Project-run rows:', len(project_runs), '/', EXPECTED_TOTAL_PROJECT_ROWS)
print('Condition-audit rows:', len(condition_audit), '/', EXPECTED_TOTAL_AUDIT_ROWS)
print('Training-median rows:', len(training_medians), '/', EXPECTED_TOTAL_MEDIAN_ROWS)
print('\nAnalysis-ready aggregates:')
print('Noise-technique summary rows:', len(noise_summary))
print('Seed-level noise-delta rows:', len(seed_deltas))
print('Noise-delta summary rows:', len(delta_summary))
print('Sample SD calculated with ddof=1:', True)
print('Ranking-level baseline-invariance failures:', ranking_baseline_fail)
print('Project-metric baseline-invariance failures:', metric_baseline_fail)
print('Maximum within-metric baseline range:', metric_baseline_max_range)
print('\nImmutability and isolation:')
print('Completion registry unchanged:', registry_sha_after == registry_sha_before)
print('Registry Project 11 rows:', int(pnums.eq(11).sum()))
print('Projects 1–10 modified:', 0)
print('Project 12 accessed:', False)
print('Project 12 write attempted:', False)
print('Conditions rerun:', False)
print('Models fitted:', False)
print('\nRuntime:')
print('Raw hashing seconds:', round(hash_seconds, 2))
print('Condition revalidation seconds:', round(condition_seconds, 2))
print('Compact aggregation seconds:', round(agg_seconds, 2))
print('\nValidation:')
print('Checks:', len(validation))
print('Failed checks:', len(failed))
print('\nProject 11 Step 5B checkpoint:')
print(CHECKPOINT_PATH)
print('Checkpoint SHA-256:', checkpoint_sha)
print('\nSTATUS:', STEP5B_STATUS)
print('=' * 136)

=== PROJECT 11 CELL 10 / STEP 5B V2: CORRECTED PER-METRIC BASELINE INVARIANCE ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Independently hashing all 2,160 raw files.
  Raw hashing progress: 200 / 2160 files
  Raw hashing progress: 400 / 2160 files
  Raw hashing progress: 600 / 2160 files
  Raw hashing progress: 800 / 2160 files
  Raw hashing progress: 1000 / 2160 files
  Raw hashing progress: 1200 / 2160 files
  Raw hashing progress: 1400 / 2160 files
  Raw hashing progress: 1600 / 2160 files
  Raw hashing progress: 1800 / 2160 files
  Raw hashing progress: 2000 / 2160 files
  Raw hashing progress: 2160 / 2160 files

Revalidating all 270 condition directories.
  Condition revalidation progress: 30 / 270
  Condition revalidation progress: 60 / 270
  Condition revalidation progress: 90 / 270
  Condition revalidation progress: 120 / 270
  Condition revalidation progress: 150 / 270
  Condition revalida

,Check,Expected,Actual,Pass
0,Step 5A status,PASS_PROJECT_11_FULL_270_CONDITION_EXPERIMENT_...,PASS_PROJECT_11_FULL_270_CONDITION_EXPERIMENT_...,True
1,Step 5A checkpoint SHA-256,4b0b53c91c25d2f3900a5c9df2b313c495545ea65036ae...,4b0b53c91c25d2f3900a5c9df2b313c495545ea65036ae...,True
2,Frozen raw-root SHA-256,6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e8...,6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e8...,True
3,Independent current raw-root SHA-256,6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e8...,6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e8...,True
4,Step 5A aggregate-manifest failures,0,0,True
5,Condition marker failures,0,0,True
6,Condition summary failures,0,0,True
7,Condition file-set failures,0,0,True
8,Embedded output-manifest failures,0,0,True
9,Conditions,270,270,True



Noise-technique summary:


,NoisePercent,Technique,Runs,Seeds,Mean_MeanAPFDc,SD_MeanAPFDc,Median_MeanAPFDc,Mean_MedianAPFDc,SD_MedianAPFDc,Median_MedianAPFDc,Mean_MeanAPFD,SD_MeanAPFD,Median_MeanAPFD,Mean_MedianAPFD,SD_MedianAPFD,Median_MedianAPFD
0,0,LatestFail,30,30,0.117362,0.000000,0.117362,0.039770,0.000000,0.039770,0.038628,0.000000,0.038628,0.017391,0.000000,0.017391
1,0,LightGBM,30,30,0.956119,0.000000,0.956119,0.992583,0.000000,0.992583,0.976796,0.000000,0.976796,0.989297,0.000000,0.989297
2,0,NaiveBayes,30,30,0.801712,0.000000,0.801712,0.902729,0.000000,0.902729,0.952807,0.000000,0.952807,0.977501,0.000000,0.977501
3,0,QTF-Avg,30,30,0.873847,0.000000,0.873847,0.867396,0.000000,0.867396,0.289495,0.000000,0.289495,0.297862,0.000000,0.297862
4,0,Random,30,30,0.496849,0.020095,0.498377,0.493450,0.027435,0.491114,0.496935,0.019359,0.498200,0.499485,0.026817,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,50,NaiveBayes,30,30,0.471575,0.375123,0.373314,0.475547,0.398716,0.436739,0.474009,0.411448,0.253589,0.479128,0.443817,0.280472
59,50,QTF-Avg,30,30,0.873847,0.000000,0.873847,0.867396,0.000000,0.867396,0.289495,0.000000,0.289495,0.297862,0.000000,0.297862
60,50,Random,30,30,0.496849,0.020095,0.498377,0.493450,0.027435,0.491114,0.496935,0.019359,0.498200,0.499485,0.026817,0.500000
61,50,RandomForest,30,30,0.355996,0.080369,0.361550,0.328744,0.105499,0.321031,0.354230,0.071891,0.367372,0.324962,0.088079,0.340530



Noise-delta summary:


,NoisePercent,Technique,Seeds,Mean_Delta_MeanAPFDc,SD_Delta_MeanAPFDc,Median_Delta_MeanAPFDc,Mean_Delta_MedianAPFDc,SD_Delta_MedianAPFDc,Median_Delta_MedianAPFDc,Mean_Delta_MeanAPFD,SD_Delta_MeanAPFD,Median_Delta_MeanAPFD,Mean_Delta_MedianAPFD,SD_Delta_MedianAPFD,Median_Delta_MedianAPFD
0,0,LatestFail,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,LightGBM,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0,NaiveBayes,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0,QTF-Avg,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0,Random,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,50,NaiveBayes,30,-0.330137,0.375123,-0.428398,-0.427182,0.398716,-0.465990,-0.478797,0.411448,-0.699218,-0.498373,0.443817,-0.697029
59,50,QTF-Avg,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
60,50,Random,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
61,50,RandomForest,30,-0.630327,0.080184,-0.625612,-0.663286,0.105462,-0.671143,-0.636383,0.071607,-0.623410,-0.664582,0.087965,-0.648769



=== PROJECT 11 CELL 10 / STEP 5B RESULT ===
Project: apache@shardingsphere
Step 5A checkpoint SHA-256: 4b0b53c91c25d2f3900a5c9df2b313c495545ea65036aeba39741a88770b5862
Frozen raw-root SHA-256: 6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e88642a299343f2a73a1
Independent current raw-root SHA-256: 6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e88642a299343f2a73a1

Raw-output revalidation:
Conditions: 270 / 270
Raw files: 2160 / 2160
Raw bytes: 386769592 / 386769592
Missing / unexpected / size / SHA mismatches: 0 / 0 / 0 / 0
Embedded output-manifest failures: 0

Experiment totals:
ML fits: 1080 / 1080
Ranking rows: 24583230 / 24583230
Build-metric rows: 47250 / 47250
Project-run rows: 1890 / 1890
Condition-audit rows: 270 / 270
Training-median rows: 40770 / 40770

Analysis-ready aggregates:
Noise-technique summary rows: 63
Seed-level noise-delta rows: 1890
Noise-delta summary rows: 63
Sample SD calculated with ddof=1: True
Ranking-level baseline-invariance failures: 0
Project-metric baselin

In [5]:
# ==================================================================================================
# PROJECT 11 — CELL 11 / STEP 5C
# FINAL COMPACT PACKAGE FREEZE AND COMPLETION-REGISTRY REGISTRATION
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import hashlib, json, os, re, shutil, tempfile
import pandas as pd

print("=" * 136)
print("=== PROJECT 11 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===")
print("=" * 136)

# Frozen identity and hashes.
PROJECT_NUMBER = 11
PROJECT_NAME = "apache@shardingsphere"
PROJECT_SLUG = "apache__shardingsphere"
PROJECT_SHORT = "shardingsphere"
COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
STEP5C_STATUS = "PASS_PROJECT_11_FINAL_PACKAGE_FROZEN_AND_REGISTERED"
STEP5A_STATUS = "PASS_PROJECT_11_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
STEP5B_STATUS = "PASS_PROJECT_11_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN"
REGISTRY_SHA_BEFORE_EXPECTED = "847bdcbee16c8757fde34e9489cd3abbf0781a319644175472597265b1df2750"
STEP5A_SHA_EXPECTED = "4b0b53c91c25d2f3900a5c9df2b313c495545ea65036aeba39741a88770b5862"
STEP5B_SHA_EXPECTED = "cfbcbf39431c8eaa2196fb069b2ee0f203703379696a2cdc52c44902383d73ee"
SOURCE_ROOT_SHA = "3e6e0a60d58bff61690d6e2ac2bac18371414819eff601f3ea347ecb8ad8b0b5"
RAW_ROOT_SHA = "6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e88642a299343f2a73a1"

COUNTS = {
    "RawFiles": 2160, "RawBytes": 386769592, "Conditions": 270, "MLFits": 1080,
    "RankingRows": 24583230, "BuildMetricRows": 47250, "ProjectRunRows": 1890,
    "ConditionAuditRows": 270, "TrainingMedianRows": 40770, "Builds": 1049,
    "TrainingBuilds": 786, "EvaluationBuilds": 263, "RawRows": 833541,
    "RawTrainingRows": 609619, "RawEvaluationRows": 223922,
    "RawTrainingFailures": 1193, "RawEvaluationFailures": 171, "ModelRows": 91042,
    "ModelTrainingRows": 78035, "ModelEvaluationRows": 13007,
    "ModelTrainingFailures": 1188, "ModelEvaluationFailures": 171,
    "ModelFailingEvaluationBuilds": 25, "Predictors": 151, "RECFeatures": 19,
}

drive.mount("/content/drive", force_remount=False)
ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"
REGISTRY = NOTES / "completed_project_registry.csv"
PROJECT_ROOT = RESULTS / "Aggregated" / PROJECT_SLUG
RAW_ROOT = RESULTS / "Raw" / PROJECT_SLUG
FINAL_ROOT = RESULTS / "Final" / PROJECT_SLUG
MANIFEST_PATH = FINAL_ROOT / "final_package_manifest.csv"
SUMMARY_PATH = FINAL_ROOT / "final_package_summary.json"
README_PATH = FINAL_ROOT / "README.txt"
STEP5C_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_step5c"
VALIDATION_PATH = STEP5C_ROOT / f"{PROJECT_SHORT}_step5c_validation.csv"
REPORT_PATH = STEP5C_ROOT / f"{PROJECT_SHORT}_step5c_report.json"
STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5c_status.json"
CHECKPOINT_PATH = NOTES / "project_11_step5c_checkpoint.json"
BACKUP_PATH = NOTES / "completed_project_registry_before_project_11.csv"
STEP5A_CP = NOTES / "project_11_step5a_checkpoint.json"
STEP5B_CP = NOTES / "project_11_step5b_checkpoint.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5b_status.json"
UPSTREAM_CPS = [
    NOTES / "project_11_selection_checkpoint.json",
    NOTES / "project_11_rec_reconstruction_checkpoint.json",
    NOTES / "project_11_noise_plan_checkpoint.json",
    NOTES / "project_11_runtime_contract_checkpoint.json",
    NOTES / "project_11_smoke_test_checkpoint.json",
    STEP5A_CP, STEP5B_CP,
]


def sha(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, ensure_ascii=False, default=str); f.write("\n")
    os.replace(tmp, path)


def atomic_csv(path, df):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    df.to_csv(tmp, index=False, lineterminator="\n")
    os.replace(tmp, path)


def atomic_text(path, text):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    tmp.write_text(text, encoding="utf-8"); os.replace(tmp, path)


def resolve(cols, expected):
    matches = [c for c in cols if str(c).strip().lower() == expected.lower()]
    if len(matches) != 1:
        raise RuntimeError(f"Could not resolve registry column {expected!r}; matches={matches}; columns={list(cols)}")
    return matches[0]


def norm(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


def manifest(root, exclude=()):
    root = Path(root); exclude = set(exclude); rows = []
    for p in sorted((x for x in root.rglob("*") if x.is_file()), key=lambda x: x.relative_to(root).as_posix()):
        rel = p.relative_to(root).as_posix()
        if rel not in exclude:
            rows.append({"RelativePath": rel, "Bytes": int(p.stat().st_size), "SHA256": sha(p)})
    return pd.DataFrame(rows, columns=["RelativePath", "Bytes", "SHA256"])


def root_hash(df):
    h = hashlib.sha256()
    for r in df.sort_values("RelativePath", kind="mergesort").itertuples(index=False):
        h.update(f"{r.RelativePath}\0{int(r.Bytes)}\0{str(r.SHA256).lower()}\n".encode())
    return h.hexdigest()


def verify_manifest(items, label):
    if not isinstance(items, list) or not items:
        raise RuntimeError(f"{label} has no output manifest.")
    rows = []
    for item in items:
        p = Path(item["Path"]); exists = p.is_file()
        eb, es = int(item["Bytes"]), str(item["SHA256"]).lower()
        ab, ac = (int(p.stat().st_size), sha(p)) if exists else (-1, "MISSING")
        rows.append({"Path": str(p), "ExpectedBytes": eb, "ActualBytes": ab,
                     "ExpectedSHA256": es, "ActualSHA256": ac,
                     "Pass": bool(exists and eb == ab and es == ac)})
    out = pd.DataFrame(rows)
    if not out["Pass"].all():
        display(out.loc[~out["Pass"]]); raise RuntimeError(f"{label} manifest verification failed.")
    return out


def check(rows, name, expected, actual, passed):
    rows.append({"Check": name, "Expected": expected, "Actual": actual, "Pass": bool(passed)})


# Verify all inputs before any package or registry write.
required = [REGISTRY, RAW_ROOT, STEP5A_CP, STEP5B_CP, STEP5A_STATUS_PATH, STEP5B_STATUS_PATH, *UPSTREAM_CPS]
missing = [str(p) for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing Project 11 Step 5C inputs:\n" + "\n".join(missing))

step5a_sha, step5b_sha = sha(STEP5A_CP), sha(STEP5B_CP)
if step5a_sha != STEP5A_SHA_EXPECTED:
    raise RuntimeError(f"Step 5A checkpoint SHA differs: {step5a_sha}")
if step5b_sha != STEP5B_SHA_EXPECTED:
    raise RuntimeError(f"Step 5B checkpoint SHA differs: {step5b_sha}")
step5a, step5b = load_json(STEP5A_CP), load_json(STEP5B_CP)
for label, payload, expected in [
    ("Step 5A checkpoint", step5a, STEP5A_STATUS),
    ("Step 5A status", load_json(STEP5A_STATUS_PATH), STEP5A_STATUS),
    ("Step 5B checkpoint", step5b, STEP5B_STATUS),
    ("Step 5B status", load_json(STEP5B_STATUS_PATH), STEP5B_STATUS),
]:
    if payload.get("Status") != expected:
        raise RuntimeError(f"{label} is not in expected PASS state.")
if step5a.get("SourceRootSHA256") != SOURCE_ROOT_SHA or step5a.get("RawRootSHA256") != RAW_ROOT_SHA:
    raise RuntimeError("Step 5A source/raw root differs.")
if step5b.get("IndependentRawRootSHA256") != RAW_ROOT_SHA:
    raise RuntimeError("Step 5B independent raw root differs.")
step5a_audit = verify_manifest(step5a.get("AggregateOutputManifest", []), "Step 5A aggregate")
step5b_audit = verify_manifest(step5b.get("OutputManifest", []), "Step 5B")

# Registry must still be exactly Projects 1–10.
registry_sha_before = sha(REGISTRY)
if registry_sha_before != REGISTRY_SHA_BEFORE_EXPECTED:
    raise RuntimeError(f"Registry SHA differs before Project 11 registration: {registry_sha_before}")
reg_before = pd.read_csv(REGISTRY, dtype=str).fillna("")
pn_col, project_col, status_col = resolve(reg_before.columns, "ProjectNumber"), resolve(reg_before.columns, "Project"), resolve(reg_before.columns, "Status")
pnums = pd.to_numeric(reg_before[pn_col], errors="raise").astype(int)
if len(reg_before) != 10 or sorted(pnums.tolist()) != list(range(1, 11)):
    raise RuntimeError("Registry must contain exactly Projects 1–10.")
if not reg_before[status_col].eq(COMPLETE_STATUS).all() or pnums.eq(11).any() or reg_before[project_col].eq(PROJECT_NAME).any():
    raise RuntimeError("Registry state is invalid for Project 11 registration.")
if not BACKUP_PATH.exists():
    shutil.copy2(REGISTRY, BACKUP_PATH)
if sha(BACKUP_PATH) != registry_sha_before:
    raise RuntimeError("Pre-Project-11 registry backup does not match live registry.")

# Assemble compact package from all upstream checkpoints plus Step 5A/5B frozen manifest outputs.
sources = set(Path(p) for p in UPSTREAM_CPS)
sources.update([STEP5A_STATUS_PATH, STEP5B_STATUS_PATH])
sources.update(Path(item["Path"]) for item in step5a.get("AggregateOutputManifest", []))
sources.update(Path(item["Path"]) for item in step5b.get("OutputManifest", []))
sources = sorted(sources, key=str)
missing_sources = [str(p) for p in sources if not p.is_file()]
if missing_sources:
    raise FileNotFoundError("Missing compact package sources:\n" + "\n".join(missing_sources))

created_at = datetime.now(timezone.utc).isoformat()
tmp_root = Path(tempfile.mkdtemp(prefix="project11_package_", dir="/content"))
try:
    for src in sources:
        try:
            rel = src.relative_to(ROOT)
        except ValueError as exc:
            raise RuntimeError(f"Package source is outside thesis root: {src}") from exc
        dst = tmp_root / rel; dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst)
    atomic_text(tmp_root / "README.txt", f"""PROJECT 11 FINAL COMPACT PACKAGE

Project number: {PROJECT_NUMBER}
Project: {PROJECT_NAME}
Project slug: {PROJECT_SLUG}
Status: {COMPLETE_STATUS}
Created at UTC: {created_at}

The raw 2,160 condition files are not duplicated here.
Raw results: {RAW_ROOT}
Raw-root SHA-256: {RAW_ROOT_SHA}
Primary metric: APFDc
Secondary metric: APFD
""")
    before_summary = manifest(tmp_root)
    atomic_json(tmp_root / "final_package_summary.json", {
        "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
        "Status": COMPLETE_STATUS, "CreatedAtUTC": created_at, "SourceRootSHA256": SOURCE_ROOT_SHA,
        "RawRootSHA256": RAW_ROOT_SHA, **COUNTS, "Step5ACheckpointSHA256": step5a_sha,
        "Step5BCheckpointSHA256": step5b_sha, "PayloadRootSHA256BeforeSummary": root_hash(before_summary),
        "RawResultsDuplicatedIntoPackage": False,
    })
    candidate_manifest = manifest(tmp_root, {"final_package_manifest.csv"})
    package_root_sha = root_hash(candidate_manifest)
    atomic_csv(tmp_root / "final_package_manifest.csv", candidate_manifest)
    package_files = len(candidate_manifest) + 1
    package_bytes = int(candidate_manifest["Bytes"].sum() + (tmp_root / "final_package_manifest.csv").stat().st_size)
    if FINAL_ROOT.exists():
        if not MANIFEST_PATH.is_file():
            raise RuntimeError("Existing final-package directory has no manifest and was not modified.")
        existing = pd.read_csv(MANIFEST_PATH, low_memory=False)
        if root_hash(existing) != package_root_sha:
            raise RuntimeError("A different Project 11 final package already exists and was not modified.")
        shutil.rmtree(tmp_root); package_already_frozen = True
    else:
        FINAL_ROOT.parent.mkdir(parents=True, exist_ok=True); os.replace(tmp_root, FINAL_ROOT); package_already_frozen = False
except Exception:
    if tmp_root.exists(): shutil.rmtree(tmp_root, ignore_errors=True)
    raise

# Read back every packaged file.
pkg_manifest = pd.read_csv(MANIFEST_PATH, low_memory=False)
manifest_paths = set(pkg_manifest["RelativePath"].astype(str))
actual_paths = {p.relative_to(FINAL_ROOT).as_posix() for p in FINAL_ROOT.rglob("*") if p.is_file()}
expected_paths = manifest_paths | {"final_package_manifest.csv"}
missing_pkg, unexpected_pkg, size_bad, hash_bad = len(expected_paths - actual_paths), len(actual_paths - expected_paths), 0, 0
for r in pkg_manifest.itertuples(index=False):
    p = FINAL_ROOT / str(r.RelativePath)
    if p.is_file():
        size_bad += int(p.stat().st_size != int(r.Bytes)); hash_bad += int(sha(p) != str(r.SHA256))
package_root_readback = root_hash(pkg_manifest)
if package_root_readback != package_root_sha or any([missing_pkg, unexpected_pkg, size_bad, hash_bad]):
    raise RuntimeError("Final Project 11 package failed readback validation.")

# Build registry row dynamically while preserving unknown global constants.
values = {
    "projectnumber": PROJECT_NUMBER, "projectno": PROJECT_NUMBER, "project": PROJECT_NAME,
    "projectname": PROJECT_NAME, "projectslug": PROJECT_SLUG, "slug": PROJECT_SLUG,
    "status": COMPLETE_STATUS, "completionstatus": COMPLETE_STATUS,
    "completedatutc": created_at, "completedat": created_at, "frozenatutc": created_at,
    "frozenat": created_at, "registeredatutc": created_at, "registeredat": created_at,
    "sourcerootsha256": SOURCE_ROOT_SHA, "rawrootsha256": RAW_ROOT_SHA,
    "rawresultrootsha256": RAW_ROOT_SHA, "rawroot": str(RAW_ROOT), "rawresultroot": str(RAW_ROOT),
    "finalpackagepath": str(FINAL_ROOT), "packagepath": str(FINAL_ROOT), "finalpackageroot": str(FINAL_ROOT),
    "finalpackagerootsha256": package_root_sha, "packagerootsha256": package_root_sha,
    "packagesha256": package_root_sha, "packagefiles": package_files, "packagefilecount": package_files,
    "packagebytes": package_bytes, "step5acheckpointsha256": step5a_sha, "step5bcheckpointsha256": step5b_sha,
}
for key, val in COUNTS.items():
    values[norm(key)] = val
values.update({
    "rawfilecount": COUNTS["RawFiles"], "conditioncount": COUNTS["Conditions"],
    "modelfits": COUNTS["MLFits"], "modelreadyrows": COUNTS["ModelRows"],
    "predictorcount": COUNTS["Predictors"], "recfeaturecount": COUNTS["RECFeatures"],
    "rawexecutionrows": COUNTS["RawRows"],
})

new_row, unresolved = {}, []
for col in reg_before.columns:
    n = norm(col)
    if n in values:
        new_row[col] = str(values[n])
    else:
        unique_nonempty = sorted(set(v for v in reg_before[col].astype(str).str.strip() if v))
        if len(unique_nonempty) == 1:
            new_row[col] = unique_nonempty[0]  # preserve a global protocol constant
        elif reg_before[col].astype(str).str.strip().eq("").all():
            new_row[col] = ""
        else:
            new_row[col] = ""; unresolved.append(col)
new_row[pn_col], new_row[project_col], new_row[status_col] = str(PROJECT_NUMBER), PROJECT_NAME, COMPLETE_STATUS
if unresolved:
    raise RuntimeError("Unresolved variable registry columns; no registry write attempted:\n" + "\n".join(unresolved))

reg_candidate = pd.concat([reg_before, pd.DataFrame([new_row])], ignore_index=True)
reg_candidate[pn_col] = pd.to_numeric(reg_candidate[pn_col], errors="raise").astype(int).astype(str)
candidate_nums = pd.to_numeric(reg_candidate[pn_col], errors="raise").astype(int)
project11_candidate = reg_candidate.loc[candidate_nums.eq(11)]
if len(reg_candidate) != 11 or sorted(candidate_nums.tolist()) != list(range(1, 12)):
    raise RuntimeError("Candidate registry does not contain exactly Projects 1–11.")
if not reg_candidate[status_col].eq(COMPLETE_STATUS).all() or len(project11_candidate) != 1:
    raise RuntimeError("Candidate registry failed status/identity validation.")

rows = []
check(rows, "Step 5A checkpoint SHA-256", STEP5A_SHA_EXPECTED, step5a_sha, step5a_sha == STEP5A_SHA_EXPECTED)
check(rows, "Step 5B checkpoint SHA-256", STEP5B_SHA_EXPECTED, step5b_sha, step5b_sha == STEP5B_SHA_EXPECTED)
check(rows, "Step 5A manifest failures", 0, int((~step5a_audit["Pass"]).sum()), step5a_audit["Pass"].all())
check(rows, "Step 5B manifest failures", 0, int((~step5b_audit["Pass"]).sum()), step5b_audit["Pass"].all())
check(rows, "Package missing files", 0, missing_pkg, missing_pkg == 0)
check(rows, "Package unexpected files", 0, unexpected_pkg, unexpected_pkg == 0)
check(rows, "Package size mismatches", 0, size_bad, size_bad == 0)
check(rows, "Package SHA-256 mismatches", 0, hash_bad, hash_bad == 0)
check(rows, "Registry rows before", 10, len(reg_before), len(reg_before) == 10)
check(rows, "Registry rows candidate", 11, len(reg_candidate), len(reg_candidate) == 11)
check(rows, "Candidate Project 11 rows", 1, len(project11_candidate), len(project11_candidate) == 1)
check(rows, "Unresolved variable registry columns", 0, len(unresolved), len(unresolved) == 0)
pre = pd.DataFrame(rows)
print("\nProject 11 Step 5C pre-write validation:"); display(pre)
print("\nProject 11 registry row candidate:"); display(project11_candidate)
if not pre["Pass"].all():
    raise RuntimeError("PROJECT 11 STEP 5C PRE-WRITE VALIDATION FAILED. Registry not modified.")

# Atomic registry write only after all package checks pass.
tmp_reg = REGISTRY.with_name(f".{REGISTRY.name}.project11_{os.getpid()}")
reg_candidate.to_csv(tmp_reg, index=False, lineterminator="\n")
tmp_read = pd.read_csv(tmp_reg, dtype=str).fillna("")
tmp_nums = pd.to_numeric(tmp_read[pn_col], errors="raise").astype(int)
if len(tmp_read) != 11 or sorted(tmp_nums.tolist()) != list(range(1, 12)) or not tmp_read[status_col].eq(COMPLETE_STATUS).all():
    tmp_reg.unlink(missing_ok=True); raise RuntimeError("Temporary registry failed readback; live registry unchanged.")
os.replace(tmp_reg, REGISTRY)

reg_after = pd.read_csv(REGISTRY, dtype=str).fillna("")
after_nums = pd.to_numeric(reg_after[pn_col], errors="raise").astype(int)
project11_after = reg_after.loc[after_nums.eq(11)]
registry_sha_after = sha(REGISTRY)
if len(reg_after) != 11 or sorted(after_nums.tolist()) != list(range(1, 12)) or not reg_after[status_col].eq(COMPLETE_STATUS).all() or len(project11_after) != 1:
    raise RuntimeError(f"Live registry failed post-write validation. Backup: {BACKUP_PATH}")

check(rows, "Registry rows after", 11, len(reg_after), len(reg_after) == 11)
check(rows, "COMPLETE_AND_FROZEN projects after", 11, int(reg_after[status_col].eq(COMPLETE_STATUS).sum()), reg_after[status_col].eq(COMPLETE_STATUS).sum() == 11)
check(rows, "Registry Project 11 rows after", 1, len(project11_after), len(project11_after) == 1)
check(rows, "Registry SHA changed", True, registry_sha_after != registry_sha_before, registry_sha_after != registry_sha_before)
validation = pd.DataFrame(rows)
failed = validation.loc[~validation["Pass"]]
if not failed.empty:
    display(failed); raise RuntimeError("PROJECT 11 STEP 5C POST-WRITE VALIDATION FAILED.")

STEP5C_ROOT.mkdir(parents=True, exist_ok=True)
atomic_csv(VALIDATION_PATH, validation)
report = {
    "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS, "CompletedAtUTC": created_at, "SourceRootSHA256": SOURCE_ROOT_SHA,
    "RawRootSHA256": RAW_ROOT_SHA, **COUNTS, "FinalPackageRoot": str(FINAL_ROOT),
    "FinalPackageFiles": package_files, "FinalPackageBytes": package_bytes,
    "FinalPackageRootSHA256": package_root_sha, "PackageAlreadyFrozenBeforeThisCell": package_already_frozen,
    "PackageMissingFiles": missing_pkg, "PackageUnexpectedFiles": unexpected_pkg,
    "PackageSizeMismatches": size_bad, "PackageSHA256Mismatches": hash_bad,
    "RegistrySHA256Before": registry_sha_before, "RegistrySHA256After": registry_sha_after,
    "RegistryRowsBefore": len(reg_before), "RegistryRowsAfter": len(reg_after),
    "Project11RegistryRowsAfter": len(project11_after), "RegistryBackup": str(BACKUP_PATH),
    "Step5ACheckpointSHA256": step5a_sha, "Step5BCheckpointSHA256": step5b_sha,
    "ValidationChecks": len(validation), "FailedValidationChecks": len(failed),
    "ConditionsRerun": False, "ModelsFitted": False, "RawResultsModified": False,
    "Project12Accessed": False, "Project12WriteAttempted": False,
}
atomic_json(REPORT_PATH, report)
atomic_json(CHECKPOINT_PATH, {**report, "CheckpointVersion": 1, "FinalPackageFrozen": True,
                              "CompletionRegistryUpdated": True, "ProjectCompleteAndFrozen": True})
step5c_sha = sha(CHECKPOINT_PATH)
atomic_json(STATUS_PATH, {
    "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS, "CompletedAtUTC": created_at, "FinalPackageRoot": str(FINAL_ROOT),
    "FinalPackageRootSHA256": package_root_sha, "RegistryRows": len(reg_after),
    "RegistrySHA256": registry_sha_after, "Checkpoint": str(CHECKPOINT_PATH),
    "CheckpointSHA256": step5c_sha, "ProjectCompleteAndFrozen": True, "Project12Accessed": False,
})
if load_json(CHECKPOINT_PATH).get("Status") != STEP5C_STATUS or load_json(STATUS_PATH).get("Status") != STEP5C_STATUS:
    raise RuntimeError("Step 5C checkpoint/status readback failed.")
if sha(REGISTRY) != registry_sha_after or root_hash(pd.read_csv(MANIFEST_PATH)) != package_root_sha:
    raise RuntimeError("Registry or final package changed after finalisation.")

print("\n" + "=" * 136)
print("=== PROJECT 11 CELL 11 / STEP 5C RESULT ===")
print("=" * 136)
print("Project number:", PROJECT_NUMBER)
print("Project:", PROJECT_NAME)
print("Project slug:", PROJECT_SLUG)
print("\nRaw result freeze:")
print("Conditions:", COUNTS["Conditions"])
print("ML fits:", COUNTS["MLFits"])
print("Raw files:", COUNTS["RawFiles"])
print("Raw bytes:", COUNTS["RawBytes"])
print("Raw root SHA-256:", RAW_ROOT_SHA)
print("\nFinal package freeze:")
print("Package root:", FINAL_ROOT)
print("Package files:", package_files)
print("Package bytes:", package_bytes)
print("Missing package files:", missing_pkg)
print("Unexpected package files:", unexpected_pkg)
print("Package size mismatches:", size_bad)
print("Package SHA-256 mismatches:", hash_bad)
print("Final package root SHA-256:", package_root_sha)
print("\nCompletion registry:")
print("Registry rows:", len(reg_after))
print("COMPLETE_AND_FROZEN projects:", int(reg_after[status_col].eq(COMPLETE_STATUS).sum()))
print("Project 11 registry rows:", len(project11_after))
print("Registry SHA-256 before:", registry_sha_before)
print("Registry SHA-256 after:", registry_sha_after)
print("Package already frozen before this cell:", package_already_frozen)
print("\nIsolation:")
print("Conditions rerun:", False)
print("Models fitted:", False)
print("Raw results modified:", False)
print("Project 12 accessed:", False)
print("Project 12 write attempted:", False)
print("\nValidation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed))
print("\nProject 11 Step 5C checkpoint:")
print(CHECKPOINT_PATH)
print("Checkpoint SHA-256:", step5c_sha)
print("\nSTATUS:", STEP5C_STATUS)
print("=" * 136)

=== PROJECT 11 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


OSError: [Errno 18] Invalid cross-device link: '/content/project11_package_bna81229' -> '/content/drive/MyDrive/Thesis_Experiment/Results/Final/apache__shardingsphere'

In [6]:
# ==================================================================================================
# PROJECT 11 — CELL 11 / STEP 5C V2
# CROSS-FILESYSTEM-SAFE FINAL PACKAGE FREEZE AND COMPLETION-REGISTRY REGISTRATION
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import hashlib, json, os, re, shutil, tempfile
import pandas as pd

print("=" * 136)
print("=== PROJECT 11 CELL 11 / STEP 5C V2: CROSS-FILESYSTEM-SAFE PACKAGE FREEZE AND REGISTRATION ===")
print("=" * 136)

# Frozen identity and hashes.
PROJECT_NUMBER = 11
PROJECT_NAME = "apache@shardingsphere"
PROJECT_SLUG = "apache__shardingsphere"
PROJECT_SHORT = "shardingsphere"
COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
STEP5C_STATUS = "PASS_PROJECT_11_FINAL_PACKAGE_FROZEN_AND_REGISTERED"
STEP5A_STATUS = "PASS_PROJECT_11_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
STEP5B_STATUS = "PASS_PROJECT_11_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN"
REGISTRY_SHA_BEFORE_EXPECTED = "847bdcbee16c8757fde34e9489cd3abbf0781a319644175472597265b1df2750"
STEP5A_SHA_EXPECTED = "4b0b53c91c25d2f3900a5c9df2b313c495545ea65036aeba39741a88770b5862"
STEP5B_SHA_EXPECTED = "cfbcbf39431c8eaa2196fb069b2ee0f203703379696a2cdc52c44902383d73ee"
SOURCE_ROOT_SHA = "3e6e0a60d58bff61690d6e2ac2bac18371414819eff601f3ea347ecb8ad8b0b5"
RAW_ROOT_SHA = "6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e88642a299343f2a73a1"

COUNTS = {
    "RawFiles": 2160, "RawBytes": 386769592, "Conditions": 270, "MLFits": 1080,
    "RankingRows": 24583230, "BuildMetricRows": 47250, "ProjectRunRows": 1890,
    "ConditionAuditRows": 270, "TrainingMedianRows": 40770, "Builds": 1049,
    "TrainingBuilds": 786, "EvaluationBuilds": 263, "RawRows": 833541,
    "RawTrainingRows": 609619, "RawEvaluationRows": 223922,
    "RawTrainingFailures": 1193, "RawEvaluationFailures": 171, "ModelRows": 91042,
    "ModelTrainingRows": 78035, "ModelEvaluationRows": 13007,
    "ModelTrainingFailures": 1188, "ModelEvaluationFailures": 171,
    "ModelFailingEvaluationBuilds": 25, "Predictors": 151, "RECFeatures": 19,
}

drive.mount("/content/drive", force_remount=False)
ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"
REGISTRY = NOTES / "completed_project_registry.csv"
PROJECT_ROOT = RESULTS / "Aggregated" / PROJECT_SLUG
RAW_ROOT = RESULTS / "Raw" / PROJECT_SLUG
FINAL_ROOT = RESULTS / "Final" / PROJECT_SLUG
MANIFEST_PATH = FINAL_ROOT / "final_package_manifest.csv"
SUMMARY_PATH = FINAL_ROOT / "final_package_summary.json"
README_PATH = FINAL_ROOT / "README.txt"
STEP5C_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_step5c"
VALIDATION_PATH = STEP5C_ROOT / f"{PROJECT_SHORT}_step5c_validation.csv"
REPORT_PATH = STEP5C_ROOT / f"{PROJECT_SHORT}_step5c_report.json"
STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5c_status.json"
CHECKPOINT_PATH = NOTES / "project_11_step5c_checkpoint.json"
BACKUP_PATH = NOTES / "completed_project_registry_before_project_11.csv"
STEP5A_CP = NOTES / "project_11_step5a_checkpoint.json"
STEP5B_CP = NOTES / "project_11_step5b_checkpoint.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5b_status.json"
UPSTREAM_CPS = [
    NOTES / "project_11_selection_checkpoint.json",
    NOTES / "project_11_rec_reconstruction_checkpoint.json",
    NOTES / "project_11_noise_plan_checkpoint.json",
    NOTES / "project_11_runtime_contract_checkpoint.json",
    NOTES / "project_11_smoke_test_checkpoint.json",
    STEP5A_CP, STEP5B_CP,
]


def sha(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, ensure_ascii=False, default=str); f.write("\n")
    os.replace(tmp, path)


def atomic_csv(path, df):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    df.to_csv(tmp, index=False, lineterminator="\n")
    os.replace(tmp, path)


def atomic_text(path, text):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    tmp.write_text(text, encoding="utf-8"); os.replace(tmp, path)


def resolve(cols, expected):
    matches = [c for c in cols if str(c).strip().lower() == expected.lower()]
    if len(matches) != 1:
        raise RuntimeError(f"Could not resolve registry column {expected!r}; matches={matches}; columns={list(cols)}")
    return matches[0]


def norm(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


def manifest(root, exclude=()):
    root = Path(root); exclude = set(exclude); rows = []
    for p in sorted((x for x in root.rglob("*") if x.is_file()), key=lambda x: x.relative_to(root).as_posix()):
        rel = p.relative_to(root).as_posix()
        if rel not in exclude:
            rows.append({"RelativePath": rel, "Bytes": int(p.stat().st_size), "SHA256": sha(p)})
    return pd.DataFrame(rows, columns=["RelativePath", "Bytes", "SHA256"])


def root_hash(df):
    h = hashlib.sha256()
    for r in df.sort_values("RelativePath", kind="mergesort").itertuples(index=False):
        h.update(f"{r.RelativePath}\0{int(r.Bytes)}\0{str(r.SHA256).lower()}\n".encode())
    return h.hexdigest()


def verify_manifest(items, label):
    if not isinstance(items, list) or not items:
        raise RuntimeError(f"{label} has no output manifest.")
    rows = []
    for item in items:
        p = Path(item["Path"]); exists = p.is_file()
        eb, es = int(item["Bytes"]), str(item["SHA256"]).lower()
        ab, ac = (int(p.stat().st_size), sha(p)) if exists else (-1, "MISSING")
        rows.append({"Path": str(p), "ExpectedBytes": eb, "ActualBytes": ab,
                     "ExpectedSHA256": es, "ActualSHA256": ac,
                     "Pass": bool(exists and eb == ab and es == ac)})
    out = pd.DataFrame(rows)
    if not out["Pass"].all():
        display(out.loc[~out["Pass"]]); raise RuntimeError(f"{label} manifest verification failed.")
    return out


def check(rows, name, expected, actual, passed):
    rows.append({"Check": name, "Expected": expected, "Actual": actual, "Pass": bool(passed)})


# Verify all inputs before any package or registry write.
required = [REGISTRY, RAW_ROOT, STEP5A_CP, STEP5B_CP, STEP5A_STATUS_PATH, STEP5B_STATUS_PATH, *UPSTREAM_CPS]
missing = [str(p) for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing Project 11 Step 5C inputs:\n" + "\n".join(missing))

step5a_sha, step5b_sha = sha(STEP5A_CP), sha(STEP5B_CP)
if step5a_sha != STEP5A_SHA_EXPECTED:
    raise RuntimeError(f"Step 5A checkpoint SHA differs: {step5a_sha}")
if step5b_sha != STEP5B_SHA_EXPECTED:
    raise RuntimeError(f"Step 5B checkpoint SHA differs: {step5b_sha}")
step5a, step5b = load_json(STEP5A_CP), load_json(STEP5B_CP)
for label, payload, expected in [
    ("Step 5A checkpoint", step5a, STEP5A_STATUS),
    ("Step 5A status", load_json(STEP5A_STATUS_PATH), STEP5A_STATUS),
    ("Step 5B checkpoint", step5b, STEP5B_STATUS),
    ("Step 5B status", load_json(STEP5B_STATUS_PATH), STEP5B_STATUS),
]:
    if payload.get("Status") != expected:
        raise RuntimeError(f"{label} is not in expected PASS state.")
if step5a.get("SourceRootSHA256") != SOURCE_ROOT_SHA or step5a.get("RawRootSHA256") != RAW_ROOT_SHA:
    raise RuntimeError("Step 5A source/raw root differs.")
if step5b.get("IndependentRawRootSHA256") != RAW_ROOT_SHA:
    raise RuntimeError("Step 5B independent raw root differs.")
step5a_audit = verify_manifest(step5a.get("AggregateOutputManifest", []), "Step 5A aggregate")
step5b_audit = verify_manifest(step5b.get("OutputManifest", []), "Step 5B")

# Registry must still be exactly Projects 1–10.
registry_sha_before = sha(REGISTRY)
if registry_sha_before != REGISTRY_SHA_BEFORE_EXPECTED:
    raise RuntimeError(f"Registry SHA differs before Project 11 registration: {registry_sha_before}")
reg_before = pd.read_csv(REGISTRY, dtype=str).fillna("")
pn_col, project_col, status_col = resolve(reg_before.columns, "ProjectNumber"), resolve(reg_before.columns, "Project"), resolve(reg_before.columns, "Status")
pnums = pd.to_numeric(reg_before[pn_col], errors="raise").astype(int)
if len(reg_before) != 10 or sorted(pnums.tolist()) != list(range(1, 11)):
    raise RuntimeError("Registry must contain exactly Projects 1–10.")
if not reg_before[status_col].eq(COMPLETE_STATUS).all() or pnums.eq(11).any() or reg_before[project_col].eq(PROJECT_NAME).any():
    raise RuntimeError("Registry state is invalid for Project 11 registration.")
if not BACKUP_PATH.exists():
    shutil.copy2(REGISTRY, BACKUP_PATH)
if sha(BACKUP_PATH) != registry_sha_before:
    raise RuntimeError("Pre-Project-11 registry backup does not match live registry.")

# Assemble compact package from all upstream checkpoints plus Step 5A/5B frozen manifest outputs.
sources = set(Path(p) for p in UPSTREAM_CPS)
sources.update([STEP5A_STATUS_PATH, STEP5B_STATUS_PATH])
sources.update(Path(item["Path"]) for item in step5a.get("AggregateOutputManifest", []))
sources.update(Path(item["Path"]) for item in step5b.get("OutputManifest", []))
sources = sorted(sources, key=str)
missing_sources = [str(p) for p in sources if not p.is_file()]
if missing_sources:
    raise FileNotFoundError("Missing compact package sources:\n" + "\n".join(missing_sources))

# Use the frozen Step 5B completion timestamp so the package is deterministic
# across safe reruns.
created_at = str(
    step5b.get(
        "CompletedAtUTC",
        ""
    )
).strip()

if not created_at:
    raise RuntimeError(
        "The frozen Step 5B checkpoint contains no CompletedAtUTC timestamp."
    )

tmp_root = Path(
    tempfile.mkdtemp(
        prefix="project11_package_",
        dir="/content",
    )
)

drive_staging_root = None

try:
    for src in sources:
        try:
            rel = src.relative_to(ROOT)
        except ValueError as exc:
            raise RuntimeError(f"Package source is outside thesis root: {src}") from exc
        dst = tmp_root / rel; dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst)
    atomic_text(tmp_root / "README.txt", f"""PROJECT 11 FINAL COMPACT PACKAGE

Project number: {PROJECT_NUMBER}
Project: {PROJECT_NAME}
Project slug: {PROJECT_SLUG}
Status: {COMPLETE_STATUS}
Created at UTC: {created_at}

The raw 2,160 condition files are not duplicated here.
Raw results: {RAW_ROOT}
Raw-root SHA-256: {RAW_ROOT_SHA}
Primary metric: APFDc
Secondary metric: APFD
""")
    before_summary = manifest(tmp_root)
    atomic_json(tmp_root / "final_package_summary.json", {
        "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
        "Status": COMPLETE_STATUS, "CreatedAtUTC": created_at, "SourceRootSHA256": SOURCE_ROOT_SHA,
        "RawRootSHA256": RAW_ROOT_SHA, **COUNTS, "Step5ACheckpointSHA256": step5a_sha,
        "Step5BCheckpointSHA256": step5b_sha, "PayloadRootSHA256BeforeSummary": root_hash(before_summary),
        "RawResultsDuplicatedIntoPackage": False,
    })
    candidate_manifest = manifest(tmp_root, {"final_package_manifest.csv"})
    package_root_sha = root_hash(candidate_manifest)
    atomic_csv(tmp_root / "final_package_manifest.csv", candidate_manifest)
    package_files = len(candidate_manifest) + 1
    package_bytes = int(candidate_manifest["Bytes"].sum() + (tmp_root / "final_package_manifest.csv").stat().st_size)
    if FINAL_ROOT.exists():
        if not MANIFEST_PATH.is_file():
            raise RuntimeError(
                "An existing Project 11 final-package directory has no manifest "
                "and was not modified."
            )

        existing = pd.read_csv(
            MANIFEST_PATH,
            low_memory=False,
        )

        if root_hash(
            existing
        ) != package_root_sha:
            raise RuntimeError(
                "A different Project 11 final package already exists and "
                "was not modified."
            )

        shutil.rmtree(
            tmp_root,
        )

        package_already_frozen = True

    else:
        # /content and Google Drive are different filesystems. A direct
        # os.replace(tmp_root, FINAL_ROOT) therefore raises EXDEV. Publish in
        # two stages:
        #   1. copy the completed local package to a sibling staging directory
        #      on Google Drive;
        #   2. verify every staged payload file;
        #   3. rename the staging directory to FINAL_ROOT within Google Drive.
        FINAL_ROOT.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        drive_staging_root = FINAL_ROOT.with_name(
            f"{FINAL_ROOT.name}__staging_{os.getpid()}"
        )

        if drive_staging_root.exists():
            shutil.rmtree(
                drive_staging_root,
            )

        shutil.copytree(
            tmp_root,
            drive_staging_root,
            copy_function=shutil.copy2,
        )

        staged_manifest_path = (
            drive_staging_root
            / "final_package_manifest.csv"
        )

        if not staged_manifest_path.is_file():
            raise RuntimeError(
                "The Google Drive staging package has no manifest."
            )

        staged_manifest = pd.read_csv(
            staged_manifest_path,
            low_memory=False,
        )

        staged_root_sha = root_hash(
            staged_manifest
        )

        if staged_root_sha != package_root_sha:
            raise RuntimeError(
                "The Google Drive staging package root SHA-256 differs.\n"
                f"Expected: {package_root_sha}\n"
                f"Actual:   {staged_root_sha}"
            )

        staged_payload_manifest = manifest(
            drive_staging_root,
            {
                "final_package_manifest.csv",
            },
        )

        candidate_payload_manifest = (
            candidate_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        staged_payload_manifest = (
            staged_payload_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        if not staged_payload_manifest.equals(
            candidate_payload_manifest
        ):
            comparison = candidate_payload_manifest.merge(
                staged_payload_manifest,
                on="RelativePath",
                how="outer",
                suffixes=(
                    "_candidate",
                    "_staged",
                ),
                indicator=True,
            )

            raise RuntimeError(
                "The Google Drive staging package failed exact file-level "
                "verification.\n"
                + comparison.loc[
                    (
                        comparison["_merge"].ne(
                            "both"
                        )
                        | comparison[
                            "Bytes_candidate"
                        ].ne(
                            comparison[
                                "Bytes_staged"
                            ]
                        )
                        | comparison[
                            "SHA256_candidate"
                        ].ne(
                            comparison[
                                "SHA256_staged"
                            ]
                        )
                    )
                ].head(
                    20
                ).to_string(
                    index=False
                )
            )

        # This rename is within the Google Drive filesystem, so it does not
        # cross a device boundary.
        os.replace(
            drive_staging_root,
            FINAL_ROOT,
        )

        drive_staging_root = None

        shutil.rmtree(
            tmp_root,
        )

        package_already_frozen = False

except Exception:
    if tmp_root.exists():
        shutil.rmtree(
            tmp_root,
            ignore_errors=True,
        )

    if (
        drive_staging_root is not None
        and drive_staging_root.exists()
    ):
        shutil.rmtree(
            drive_staging_root,
            ignore_errors=True,
        )

    raise

# Read back every packaged file.
pkg_manifest = pd.read_csv(MANIFEST_PATH, low_memory=False)
manifest_paths = set(pkg_manifest["RelativePath"].astype(str))
actual_paths = {p.relative_to(FINAL_ROOT).as_posix() for p in FINAL_ROOT.rglob("*") if p.is_file()}
expected_paths = manifest_paths | {"final_package_manifest.csv"}
missing_pkg, unexpected_pkg, size_bad, hash_bad = len(expected_paths - actual_paths), len(actual_paths - expected_paths), 0, 0
for r in pkg_manifest.itertuples(index=False):
    p = FINAL_ROOT / str(r.RelativePath)
    if p.is_file():
        size_bad += int(p.stat().st_size != int(r.Bytes)); hash_bad += int(sha(p) != str(r.SHA256))
package_root_readback = root_hash(pkg_manifest)
if package_root_readback != package_root_sha or any([missing_pkg, unexpected_pkg, size_bad, hash_bad]):
    raise RuntimeError("Final Project 11 package failed readback validation.")

# Build registry row dynamically while preserving unknown global constants.
values = {
    "projectnumber": PROJECT_NUMBER, "projectno": PROJECT_NUMBER, "project": PROJECT_NAME,
    "projectname": PROJECT_NAME, "projectslug": PROJECT_SLUG, "slug": PROJECT_SLUG,
    "status": COMPLETE_STATUS, "completionstatus": COMPLETE_STATUS,
    "completedatutc": created_at, "completedat": created_at, "frozenatutc": created_at,
    "frozenat": created_at, "registeredatutc": created_at, "registeredat": created_at,
    "sourcerootsha256": SOURCE_ROOT_SHA, "rawrootsha256": RAW_ROOT_SHA,
    "rawresultrootsha256": RAW_ROOT_SHA, "rawroot": str(RAW_ROOT), "rawresultroot": str(RAW_ROOT),
    "finalpackagepath": str(FINAL_ROOT), "packagepath": str(FINAL_ROOT), "finalpackageroot": str(FINAL_ROOT),
    "finalpackagerootsha256": package_root_sha, "packagerootsha256": package_root_sha,
    "packagesha256": package_root_sha, "packagefiles": package_files, "packagefilecount": package_files,
    "packagebytes": package_bytes, "step5acheckpointsha256": step5a_sha, "step5bcheckpointsha256": step5b_sha,
}
for key, val in COUNTS.items():
    values[norm(key)] = val
values.update({
    "rawfilecount": COUNTS["RawFiles"], "conditioncount": COUNTS["Conditions"],
    "modelfits": COUNTS["MLFits"], "modelreadyrows": COUNTS["ModelRows"],
    "predictorcount": COUNTS["Predictors"], "recfeaturecount": COUNTS["RECFeatures"],
    "rawexecutionrows": COUNTS["RawRows"],
})

new_row, unresolved = {}, []
for col in reg_before.columns:
    n = norm(col)
    if n in values:
        new_row[col] = str(values[n])
    else:
        unique_nonempty = sorted(set(v for v in reg_before[col].astype(str).str.strip() if v))
        if len(unique_nonempty) == 1:
            new_row[col] = unique_nonempty[0]  # preserve a global protocol constant
        elif reg_before[col].astype(str).str.strip().eq("").all():
            new_row[col] = ""
        else:
            new_row[col] = ""; unresolved.append(col)
new_row[pn_col], new_row[project_col], new_row[status_col] = str(PROJECT_NUMBER), PROJECT_NAME, COMPLETE_STATUS
if unresolved:
    raise RuntimeError("Unresolved variable registry columns; no registry write attempted:\n" + "\n".join(unresolved))

reg_candidate = pd.concat([reg_before, pd.DataFrame([new_row])], ignore_index=True)
reg_candidate[pn_col] = pd.to_numeric(reg_candidate[pn_col], errors="raise").astype(int).astype(str)
candidate_nums = pd.to_numeric(reg_candidate[pn_col], errors="raise").astype(int)
project11_candidate = reg_candidate.loc[candidate_nums.eq(11)]
if len(reg_candidate) != 11 or sorted(candidate_nums.tolist()) != list(range(1, 12)):
    raise RuntimeError("Candidate registry does not contain exactly Projects 1–11.")
if not reg_candidate[status_col].eq(COMPLETE_STATUS).all() or len(project11_candidate) != 1:
    raise RuntimeError("Candidate registry failed status/identity validation.")

rows = []
check(rows, "Step 5A checkpoint SHA-256", STEP5A_SHA_EXPECTED, step5a_sha, step5a_sha == STEP5A_SHA_EXPECTED)
check(rows, "Step 5B checkpoint SHA-256", STEP5B_SHA_EXPECTED, step5b_sha, step5b_sha == STEP5B_SHA_EXPECTED)
check(rows, "Step 5A manifest failures", 0, int((~step5a_audit["Pass"]).sum()), step5a_audit["Pass"].all())
check(rows, "Step 5B manifest failures", 0, int((~step5b_audit["Pass"]).sum()), step5b_audit["Pass"].all())
check(rows, "Package missing files", 0, missing_pkg, missing_pkg == 0)
check(rows, "Package unexpected files", 0, unexpected_pkg, unexpected_pkg == 0)
check(rows, "Package size mismatches", 0, size_bad, size_bad == 0)
check(rows, "Package SHA-256 mismatches", 0, hash_bad, hash_bad == 0)
check(rows, "Registry rows before", 10, len(reg_before), len(reg_before) == 10)
check(rows, "Registry rows candidate", 11, len(reg_candidate), len(reg_candidate) == 11)
check(rows, "Candidate Project 11 rows", 1, len(project11_candidate), len(project11_candidate) == 1)
check(rows, "Unresolved variable registry columns", 0, len(unresolved), len(unresolved) == 0)
pre = pd.DataFrame(rows)
print("\nProject 11 Step 5C pre-write validation:"); display(pre)
print("\nProject 11 registry row candidate:"); display(project11_candidate)
if not pre["Pass"].all():
    raise RuntimeError("PROJECT 11 STEP 5C PRE-WRITE VALIDATION FAILED. Registry not modified.")

# Atomic registry write only after all package checks pass.
tmp_reg = REGISTRY.with_name(f".{REGISTRY.name}.project11_{os.getpid()}")
reg_candidate.to_csv(tmp_reg, index=False, lineterminator="\n")
tmp_read = pd.read_csv(tmp_reg, dtype=str).fillna("")
tmp_nums = pd.to_numeric(tmp_read[pn_col], errors="raise").astype(int)
if len(tmp_read) != 11 or sorted(tmp_nums.tolist()) != list(range(1, 12)) or not tmp_read[status_col].eq(COMPLETE_STATUS).all():
    tmp_reg.unlink(missing_ok=True); raise RuntimeError("Temporary registry failed readback; live registry unchanged.")
os.replace(tmp_reg, REGISTRY)

reg_after = pd.read_csv(REGISTRY, dtype=str).fillna("")
after_nums = pd.to_numeric(reg_after[pn_col], errors="raise").astype(int)
project11_after = reg_after.loc[after_nums.eq(11)]
registry_sha_after = sha(REGISTRY)
if len(reg_after) != 11 or sorted(after_nums.tolist()) != list(range(1, 12)) or not reg_after[status_col].eq(COMPLETE_STATUS).all() or len(project11_after) != 1:
    raise RuntimeError(f"Live registry failed post-write validation. Backup: {BACKUP_PATH}")

check(rows, "Registry rows after", 11, len(reg_after), len(reg_after) == 11)
check(rows, "COMPLETE_AND_FROZEN projects after", 11, int(reg_after[status_col].eq(COMPLETE_STATUS).sum()), reg_after[status_col].eq(COMPLETE_STATUS).sum() == 11)
check(rows, "Registry Project 11 rows after", 1, len(project11_after), len(project11_after) == 1)
check(rows, "Registry SHA changed", True, registry_sha_after != registry_sha_before, registry_sha_after != registry_sha_before)
validation = pd.DataFrame(rows)
failed = validation.loc[~validation["Pass"]]
if not failed.empty:
    display(failed); raise RuntimeError("PROJECT 11 STEP 5C POST-WRITE VALIDATION FAILED.")

STEP5C_ROOT.mkdir(parents=True, exist_ok=True)
atomic_csv(VALIDATION_PATH, validation)
report = {
    "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS, "CompletedAtUTC": created_at, "SourceRootSHA256": SOURCE_ROOT_SHA,
    "RawRootSHA256": RAW_ROOT_SHA, **COUNTS, "FinalPackageRoot": str(FINAL_ROOT),
    "FinalPackageFiles": package_files, "FinalPackageBytes": package_bytes,
    "FinalPackageRootSHA256": package_root_sha, "PackageAlreadyFrozenBeforeThisCell": package_already_frozen,
    "PackageMissingFiles": missing_pkg, "PackageUnexpectedFiles": unexpected_pkg,
    "PackageSizeMismatches": size_bad, "PackageSHA256Mismatches": hash_bad,
    "RegistrySHA256Before": registry_sha_before, "RegistrySHA256After": registry_sha_after,
    "RegistryRowsBefore": len(reg_before), "RegistryRowsAfter": len(reg_after),
    "Project11RegistryRowsAfter": len(project11_after), "RegistryBackup": str(BACKUP_PATH),
    "Step5ACheckpointSHA256": step5a_sha, "Step5BCheckpointSHA256": step5b_sha,
    "ValidationChecks": len(validation), "FailedValidationChecks": len(failed),
    "ConditionsRerun": False, "ModelsFitted": False, "RawResultsModified": False,
    "Project12Accessed": False, "Project12WriteAttempted": False,
}
atomic_json(REPORT_PATH, report)
atomic_json(CHECKPOINT_PATH, {**report, "CheckpointVersion": 1, "FinalPackageFrozen": True,
                              "CompletionRegistryUpdated": True, "ProjectCompleteAndFrozen": True})
step5c_sha = sha(CHECKPOINT_PATH)
atomic_json(STATUS_PATH, {
    "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS, "CompletedAtUTC": created_at, "FinalPackageRoot": str(FINAL_ROOT),
    "FinalPackageRootSHA256": package_root_sha, "RegistryRows": len(reg_after),
    "RegistrySHA256": registry_sha_after, "Checkpoint": str(CHECKPOINT_PATH),
    "CheckpointSHA256": step5c_sha, "ProjectCompleteAndFrozen": True, "Project12Accessed": False,
})
if load_json(CHECKPOINT_PATH).get("Status") != STEP5C_STATUS or load_json(STATUS_PATH).get("Status") != STEP5C_STATUS:
    raise RuntimeError("Step 5C checkpoint/status readback failed.")
if sha(REGISTRY) != registry_sha_after or root_hash(pd.read_csv(MANIFEST_PATH)) != package_root_sha:
    raise RuntimeError("Registry or final package changed after finalisation.")

print("\n" + "=" * 136)
print("=== PROJECT 11 CELL 11 / STEP 5C RESULT ===")
print("=" * 136)
print("Project number:", PROJECT_NUMBER)
print("Project:", PROJECT_NAME)
print("Project slug:", PROJECT_SLUG)
print("\nRaw result freeze:")
print("Conditions:", COUNTS["Conditions"])
print("ML fits:", COUNTS["MLFits"])
print("Raw files:", COUNTS["RawFiles"])
print("Raw bytes:", COUNTS["RawBytes"])
print("Raw root SHA-256:", RAW_ROOT_SHA)
print("\nFinal package freeze:")
print("Package root:", FINAL_ROOT)
print("Package files:", package_files)
print("Package bytes:", package_bytes)
print("Missing package files:", missing_pkg)
print("Unexpected package files:", unexpected_pkg)
print("Package size mismatches:", size_bad)
print("Package SHA-256 mismatches:", hash_bad)
print("Final package root SHA-256:", package_root_sha)
print("\nCompletion registry:")
print("Registry rows:", len(reg_after))
print("COMPLETE_AND_FROZEN projects:", int(reg_after[status_col].eq(COMPLETE_STATUS).sum()))
print("Project 11 registry rows:", len(project11_after))
print("Registry SHA-256 before:", registry_sha_before)
print("Registry SHA-256 after:", registry_sha_after)
print("Package already frozen before this cell:", package_already_frozen)
print("\nIsolation:")
print("Conditions rerun:", False)
print("Models fitted:", False)
print("Raw results modified:", False)
print("Project 12 accessed:", False)
print("Project 12 write attempted:", False)
print("\nValidation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed))
print("\nProject 11 Step 5C checkpoint:")
print(CHECKPOINT_PATH)
print("Checkpoint SHA-256:", step5c_sha)
print("\nSTATUS:", STEP5C_STATUS)
print("=" * 136)

=== PROJECT 11 CELL 11 / STEP 5C V2: CROSS-FILESYSTEM-SAFE PACKAGE FREEZE AND REGISTRATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


RuntimeError: Unresolved variable registry columns; no registry write attempted:
Seeds
NoiseLevels
Techniques
EvaluationRows
EvaluationFailures
FinalDirectory
FinalAuditReport
DoNotRerun
FreezeRecord
RawResultsManifest
FinalPackageManifest
RawResultsRootSHA256
FinalAuditStatus

In [7]:
# ==================================================================================================
# PROJECT 11 — CELL 11 / STEP 5C V3
# REGISTRY-SCHEMA-COMPLETE, CROSS-FILESYSTEM-SAFE FINALISATION
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import hashlib, json, os, re, shutil, tempfile
import pandas as pd

print("=" * 136)
print("=== PROJECT 11 CELL 11 / STEP 5C V3: REGISTRY-SCHEMA-COMPLETE FINALISATION ===")
print("=" * 136)

# Frozen identity and hashes.
PROJECT_NUMBER = 11
PROJECT_NAME = "apache@shardingsphere"
PROJECT_SLUG = "apache__shardingsphere"
PROJECT_SHORT = "shardingsphere"
COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
STEP5C_STATUS = "PASS_PROJECT_11_FINAL_PACKAGE_FROZEN_AND_REGISTERED"
STEP5A_STATUS = "PASS_PROJECT_11_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
STEP5B_STATUS = "PASS_PROJECT_11_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN"
REGISTRY_SHA_BEFORE_EXPECTED = "847bdcbee16c8757fde34e9489cd3abbf0781a319644175472597265b1df2750"
STEP5A_SHA_EXPECTED = "4b0b53c91c25d2f3900a5c9df2b313c495545ea65036aeba39741a88770b5862"
STEP5B_SHA_EXPECTED = "cfbcbf39431c8eaa2196fb069b2ee0f203703379696a2cdc52c44902383d73ee"
SOURCE_ROOT_SHA = "3e6e0a60d58bff61690d6e2ac2bac18371414819eff601f3ea347ecb8ad8b0b5"
RAW_ROOT_SHA = "6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e88642a299343f2a73a1"

COUNTS = {
    "RawFiles": 2160, "RawBytes": 386769592, "Conditions": 270, "MLFits": 1080,
    "RankingRows": 24583230, "BuildMetricRows": 47250, "ProjectRunRows": 1890,
    "ConditionAuditRows": 270, "TrainingMedianRows": 40770, "Builds": 1049,
    "TrainingBuilds": 786, "EvaluationBuilds": 263, "RawRows": 833541,
    "RawTrainingRows": 609619, "RawEvaluationRows": 223922,
    "RawTrainingFailures": 1193, "RawEvaluationFailures": 171, "ModelRows": 91042,
    "ModelTrainingRows": 78035, "ModelEvaluationRows": 13007,
    "ModelTrainingFailures": 1188, "ModelEvaluationFailures": 171,
    "ModelFailingEvaluationBuilds": 25, "Predictors": 151, "RECFeatures": 19,
}

drive.mount("/content/drive", force_remount=False)
ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"
REGISTRY = NOTES / "completed_project_registry.csv"
PROJECT_ROOT = RESULTS / "Aggregated" / PROJECT_SLUG
RAW_ROOT = RESULTS / "Raw" / PROJECT_SLUG
FINAL_ROOT = RESULTS / "Final" / PROJECT_SLUG
MANIFEST_PATH = FINAL_ROOT / "final_package_manifest.csv"
SUMMARY_PATH = FINAL_ROOT / "final_package_summary.json"
README_PATH = FINAL_ROOT / "README.txt"
STEP5C_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_step5c"
VALIDATION_PATH = STEP5C_ROOT / f"{PROJECT_SHORT}_step5c_validation.csv"
REPORT_PATH = STEP5C_ROOT / f"{PROJECT_SHORT}_step5c_report.json"
STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5c_status.json"
CHECKPOINT_PATH = NOTES / "project_11_step5c_checkpoint.json"
BACKUP_PATH = NOTES / "completed_project_registry_before_project_11.csv"

STEP5B_RAW_MANIFEST_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5b"
    / f"{PROJECT_SHORT}_independent_raw_manifest.csv"
)
STEP5A_CP = NOTES / "project_11_step5a_checkpoint.json"
STEP5B_CP = NOTES / "project_11_step5b_checkpoint.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5b_status.json"
UPSTREAM_CPS = [
    NOTES / "project_11_selection_checkpoint.json",
    NOTES / "project_11_rec_reconstruction_checkpoint.json",
    NOTES / "project_11_noise_plan_checkpoint.json",
    NOTES / "project_11_runtime_contract_checkpoint.json",
    NOTES / "project_11_smoke_test_checkpoint.json",
    STEP5A_CP, STEP5B_CP,
]


def sha(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, ensure_ascii=False, default=str); f.write("\n")
    os.replace(tmp, path)


def atomic_csv(path, df):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    df.to_csv(tmp, index=False, lineterminator="\n")
    os.replace(tmp, path)


def atomic_text(path, text):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    tmp.write_text(text, encoding="utf-8"); os.replace(tmp, path)


def resolve(cols, expected):
    matches = [c for c in cols if str(c).strip().lower() == expected.lower()]
    if len(matches) != 1:
        raise RuntimeError(f"Could not resolve registry column {expected!r}; matches={matches}; columns={list(cols)}")
    return matches[0]


def norm(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


def manifest(root, exclude=()):
    root = Path(root); exclude = set(exclude); rows = []
    for p in sorted((x for x in root.rglob("*") if x.is_file()), key=lambda x: x.relative_to(root).as_posix()):
        rel = p.relative_to(root).as_posix()
        if rel not in exclude:
            rows.append({"RelativePath": rel, "Bytes": int(p.stat().st_size), "SHA256": sha(p)})
    return pd.DataFrame(rows, columns=["RelativePath", "Bytes", "SHA256"])


def root_hash(df):
    h = hashlib.sha256()
    for r in df.sort_values("RelativePath", kind="mergesort").itertuples(index=False):
        h.update(f"{r.RelativePath}\0{int(r.Bytes)}\0{str(r.SHA256).lower()}\n".encode())
    return h.hexdigest()


def verify_manifest(items, label):
    if not isinstance(items, list) or not items:
        raise RuntimeError(f"{label} has no output manifest.")
    rows = []
    for item in items:
        p = Path(item["Path"]); exists = p.is_file()
        eb, es = int(item["Bytes"]), str(item["SHA256"]).lower()
        ab, ac = (int(p.stat().st_size), sha(p)) if exists else (-1, "MISSING")
        rows.append({"Path": str(p), "ExpectedBytes": eb, "ActualBytes": ab,
                     "ExpectedSHA256": es, "ActualSHA256": ac,
                     "Pass": bool(exists and eb == ab and es == ac)})
    out = pd.DataFrame(rows)
    if not out["Pass"].all():
        display(out.loc[~out["Pass"]]); raise RuntimeError(f"{label} manifest verification failed.")
    return out


def check(rows, name, expected, actual, passed):
    rows.append({"Check": name, "Expected": expected, "Actual": actual, "Pass": bool(passed)})


# Verify all inputs before any package or registry write.
required = [REGISTRY, RAW_ROOT, STEP5A_CP, STEP5B_CP, STEP5A_STATUS_PATH, STEP5B_STATUS_PATH, *UPSTREAM_CPS]
missing = [str(p) for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing Project 11 Step 5C inputs:\n" + "\n".join(missing))

step5a_sha, step5b_sha = sha(STEP5A_CP), sha(STEP5B_CP)
if step5a_sha != STEP5A_SHA_EXPECTED:
    raise RuntimeError(f"Step 5A checkpoint SHA differs: {step5a_sha}")
if step5b_sha != STEP5B_SHA_EXPECTED:
    raise RuntimeError(f"Step 5B checkpoint SHA differs: {step5b_sha}")
step5a, step5b = load_json(STEP5A_CP), load_json(STEP5B_CP)
for label, payload, expected in [
    ("Step 5A checkpoint", step5a, STEP5A_STATUS),
    ("Step 5A status", load_json(STEP5A_STATUS_PATH), STEP5A_STATUS),
    ("Step 5B checkpoint", step5b, STEP5B_STATUS),
    ("Step 5B status", load_json(STEP5B_STATUS_PATH), STEP5B_STATUS),
]:
    if payload.get("Status") != expected:
        raise RuntimeError(f"{label} is not in expected PASS state.")
if step5a.get("SourceRootSHA256") != SOURCE_ROOT_SHA or step5a.get("RawRootSHA256") != RAW_ROOT_SHA:
    raise RuntimeError("Step 5A source/raw root differs.")
if step5b.get("IndependentRawRootSHA256") != RAW_ROOT_SHA:
    raise RuntimeError("Step 5B independent raw root differs.")
step5a_audit = verify_manifest(step5a.get("AggregateOutputManifest", []), "Step 5A aggregate")
step5b_audit = verify_manifest(step5b.get("OutputManifest", []), "Step 5B")

# Registry must still be exactly Projects 1–10.
registry_sha_before = sha(REGISTRY)
if registry_sha_before != REGISTRY_SHA_BEFORE_EXPECTED:
    raise RuntimeError(f"Registry SHA differs before Project 11 registration: {registry_sha_before}")
reg_before = pd.read_csv(REGISTRY, dtype=str).fillna("")
pn_col, project_col, status_col = resolve(reg_before.columns, "ProjectNumber"), resolve(reg_before.columns, "Project"), resolve(reg_before.columns, "Status")
pnums = pd.to_numeric(reg_before[pn_col], errors="raise").astype(int)
if len(reg_before) != 10 or sorted(pnums.tolist()) != list(range(1, 11)):
    raise RuntimeError("Registry must contain exactly Projects 1–10.")
if not reg_before[status_col].eq(COMPLETE_STATUS).all() or pnums.eq(11).any() or reg_before[project_col].eq(PROJECT_NAME).any():
    raise RuntimeError("Registry state is invalid for Project 11 registration.")
if not BACKUP_PATH.exists():
    shutil.copy2(REGISTRY, BACKUP_PATH)
if sha(BACKUP_PATH) != registry_sha_before:
    raise RuntimeError("Pre-Project-11 registry backup does not match live registry.")

# Assemble compact package from all upstream checkpoints plus Step 5A/5B frozen manifest outputs.
sources = set(Path(p) for p in UPSTREAM_CPS)
sources.update([STEP5A_STATUS_PATH, STEP5B_STATUS_PATH])
sources.update(Path(item["Path"]) for item in step5a.get("AggregateOutputManifest", []))
sources.update(Path(item["Path"]) for item in step5b.get("OutputManifest", []))
sources = sorted(sources, key=str)
missing_sources = [str(p) for p in sources if not p.is_file()]
if missing_sources:
    raise FileNotFoundError("Missing compact package sources:\n" + "\n".join(missing_sources))

# Use the frozen Step 5B completion timestamp so the package is deterministic
# across safe reruns.
created_at = str(
    step5b.get(
        "CompletedAtUTC",
        ""
    )
).strip()

if not created_at:
    raise RuntimeError(
        "The frozen Step 5B checkpoint contains no CompletedAtUTC timestamp."
    )

tmp_root = Path(
    tempfile.mkdtemp(
        prefix="project11_package_",
        dir="/content",
    )
)

drive_staging_root = None

try:
    for src in sources:
        try:
            rel = src.relative_to(ROOT)
        except ValueError as exc:
            raise RuntimeError(f"Package source is outside thesis root: {src}") from exc
        dst = tmp_root / rel; dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst)
    atomic_text(tmp_root / "README.txt", f"""PROJECT 11 FINAL COMPACT PACKAGE

Project number: {PROJECT_NUMBER}
Project: {PROJECT_NAME}
Project slug: {PROJECT_SLUG}
Status: {COMPLETE_STATUS}
Created at UTC: {created_at}

The raw 2,160 condition files are not duplicated here.
Raw results: {RAW_ROOT}
Raw-root SHA-256: {RAW_ROOT_SHA}
Primary metric: APFDc
Secondary metric: APFD
""")
    before_summary = manifest(tmp_root)
    atomic_json(tmp_root / "final_package_summary.json", {
        "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
        "Status": COMPLETE_STATUS, "CreatedAtUTC": created_at, "SourceRootSHA256": SOURCE_ROOT_SHA,
        "RawRootSHA256": RAW_ROOT_SHA, **COUNTS, "Step5ACheckpointSHA256": step5a_sha,
        "Step5BCheckpointSHA256": step5b_sha, "PayloadRootSHA256BeforeSummary": root_hash(before_summary),
        "RawResultsDuplicatedIntoPackage": False,
    })
    candidate_manifest = manifest(tmp_root, {"final_package_manifest.csv"})
    package_root_sha = root_hash(candidate_manifest)
    atomic_csv(tmp_root / "final_package_manifest.csv", candidate_manifest)
    package_files = len(candidate_manifest) + 1
    package_bytes = int(candidate_manifest["Bytes"].sum() + (tmp_root / "final_package_manifest.csv").stat().st_size)
    if FINAL_ROOT.exists():
        if not MANIFEST_PATH.is_file():
            raise RuntimeError(
                "An existing Project 11 final-package directory has no manifest "
                "and was not modified."
            )

        existing = pd.read_csv(
            MANIFEST_PATH,
            low_memory=False,
        )

        if root_hash(
            existing
        ) != package_root_sha:
            raise RuntimeError(
                "A different Project 11 final package already exists and "
                "was not modified."
            )

        shutil.rmtree(
            tmp_root,
        )

        package_already_frozen = True

    else:
        # /content and Google Drive are different filesystems. A direct
        # os.replace(tmp_root, FINAL_ROOT) therefore raises EXDEV. Publish in
        # two stages:
        #   1. copy the completed local package to a sibling staging directory
        #      on Google Drive;
        #   2. verify every staged payload file;
        #   3. rename the staging directory to FINAL_ROOT within Google Drive.
        FINAL_ROOT.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        drive_staging_root = FINAL_ROOT.with_name(
            f"{FINAL_ROOT.name}__staging_{os.getpid()}"
        )

        if drive_staging_root.exists():
            shutil.rmtree(
                drive_staging_root,
            )

        shutil.copytree(
            tmp_root,
            drive_staging_root,
            copy_function=shutil.copy2,
        )

        staged_manifest_path = (
            drive_staging_root
            / "final_package_manifest.csv"
        )

        if not staged_manifest_path.is_file():
            raise RuntimeError(
                "The Google Drive staging package has no manifest."
            )

        staged_manifest = pd.read_csv(
            staged_manifest_path,
            low_memory=False,
        )

        staged_root_sha = root_hash(
            staged_manifest
        )

        if staged_root_sha != package_root_sha:
            raise RuntimeError(
                "The Google Drive staging package root SHA-256 differs.\n"
                f"Expected: {package_root_sha}\n"
                f"Actual:   {staged_root_sha}"
            )

        staged_payload_manifest = manifest(
            drive_staging_root,
            {
                "final_package_manifest.csv",
            },
        )

        candidate_payload_manifest = (
            candidate_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        staged_payload_manifest = (
            staged_payload_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        if not staged_payload_manifest.equals(
            candidate_payload_manifest
        ):
            comparison = candidate_payload_manifest.merge(
                staged_payload_manifest,
                on="RelativePath",
                how="outer",
                suffixes=(
                    "_candidate",
                    "_staged",
                ),
                indicator=True,
            )

            raise RuntimeError(
                "The Google Drive staging package failed exact file-level "
                "verification.\n"
                + comparison.loc[
                    (
                        comparison["_merge"].ne(
                            "both"
                        )
                        | comparison[
                            "Bytes_candidate"
                        ].ne(
                            comparison[
                                "Bytes_staged"
                            ]
                        )
                        | comparison[
                            "SHA256_candidate"
                        ].ne(
                            comparison[
                                "SHA256_staged"
                            ]
                        )
                    )
                ].head(
                    20
                ).to_string(
                    index=False
                )
            )

        # This rename is within the Google Drive filesystem, so it does not
        # cross a device boundary.
        os.replace(
            drive_staging_root,
            FINAL_ROOT,
        )

        drive_staging_root = None

        shutil.rmtree(
            tmp_root,
        )

        package_already_frozen = False

except Exception:
    if tmp_root.exists():
        shutil.rmtree(
            tmp_root,
            ignore_errors=True,
        )

    if (
        drive_staging_root is not None
        and drive_staging_root.exists()
    ):
        shutil.rmtree(
            drive_staging_root,
            ignore_errors=True,
        )

    raise

# Read back every packaged file.
pkg_manifest = pd.read_csv(MANIFEST_PATH, low_memory=False)
manifest_paths = set(pkg_manifest["RelativePath"].astype(str))
actual_paths = {p.relative_to(FINAL_ROOT).as_posix() for p in FINAL_ROOT.rglob("*") if p.is_file()}
expected_paths = manifest_paths | {"final_package_manifest.csv"}
missing_pkg, unexpected_pkg, size_bad, hash_bad = len(expected_paths - actual_paths), len(actual_paths - expected_paths), 0, 0
for r in pkg_manifest.itertuples(index=False):
    p = FINAL_ROOT / str(r.RelativePath)
    if p.is_file():
        size_bad += int(p.stat().st_size != int(r.Bytes)); hash_bad += int(sha(p) != str(r.SHA256))
package_root_readback = root_hash(pkg_manifest)
if package_root_readback != package_root_sha or any([missing_pkg, unexpected_pkg, size_bad, hash_bad]):
    raise RuntimeError("Final Project 11 package failed readback validation.")

# Build a complete Project 11 registry row.
#
# The registry has evolved across Projects 1–10. Some columns are protocol
# descriptors, some are project-specific counts, and some are paths to frozen
# audit artefacts. V2 deliberately stopped because it did not map every
# variable column. V3 handles the complete observed schema explicitly.
#
# For protocol fields whose textual formatting has varied historically
# (Seeds, NoiseLevels, Techniques, DoNotRerun), use the exact Project 10
# representation because Project 10 and Project 11 use the same frozen
# protocol. The actual Project 11 values are also validated below.
project_10_rows = reg_before.loc[
    pd.to_numeric(
        reg_before[
            pn_col
        ],
        errors="raise",
    ).astype(
        int
    ).eq(
        10
    )
]

if len(
    project_10_rows
) != 1:
    raise RuntimeError(
        "Could not resolve exactly one Project 10 registry template row."
    )

project_10_template = project_10_rows.iloc[
    0
]

protocol_template_values = {}

for registry_column in reg_before.columns:
    normalised_column = norm(
        registry_column
    )

    if normalised_column in {
        "seeds",
        "noiselevels",
        "techniques",
        "donotrerun",
    }:
        protocol_template_values[
            normalised_column
        ] = str(
            project_10_template[
                registry_column
            ]
        ).strip()


protocol_fallback_values = {
    "seeds":
        json.dumps(
            list(
                range(
                    1,
                    31,
                )
            ),
            separators=(
                ",",
                ":",
            ),
        ),

    "noiselevels":
        json.dumps(
            [
                0,
                5,
                10,
                15,
                20,
                25,
                30,
                40,
                50,
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "techniques":
        json.dumps(
            [
                "RandomForest",
                "XGBoost",
                "LightGBM",
                "NaiveBayes",
                "Random",
                "LatestFail",
                "QTF-Avg",
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "donotrerun":
        "True",
}


for protocol_key, fallback_value in protocol_fallback_values.items():
    if not protocol_template_values.get(
        protocol_key,
        ""
    ):
        protocol_template_values[
            protocol_key
        ] = fallback_value


values = {
    "projectnumber": PROJECT_NUMBER, "projectno": PROJECT_NUMBER, "project": PROJECT_NAME,
    "projectname": PROJECT_NAME, "projectslug": PROJECT_SLUG, "slug": PROJECT_SLUG,
    "status": COMPLETE_STATUS, "completionstatus": COMPLETE_STATUS,
    "completedatutc": created_at, "completedat": created_at, "frozenatutc": created_at,
    "frozenat": created_at, "registeredatutc": created_at, "registeredat": created_at,
    "sourcerootsha256": SOURCE_ROOT_SHA, "rawrootsha256": RAW_ROOT_SHA,
    "rawresultrootsha256": RAW_ROOT_SHA, "rawroot": str(RAW_ROOT), "rawresultroot": str(RAW_ROOT),
    "finalpackagepath": str(FINAL_ROOT), "packagepath": str(FINAL_ROOT), "finalpackageroot": str(FINAL_ROOT),
    "finalpackagerootsha256": package_root_sha, "packagerootsha256": package_root_sha,
    "packagesha256": package_root_sha, "packagefiles": package_files, "packagefilecount": package_files,
    "packagebytes": package_bytes, "step5acheckpointsha256": step5a_sha, "step5bcheckpointsha256": step5b_sha,

    # Complete observed registry schema.
    "seeds": protocol_template_values["seeds"],
    "noiselevels": protocol_template_values["noiselevels"],
    "techniques": protocol_template_values["techniques"],
    "evaluationrows": COUNTS["ModelEvaluationRows"],
    "evaluationfailures": COUNTS["ModelEvaluationFailures"],
    "finaldirectory": str(FINAL_ROOT),
    "finalauditreport": str(REPORT_PATH),
    "donotrerun": protocol_template_values["donotrerun"],
    "freezerecord": str(CHECKPOINT_PATH),
    "rawresultsmanifest": str(STEP5B_RAW_MANIFEST_PATH),
    "finalpackagemanifest": str(MANIFEST_PATH),
    "rawresultsrootsha256": RAW_ROOT_SHA,
    "finalauditstatus": STEP5C_STATUS,
}
for key, val in COUNTS.items():
    values[norm(key)] = val
values.update({
    "rawfilecount": COUNTS["RawFiles"], "conditioncount": COUNTS["Conditions"],
    "modelfits": COUNTS["MLFits"], "modelreadyrows": COUNTS["ModelRows"],
    "predictorcount": COUNTS["Predictors"], "recfeaturecount": COUNTS["RECFeatures"],
    "rawexecutionrows": COUNTS["RawRows"],
})

if not STEP5B_RAW_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "The independently frozen Step 5B raw-results manifest is missing:\n"
        f"{STEP5B_RAW_MANIFEST_PATH}"
    )


new_row, unresolved = {}, []
for col in reg_before.columns:
    n = norm(col)
    if n in values:
        new_row[col] = str(values[n])
    else:
        unique_nonempty = sorted(set(v for v in reg_before[col].astype(str).str.strip() if v))
        if len(unique_nonempty) == 1:
            new_row[col] = unique_nonempty[0]  # preserve a global protocol constant
        elif reg_before[col].astype(str).str.strip().eq("").all():
            new_row[col] = ""
        else:
            new_row[col] = ""; unresolved.append(col)
new_row[pn_col], new_row[project_col], new_row[status_col] = str(PROJECT_NUMBER), PROJECT_NAME, COMPLETE_STATUS
if unresolved:
    raise RuntimeError(
        "Unexpected unmapped registry columns remain; no registry write "
        "was attempted:\n"
        + "\n".join(
            unresolved
        )
    )

reg_candidate = pd.concat([reg_before, pd.DataFrame([new_row])], ignore_index=True)
reg_candidate[pn_col] = pd.to_numeric(reg_candidate[pn_col], errors="raise").astype(int).astype(str)
candidate_nums = pd.to_numeric(reg_candidate[pn_col], errors="raise").astype(int)
project11_candidate = reg_candidate.loc[candidate_nums.eq(11)]
if len(reg_candidate) != 11 or sorted(candidate_nums.tolist()) != list(range(1, 12)):
    raise RuntimeError("Candidate registry does not contain exactly Projects 1–11.")
if not reg_candidate[status_col].eq(COMPLETE_STATUS).all() or len(project11_candidate) != 1:
    raise RuntimeError("Candidate registry failed status/identity validation.")

rows = []
check(rows, "Step 5A checkpoint SHA-256", STEP5A_SHA_EXPECTED, step5a_sha, step5a_sha == STEP5A_SHA_EXPECTED)
check(rows, "Step 5B checkpoint SHA-256", STEP5B_SHA_EXPECTED, step5b_sha, step5b_sha == STEP5B_SHA_EXPECTED)
check(rows, "Step 5A manifest failures", 0, int((~step5a_audit["Pass"]).sum()), step5a_audit["Pass"].all())
check(rows, "Step 5B manifest failures", 0, int((~step5b_audit["Pass"]).sum()), step5b_audit["Pass"].all())
check(rows, "Package missing files", 0, missing_pkg, missing_pkg == 0)
check(rows, "Package unexpected files", 0, unexpected_pkg, unexpected_pkg == 0)
check(rows, "Package size mismatches", 0, size_bad, size_bad == 0)
check(rows, "Package SHA-256 mismatches", 0, hash_bad, hash_bad == 0)
check(rows, "Registry rows before", 10, len(reg_before), len(reg_before) == 10)
check(rows, "Registry rows candidate", 11, len(reg_candidate), len(reg_candidate) == 11)
check(rows, "Candidate Project 11 rows", 1, len(project11_candidate), len(project11_candidate) == 1)
check(rows, "Unresolved variable registry columns", 0, len(unresolved), len(unresolved) == 0)

required_registry_field_expectations = {
    "Seeds":
        protocol_template_values[
            "seeds"
        ],

    "NoiseLevels":
        protocol_template_values[
            "noiselevels"
        ],

    "Techniques":
        protocol_template_values[
            "techniques"
        ],

    "EvaluationRows":
        str(
            COUNTS[
                "ModelEvaluationRows"
            ]
        ),

    "EvaluationFailures":
        str(
            COUNTS[
                "ModelEvaluationFailures"
            ]
        ),

    "FinalDirectory":
        str(
            FINAL_ROOT
        ),

    "FinalAuditReport":
        str(
            REPORT_PATH
        ),

    "DoNotRerun":
        protocol_template_values[
            "donotrerun"
        ],

    "FreezeRecord":
        str(
            CHECKPOINT_PATH
        ),

    "RawResultsManifest":
        str(
            STEP5B_RAW_MANIFEST_PATH
        ),

    "FinalPackageManifest":
        str(
            MANIFEST_PATH
        ),

    "RawResultsRootSHA256":
        RAW_ROOT_SHA,

    "FinalAuditStatus":
        STEP5C_STATUS,
}


registry_field_validation_failures = 0

for expected_column_name, expected_value in required_registry_field_expectations.items():
    matching_columns = [
        column
        for column in reg_before.columns
        if norm(
            column
        )
        == norm(
            expected_column_name
        )
    ]

    if len(
        matching_columns
    ) != 1:
        registry_field_validation_failures += 1
        continue

    actual_value = str(
        project11_candidate.iloc[
            0
        ][
            matching_columns[
                0
            ]
        ]
    )

    registry_field_validation_failures += int(
        actual_value
        != str(
            expected_value
        )
    )


check(
    rows,
    "Explicit Project 11 registry-field failures",
    0,
    registry_field_validation_failures,
    registry_field_validation_failures
    == 0,
)

pre = pd.DataFrame(rows)
print("\nProject 11 Step 5C pre-write validation:"); display(pre)
print("\nProject 11 registry row candidate:"); display(project11_candidate)
if not pre["Pass"].all():
    raise RuntimeError("PROJECT 11 STEP 5C PRE-WRITE VALIDATION FAILED. Registry not modified.")

# Atomic registry write only after all package checks pass.
tmp_reg = REGISTRY.with_name(f".{REGISTRY.name}.project11_{os.getpid()}")
reg_candidate.to_csv(tmp_reg, index=False, lineterminator="\n")
tmp_read = pd.read_csv(tmp_reg, dtype=str).fillna("")
tmp_nums = pd.to_numeric(tmp_read[pn_col], errors="raise").astype(int)
if len(tmp_read) != 11 or sorted(tmp_nums.tolist()) != list(range(1, 12)) or not tmp_read[status_col].eq(COMPLETE_STATUS).all():
    tmp_reg.unlink(missing_ok=True); raise RuntimeError("Temporary registry failed readback; live registry unchanged.")
os.replace(tmp_reg, REGISTRY)

reg_after = pd.read_csv(REGISTRY, dtype=str).fillna("")
after_nums = pd.to_numeric(reg_after[pn_col], errors="raise").astype(int)
project11_after = reg_after.loc[after_nums.eq(11)]
registry_sha_after = sha(REGISTRY)
if len(reg_after) != 11 or sorted(after_nums.tolist()) != list(range(1, 12)) or not reg_after[status_col].eq(COMPLETE_STATUS).all() or len(project11_after) != 1:
    raise RuntimeError(f"Live registry failed post-write validation. Backup: {BACKUP_PATH}")

check(rows, "Registry rows after", 11, len(reg_after), len(reg_after) == 11)
check(rows, "COMPLETE_AND_FROZEN projects after", 11, int(reg_after[status_col].eq(COMPLETE_STATUS).sum()), reg_after[status_col].eq(COMPLETE_STATUS).sum() == 11)
check(rows, "Registry Project 11 rows after", 1, len(project11_after), len(project11_after) == 1)
check(rows, "Registry SHA changed", True, registry_sha_after != registry_sha_before, registry_sha_after != registry_sha_before)
validation = pd.DataFrame(rows)
failed = validation.loc[~validation["Pass"]]
if not failed.empty:
    display(failed); raise RuntimeError("PROJECT 11 STEP 5C POST-WRITE VALIDATION FAILED.")

STEP5C_ROOT.mkdir(parents=True, exist_ok=True)
atomic_csv(VALIDATION_PATH, validation)
report = {
    "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS, "CompletedAtUTC": created_at, "SourceRootSHA256": SOURCE_ROOT_SHA,
    "RawRootSHA256": RAW_ROOT_SHA, **COUNTS, "FinalPackageRoot": str(FINAL_ROOT),
    "FinalPackageFiles": package_files, "FinalPackageBytes": package_bytes,
    "FinalPackageRootSHA256": package_root_sha, "PackageAlreadyFrozenBeforeThisCell": package_already_frozen,
    "PackageMissingFiles": missing_pkg, "PackageUnexpectedFiles": unexpected_pkg,
    "PackageSizeMismatches": size_bad, "PackageSHA256Mismatches": hash_bad,
    "RegistrySHA256Before": registry_sha_before, "RegistrySHA256After": registry_sha_after,
    "RegistryRowsBefore": len(reg_before), "RegistryRowsAfter": len(reg_after),
    "Project11RegistryRowsAfter": len(project11_after), "RegistryBackup": str(BACKUP_PATH),
    "Step5ACheckpointSHA256": step5a_sha, "Step5BCheckpointSHA256": step5b_sha,
    "ValidationChecks": len(validation), "FailedValidationChecks": len(failed),
    "ConditionsRerun": False, "ModelsFitted": False, "RawResultsModified": False,
    "Project12Accessed": False, "Project12WriteAttempted": False,
}
atomic_json(REPORT_PATH, report)
atomic_json(CHECKPOINT_PATH, {**report, "CheckpointVersion": 1, "FinalPackageFrozen": True,
                              "CompletionRegistryUpdated": True, "ProjectCompleteAndFrozen": True})
step5c_sha = sha(CHECKPOINT_PATH)
atomic_json(STATUS_PATH, {
    "ProjectNumber": PROJECT_NUMBER, "Project": PROJECT_NAME, "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS, "CompletedAtUTC": created_at, "FinalPackageRoot": str(FINAL_ROOT),
    "FinalPackageRootSHA256": package_root_sha, "RegistryRows": len(reg_after),
    "RegistrySHA256": registry_sha_after, "Checkpoint": str(CHECKPOINT_PATH),
    "CheckpointSHA256": step5c_sha, "ProjectCompleteAndFrozen": True, "Project12Accessed": False,
})
if load_json(CHECKPOINT_PATH).get("Status") != STEP5C_STATUS or load_json(STATUS_PATH).get("Status") != STEP5C_STATUS:
    raise RuntimeError("Step 5C checkpoint/status readback failed.")
if sha(REGISTRY) != registry_sha_after or root_hash(pd.read_csv(MANIFEST_PATH)) != package_root_sha:
    raise RuntimeError("Registry or final package changed after finalisation.")

print("\n" + "=" * 136)
print("=== PROJECT 11 CELL 11 / STEP 5C RESULT ===")
print("=" * 136)
print("Project number:", PROJECT_NUMBER)
print("Project:", PROJECT_NAME)
print("Project slug:", PROJECT_SLUG)
print("\nRaw result freeze:")
print("Conditions:", COUNTS["Conditions"])
print("ML fits:", COUNTS["MLFits"])
print("Raw files:", COUNTS["RawFiles"])
print("Raw bytes:", COUNTS["RawBytes"])
print("Raw root SHA-256:", RAW_ROOT_SHA)
print("\nFinal package freeze:")
print("Package root:", FINAL_ROOT)
print("Package files:", package_files)
print("Package bytes:", package_bytes)
print("Missing package files:", missing_pkg)
print("Unexpected package files:", unexpected_pkg)
print("Package size mismatches:", size_bad)
print("Package SHA-256 mismatches:", hash_bad)
print("Final package root SHA-256:", package_root_sha)
print("\nCompletion registry:")
print("Registry rows:", len(reg_after))
print("COMPLETE_AND_FROZEN projects:", int(reg_after[status_col].eq(COMPLETE_STATUS).sum()))
print("Project 11 registry rows:", len(project11_after))
print("Registry SHA-256 before:", registry_sha_before)
print("Registry SHA-256 after:", registry_sha_after)
print("Package already frozen before this cell:", package_already_frozen)
print("\nIsolation:")
print("Conditions rerun:", False)
print("Models fitted:", False)
print("Raw results modified:", False)
print("Project 12 accessed:", False)
print("Project 12 write attempted:", False)
print("\nValidation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed))
print("\nProject 11 Step 5C checkpoint:")
print(CHECKPOINT_PATH)
print("Checkpoint SHA-256:", step5c_sha)
print(
    "Explicit registry-schema fields validated:",
    len(
        required_registry_field_expectations
    ),
)

print("\nSTATUS:", STEP5C_STATUS)
print("=" * 136)

=== PROJECT 11 CELL 11 / STEP 5C V3: REGISTRY-SCHEMA-COMPLETE FINALISATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Project 11 Step 5C pre-write validation:


,Check,Expected,Actual,Pass
0,Step 5A checkpoint SHA-256,4b0b53c91c25d2f3900a5c9df2b313c495545ea65036ae...,4b0b53c91c25d2f3900a5c9df2b313c495545ea65036ae...,True
1,Step 5B checkpoint SHA-256,cfbcbf39431c8eaa2196fb069b2ee0f203703379696a2c...,cfbcbf39431c8eaa2196fb069b2ee0f203703379696a2c...,True
2,Step 5A manifest failures,0,0,True
3,Step 5B manifest failures,0,0,True
4,Package missing files,0,0,True
5,Package unexpected files,0,0,True
6,Package size mismatches,0,0,True
7,Package SHA-256 mismatches,0,0,True
8,Registry rows before,10,10,True
9,Registry rows candidate,11,11,True



Project 11 registry row candidate:


,ProjectNumber,Project,ProjectSlug,Status,Conditions,Seeds,NoiseLevels,Techniques,EvaluationBuilds,EvaluationRows,...,FreezeRecord,ChecksumManifest,LastFreezeValidationAtUTC,RawResultsManifest,FinalPackageManifest,RawResultsRootSHA256,FinalPackageRootSHA256,ModelFits,ManifestRowsAudited,FinalAuditStatus
10,11,apache@shardingsphere,apache__shardingsphere,COMPLETE_AND_FROZEN,270,30,9,7,263,13007,...,/content/drive/MyDrive/Thesis_Experiment/Notes...,/content/drive/MyDrive/Thesis_Experiment/Resul...,2026-07-25T03:49:02.436302+00:00,/content/drive/MyDrive/Thesis_Experiment/Resul...,/content/drive/MyDrive/Thesis_Experiment/Resul...,6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e8...,e9d56c6320e0db6457e835c2d3260a70bf60bafdea466e...,1080,5358150.0,PASS_PROJECT_11_FINAL_PACKAGE_FROZEN_AND_REGIS...



=== PROJECT 11 CELL 11 / STEP 5C RESULT ===
Project number: 11
Project: apache@shardingsphere
Project slug: apache__shardingsphere

Raw result freeze:
Conditions: 270
ML fits: 1080
Raw files: 2160
Raw bytes: 386769592
Raw root SHA-256: 6b5b74c4806fcbf65f97b6097aa1bdc5031696af4a73e88642a299343f2a73a1

Final package freeze:
Package root: /content/drive/MyDrive/Thesis_Experiment/Results/Final/apache__shardingsphere
Package files: 32
Package bytes: 20687283
Missing package files: 0
Unexpected package files: 0
Package size mismatches: 0
Package SHA-256 mismatches: 0
Final package root SHA-256: e9d56c6320e0db6457e835c2d3260a70bf60bafdea466e37a62b46a95cc2a04c

Completion registry:
Registry rows: 11
COMPLETE_AND_FROZEN projects: 11
Project 11 registry rows: 1
Registry SHA-256 before: 847bdcbee16c8757fde34e9489cd3abbf0781a319644175472597265b1df2750
Registry SHA-256 after: 54ae2cb3036a869a5068032ffb30e5ab77cf489ecd9ec0dc6c20b2db9ce493b0
Package already frozen before this cell: True

Isolation:
